# PRISM v17 — DRIFT: the student learns from its own mistakes

v16.2 made "≤ 3.0B per token" strict and brought the student back to PPL 33.4 under that law (v15 read 31.3 only with an
uncounted dense head). Its arithmetic went 1/20 → 2/20, and the wrong answers became near misses (67+25 → 91,
50−16 → 32): the student is close, and it fails at the one token that decides. v17 goes after exactly that, and after
three limits that v16.2 measured on itself.

| step | what | why |
|---|---|---|
| **0. DRIFT** (the gamble) | Every other micro-batch the student first reads the batch itself (served, no gradient). Where its first choice differs from the text — its own fork — the next token is replaced by the student's choice (half of the forks, at most 6% of the positions). The dense teacher then labels the batch as it now reads. | Teacher forcing never shows a student its own errors, and long answers in code, science and math break exactly after the first one. DRIFT trains every kind of text on recovery, for one extra forward and without generating anything. Nobody knows yet how much it gives at 3B active; that is the bet. |
| **0b. FULL KL** | The loss is the KL to the teacher's **whole** 248k-token distribution, computed chunk by chunk inside the served head (exact gradient, checked against autograd). The top-64 KL is still logged. | Until v16 the student saw only the teacher's top 64 tokens and one number for the rest. The teacher already runs in the step, so its whole distribution is free. |
| **0c. Looser anchor** | The stream anchor weight 1.0 → 0.25. | v16.2's x-ray: every part drifted from its dense original while the output got much closer to the teacher. The anchor pulls against that. |
| **1. LOOP** | Once every three updates the student **writes its own answers** to fresh arithmetic problems (batched, cached decoding, the served model). The exchanges — prompt, the student's own tokens, end of turn — become a training row, and the dense teacher grades **every token the student wrote**, given the student's own prefix. | Until v16 the student only ever read text someone else wrote. It never saw the state it puts itself in after its own first wrong digit. LOOP trains it there. The probe's own problems are never practised. |
| **2. The teacher in the step** | The dense teacher already ran in every training step (for the stream anchor). Now that same pass also produces the top-64 targets of the batch. No cache is built. | The data could only grow as far as a cache could be built and stored. Now it can grow freely: **3× the reasoning traces** (math, reasoning, science, code, other) and **1,024 blocks of fresh text** no version has read (web-edu, code, Korean Wikipedia, chat). Each sequence is seen about once instead of twice. |
| **3. The wallet spends the budget, and the ranges open** | The price level follows what the tokens **leave unspent**, not what they want. A token may now read 0–2× the FFN quota of a layer (was 0.5–1.5×), up to 6 extra row-blocks (was 3) and up to 8 extra memory writes (was 4). | v16.2 left ~157M per token unspent (mean 2.84B of a 2.999B ceiling): the wants came from tokens whose wallets were already empty, so prices rose and everyone else sold. And the hungriest tokens sat exactly on every upper cap (FFN 723 = 1.5 × 482, memory writes 7.8 of 8, row-blocks 4.7 of 4.8). |
| **4. Long context, measured** | One long document is read up to 32K tokens before and after training, by the teacher and by the student as served; KL and first-token agreement are reported by position (0–1K, 1K–4K, 4K–8K, 8K–16K, 16K–32K). | Coding and math need long inputs and long answers. All training and evaluation so far used 1,024-token blocks. From v17 every version reports this curve. |
| **5. KL by source** | Every 25 updates the training log shows the KL of text, problems, traces, fresh text and LOOP rows separately, and how often the student's own answers were right. | To see which kind of data is hard, and whether LOOP moves the student. |

Kept from v16.2: the strict head + LIFT, strict memory writes (log-score straight-through), own neurons, fork loss, the
wallet, the fuse. The state restored is v16.2's; a finished v17 state is restored first if there is one.


## Setup

In [ ]:
# Storage + environment: Colab (Google Drive)  |  Lightning AI Studio  |  any other Linux GPU machine
import os, shutil, subprocess
try:
    from google.colab import drive
    _COLAB = True
except Exception:
    _COLAB = False
PRISM_ENV = "colab" if _COLAB else ("lightning" if (os.path.isdir("/teamspace") or os.environ.get("LIGHTNING_CLOUD_PROJECT_ID"))
                                    else "other")


def _free_gb(p):
    try:
        return shutil.disk_usage(p).free / 2**30
    except Exception:
        return -1.0


def _fstype(p):
    """File-system type of the mount that holds p (so a RAM disk is never used for 54 GB of weights)."""
    best, typ = "", "?"
    try:
        rp = os.path.realpath(p)
        with open("/proc/mounts") as f:
            for line in f:
                parts = line.split()
                if len(parts) >= 3 and (rp == parts[1] or rp.startswith(parts[1].rstrip("/") + "/")) and len(parts[1]) >= len(best):
                    best, typ = parts[1], parts[2]
    except Exception:
        pass
    return typ


if _COLAB:
    if not os.path.ismount("/content/drive"):
        # a previous cell run before mounting may have left EMPTY local folders here; rmdir never deletes a file
        for _root, _dirs, _files in os.walk("/content/drive", topdown=False):
            if not _files:
                try:
                    os.rmdir(_root)
                except OSError:
                    pass
        drive.mount("/content/drive")
    print("Drive mounted:", os.path.ismount("/content/drive"))
    _root = os.environ.get("PRISM_ROOT") or "/content/drive/MyDrive/PRISM-full"
    os.makedirs(_root, exist_ok=True)
    _q = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits", shell=True,
                        capture_output=True, text=True).stdout.strip().splitlines()
    if _q:
        _gname, _gmem = [s.strip() for s in _q[0].split(",")][:2]
        print(f"GPU: {_gname} | {float(_gmem) / 1024:.0f} GiB")
        if float(_gmem) < 70000 and not globals().get("ALLOW_SMALL_GPU", False):
            raise RuntimeError(f"이 런타임의 GPU는 {float(_gmem) / 1024:.0f}GB입니다. 27B 모델에는 A100 80GB가 필요합니다 "
                               "(런타임 유형 변경 -> A100 + 고용량 RAM; 40GB A100이 잡히면 연결을 끊고 다시 잡아야 합니다). "
                               "아직 아무것도 내려받지 않았습니다.")
else:
    # ---- 1) the GPU: this notebook needs ONE GPU with >= 80 GB (the dense weights alone are 54 GB)
    _q = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits", shell=True,
                        capture_output=True, text=True).stdout.strip().splitlines()
    if _q:
        _gname, _gmem = [s.strip() for s in _q[0].split(",")][:2]
        print(f"GPU: {_gname} | {float(_gmem) / 1024:.0f} GiB" + (f" (+{len(_q) - 1} more)" if len(_q) > 1 else ""))
        if float(_gmem) < 70000 and not globals().get("ALLOW_SMALL_GPU", False):
            raise RuntimeError(f"이 GPU는 {float(_gmem) / 1024:.0f}GB입니다. 27B 모델에는 80GB급(A100 80GB / H100 / H200)이 "
                               "필요합니다. 머신을 바꾼 뒤 다시 실행하세요. (아직 아무것도 내려받지 않았습니다.)")
    elif not globals().get("ALLOW_NO_GPU", False):
        raise RuntimeError("GPU가 없습니다. 80GB급 GPU 머신(A100 80GB 등)으로 바꾼 뒤 다시 실행하세요. "
                           "(아직 아무것도 내려받지 않았습니다.)")
    # ---- 2) small persistent things (v10 state, teacher cache, results): the home that survives a restart
    _home = "/teamspace/studios/this_studio" if os.path.isdir("/teamspace/studios/this_studio") else os.path.expanduser("~")
    os.environ.setdefault("PRISM_ROOT", os.path.join(_home, "prism", "PRISM-full"))
    _root = os.environ["PRISM_ROOT"]
    os.makedirs(_root, exist_ok=True)
    # ---- 3) the 54 GB of dense weights: a disk with room, preferably NOT the persistent one (free plans cap it)
    _need = float(globals().get("HF_CACHE_NEED_GB", 62))
    _default_hub = os.path.join(os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface")), "hub")
    _have = os.path.isdir(_default_hub) and any(d.startswith("models--Qwen--") and "27B" in d for d in os.listdir(_default_hub))
    if "HF_HOME" in os.environ or _have:
        print(f"HF cache: {os.environ.get('HF_HOME', '~/.cache/huggingface')} (kept as it is"
              f"{' — the model is already there' if _have else ''}) | free {_free_gb(os.path.dirname(_default_hub) if os.path.isdir(os.path.dirname(_default_hub)) else _home):.0f} GB")
    else:
        _cands = [c for c in ("/tmp", "/cache", "/scratch", "/mnt", "/workspace", "/var/tmp", _home) if os.path.isdir(c)]
        print("where the model weights can go (free space):")
        _pick = None
        for c in _cands:
            _ft, _fg = _fstype(c), _free_gb(c)
            _ok = _fg >= _need and _ft not in ("tmpfs", "ramfs", "devtmpfs") and os.access(c, os.W_OK)
            print(f"   {c:<32} {_fg:7.0f} GB free | {_ft:<8} {'<- ok' if _ok else ''}")
            if _ok and _pick is None:
                _pick = c
        if _pick is None:
            raise RuntimeError(f"모델 가중치(약 54GB)를 둘 디스크가 없습니다: 위 목록에 {_need:.0f}GB 이상 빈 곳이 없습니다. "
                               "이 출력을 그대로 보내 주세요. (공간이 있는 경로를 알면 그 위에 "
                               "os.environ['HF_HOME']='/경로' 한 줄을 넣고 다시 실행하면 됩니다.)")
        if _pick != _home:
            os.environ["HF_HOME"] = os.path.join(_pick, "hf_cache")
            os.makedirs(os.environ["HF_HOME"], exist_ok=True)
        print(f"HF cache -> {os.environ.get('HF_HOME', '~/.cache/huggingface (home)')}"
              + ("  (임시 디스크: Studio가 꺼지면 다음에 다시 내려받습니다)" if _pick != _home else
                 "  (영구 저장소: 무료 플랜의 저장 한도에 걸리면 내려받기가 중간에 실패할 수 있습니다 — 그러면 그 오류를 보내 주세요)"))

# ---- the starting state lives in YOUR private Hub repo (v15 was saved there); so will the result of this run
import glob
from huggingface_hub import HfApi, hf_hub_download, login


def _hf_login():
    try:
        return HfApi().whoami()["name"]
    except Exception:
        pass
    _tok = os.environ.get("HF_TOKEN")
    if not _tok and _COLAB:
        try:
            from google.colab import userdata
            _tok = userdata.get("HF_TOKEN")              # Colab "Secrets" (the key icon), if you stored it there
        except Exception:
            _tok = None
    if not _tok:
        from getpass import getpass
        _tok = getpass("Hugging Face 토큰 (write 권한) 을 붙여 넣고 Enter: ")
    _tok = _tok.strip()
    login(token=_tok, add_to_git_credential=False)
    os.environ["HF_TOKEN"] = _tok
    return HfApi().whoami()["name"]


_me = _hf_login()
_repo = os.environ.get("PRISM_STATE_REPO") or (_me + "/prism-v10-state")
os.environ["PRISM_PUSH_REPO"] = _repo
_alt = "/content/prism_local/PRISM-full" if _COLAB else os.path.join(_root, "_local")
_ORDER = ("v17_state", "v16_state", "v15_state", "v14_state", "v13_state", "v12_state", "v10_state")


def _local_state():
    for d in _ORDER:
        for base in (_root, _alt):
            if os.path.exists(os.path.join(base, d, d + ".pt")):
                return os.path.join(base, d, d + ".pt")
    return None


def _fetch(path_in_repo, min_free_gb):
    """One file of the repo -> the persistent root when it has room, else the session-local folder."""
    dest = _alt if (_COLAB or _free_gb(_root) < min_free_gb) else _root     # Colab: local SSD, never through the Drive mount
    os.makedirs(dest, exist_ok=True)
    hf_hub_download(_repo, path_in_repo, local_dir=dest)
    return os.path.join(dest, path_in_repo)


try:
    _files = set(HfApi().list_repo_files(_repo))
except Exception as _e:
    _files = set()
    print(f"(could not list {_repo}: {type(_e).__name__}: {str(_e)[:100]})")
_have = _local_state()
_best_remote = next((d for d in _ORDER if f"{d}/{d}.pt" in _files), None)
_rank = lambda d: _ORDER.index(d) if d in _ORDER else len(_ORDER)
if _best_remote and (_have is None or _rank(_best_remote) < _rank(os.path.basename(os.path.dirname(_have)))):
    print(f"downloading {_repo}/{_best_remote} ...")
    _have = _fetch(f"{_best_remote}/{_best_remote}.pt", 14)
    if f"{_best_remote}/ppl_after.json" in _files:
        hf_hub_download(_repo, f"{_best_remote}/ppl_after.json", local_dir=os.path.dirname(os.path.dirname(_have)))
if "v17_state/resume.pt" in _files and not any(os.path.exists(os.path.join(b, "v17_state", "resume.pt")) for b in (_root, _alt)):
    print(f"downloading {_repo}/v17_state/resume.pt (a v17 session that was cut) ...")
    _fetch("v17_state/resume.pt", 14)
if not glob.glob(os.path.join(_root, "cache", "teacher_top*.pt")):
    for _f in sorted(f for f in _files if f.startswith("cache/teacher_top")):     # the text teacher cache (else it is rebuilt: ~15 min)
        try:
            _p = _fetch(_f, 2)
            if not _p.startswith(_root):
                os.makedirs(os.path.join(_root, "cache"), exist_ok=True)
                shutil.copy(_p, os.path.join(_root, _f))
        except Exception as _e:
            print(f"(teacher cache not fetched: {type(_e).__name__}: {str(_e)[:100]} — it will be rebuilt)")
os.environ["PRISM_STATE_ALT"] = _alt
print(f"environment: {PRISM_ENV} | storage root {_root} ({_free_gb(_root):.0f} GB free) | state repo {_repo} | "
      f"starting state: {_have} | text teacher cache present: {bool(glob.glob(os.path.join(_root, 'cache', 'teacher_top*.pt')))}")
if _have is None:
    raise RuntimeError(f"시작할 저장본이 없습니다: {_repo} 에 v15_state/v15_state.pt 가 있어야 합니다. 저장소 이름과 토큰 권한을 확인하세요.")
if "v15_state" not in _have and "v16_state" not in _have and "v17_state" not in _have:
    print("!! v15/v16 저장본을 찾지 못해 더 오래된 상태에서 시작합니다 — v15까지의 개선이 빠진 출발점입니다.")


In [ ]:
# Setup — packages, allocator config, memory helpers (v9.0)
# v16: torchao is optional (nothing is quantized here)
import importlib.metadata as _ilm, sys as _sys
def _pkg_ver(_p):
    try:
        return _ilm.version(_p)
    except Exception:
        return None
_torch_before = _pkg_ver('torch')
_pin = f' "torch=={_torch_before}"' if _torch_before else ''          # pip may add packages; it may not move torch
get_ipython().system(f'pip install -q -U "transformers>=5.8.0" accelerate datasets safetensors sentencepiece einops{_pin} 2>&1 | tail -2')
# vLLM is installed later in the serving section (heavy install).

# v8.0.4 [SPEED]: fla-core = the optimized Triton kernels for the Gated-DeltaNet
# layers (48 of the 64 layers). Without it transformers falls back to a readable
# PyTorch reference that is >10x slower AND keeps far more activations for backward
# — the largest single cost of the calibration, teacher-cache and training phases.
# Installed on GPU runtimes only. If it cannot be imported, transformers silently
# keeps the reference path, so this can only help. (Set USE_FLA = False to opt out.)
USE_FLA = globals().get("USE_FLA", True)
if USE_FLA:
    get_ipython().system(f'nvidia-smi -L > /dev/null 2>&1 && pip install -q -U fla-core{_pin} 2>&1 | tail -2')
_RESTART = ('torch가 이 세션 안에서 바뀌었거나 두 번 로드되었습니다. 메뉴에서 [런타임 → 세션 다시 시작]을 누른 뒤 [모두 실행]을 '
            '다시 눌러 주세요. 내려받은 모델과 저장본은 그대로 남아 있어 다시 받지 않습니다.')
if _torch_before and _pkg_ver('torch') != _torch_before:
    raise RuntimeError(f'pip가 torch를 {_torch_before} -> {_pkg_ver("torch")} 로 바꿨습니다. ' + _RESTART)

import importlib, subprocess, sys, os, math, json, time, random

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

gpu_name = _sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null") or "NO GPU (CPU-only)"
print("GPU:", gpu_name)

import os
# Allocator config BEFORE the first CUDA allocation (PyTorch docs:
# expandable_segments must be active before the allocator is created):
# segments grow in place instead of minting un-splittable cudaMalloc blocks,
# which is what fragmented the A100 after the big calibration allocations.
# Forced (not setdefault) so a stale env value can never re-enable the
# fragmenting allocator.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
try:
    import torch
except RuntimeError as _e:
    if 'override a python impl' in str(_e):
        raise RuntimeError(_RESTART) from None
    raise
import transformers
try:
    import torchao
    _tao = torchao.__version__
except Exception:
    torchao, _tao = None, "not installed (v16 does not quantize)"
print(f"torch {torch.__version__} | transformers {transformers.__version__} | torchao {_tao}")

from packaging import version
assert version.parse(transformers.__version__) >= version.parse("5.8.0"), \
    "transformers>=5.8.0 required (qwen3_5 / qwen3_5_moe architectures)"

# v8.0.4: which GDN kernel will the model use? transformers binds it when the
# qwen3_5 modeling module is first imported (= the model load, after this cell).
FLA_ACTIVE = False
try:
    importlib.invalidate_caches()          # fla-core was pip-installed a moment ago
    import fla as _fla
    import fla.ops.gated_delta_rule as _fla_gdr          # what transformers resolves
    FLA_ACTIVE = bool(torch.cuda.is_available())
    print(f"GDN kernel: fla-core {getattr(_fla, '__version__', '?')} (Triton) "
          f"{'- ACTIVE' if FLA_ACTIVE else '- CPU runtime: reference path is used'}")
except Exception as _e:
    print("GDN kernel: PyTorch reference path (much slower) — fla-core not usable: "
          f"{type(_e).__name__}: {str(_e)[:80]}")

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    print(f"CUDA compute capability: sm_{major}{minor}",
          "(FP8 training eligible)" if major >= 9 else "(FP8 not available; int8/int4 path)")


# ---- memory helpers (used at every phase boundary) ----------
# ============================================================================
# v8.0.3 OOM hardening (nothing in this block changes any model math)
#   * gpu_mem also prints HOST RAM: on Colab the "OOM" is often the VM itself
#     (session dies when host RAM runs out), not the GPU
#   * is_oom / oom_backoff: classify CUDA OOMs and retry with a smaller, purely
#     batching knob after scrubbing the heap OUTSIDE the except block
#   * release_previous_run: pressing Run on a cell twice must not stack a 2nd
#     copy of the model on top of the 1st
# ============================================================================
OOMFIX_VER = "v9.0"
try:
    import psutil
except Exception:
    psutil = None


def host_ram_gib():
    """(available, total) host RAM in GiB; (inf, inf) when psutil is missing."""
    if psutil is None:
        return float("inf"), float("inf")
    vm = psutil.virtual_memory()
    return vm.available / 2**30, vm.total / 2**30


def vram_gib():
    """(free, total) GPU memory in GiB; (0, 0) on a CPU-only runtime."""
    if not torch.cuda.is_available():
        return 0.0, 0.0
    free, total = torch.cuda.mem_get_info()
    return free / 2**30, total / 2**30


def gpu_mem(tag: str = "") -> None:
    """One-line census: VRAM allocated | reserved | truly free  +  host RAM."""
    ram_av, ram_tot = host_ram_gib()
    ram = (f" | host RAM {ram_av:.0f}/{ram_tot:.0f} GiB free"
           if ram_tot != float("inf") else "")
    if not torch.cuda.is_available():
        print(f"[mem{(' ' + tag) if tag else ''}] CPU-only runtime{ram}")
        return
    alloc = torch.cuda.memory_allocated() / 2**30
    res = torch.cuda.memory_reserved() / 2**30
    free, total = torch.cuda.mem_get_info()
    print(f"[mem{(' ' + tag) if tag else ''}] alloc {alloc:6.1f} GiB | "
          f"reserved {res:6.1f} GiB | free {free / 2**30:5.1f} / "
          f"{total / 2**30:.0f} GiB{ram}")


def free_cuda(tag: str = "") -> None:
    """Call-site `del`s happen first; this releases what they left behind:
    gc.collect() (breaks autograd/reference cycles) THEN empty_cache()
    (returns reserved-but-unallocated blocks to the driver so the NEXT
    phase — export reload, vLLM — can actually claim them)."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gpu_mem(tag)


def is_oom(e: BaseException) -> bool:
    """True for a CUDA OOM, including the allocator-failure RuntimeError flavours."""
    if isinstance(e, torch.cuda.OutOfMemoryError):
        return True
    s = str(e).lower()
    return isinstance(e, RuntimeError) and (
        "out of memory" in s or "cublas_status_alloc_failed" in s)


def oom_backoff(fn, sizes, tag: str = ""):
    """Run fn(size), largest size first. On a CUDA OOM: scrub the heap and retry
    with the next (smaller) size. `size` must be a PURE batching knob
    (micro-batch, chunk rows, ...) so the result is identical at every size.
    The scrub happens after the except block has exited, so the failed
    attempt's graph / traceback is already releasable."""
    sizes = list(sizes)
    for i, s in enumerate(sizes):
        failed, msg = False, ""
        try:
            return fn(s)
        except RuntimeError as e:
            if not is_oom(e):
                raise
            failed, msg = True, str(e)[:300]
        if failed:
            gpu_mem(f"OOM in {tag or getattr(fn, '__name__', 'fn')} at size={s}")
            free_cuda()
            if i == len(sizes) - 1:
                raise torch.cuda.OutOfMemoryError(
                    f"{tag}: still OOM at the smallest size ({s}). {msg}")
            print(f"[oom-backoff] {tag}: retrying with size={sizes[i + 1]}")


def release_previous_run(verbose: bool = True) -> None:
    """Re-run guard: drop everything a previous pass of this notebook still
    holds on the GPU (the model, the module lists that ADOPT its weights,
    optimizer / trainer / vLLM handles) before a fresh load allocates another
    ~52 GiB on top of it — the classic 'OOM the second time I press Run'."""
    g = globals()
    held = []
    for nm in ("PRISM_MODULES", "RESONA_LINEARS", "RESONA_SECTIONED", "RESONA_MORM",
               "RESONA_FUNNEL", "RESONA_MORM_GATES", "_FORCED_CKPT", "_PARKED"):
        obj = g.get(nm)
        if isinstance(obj, list) and obj:
            held.append(f"{nm}[{len(obj)}]")
            obj.clear()
    for nm in ("trainer", "optimizer", "scheduler", "reloaded", "llm", "model",
               "funnel", "MAST", "ORIG_LOGITS", "dense_sd", "ref_sd", "STORE"):
        if g.get(nm) is not None:
            held.append(nm)
            g[nm] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if held and verbose:
        print(f"[re-run guard] released previous-run state: {', '.join(held)}")
        gpu_mem("after release")


# ---- v8.0.4: safety net for CPU/CUDA index mismatches -------------------------
# "indices should be either on cpu or on the same device as the indexed tensor"
# kills a multi-hour run over a one-line device slip (it did, in the surgery cell).
# The only sensible meaning of such an index is "use it where the tensor lives",
# so on exactly that error the index is moved and the op is retried (once per
# call-site a note is printed so the line can still be fixed). It costs nothing
# unless the error fires. Set INDEX_AUTOHEAL = False to opt out.
INDEX_AUTOHEAL = globals().get("INDEX_AUTOHEAL", True)
_IDX_MSG = "indices should be either on cpu or on the same device as the indexed tensor"
_idx_heal_sites = []


def _idx_to(idx, dev):
    if torch.is_tensor(idx):
        return idx if idx.device == dev else idx.to(dev)
    if isinstance(idx, tuple):
        return tuple(_idx_to(i, dev) for i in idx)
    if isinstance(idx, list):
        return [_idx_to(i, dev) for i in idx]
    return idx


def _install_index_autoheal():
    T = torch.Tensor
    if getattr(T, "_prism_idx_heal", False):
        return
    _get, _set = T.__getitem__, T.__setitem__

    def _note(kind):
        try:
            f = sys._getframe(2)
            site = f"{os.path.basename(f.f_code.co_filename)}:{f.f_lineno}"
        except Exception:
            site = "?"
        if site not in _idx_heal_sites and len(_idx_heal_sites) < 8:
            _idx_heal_sites.append(site)
            print(f"[index-heal] {kind}: index tensor was on a different device than the "
                  f"indexed tensor — moved it and retried (first seen at {site})")

    def __getitem__(self, idx):
        try:
            return _get(self, idx)
        except RuntimeError as e:
            if _IDX_MSG not in str(e):
                raise
        _note("getitem")
        return _get(self, _idx_to(idx, self.device))

    def __setitem__(self, idx, val):
        try:
            return _set(self, idx, val)
        except RuntimeError as e:
            if _IDX_MSG not in str(e):
                raise
        _note("setitem")
        return _set(self, _idx_to(idx, self.device), val)

    T.__getitem__, T.__setitem__ = __getitem__, __setitem__
    T._prism_idx_heal = True


if INDEX_AUTOHEAL:
    _install_index_autoheal()


torch.manual_seed(0); random.seed(0)
print("Setup OK.")
_f0, _t0 = vram_gib(); _r0, _rt0 = host_ram_gib()
print(f"[{OOMFIX_VER}] GPU {_t0:.0f} GiB total / {_f0:.0f} free | "
      f"host RAM {_rt0:.0f} GiB total / {_r0:.0f} free")


## Configuration

In [ ]:
# Configuration — every knob in the pipeline lives here; the same code
# runs the demo model and the 27B target (Drive is mounted in cell 0).

# ------------------------------- RUN MODE -----------------------------------
RUN_MODE = "full"          # "demo" (Qwen3.5-0.8B, runs anywhere) | "full" (Qwen3.8-27B)
QUALITY_MODE = "max_budget_use"   # "max_budget_use" (spend all slack under 3.0B) | "headroom" (target 2.75B) | "safety" (no write-cap request: MoRM may use every head the budget allows)

# One version constant: every banner and label in the notebook derives
# from this single name, so the label can never drift from the code.
__PRISM_VER__ = "v17.0"


MODELS = {
    "demo": "Qwen/Qwen3.5-0.8B",      # same qwen3_5 family, dense, tiny
    "full": "Qwen/Qwen3.8-27B",       # the target: dense VL hybrid 27B, 262K ctx
}
MODEL_ID = MODELS[RUN_MODE]

# --------------------------- PRISM structure (FFN) ---------------------------
# FFN (E, k) candidates; the v2 solver refines these against the ACTIVE budget.
CANDIDATES = {
    "demo": ((16, 2), (16, 4), (8, 2), (8, 4), (32, 2)),
    "full": ((256, 8), (128, 8), (128, 4), (64, 8), (64, 4), (64, 2),
             (32, 4), (32, 2)),
}[RUN_MODE]
SHARED_SIZE = 512 if RUN_MODE == "full" else 32     # REQUESTED always-on shared-core neurons (v9: the budget solver may trim it; the effective value is re-bound after the solve)
FFN_EXPERTS = 128 if RUN_MODE == "full" else 32   # v10: FINER experts (132 neurons each on the 27B) ...
FFN_TOPK    = 4 if RUN_MODE == "full" else 2      # ... and more of them per token: the same routed-neuron budget
#   as v9's top-1-of-32, but the selection adapts to the token in 4 independent pieces instead of 1.
COMPLETION_RANK = 64 if RUN_MODE == "full" else 16  # ZSSI field rank R per layer

# --------------------------- RESONA execution (v2) ---------------------------
TARGET_ACTIVE = {"demo": None,                     # demo: minimize & report
                 "full": 3.0e9}[RUN_MODE]          # user MANDATE (hard assert)
if QUALITY_MODE == "headroom" and RUN_MODE == "full":
    TARGET_ACTIVE = 2.75e9
MAST_S        = 256                 # AntennaMast sketch dim (once per token)
LIN_BLOCKS    = 64 if RUN_MODE == "full" else 16    # resonant row-blocks per routed Linear (v10: fewer exact rows,
#                                     the budget goes to the shadows instead)
# ---- v10 SHADOW EXECUTION: nothing is skipped — what is not computed exactly takes the value of the best
#      rank-r predictor of that site's output (see 'SHADOW EXECUTION' in the core cell). 0 = off (v9).
SHADOW_RANK_LIN = 256 if RUN_MODE == "full" else 32   # routed projections (q / z / o / out) and DeltaNet v
SHADOW_RANK_FFN = 128 if RUN_MODE == "full" else 16   # affine low-rank model of the unselected FFN experts
SHADOW_V        = True     # DeltaNet: every head writes (shadow value); the MoRM critic picks the EXACT heads
M_WRITE_FLOOR   = 4        # exact heads / token the plan starts from when SHADOW_V is on (the solver may add more)
ROUTER_ST       = True     # straight-through counterfactual gradient for the top-k block routers
LIN_TOPK      = 1                   # blocks activated per token
CARRIER_FRAC  = 0.01                # always-on carrier rows (fraction of rows; the highest RESIDUAL rows)
M_WRITE_FRAC  = None if QUALITY_MODE == "safety" else 0.56   # MoRM write-cap REQUEST (fraction of heads); the solver returns the m_write the budget really affords (<= this cap)
FUNNEL_RANK   = 256                 # Spectral Head Funnel proxy rank
FUNNEL_SHORTLIST = 8192             # exact head rows evaluated per token (a CAP: the solver may use fewer)

# ------------------------------ calibration ---------------------------------
CAL_TOKENS      = 65536 if RUN_MODE == "full" else 16384
SKETCH_DIM      = 512
GAIN_CAL_TOKENS = 2048  if RUN_MODE == "full" else 256   # rows per resonant-linear / MoRM stash (spread over blocks)
GAIN_CAL_STEPS  = 300                                     # v8 Adam gains (only used when FFN_CAL_MODE='v8' / as fallback)

# ------------------------- v9 formula switches ------------------------------
# Every v9 upgrade is individually switchable.  The "v9 contracts" cell runs CPU
# self-checks of each formula and, if one fails, flips its switch back to the v8
# behaviour (loudly) — a new formula can never be the reason a long run dies.
FFN_CAL_MODE      = "v9"        # "v9": validated router + serving-consistent NNLS gains | "v8": prototype router + Adam
FFN_ROUTER_MODE   = "auto"      # "auto" (best on held-out tokens) | "prototype" | "ridge"
FFN_CAL_TOKENS    = 4096 if RUN_MODE == "full" else 1024   # FFN calibration rows per layer (stratified over blocks)
KL_MODE           = "partition" # "partition": KL on {teacher top-K ids} + {rest}  |  "support": v8 renormalised KL
LAMBDA_EMRD       = 0.05        # energy-mass router distillation (0 = off)
LAMBDA_LTD        = 0.25        # layer-local dense-twin loss, hard phase only (0 = off)
FUNNEL_USE_RHO    = True        # residual-aware head shortlist (V extra reads/token — counted in the budget)
FUNNEL_FIT_BLOCKS = 8           # blocks x tokens used to fit kappa and to measure shortlist recall
FUNNEL_FIT_TOK    = 256
PRECISION_PLAN    = "bf16"      # "bf16" (default) | "int8_frozen" (opt-in; see the precision cell)
ENFORCE_BUDGET    = True        # stop when the census of the REAL module graph exceeds TARGET_ACTIVE
MAST_ACCOUNTING   = "applications"  # "applications": every phi evaluation is charged (strict) | "distinct": each distinct sketch matrix once (v8 convention)
MAST_S_MIN        = 192         # the budget solver never trims the antenna sketch below this

# ------------------------------- training -----------------------------------
STEPS          = 400 if RUN_MODE == "full" else 120
WARMUP_STEPS   = 24                  # warmup_ratio is DEPRECATED -> warmup_steps
LR_FIELDS      = 3e-4
LR_ROUTER      = 1e-4
LR_TAU         = 1e-3
LAMBDA_CE      = 0.0      # v9.1: hard-label CE weight. 0 = PURE TEACHER DISTILLATION (targets = dense Qwen3.8, not the corpus)
LAMBDA_KL      = 1.0
TRAIN_PATH     = "hard"   # v10 default: train the SERVED function. "auto" (pick by measured held-out PPL) | "hard" (train the served path) | "soft_then_hard" (v8)
LAMBDA_BAL     = 0.01
LAMBDA_Z       = 1e-4
LAMBDA_ZS      = 0.05
# v9 loss terms (LAMBDA_EMRD / LAMBDA_LTD and KL_MODE live in the v9 block above)
SEQ_LEN        = 1024
# micro-batch 1 x accum 16 = the SAME effective batch 16, while every
# T-proportional consumer (GDN scan saves, soft-mode section outputs,
# head chunks) halves at zero training-semantics cost — and with the
# head GEMM chunked INSIDE the loss (_ChunkedHeadLoss), LOSS_CHUNK
# governs the vocab GEMM itself instead of chunking an
# already-materialized tensor.
BATCH_SIZE     = 1 if RUN_MODE == "full" else 4
GRAD_ACCUM     = 16 if RUN_MODE == "full" else 2
LOSS_CHUNK     = 256      # vocab-GEMM rows per fp32 head-loss chunk
TEACHER_TOPK   = 64
ANNEAL_FRAC    = 0.75
VLLM_GPU_FRAC  = 0.80   # fraction of TOTAL VRAM vLLM may claim (the student is parked on CPU RAM first)

# ----------------------- MoRM 2.0 (RESONA v8) --------------------------------
# Elastic write budget: mean writes per token (m_bar) defaults to the
# solver's m_write (same average cost as v7's fixed top-m); m_max caps the
# per-token spike; K bounds deferred-write staleness; the WIK window is the
# influence-label horizon. BETA_LOGIT_FLOOR drives masked heads to
# sigma(-60) ~ 9e-27 (bit-exact 0 when the kernel takes post-sigmoid beta).
MORM2_M_MAX_FRAC  = 0.75     # per-token write cap (fraction of n_heads)
MORM2_STALENESS_K = 64       # deferred-write staleness bound (tokens)
MORM2_WIK_WINDOW  = 1024     # WIK label horizon (tokens)
MORM2_CTRL_LR     = 0.02     # elastic rate-controller lr (per batch)
MORM2_STE_TAU     = 1.0      # straight-through softness
MORM2_LAMBDA_CRITIC = 0.10   # critic label-regression aux weight
MORM2_CRITIC_BIAS = True     # v9: per-head intercept in the influence critic (v8's bias-free warm start lost each head's mean level)
MORM2_BETA_LOGIT_FLOOR = -60.0
MORM2_LABEL_BLOCKS = 16 if RUN_MODE == "full" else 4   # blocks for WIK labels

# ------------------------------ persistence ---------------------------------
DRIVE_DIR = os.environ.get("PRISM_ROOT") or f"/content/drive/MyDrive/PRISM-{RUN_MODE}"   # PRISM_ROOT: machines without Drive
os.makedirs(DRIVE_DIR, exist_ok=True)
EXPORT_DIR = os.path.join(DRIVE_DIR, "PRISM-Qwen3.8-27B-A3B" if RUN_MODE == "full"
                          else "PRISM-demo-A3B")                 # COMPAT artifact
NATIVE_DIR = EXPORT_DIR + "-RESONA-native"                        # NATIVE artifact
CACHE_DIR  = os.path.join(DRIVE_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)

print(json.dumps({"PRISM_VER": __PRISM_VER__, "RUN_MODE": RUN_MODE,
                  "MODEL_ID": MODEL_ID,
                  "QUALITY_MODE": QUALITY_MODE, "TARGET_ACTIVE": TARGET_ACTIVE,
                  "MAST_S": MAST_S, "LIN_BLOCKS": LIN_BLOCKS,
                  "LIN_TOPK": LIN_TOPK, "CARRIER_FRAC": CARRIER_FRAC,
                  "FUNNEL_RANK": FUNNEL_RANK,
                  "FUNNEL_SHORTLIST": FUNNEL_SHORTLIST,
                  "FFN_CAL_MODE": FFN_CAL_MODE, "FFN_CAL_TOKENS": FFN_CAL_TOKENS,
                  "KL_MODE": KL_MODE, "LAMBDA_EMRD": LAMBDA_EMRD, "LAMBDA_LTD": LAMBDA_LTD,
                  "FUNNEL_USE_RHO": FUNNEL_USE_RHO, "PRECISION_PLAN": PRECISION_PLAN,
                  "ENFORCE_BUDGET": ENFORCE_BUDGET, "MAST_ACCOUNTING": MAST_ACCOUNTING,
                  "STEPS": STEPS, "WARMUP_STEPS": WARMUP_STEPS,
                  "DRIVE_DIR": DRIVE_DIR}, indent=1, default=str))


## Core modules (v10; the routed projections carry the market's probes)

In [ ]:
# PRISM core — fission math + PrismMoeMLP (pure PyTorch, self-contained)
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as _torch_checkpoint

# ============================================================================
# PRISM CORE — Prototype-Resonant Interleaved-Spectral Mixture
# Canonical math + modules for dense -> param-preserving fine-grained MoE
# ("function-field fission") upcycling. Pure PyTorch, self-contained.
#
# Components
#   1. sinkhorn_balanced                        entropic-OT balanced assignment
#                                              (the router training signal)
#   2. neuron_sketches                         JL co-firing sketches
#                                              (memory-safe fission statistics)
#   3. spectral_seeds / balanced_assign /
#      fission_layer                           spectral-seeded BALANCED fission
#                                              (equal-size experts + shared core)
#   4. zero_sum_gains / zero_sum_penalty       ZSSI completion fields
#                                              (Theorem 2)
#   5. router_prototypes / calibrate_gains     energy-derived router init
#   6. PrismMoeMLP                             uniform / soft / topk modes,
#                                              export materialization
#                                              (Theorems 1 and 3)
#   7. count_params / autotune                 parameter + active accounting
# ============================================================================


def _dm(x: torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
    """Dtype-boundary guard: cast x to ref's dtype (no-op when equal).

    The dense substrate is loaded in bf16 on GPU while the ZSSI completion
    field / router stack stay fp32 (fitting + training precision).  Every
    matmul between the two worlds must pass through this guard — bf16 @ fp32
    raises RuntimeError otherwise.  Activations are upcast into fp32 field
    math; results are cast back into the substrate compute dtype so nothing
    leaks fp32 into the model stream (which would double memory on A100).
    """
    return x if x.dtype == ref.dtype else x.to(ref.dtype)


# ----------------------------------------------------------------------------
# 1. Sinkhorn-OT balanced assignment
# ----------------------------------------------------------------------------
def sinkhorn_balanced(logits: torch.Tensor, n_iter: int = 5, tau: float = 1.0) -> torch.Tensor:
    """Entropic OT between tokens (uniform mass) and experts (uniform capacity).

    Given router logits [T, E], returns a soft plan P [T, E] whose columns are
    (approximately) balanced: sum_t P[t, e] ~= T/E for every e, rows sum to 1.
    Used as the *balanced target* that the top-k router is distilled toward:
    a projection, not a penalty (smoother than Switch-style aux losses).

    Args:
        logits: [T, E] router scores.
        n_iter: Sinkhorn row/col normalization sweeps.
        tau:    temperature applied to logits before OT.
    Returns:
        P: [T, E] balanced soft assignment plan.
    """
    if tau <= 0:
        raise ValueError(f"tau must be > 0, got {tau}")
    logK = logits / tau
    T, E = logK.shape
    f = torch.zeros(T, device=logK.device, dtype=logK.dtype)
    g = torch.zeros(E, device=logK.device, dtype=logK.dtype)
    for _ in range(n_iter):
        g = (math.log(T / E)) - torch.logsumexp(logK + f[:, None], dim=0)  # col marginals
        f = -torch.logsumexp(logK + g[None, :], dim=1)                     # row marginals
    P = torch.exp(logK + f[:, None] + g[None, :])
    return P


# ----------------------------------------------------------------------------
# 2. Neuron co-firing sketches (Johnson-Lindenstrauss)
# ----------------------------------------------------------------------------
def neuron_sketches(A: torch.Tensor, sketch_dim: int = 512,
                    generator: torch.Generator | None = None) -> torch.Tensor:
    """Sketch neuron activation patterns so inner products of sketch rows
    approximate token-averaged co-firing correlations between neurons.

    Given A [T, D] (tokens x neurons), returns S [D, s] with
        S_i . S_j  ~=  corr(a_i, a_j)
    Memory: O(D * s) instead of O(D^2) / O(T * D).

    Args:
        A: [T, D] activation matrix (centered internally).
        sketch_dim: JL projection dimension (>= 256 recommended).
        generator: optional torch.Generator for reproducibility.
    Returns:
        S: [D, sketch_dim] unit-norm sketch rows.
    """
    if A.dim() != 2:
        raise ValueError(f"A must be 2-D [T, D], got shape {tuple(A.shape)}")
    Ac = A - A.mean(dim=0, keepdim=True)
    sd = Ac.std(dim=0, keepdim=True).clamp_min(1e-6)
    Az = Ac / sd
    T = A.shape[0]
    Phi = torch.randn(T, sketch_dim, generator=generator, device=A.device, dtype=A.dtype)
    Phi = Phi / math.sqrt(sketch_dim)
    S = Az.t() @ Phi
    S = S / S.norm(dim=1, keepdim=True).clamp_min(1e-8)
    return S


# ----------------------------------------------------------------------------
# 3. Spectral-seeded BALANCED fission
# ----------------------------------------------------------------------------
def _kmeans(X: torch.Tensor, k: int, iters: int = 25, seed: int = 0) -> torch.Tensor:
    """Plain k-means (torch) returning centroids [k, d]. X: [N, d]."""
    g = torch.Generator(device="cpu").manual_seed(seed)
    idx = torch.randperm(X.shape[0], generator=g)[:k]
    C = X[idx.to(X.device)].clone()
    for _ in range(iters):
        a = torch.cdist(X, C).argmin(dim=1)
        for j in range(k):
            m = a == j
            if m.any():
                C[j] = X[m].mean(dim=0)
    return C


def spectral_seeds(S: torch.Tensor, E: int, subsample: int = 2048, seed: int = 0) -> torch.Tensor:
    """Spectral seeding: eigendecompose the co-firing affinity of a neuron
    subsample, embed in the top modes, k-means -> E prototype seeds in sketch
    space.

    Args:
        S: [D, s] unit-norm neuron sketches.
        E: number of routed experts.
        subsample: neurons used for the spectral stage.
        seed: RNG seed.
    Returns:
        seeds: [E, s] seed rows in sketch space.
    """
    D = S.shape[0]
    g = torch.Generator(device="cpu").manual_seed(seed)
    pick = torch.randperm(D, generator=g)[: min(subsample, D)].to(S.device)
    Ssub = S[pick]
    Aff = Ssub @ Ssub.t()                                        # co-firing affinity
    deg = Aff.sum(dim=1).clamp_min(1e-8)
    L = Aff / deg.sqrt()[:, None] / deg.sqrt()[None, :]          # normalized
    m = min(E, 16)
    evals, evecs = torch.linalg.eigh(L)                          # ascending
    emb = evecs[:, -m:] * evals[-m:].clamp_min(0).sqrt()
    emb = emb / emb.norm(dim=1, keepdim=True).clamp_min(1e-8)
    C_emb = _kmeans(emb, E, seed=seed)
    nearest = torch.cdist(emb, C_emb).argmin(dim=0)              # [E]
    return Ssub[nearest]


def balanced_assign(S: torch.Tensor, seeds: torch.Tensor, iters: int = 40) -> torch.Tensor:
    """BALANCED assignment of neurons to experts (equal cluster sizes),
    by margin-based repair of over-full clusters. Deterministic.

    Args:
        S: [D, s] unit-norm neuron sketches.
        seeds: [E, s] seeds.
        iters: max repair iterations.
    Returns:
        assign: [D] expert ids with sizes equal (max - min <= 1).
    """
    D, E = S.shape[0], seeds.shape[0]
    base = D // E
    rem = D - base * E
    targets = torch.full((E,), base, device=S.device)
    targets[:rem] += 1
    sim = S @ seeds.t()                                          # [D, E]
    assign = sim.argmax(dim=1)
    for _ in range(iters):
        sizes = torch.bincount(assign, minlength=E)
        diff = sizes - targets
        if int(diff.max()) <= 0:
            break
        j = int(diff.argmax())                                   # most over-full
        margin = sim.topk(2, dim=1).values
        slack = margin[:, 0] - margin[:, 1]                      # confidence
        n_move = int(diff[j])
        in_j = assign == j
        cand = torch.where(in_j, slack, torch.full_like(slack, float("inf")))
        movers = cand.topk(n_move, largest=False).indices        # least confident
        # destinations: under-full clusters, best sim for each mover
        room = (targets - sizes).clamp_min(0)
        room[j] = 0
        for mi in movers.tolist():
            sims_m = sim[mi].clone()
            sims_m[room <= 0] = float("-inf")
            dest = int(sims_m.argmax())
            assign[mi] = dest
            sizes[dest] += 1
            room[dest] -= 1
    return assign


def fission_layer(duty: torch.Tensor, S: torch.Tensor, E: int, shared_size: int,
                  seed: int = 0, min_expert_size: int = 64) -> dict:
    """Fission one FFN's intermediate dimension:
      1. shared core  = `shared_size` highest-importance neurons (always-on),
      2. routed experts = spectral-seeded BALANCED clustering of the rest.

    Args:
        duty: [D] neuron importance used to rank the always-on shared core.
              v9 passes E[a^2]*||W_down[:, r]||^2 (output energy, ffn_importance); v8 passed
              the SIGNED mean activation (blind to magnitude, cancels for sign-flipping neurons).
        S: [D, s] unit-norm co-firing sketches (may include all neurons).
        E: number of routed experts.
        shared_size: neurons for the always-on shared expert.
        seed: RNG seed.
        min_expert_size: sanity floor on neurons per routed expert.
    Returns:
        dict(assign [D] long: -1 = shared, 0..E-1 = routed expert,
             duty [D], shared_idx [<=shared_size])
    """
    D = S.shape[0]
    if E <= 0 or D - shared_size < E * min_expert_size:
        raise ValueError("Need (D - shared_size) >= E*min_expert_size neurons")
    shared_order = duty.topk(shared_size).indices
    shared_mask = torch.zeros(D, dtype=torch.bool, device=S.device)
    shared_mask[shared_order] = True
    Sr = S[~shared_mask]
    seeds = spectral_seeds(Sr, E, seed=seed)
    assign_r = balanced_assign(Sr, seeds)
    assign = torch.full((D,), -1, dtype=torch.long, device=S.device)
    assign[~shared_mask] = assign_r
    return {"assign": assign, "duty": duty, "shared_idx": shared_order}


# ----------------------------------------------------------------------------
# 4. Zero-Sum Superposition Init (ZSSI)
# ----------------------------------------------------------------------------
def zero_sum_gains(E: int, R: int, scale: float, seed: int = 0) -> torch.Tensor:
    """Zero-column-sum gain matrix G [E, R] used to warm-start or regularize
    the ZSSI completion field: the *mean expert* carries no completion, and
    capacity grows as antisymmetric deviations around the dense function.

    NOTE (honest math): because experts own disjoint neuron rows, zero column
    sums do NOT make the scattered field vanish row-wise; exactness at init is
    guaranteed instead by initializing gains to exactly 0 (see PrismMoeMLP).
    Zero-sum structure constrains GROWTH during training (Theorem 2).
    """
    g = torch.Generator().manual_seed(seed)
    G = torch.randn(E, R, generator=g) * scale
    G = G - G.mean(dim=0, keepdim=True)
    return G


def zero_sum_penalty(gains: torch.Tensor) -> torch.Tensor:
    """Differentiable penalty enforcing zero column sums on [E, R] gains
    (Theorem 2: antisymmetric capacity growth around the dense function)."""
    return gains.mean(dim=0).pow(2).sum()


# ----------------------------------------------------------------------------
# 5. Prototype-Resonant router init + calibrated gains
# ----------------------------------------------------------------------------
def router_prototypes(H: torch.Tensor, energy: torch.Tensor) -> torch.Tensor:
    """Functional prototypes: energy-weighted centroids of layer inputs.

    Args:
        H: [T, d] pre-FFN hidden states (subsample of calibration tokens).
        energy: [T, E] per-token activation energy owned by each expert.
    Returns:
        W_r: [E, d] unit-norm router rows (prototype directions).
    """
    tot = energy.sum(dim=0).clamp_min(1e-8)
    W_r = (energy.t() @ H) / tot[:, None]
    return W_r / W_r.norm(dim=1, keepdim=True).clamp_min(1e-8)


def calibrate_gains(expert_outputs: torch.Tensor, y_dense: torch.Tensor,
                    topk_idx: torch.Tensor, steps: int = 300, lr: float = 0.05,
                    init: torch.Tensor | None = None) -> torch.Tensor:
    """Fit per-expert scalar gains c >= 0 so the top-k serving mixture best
    reconstructs the dense output at init (projected SGD):

        min_c  sum_t || sum_{i} c_{e_i(t)} y_{e_i(t)}(t) - y_dense(t) ||^2

    Args:
        expert_outputs: [T, E, d] all expert outputs on the calibration tokens.
        y_dense: [T, d] dense targets.
        topk_idx: [T, k] serving top-k selections.
        steps, lr: optimizer settings.
        init: optional [E] starting gains.
    Returns:
        c: [E] nonnegative gains (fused multiplicatively into down_proj).
    """
    device = y_dense.device
    E = expert_outputs.shape[1]
    sel = torch.gather(expert_outputs, 1,
                       topk_idx[:, :, None].expand(-1, -1, expert_outputs.shape[-1]))
    c0 = torch.ones(E, device=device) if init is None else init.clone()
    c = torch.nn.Parameter(c0)
    opt = torch.optim.Adam([c], lr=lr)
    for _ in range(steps):
        y_hat = (sel * c[topk_idx][:, :, None]).sum(dim=1)
        loss = F.mse_loss(y_hat, y_dense.to(y_hat.dtype))   # dtype-boundary guard
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            c.clamp_(min=0.0)
    return c.detach()


# ----------------------------------------------------------------------------
# 5b. v9 calibration math — oracle energies, serving-consistent NNLS gains,
#     validated routers, ONE call per fissioned layer
# ----------------------------------------------------------------------------
import numpy as _np


def oracle_group_energy(act: torch.Tensor, dn2: torch.Tensor,
                        row_assign: torch.Tensor, n_groups: int) -> torch.Tensor:
    """Per-token OUTPUT energy owned by every neuron group.

    act [T, D] = silu(g) * u;  dn2 [D] = ||W_down[:, r]||^2;  row_assign [D] long
    in [0, n_groups] (n_groups marks the SHARED rows).  Returns [T, n_groups+1]:
        e[t, g] = sum_{r in g} act[t, r]^2 * dn2[r]
    the Gram-diagonal energy of group g's contribution to y = sum_r act_r W_down[:, r]
    (exact for ranking when the rank-1 terms are incoherent). One elementwise
    pass + one index_add; v8 used the activation energy WITHOUT the ||W_down||^2
    factor, which ranks a neuron by its pre-projection size, not by its effect."""
    a2 = act.float().pow(2) * dn2.float()[None, :]
    return a2.new_zeros(a2.shape[0], n_groups + 1).index_add_(1, row_assign, a2)


def ffn_importance(sq_mean: torch.Tensor, down_w: torch.Tensor) -> torch.Tensor:
    """[D] output-energy importance of every intermediate neuron:
    E[a_r^2] * ||W_down[:, r]||^2 (down_w is the HF [d, D] weight). v8 ranked the
    always-on shared core by the SIGNED mean activation: blind to magnitude and
    cancelling for neurons whose sign flips (a = silu(g)*u is signed)."""
    return sq_mean.float() * down_w.float().pow(2).sum(0)


def nnls_normal(A, b, max_iter=None):
    """min_{c >= 0}  0.5 c'Ac - b'c  for SPD A — Lawson-Hanson active set on the
    normal equations (finite termination; a variable dropped when it hits zero
    can re-enter, unlike the 'clip negatives and refit' shortcut)."""
    A = _np.asarray(A, dtype=_np.float64)
    b = _np.asarray(b, dtype=_np.float64)
    n = b.shape[0]
    c = _np.zeros(n)
    P = _np.zeros(n, dtype=bool)
    tol = 1e-12 * max(float(_np.abs(b).max(initial=0.0)),
                      float(_np.abs(_np.diag(A)).max(initial=0.0)), 1e-300)
    max_iter = max_iter or 60 * n + 60
    it = 0
    w = b - A @ c
    while it < max_iter:
        cand = (~P) & (w > tol)
        if not cand.any():
            break
        P[int(_np.argmax(_np.where(cand, w, -_np.inf)))] = True
        while it < max_iter:
            it += 1
            idx = _np.flatnonzero(P)
            if idx.size == 0:
                c = _np.zeros(n)
                break
            s = _np.zeros(n)
            s[idx] = _np.linalg.solve(A[_np.ix_(idx, idx)], b[idx])
            if s[idx].min() > 0.0:
                c = s
                break
            neg = P & (s <= 0.0)
            a = float(_np.min(c[neg] / (c[neg] - s[neg])))
            c = c + a * (s - c)
            P &= c > 1e-14 * max(1.0, float(c.max()))
            c[~P] = 0.0
        w = b - A @ c
    return c


def gate_select(logits: torch.Tensor, k: int, tau: float):
    """The serving gate: softmax(logits / tau) -> top-k -> renormalise (exactly the
    topk branch of PrismMoeMLP and of the stock Qwen3_5Moe router)."""
    p = torch.softmax(logits / tau, dim=-1)
    topv, topi = p.topk(k, dim=-1)
    return topi, topv / topv.sum(-1, keepdim=True).clamp_min(1e-30)


def gain_stats(Ysel: torch.Tensor, g: torch.Tensor, idx: torch.Tensor,
               target: torch.Tensor, n_experts: int, acc=None):
    """Accumulate the sufficient statistics (A, b, tt) of the SERVING objective
        f(c) = sum_t || target_t - sum_i g[t,i] c[idx[t,i]] Ysel[t,i] ||^2
             = tt - 2 b.c + c'Ac
    Ysel [t,k,d] selected expert outputs, g [t,k] gate weights as served,
    idx [t,k] expert ids, target [t,d] = y_dense - y_shared (the residual the
    routed experts must supply: v8 fitted the routed output to the FULL dense
    output, double-counting the shared core). Chunk-additive, so any number of
    calibration tokens fits in O(E^2) memory. acc = (A_flat, b, tt) to add into."""
    E = n_experts
    Z = Ysel.float() * g.float()[:, :, None]                          # [t,k,d]
    G = torch.einsum("tid,tjd->tij", Z, Z).double()                   # [t,k,k]
    r = torch.einsum("tid,td->ti", Z, target.float()).double()        # [t,k]
    if acc is None:
        acc = (torch.zeros(E * E, dtype=torch.float64, device=Ysel.device),
               torch.zeros(E, dtype=torch.float64, device=Ysel.device), 0.0)
    A, b, tt = acc
    pair = (idx[:, :, None] * E + idx[:, None, :]).reshape(-1)
    A = A.index_add(0, pair, G.reshape(-1))
    b = b.index_add(0, idx.reshape(-1), r.reshape(-1))
    return A, b, tt + float(target.float().pow(2).sum())


def solve_gains_nnls(A, b, c0, ridge_rel: float = 1e-3):
    """c = argmin_{c >= 0}  c'Ac - 2 b.c + lam ||c - c0||^2  (numpy float64).
    The ridge toward the prior c0 keeps experts that were (almost) never
    selected on the calibration set at a sane value instead of 0 or infinity."""
    A = _np.asarray(A, dtype=_np.float64)
    b = _np.asarray(b, dtype=_np.float64)
    c0 = _np.asarray(c0, dtype=_np.float64)
    n = b.shape[0]
    tr = float(_np.trace(A)) / max(n, 1)
    lam = ridge_rel * tr if tr > 0.0 else 1.0
    return nnls_normal(A + lam * _np.eye(n), b + lam * c0)


def gain_error(A, b, tt, c) -> float:
    """tt - 2 b.c + c'Ac : total squared serving error of gains c."""
    c = _np.asarray(c, dtype=_np.float64)
    return max(float(tt) - 2.0 * float(b @ c) + float(c @ A @ c), 0.0)


def router_capture(logits: torch.Tensor, en: torch.Tensor, k: int) -> float:
    """Energy-weighted share of the oracle group energy captured by the router's
    top-k set (1.0 = the selected experts own all of the routed energy)."""
    idx = logits.topk(k, dim=-1).indices
    return float(en.gather(1, idx).sum() / en.sum().clamp_min(1e-30))


def ridge_router_from_gram(K: torch.Tensor, X: torch.Tensor, Y: torch.Tensor,
                           lam_rel: float) -> torch.Tensor:
    """Bias-free dual-form ridge: W = X^T (K + lam I)^-1 Y, returned as the
    router weight [E, d] (a plain stock router has no bias, so none is fitted).
    K = X X^T [T, T]; lam is relative to the mean diagonal of K."""
    T = K.shape[0]
    lam = lam_rel * float(K.diagonal().mean())
    alpha = torch.linalg.solve(K + lam * torch.eye(T, device=K.device, dtype=K.dtype), Y)
    return (X.t() @ alpha).t().contiguous()


@torch.no_grad()
def calibrate_ffn_layer(prism, hs: torch.Tensor, k: int, router_mode: str = "auto",
                        gains_mode: str = "nnls", val_frac: float = 0.25,
                        chunk: int = 256, lam_grid=(1e-3, 1e-2, 1e-1, 1.0, 10.0),
                        tau_mult=(0.5, 1.0, 2.0), ridge_rel: float = 1e-3,
                        seed: int = 0) -> dict:
    """Training-free calibration of ONE fissioned FFN (router, temperature, gains).

    ROUTER  Candidates: the v8 prototype router (energy-weighted centroids) and
            bias-free ridge regressions of log(energy share) on the layer input
            (a small lambda grid). The winner is the candidate whose top-k set
            captures the most ORACLE energy on HELD-OUT tokens — selection is
            validated, never assumed — and is refit on all tokens.
    GAINS   Closed-form non-negative least squares of the SERVED function
            y = y_shared + sum_i g_i c_i y_i  against y_dense (gate weights g,
            temperature clamp and shared core exactly as served), instead of
            Adam on a routed-only MSE. For k > 1 the temperature is chosen from
            a small grid by held-out serving error.
    hs [T, d]: calibration inputs, stratified over documents (any float dtype).
    Returns a report dict; writes prism.router.weight / log_tau / gains."""
    dev = prism.router.weight.device
    E, d = prism.E, prism.d_model
    X = hs.to(dev).float()
    T = X.shape[0]
    perm = torch.randperm(T, generator=torch.Generator().manual_seed(1000 + seed)).to(dev)
    n_val = min(max(1, int(round(val_frac * T))), T - 1)
    va, fi = perm[:n_val], perm[n_val:]
    # ---- oracle group energies (completion-free: fresh fields are exactly 0) ----
    en = torch.empty(T, E, device=dev)
    for i in range(0, T, chunk):
        hc = X[i:i + chunk]
        gm = _dm(hc, prism.W_gate) @ prism.W_gate
        um = _dm(hc, prism.W_up) @ prism.W_up
        act = F.silu(gm.float()) * um.float()
        en[i:i + chunk] = oracle_group_energy(act, prism._dn2, prism._row_assign, E)[:, :E]
        del gm, um, act
    en_fit, en_val, X_fit, X_val = en[fi], en[va], X[fi], X[va]
    # ---- router candidates, validated on held-out tokens ----
    cands = {}
    if router_mode in ("auto", "prototype"):
        cands["prototype"] = router_prototypes(X_fit, en_fit)
    if router_mode in ("auto", "ridge"):
        share = en_fit / en_fit.sum(-1, keepdim=True).clamp_min(1e-30)
        Y_fit, K_fit = torch.log(share + 1e-4), X_fit @ X_fit.t()
        for lam in lam_grid:
            cands[f"ridge{lam:g}"] = ridge_router_from_gram(K_fit, X_fit, Y_fit, lam)
        del K_fit
    cap = {n: router_capture(X_val @ W.t(), en_val, k) for n, W in cands.items()}
    best = max(cap, key=cap.get)
    if "prototype" in cap and cap["prototype"] >= cap[best] - 1e-4:
        best = "prototype"                       # ties -> the parameter-free v8 router
    if best == "prototype":
        W = router_prototypes(X, en)
        lg = X @ W.t()
        tau0 = float((lg.std(dim=-1).mean() / max(0.5 * math.log(E), 1e-3)).clamp(0.25, 4.0))
    else:
        share = en / en.sum(-1, keepdim=True).clamp_min(1e-30)
        W = ridge_router_from_gram(X @ X.t(), X, torch.log(share + 1e-4), float(best[5:]))
        tau0 = 1.0                               # logits are already log-shares
    prism.router.weight.copy_(W.to(prism.router.weight.dtype))
    logit_all = X @ W.t()                                              # [T, E] fp32
    topi_all = logit_all.topk(k, dim=-1).indices
    load = torch.bincount(topi_all.reshape(-1), minlength=E).float()
    rep = {"router": best, "capture": cap[best], "capture_proto": cap.get("prototype"),
           "capture_all": router_capture(logit_all, en, k),
           "oracle": float(en.topk(k, dim=-1).values.sum() / en.sum().clamp_min(1e-30)),
           "load_cv": float(load.std() / load.mean().clamp_min(1e-9)), "tau": tau0,
           "rel_err": None, "rel_err_prior": None, "rel_err_all": None}
    if gains_mode != "nnls":
        prism.log_tau.fill_(math.log(min(max(tau0, 0.25), 4.0)))
        return rep
    # ---- serving-consistent NNLS gains (one pass; every candidate tau shares it) ----
    if k > 1:
        taus = sorted({round(min(max(tau0 * m, 0.25), 4.0), 6) for m in tau_mult})
    else:
        taus = [round(min(max(tau0, 0.25), 4.0), 6)]                  # k=1: gate == 1 for any tau
    is_val = torch.zeros(T, dtype=torch.bool, device=dev)
    is_val[va] = True
    stats = {(t_, s_): None for t_ in taus for s_ in ("fit", "val")}
    for i in range(0, T, chunk):
        hc = X[i:i + chunk]
        Ye = prism.expert_outputs(hc, use_completion=False)           # [t, E, d]
        tgt = prism.dense_forward(hc).float() - prism.shared_forward(hc, use_completion=False).float()
        vm = is_val[i:i + chunk]
        for tau in taus:
            topi, g = gate_select(logit_all[i:i + chunk], k, tau)
            Ysel = Ye.gather(1, topi[:, :, None].expand(-1, -1, Ye.shape[-1]))
            for split, msk in (("fit", ~vm), ("val", vm)):
                if bool(msk.any()):
                    stats[(tau, split)] = gain_stats(Ysel[msk], g[msk], topi[msk], tgt[msk], E,
                                                     stats[(tau, split)])
        del Ye, tgt
    def _np_stats(st):
        return (st[0].cpu().numpy().reshape(E, E), st[1].cpu().numpy(), float(st[2]))
    c0 = _np.full(E, float(k))                    # prior: every selected expert enters at weight 1
    best_tau, best_err = taus[0], None
    for tau in taus:
        Af, bf, ttf = _np_stats(stats[(tau, "fit")])
        Av, bv, ttv = _np_stats(stats[(tau, "val")]) if stats[(tau, "val")] is not None \
            else (Af, bf, ttf)
        c_fit = solve_gains_nnls(Af, bf, c0, ridge_rel)
        err = gain_error(Av, bv, ttv, c_fit) / max(ttv, 1e-30)
        if best_err is None or err < best_err - 1e-9:
            best_tau, best_err = tau, err
            rep["rel_err_prior"] = gain_error(Av, bv, ttv, c0) / max(ttv, 1e-30)
    Af, bf, ttf = _np_stats(stats[(best_tau, "fit")])
    if stats[(best_tau, "val")] is not None:
        Av, bv, ttv = _np_stats(stats[(best_tau, "val")])
        Af, bf, ttf = Af + Av, bf + bv, ttf + ttv
    c = solve_gains_nnls(Af, bf, c0, ridge_rel)                       # refit on ALL tokens
    prism.gains.copy_(torch.as_tensor(c, dtype=prism.gains.dtype, device=prism.gains.device))
    prism.log_tau.fill_(math.log(best_tau))
    # squared error of the SERVED routed part on ALL calibration tokens, relative to ||y_dense - y_shared||^2:
    # equals ||y_served - y_dense||^2 / ||y_dense - y_shared||^2 of the module itself (checked by the v9 contracts)
    rep.update(tau=best_tau, rel_err=best_err,
               rel_err_all=gain_error(Af, bf, ttf, c) / max(ttf, 1e-30))
    return rep


# ----------------------------------------------------------------------------
# 5d. FFN shadow (v10): an affine low-rank model of what the served FFN leaves out
# ----------------------------------------------------------------------------
@torch.no_grad()
def fit_ffn_shadow(prism, hs: torch.Tensor, k: int, rank: int, val_frac: float = 0.25,
                   chunk: int = 256, lam_grid=(1e-3, 1e-2, 1e-1, 1.0), seed: int = 0) -> dict:
    """r(h) = y_dense(h) - y_served(h)  ~  b + U (M h)      (reduced-rank kernel ridge, validated on held-out tokens)

    The served FFN evaluates ~6% of the neurons; the rest of the sum is not zero on average and is partly a
    smooth function of h. Its best affine rank-r predictor is fitted in closed form (dual ridge: one eigh of the
    [n, n] Gram matrix serves the whole lambda grid) and kept ONLY if it lowers the held-out error.
    Writes prism.sh_U [d, r], prism.sh_M [r, d], prism.sh_b [d]; returns a report."""
    dev = prism.router.weight.device
    T, d = hs.shape
    X = hs.to(dev).float()
    R = torch.empty(T, d, device=dev)
    was = getattr(prism, "use_shadow", True)
    prism.use_shadow = False
    for i in range(0, T, chunk):
        hc = hs[i:i + chunk].to(dev, dtype=prism.W_gate.dtype)
        y_s, _ = prism._forward_2d(hc, "topk", k, True, False)
        R[i:i + chunk] = prism.dense_forward(hc).float() - y_s.float()
    prism.use_shadow = was
    prism.last_aux = None
    perm = torch.randperm(T, generator=torch.Generator().manual_seed(2000 + seed)).to(dev)
    n_val = min(max(1, int(round(val_frac * T))), T - 1)
    va, fi = perm[:n_val], perm[n_val:]
    xm, rm = X[fi].mean(0), R[fi].mean(0)
    Xc, Rc = X[fi] - xm, R[fi] - rm
    Xv, Rv = X[va] - xm, R[va]
    n = Xc.shape[0]
    K = Xc @ Xc.t()
    evals, Q = torch.linalg.eigh(K)
    evals = evals.clamp_min(0)
    scale = float(evals.mean().clamp_min(1e-12))
    QtR = Q.t() @ Rc                                              # [n, d]
    KvQ = (Xv @ Xc.t()) @ Q                                       # [n_val, n]
    base = float(Rv.pow(2).sum().clamp_min(1e-30))
    err_bias = float((Rv - rm).pow(2).sum()) / base
    best_lam, best_err = None, None
    for lam in lam_grid:
        pv = rm + KvQ @ (QtR / (evals + lam * scale)[:, None])
        e = float((Rv - pv).pow(2).sum()) / base
        if best_err is None or e < best_err:
            best_lam, best_err = lam, e
    rep = {"rel_before": 1.0, "rel_bias_only": err_bias, "rel_fullrank": best_err, "lam": best_lam, "rank": 0,
           "rel_after": 1.0, "resid_over_dense": None}
    G = Q @ (QtR / (evals + best_lam * scale)[:, None])           # [n, d]:  pred(x) = rm + (x - xm) Xc^T G
    P_fit = K @ G                                                 # fitted predictions [n, d]
    r = int(max(0, min(int(rank), n, d)))
    U = M = None
    err_lr = None
    if r > 0:
        ev2, V = torch.linalg.eigh(P_fit.t() @ P_fit)             # output directions of the fitted map
        U = V[:, -r:].contiguous()                                # [d, r] top-r
        M = (G @ U).t() @ Xc                                      # [r, d]
        pv = rm + (Xv @ M.t()) @ U.t()
        err_lr = float((Rv - pv).pow(2).sum()) / base
    # keep the best of {nothing, bias only, bias + low-rank map} by held-out error
    choice, err = "none", 1.0
    if err_bias < err - 0.005:
        choice, err = "bias", err_bias
    if err_lr is not None and err_lr < err - 0.005:
        choice, err = "lowrank", err_lr
    dt = prism.W_gate.dtype
    if choice == "lowrank":
        prism.sh_U = U.to(dt)
        prism.sh_M = M.to(dt)
        prism.sh_b = (rm - (xm @ M.t()) @ U.t()).to(dt)
        rep["rank"] = r
    elif choice == "bias":
        prism.sh_U = torch.zeros(d, 1, device=dev, dtype=dt)
        prism.sh_M = torch.zeros(1, d, device=dev, dtype=dt)
        prism.sh_b = rm.to(dt)
        rep["rank"] = 0
    rep.update(choice=choice, rel_after=err)
    return rep


# ----------------------------------------------------------------------------
# 5c. Partition-KL — the distillation objective on {teacher top-K} + {rest}
# ----------------------------------------------------------------------------
def log1mexp(x: torch.Tensor) -> torch.Tensor:
    """log(1 - exp(x)) for x < 0, stable on both sides of -ln 2."""
    x = x.clamp(max=-1e-6)
    return torch.where(x > -0.6931471805599453,
                       torch.log(-torch.expm1(x)), torch.log1p(-torch.exp(x)))


def kl_support(t_lp: torch.Tensor, s_lp: torch.Tensor) -> torch.Tensor:
    """v8: KL of the two distributions RENORMALISED inside the teacher's top-K
    support (blind to how much probability mass the student puts elsewhere)."""
    tn = t_lp - t_lp.logsumexp(-1, keepdim=True)
    sn = s_lp - s_lp.logsumexp(-1, keepdim=True)
    return (tn.exp() * (tn - sn)).sum(-1)


def partition_kl(t_lp: torch.Tensor, s_lp: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """KL(P || Q) of the COARSE-GRAINED distributions on the partition
    {each of the teacher's top-K ids} U {everything else}, per position.

    t_lp / s_lp [..., K]: FULL-vocabulary log-probs of teacher / student at the
    teacher's top-K ids. By the chain rule of KL
        KL_part = P_S * KL_support(p_hat || q_hat)  +  KL_Bernoulli(P_S || Q_S)
    (P_S, Q_S = probability mass inside the support): the in-support term is
    weighted by how much of the teacher's mass lives there, and the support mass
    itself is matched — which v8's renormalised KL could not see. By data
    processing it lower-bounds the full-vocabulary KL and needs no more than
    the cached top-K."""
    t_lp = t_lp.float()
    s_lp = s_lp.float()
    p = t_lp.exp()
    t_mass = p.sum(-1).clamp(max=1.0 - eps)          # fp16 cache rounding can overshoot 1
    ls = s_lp.logsumexp(-1).clamp(max=-eps)          # log student mass inside the support
    t_rest = 1.0 - t_mass
    in_support = (p * (t_lp - s_lp)).sum(-1)
    return in_support + t_rest * (t_rest.log() - log1mexp(ls))


def emrd_loss(p: torch.Tensor, en: torch.Tensor, k: int) -> torch.Tensor:
    """Energy-Mass Router Distillation (EMRD).

    p  [T, E]  router probabilities (soft or the full softmax of the top-k path)
    en [T, E]  ORACLE output energy owned by every routed expert on this token
               (oracle_group_energy of the dense activations, no grad)
    With share_e = en_e / sum(en) and the saturating weight w_e = min(1, k * share_e)
    ("expert e is worth selecting"), minimise   -log sum_e p_e * w_e.
    Why a new term: with k = 1 the serving gate renormalises to g = 1, so the
    output loss sends NO gradient to the router's choice (only to the gains of the
    chosen expert). EMRD supplies that missing signal directly from the oracle,
    is scale-free, flat where the oracle has no preference (diffuse energy) and
    never saturates (log of the mass on the good set)."""
    share = en.float() / en.float().sum(-1, keepdim=True).clamp_min(1e-30)
    w = (float(k) * share).clamp(max=1.0)
    cap = (p.float() * w).sum(-1).clamp_min(1e-9)
    return -cap.log().mean()


def ltd_loss(y: torch.Tensor, y_dense: torch.Tensor) -> torch.Tensor:
    """Layer-local dense-twin loss: ||y - y_d||^2 / ||y_d||^2 (ratio of sums over the
    micro-batch; the dense twin y_d is the same layer's own dense FFN on the SAME
    input, so every layer gets a dense, well-conditioned local target instead of
    relying on the final-logit KL alone to reach 64 layers)."""
    yd = y_dense.detach().float()
    return (y.float() - yd).pow(2).sum() / yd.pow(2).sum().clamp_min(1e-12)


# ----------------------------------------------------------------------------
# 6. PrismMoeMLP — training-time module (fragment view of dense SwiGLU FFN)
# ----------------------------------------------------------------------------
class PrismMoeMLP(nn.Module):
    """Function-field fission of a dense SwiGLU FFN layer.

    The dense weights stay resident ONCE (frozen). Experts are equal-size
    index-views over the intermediate dimension; ZSSI completions are
    zero-sum low-rank fields on the experts' gate/up projections (exactly
    fusable into standard expert Linear weights); a shared expert holds the
    highest duty-cycle neurons; per-expert scalar gains c_e are fused into
    down_proj at export.

    Theorems (verified in the suite):
      Thm 1 (exact dense identity): with completion gains alpha=beta=0 the
          union of fragments partitions [D] exactly, so
          sum_e E_e(h) + E_0(h) == f(h) for every h.
      Thm 3 (serving parity): materialize_expert(s) fuse fragment + ZSSI
          completion + gain into standard Linear weights that reproduce the
          expert exactly (the bridge into the stock Qwen3_5Moe format).

    forward modes
      'uniform' : y = dense FFN(h)                  (Theorem 1 reference path)
      'soft'    : y = shared + sum_e p_e y_e        (all-fragments, grouped bmm)
      'topk'    : hard top-k sparse path (mask-based, serving-parity semantics)
    """

    def __init__(self, gate: torch.Tensor, up: torch.Tensor, down: torch.Tensor,
                 assign: torch.Tensor, E: int, rank: int = 64, seed: int = 0,
                 k_topk: int | None = None):
        """gate/up: [d, D], down: [d_out, D] (d_out == d for Qwen FFN),
        assign: [D] long (-1 = shared, else 0..E-1).

        NOTE: gate/up/down are NOT cloned — if Parameters are passed, they are
        ADOPTED (zero-copy surgery on a loaded dense model). Set
        requires_grad=False on them to keep the substrate frozen.
        """
        super().__init__()
        self.k_topk = k_topk                       # serving top-k (None -> caller passes k)
        assert gate.shape == up.shape and down.shape[1] == gate.shape[1]
        d, D = gate.shape
        for nm, t in (("W_gate", gate), ("W_up", up), ("W_down", down)):
            if not isinstance(t, nn.Parameter):
                t = nn.Parameter(t, requires_grad=False)
            else:
                t.requires_grad_(False)
            self.register_parameter(nm, t)
        self.register_buffer("assign", assign.clone())
        self.d_model, self.D, self.E = d, D, E
        shared_mask = assign < 0
        self.register_buffer("shared_mask", shared_mask)
        routed = torch.where(~shared_mask)[0]
        order = routed[torch.argsort(assign[routed])]
        m_e = routed.numel() // E
        if routed.numel() % E != 0:
            raise ValueError("routed neurons not divisible by E; balance the fission")
        self.m_e = m_e
        self.register_buffer("expert_idx", order.reshape(E, m_e))
        self.register_buffer("shared_idx", torch.where(shared_mask)[0])
        # ---- natural-row routing maps (derived; persistent=False -> never
        #      enter state_dict / export) ----
        # _row_assign[r]: expert of row r; E (=out of range) marks SHARED
        #   rows -> indexes the EXACT-ZERO column of the padded w_pad.
        # _row_field[r]: assign clamped to a valid expert id (shared rows
        #   -> 0); their ZSSI correction values are irrelevant because
        #   w_row is EXACTLY zero there (0 * finite = 0).
        self.register_buffer("_row_assign",
                             torch.where(assign < 0,
                                         torch.full_like(assign, E), assign),
                             persistent=False)
        self.register_buffer("_row_field", assign.clamp_min(0),
                             persistent=False)
        # ---- ZSSI completion field (trainable) ----
        # EXACT-INIT GUARANTEE (Theorem 1): gains start at exactly ZERO, so the
        # transformed model at init reproduces the dense function under full
        # (uniform) mixing. Capacity grows from zero during training; the
        # zero_sum_penalty keeps growth antisymmetric around the dense field.
        g = torch.Generator().manual_seed(seed)
        U = torch.randn(d, rank, generator=g) / math.sqrt(d)
        P = torch.randn(D, rank, generator=g) / math.sqrt(D)
        self.rank = rank
        self.U = nn.Parameter(U)                                  # [d, R]
        self.P = nn.Parameter(P)                                  # [D, R]
        self.alpha = nn.Parameter(torch.zeros(E, rank))           # [E, R] gate
        self.beta  = nn.Parameter(torch.zeros(E, rank))           # [E, R] up
        self.alpha_shared = nn.Parameter(torch.zeros(rank))       # shared expert field
        self.beta_shared  = nn.Parameter(torch.zeros(rank))
        self.register_buffer("init_scale", torch.tensor(1.0))     # for warm starts
        # ---- prototype-resonant router ----
        self.router = nn.Linear(d, E, bias=False)
        self.log_tau = nn.Parameter(torch.zeros(1))
        # per-expert output gains (calibrated; fused into down_proj at export)
        self.gains = nn.Parameter(torch.ones(E))
        # deployment path switch: 'soft' (train), 'topk' (serving parity),
        # 'uniform' (exact dense teacher — lives inside the student!)
        self.default_mode = "soft"
        # module-level self-checkpointing switch (probes/calibration can
        # disable it via m.self_ckpt = False)
        self.self_ckpt = True
        # ---- v9: oracle machinery ----
        # ||W_down[:, r]||^2 per intermediate row: the Gram-diagonal weight that turns an
        # activation a_r into its OUTPUT energy a_r^2 ||W_down[:, r]||^2 (derived buffer,
        # never enters the state_dict / export).
        with torch.no_grad():
            _wd = self.W_down.detach()
            _dn2 = torch.cat([_wd[:, i:i + 4096].float().pow(2).sum(0)
                              for i in range(0, D, 4096)])
        self.register_buffer("_dn2", _dn2, persistent=False)
        # side channels (computed under no_grad, detached, returned through aux): the trainer
        # switches them on only for its own forward (see DistillTrainer.compute_loss)
        self.side_en = False       # per-expert oracle energy of the dense activations
        self.side_ltd = False      # dense-twin output + this layer's output (layer-local loss)
        # v10 FFN shadow: served y += b + U (M h)  (fit_ffn_shadow; None = off)
        self.register_buffer("sh_U", None)
        self.register_buffer("sh_M", None)
        self.register_buffer("sh_b", None)
        self.use_shadow = True

    # ---------- completion helpers ----------
    def _corr(self, z: torch.Tensor, gains: torch.Tensor) -> torch.Tensor:
        """Per-expert gate/up corrections from the shared field.

        z: [T, R]; gains: [E, R] (or [R] for shared).
        Returns corr [T, E, m_e] for routed (or [T, m_s] for shared).
        """
        P_e = self.P[self.expert_idx.reshape(-1)].reshape(self.E, self.m_e, self.rank)
        tmp = z[:, None, :] * gains[None, :, :]                   # [T, E, R]
        return torch.einsum("emr,ter->etm", P_e, tmp)

    # ---------- expert paths ----------
    def _fragment_stack(self) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Gather (gate, up, down) fragment weights as [E, m_e, d] stacks."""
        idx = self.expert_idx.reshape(-1)
        Wg = self.W_gate.t()[idx].reshape(self.E, self.m_e, self.d_model)
        Wu = self.W_up.t()[idx].reshape(self.E, self.m_e, self.d_model)
        Wd = self.W_down.t()[idx].reshape(self.E, self.m_e, self.d_model)
        return Wg, Wu, Wd

    def expert_outputs(self, h: torch.Tensor, use_completion: bool = True) -> torch.Tensor:
        """All routed expert outputs y_e on tokens h. Returns [T, E, d].

        Reference implementation: gathers [E, m_e, d] fragment weight copies
        and materializes the all-expert output — fine for no-grad
        calibration, the export parity tests and oracles, but the stacks are
        autograd-SAVED in grad mode (~640 MiB per layer at 27B). The
        training forward paths therefore use _natural_routed, which computes
        the same function with zero materialization.
        """
        Wg, Wu, Wd = self._fragment_stack()
        z = _dm(h, self.U) @ self.U                               # [T, R] fp32 field
        if use_completion:
            ca = self._corr(z, self.alpha)                        # [E, T, m_e]
            cb = self._corr(z, self.beta)
        else:
            ca = cb = 0.0
        gate_p = torch.einsum("td,emd->etm", _dm(h, Wg), Wg) + ca
        up_p = torch.einsum("td,emd->etm", _dm(h, Wu), Wu) + cb
        act = F.silu(gate_p) * up_p                               # [E, T, m_e]
        y = torch.einsum("etm,emd->etd", _dm(act, Wd), Wd)        # [E, T, d]
        return y.transpose(0, 1)                                  # [T, E, d]

    def shared_forward(self, h: torch.Tensor, use_completion: bool = True) -> torch.Tensor:
        s = self.shared_idx
        Wg = self.W_gate[:, s].t()                                # [m_s, d]
        Wu = self.W_up[:, s].t()
        Wd = self.W_down[:, s]                                    # [d, m_s]
        if use_completion and self.rank > 0:
            zs = _dm(h, self.U) @ self.U                          # [T, R] fp32
            corr_a = ((self.P[s] * self.alpha_shared[None, :]) @ zs.t()).t()
            corr_b = ((self.P[s] * self.beta_shared[None, :]) @ zs.t()).t()
            a = F.silu(_dm(h, Wg) @ Wg.t() + corr_a) * (_dm(h, Wu) @ Wu.t() + corr_b)
        else:
            a = F.silu(_dm(h, Wg) @ Wg.t()) * (_dm(h, Wu) @ Wu.t())
        return _dm(a, Wd) @ Wd.t()

    def dense_forward(self, h: torch.Tensor) -> torch.Tensor:
        """Exact dense FFN (Theorem-1 reference path).

        BITWISE-ORIGINAL GUARANTEE (Theorem 7 anchor): W_gate/W_up are
        transposed VIEWS of the adopted HF weights, so W.t() is the original
        HF weight object — calling F.linear on it re-dispatches the EXACT
        original GEMM (same layout, same kernel, same accumulation order).
        A hand-rolled `h @ W_view` matmul is mathematically equal but picks a
        different kernel in bf16 -> ~1e-2 logit drift over many layers.
        """
        a = F.silu(F.linear(_dm(h, self.W_gate), self.W_gate.t())) * \
            F.linear(_dm(h, self.W_up), self.W_up.t())
        return F.linear(_dm(a, self.W_down), self.W_down)

    # ---------- natural-order zero-materialization routed path ----------
    def _natural_routed(self, h: torch.Tensor, z: torch.Tensor | None,
                        w_pad: torch.Tensor, use_completion: bool,
                        side: dict | None = None) -> torch.Tensor:
        """Routed experts for all tokens WITHOUT the [E, T, d] stack.

        Linearity of the expert sum is the escape:
            sum_e w[t,e] * y_e[t]  ==  sum_r act[t,r] * w_row[t,r] * Wd[:,r]
        with r the intermediate rows in NATURAL order and
        w_row[t,r] = w_pad[t, _row_assign[r]] — the padded column E gives
        the shared rows' weight EXACTLY zero. Gate/up run as the ORIGINAL
        full-view GEMMs (the pre-surgery dense kernels), the ZSSI
        corrections as row-aligned einsums, and the down projection as ONE
        (act * w_row) @ W_down.t() GEMM: nothing is copied and [E, T, d]
        never exists. Verified against expert_outputs on values and
        gradients in the suite.
        """
        g = _dm(h, self.W_gate) @ self.W_gate                     # [T, D]
        u = _dm(h, self.W_up) @ self.W_up                         # [T, D]
        if use_completion and self.rank > 0:
            if z is None:
                z = _dm(h, self.U) @ self.U                       # [T, R] fp32
            ca = torch.einsum("mr,tr->tm",
                              self.P * self.alpha[self._row_field], z)
            cb = torch.einsum("mr,tr->tm",
                              self.P * self.beta[self._row_field], z)
            act = F.silu(g + ca) * (u + cb)                       # [T, D]
        else:
            act = F.silu(g) * u
        act_b = _dm(act, self.W_down)                             # substrate dtype
        w_row = w_pad[:, self._row_assign].to(act_b.dtype)        # [T, D]
        if side is not None:
            # v9 side channels: DETACHED values computed under no_grad, so they add nothing to
            # autograd's saved set (checkpoint recompute stays consistent whatever the flags).
            with torch.no_grad():
                a0 = ((F.silu(g.float()) * u.float())             # completion-free = the DENSE neuron activations
                      if (use_completion and self.rank > 0) else act.float())
                if self.side_en:
                    side["en"] = oracle_group_energy(a0, self._dn2, self._row_assign,
                                                     self.E)[:, :self.E]
                if self.side_ltd:
                    side["y_dense"] = _dm(a0, self.W_down) @ self.W_down.t()   # dense twin [T, d]
        return (act_b * w_row) @ self.W_down.t()                  # [T, d]

    # ---------- main forward ----------
    def forward(self, h: torch.Tensor, mode: str | None = None, k: int | None = None,
                use_completion: bool = True, return_aux: bool = False):
        # shape-agnostic entry: decoders call mlp(hidden [B, L, d]); the core
        # math below is [T, d]. Flatten on entry, restore the lead dims on
        # every exit path.
        lead = h.shape[:-1]
        h2 = h.reshape(-1, h.shape[-1]) if len(lead) != 1 else h
        # Module self-checkpointing: bounds the layer's backward footprint
        # to its [T, d] input BY CONSTRUCTION, independent of any
        # Trainer/HF layer checkpointing. use_reentrant=False keeps the
        # side-channel aux (consumed by compute_loss BEFORE backward)
        # fully differentiable and is correct with frozen inputs and
        # trainable parameters (gradients reach params through the
        # recompute).
        if (self.training and torch.is_grad_enabled()
                and getattr(self, "self_ckpt", True)):
            y, aux = _torch_checkpoint(self._forward_2d, h2, mode, k,
                                       use_completion, return_aux,
                                       use_reentrant=False)
        else:
            y, aux = self._forward_2d(h2, mode, k, use_completion, return_aux)
        if len(lead) != 1:
            y = y.reshape(*lead, y.shape[-1])
        return (y, aux) if return_aux else y

    def _forward_2d(self, h: torch.Tensor, mode: str | None = None, k: int = 8,
                    use_completion: bool = True, return_aux: bool = False):
        if mode is None:
            mode = getattr(self, "default_mode", "soft")
        if k is None:
            k = self.k_topk                        # solver-chosen serving top-k
        if k is None:
            raise ValueError("serving top-k not set: pass k= or construct with k_topk=")
        if mode == "uniform":
            y = self.dense_forward(h)
            return ((y, {}) if return_aux else y, {})
        logits = self.router(_dm(h, self.router.weight)) / torch.exp(self.log_tau).clamp(0.25, 4.0)
        aux: dict = {"logits": logits}
        z = _dm(h, self.U) @ self.U if (use_completion and self.rank > 0) else None
        y_shared = self.shared_forward(h, use_completion)
        side = {} if (self.side_en or self.side_ltd) else None    # v9 side channels
        if mode == "soft":
            p = torch.softmax(logits, dim=-1)                     # [T, E]
            w = p * self.gains[None, :]                           # [T, E] fp32
            # padded column E = the shared rows' EXACT-zero weight
            w_pad = torch.cat([w, w.new_zeros(w.shape[0], 1)], dim=1)
            y = y_shared + self._natural_routed(h, z, w_pad, use_completion, side)
            aux["p"] = p
        elif mode == "topk":
            p_full = torch.softmax(logits, dim=-1)
            topv, topi = p_full.topk(k, dim=-1)
            g = topv / topv.sum(dim=-1, keepdim=True)
            w = g * self.gains[topi]                              # [T, k] fp32
            # hard selection as a scattered mask — skipped experts get
            # EXACT zero (same terms as a gather, regrouped)
            w_pad = torch.zeros(h.shape[0], self.E + 1, dtype=w.dtype,
                                device=h.device).scatter(1, topi, w)
            y = y_shared + self._natural_routed(h, z, w_pad, use_completion, side)
            if self.sh_U is not None and getattr(self, "use_shadow", True):
                # v10: what the unselected experts would have added, predicted from h
                hm = _dm(h, self.sh_M)
                y = y + ((hm @ self.sh_M.t()) @ self.sh_U.t() + self.sh_b).to(y.dtype)
            aux["p"] = p_full
            aux["load"] = torch.bincount(topi.reshape(-1), minlength=self.E).float()
        else:
            raise ValueError(f"unknown mode {mode}")
        # Telemetry contract: store aux ONLY on the trainer's forward
        # (via _store_aux) or under no-grad. The checkpoint recomputes
        # that run during backward must never store anything — holding
        # any tensor of a recomputed graph pins that graph's entire
        # saved set, which is how the training OOMs happened.
        y_out = _dm(y, h)         # never leak fp32 field math into the model stream
        if side:
            aux.update(side)      # v9: detached oracle energies / dense twin
            if "y_dense" in side:
                aux["y"] = y_out  # the trainer's LTD term compares THIS output with the dense twin
        if (not torch.is_grad_enabled()) or getattr(self, "_store_aux", False):
            self.last_aux = aux
        return y_out, aux

    # ---------- export: materialize standard expert Linear weights ----------
    def materialize_expert(self, e: int) -> dict:
        """Fuse fragment + ZSSI completion + gain into standard Linear weights.

        Returns dict: gate_proj [m_e, d], up_proj [m_e, d], down_proj [d, m_e].
        Exactly reproduces the expert path (Theorem 3: serving parity, ~2.5e-7).
        """
        idx = self.expert_idx[e]
        Wg = self.W_gate[:, idx].t().clone()
        Wu = self.W_up[:, idx].t().clone()
        P_e = self.P[idx]                                         # [m_e, R]
        A_e = P_e @ torch.diag(self.alpha[e].detach()) @ self.U.t()
        B_e = P_e @ torch.diag(self.beta[e].detach()) @ self.U.t()
        # HF Linear convention: down_proj weight is [d_out, m_e]
        Wd = (self.gains[e].detach() * self.W_down[:, idx]).clone()
        return {"gate_proj": Wg + _dm(A_e, Wg), "up_proj": Wu + _dm(B_e, Wu),
                "down_proj": Wd}

    def materialize_shared(self) -> dict:
        s = self.shared_idx
        Wg = self.W_gate[:, s].t().clone()
        Wu = self.W_up[:, s].t().clone()
        Ps = self.P[s]
        A_s = Ps @ torch.diag(self.alpha_shared.detach()) @ self.U.t()
        B_s = Ps @ torch.diag(self.beta_shared.detach()) @ self.U.t()
        Wd = self.W_down[:, s].clone()  # [d_out, m_s] HF convention
        return {"gate_proj": Wg + _dm(A_s, Wg), "up_proj": Wu + _dm(B_s, Wu),
                "down_proj": Wd}

    def materialize_experts_fused(self, device=None) -> dict:
        """Vectorized export in the REAL Qwen3_5Moe '3D fused' layout:
            gate_up_proj: [E, 2*m_e, d]  (rows 0:m_e = gate, m_e:2m_e = up)
            down_proj:    [E, d_out, m_e]
        (verified against transformers.models.qwen3_5_moe.Qwen3_5MoeExperts,
        which chunks gate_up_proj[e] into (gate, up) and applies
        F.linear(x, gate_up_proj[e]) / F.linear(a, down_proj[e]).)
        Equals stacking materialize_expert(e) over e (Theorem 3, fused form).
        """
        dev = device or self.W_gate.device
        idx = self.expert_idx.reshape(-1)
        E, m_e, d = self.E, self.m_e, self.d_model
        with torch.no_grad():
            P_stack = self.P[idx].reshape(E, m_e, self.rank).to(dev)
            Wg_stack = self.W_gate.t()[idx].reshape(E, m_e, d).to(dev)
            Wu_stack = self.W_up.t()[idx].reshape(E, m_e, d).to(dev)
            gate_stack = Wg_stack + torch.einsum(
                "emr,er,dr->emd", P_stack, self.alpha.detach().to(dev),
                self.U.detach().to(dev)).to(Wg_stack.dtype)
            up_stack = Wu_stack + torch.einsum(
                "emr,er,dr->emd", P_stack, self.beta.detach().to(dev),
                self.U.detach().to(dev)).to(Wu_stack.dtype)
            down_stack = (self.W_down.t()[idx].reshape(E, m_e, -1).to(dev)
                          * self.gains.detach().to(dev).to(Wg_stack.dtype)[:, None, None]
                          ).transpose(1, 2)
        return {"gate_up_proj": torch.cat([gate_stack, up_stack], dim=1).contiguous(),
                "down_proj": down_stack.contiguous()}


# ----------------------------------------------------------------------------
# 7. Param accounting & auto-tuner
# ----------------------------------------------------------------------------
def count_params(d_model: int, D: int, n_layers: int, E: int, k: int,
                 shared_size: int, vocab: int, attn_params: int,
                 rank: int = 64, mtp_params: int = 0, vision_params: int = 0,
                 tie_word_embeddings: bool = False) -> dict:
    """Parameter + active-parameter accounting for PRISM fission.

    Total  = dense-equivalent total + shared low-rank completion fields.
    Active = embeddings (+head) + attention + shared expert + k/E of routed
             FFN + router rows (+ MTP/vision if counted).
    """
    m_e = (D - shared_size) // E
    embed = vocab * d_model * (1 if tie_word_embeddings else 2)
    routed_total = n_layers * E * 3 * d_model * m_e
    shared_total = n_layers * 3 * d_model * shared_size
    field_per_layer = d_model * rank + D * rank + 2 * E * rank + 2 * rank
    total = embed + attn_params + routed_total + shared_total + \
        n_layers * field_per_layer + mtp_params + vision_params
    active = embed + attn_params + n_layers * 3 * d_model * shared_size + \
        n_layers * k * 3 * d_model * m_e + n_layers * E * d_model + \
        mtp_params + vision_params
    return {"total": total, "active": active,
            "ffn_total": routed_total + shared_total,
            "ffn_active": n_layers * (k * 3 * d_model * m_e + 3 * d_model * shared_size),
            "m_e": m_e}


def autotune(d_model: int, D: int, n_layers: int, vocab: int, attn_params: int,
             target_active: float, shared_size: int = 512, rank: int = 64,
             candidates=((256, 8), (128, 8), (128, 4), (64, 8), (64, 4),
                         (64, 2), (32, 4), (32, 2))) -> tuple:
    """Choose (E, k) minimizing active params subject to expert size >= 64;
    reports the architecture floor (attention + embeddings are always on)."""
    rows = []
    for E, k in candidates:
        if (D - shared_size) % E != 0:
            continue          # balanced fission requires exact divisibility
        m_e = (D - shared_size) // E
        if m_e < 64:
            continue
        r = count_params(d_model, D, n_layers, E, k, shared_size, vocab,
                         attn_params, rank)
        rows.append({"E": E, "k": k, **r})
    rows.sort(key=lambda x: x["active"])
    floor = count_params(d_model, D, n_layers, candidates[0][0], 1,
                         shared_size, vocab, attn_params, rank)
    floor["active"] -= n_layers * 3 * d_model * ((D - shared_size) // candidates[0][0])
    return rows[0], rows

PRISM_MODULES: list = []   # (name, PrismMoeMLP) — populated by the surgery cell
print("PRISM core loaded.")


In [ ]:
# RESONA core — resonant linears, MoRM, sectioned projector, head funnel,
# waterfall + the 3B budget solver
# ============================================================================
# RESONA CORE — the three execution families that break the 10.5B floor
# of FFN-only sparsity, plus the whole-model budget solver:
#
#   8.  AntennaMast        — one global random-feature sketch shared by every
#                            router in the model (one phi(h) per token).
#   9.  ResonantLinear     — row-mode routing of ANY whole Linear: always-on
#                            carrier rows + top-k resonant row-blocks.
#                            Entry-exact partition (Theorem 4): full mode ==
#                            dense exactly; per-token tail certificate (Thm 5).
#   10. MoRM               — Mixture of Recurrent Memories: write-sparse
#                            DeltaNet value-heads (top-m writes per token,
#                            read-dense). Masking v or beta zeroes the rank-1
#                            write exactly -> installs inside the stock GDN
#                            with zero kernel surgery (Prop 6 certificate).
#   11. SectionedProjector — fused DeltaNet in_proj split into contiguous
#                            sections [q|k|v|z]: q/z routed rows, k dense,
#                            v head-routed (MoRM), all zero-copy on the
#                            fused weight (total params preserved).
#   12. HeadFunnel         — Spectral Head Funnel: rank-r randomized-SVD
#                            proxy of lm_head + exact top-V_s shortlist rows.
#   13. active_waterfall / solve_budget_v2 — STRICT touched-weights
#                            accounting + coordinate solver for the budget.
#
# Conventions (STRICT, honest): "active params" = weight elements actually
# READ/EVALUATED to compute one text token's forward pass. Embedding = 1
# row. lm_head = funnel proxy + shortlist rows. Vision tower & MTP draft
# excluded from text-token active (reported separately).
# ============================================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as _torch_checkpoint




# ----------------------------------------------------------------------------
# 8. AntennaMast — one shared random-feature sketch for all routers
# ----------------------------------------------------------------------------
class AntennaMast(nn.Module):
    """phi(h) = silu(R h) with frozen random R [S, d].

    One mast serves EVERY sparse site in the model: each site owns only a tiny
    linear head A [n_modes, S]. Cost per token: S x d (once per token, not per
    site). Random features are distribution-agnostic; heads are ridge- or
    distillation-trained per site.
    """

    def __init__(self, d_model: int, sketch_dim: int = 512, seed: int = 0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        R = torch.randn(sketch_dim, d_model, generator=g) / math.sqrt(d_model)
        self.register_buffer("R", R)
        self.S = sketch_dim
        self.d_model = d_model
        self.seed = seed

    def _R_for(self, d_in: int, device: torch.device) -> torch.Tensor:
        """Sketch matrix for input width d_in. Widths != d_model (e.g. the
        DeltaNet out_proj input, whose width is value_dim) get a DETERMINISTIC
        sibling sketch derived from the master seed — so calibration, serving
        and the exported kernel all generate identical R without serializing
        every width."""
        if d_in == self.d_model:
            return self.R
        key = f"R{d_in}"
        R = getattr(self, key, None)
        if R is None or R.device != device:
            g = torch.Generator().manual_seed(self.seed * 1000003 + d_in)
            R = torch.randn(self.S, d_in, generator=g) / math.sqrt(d_in)
            self.register_buffer(key, R.to(device=device, dtype=self.R.dtype))
        return getattr(self, key)

    def forward(self, h: torch.Tensor) -> torch.Tensor:
        """h [..., d] -> phi [..., S]  (any input width d)"""
        R = self._R_for(h.shape[-1], h.device)
        return F.silu(h.to(R.dtype) @ R.t())


# ----------------------------------------------------------------------------
# 9. ResonantLinear — row-mode routing for a WHOLE Linear (Theorem 4/5)
# ----------------------------------------------------------------------------
def plan_row_modes(out_rows: int, in_rows: int, n_blocks: int,
                   carrier_frac: float,
                   row_energy: torch.Tensor | None = None) -> dict:
    """Partition `out_rows` output rows into carrier + equal row-blocks.

    carrier absorbs the division remainder so blocks are EXACTLY equal
    (regular grouped-GEMM export). Carrier rows = highest total energy mass
    (row_energy[i] = E_calib[(w_i . x)^2]; uniform if None).

    Returns dict(carrier_idx [c], block_idx [B, r]) — indices are sorted
    (cache-friendly) and cover EVERY row exactly once (entry partition).
    """
    if row_energy is None:
        row_energy = torch.ones(out_rows)
    device = row_energy.device
    n_carrier0 = max(0, min(int(round(carrier_frac * out_rows)), out_rows // 4))
    n_blocks = max(1, min(n_blocks, (out_rows - n_carrier0)))
    r = (out_rows - n_carrier0) // n_blocks
    if r == 0:
        n_blocks, r = 1, out_rows - max(0, out_rows // 4)
    n_carrier = out_rows - n_blocks * r          # carrier absorbs remainder
    order = torch.argsort(row_energy.to(device), descending=True)
    carrier_idx = torch.sort(order[:n_carrier]).values
    rest_idx = torch.sort(order[n_carrier:]).values
    block_idx = rest_idx.reshape(n_blocks, r)
    assert int(block_idx.numel() + carrier_idx.numel()) == out_rows
    return {"carrier_idx": carrier_idx, "block_idx": block_idx}


# ----------------------------------------------------------------------------
# 8b. SHADOW EXECUTION (v10) — control-variate sparsity
# ----------------------------------------------------------------------------
# Up to v9 everything that was not computed exactly was ZEROED: ~90% of the output rows of every routed
# projection, every unselected value head. The served student therefore started at a held-out PPL of ~2e5.
# Shadow execution never skips:
#       y  =  S(x)  +  1[selected] * (exact(x) - S(x)),          S(x) = U (U^T W) x
# S is the best rank-r predictor of THIS SITE'S OUTPUT under the calibration distribution (U = top principal
# directions of the dense outputs), so the error of an unselected row falls from its full energy to its residual
# energy. The router is calibrated on the RESIDUAL (it picks the block the shadow explains worst), and in grad
# mode it gets the exact counterfactual gradient  dL/dw_b = <dL/dy_b, exact_b - S_b>  for EVERY block through a
# straight-through gate (v9's top-1 routers received no gradient at all on the served path).
@torch.no_grad()
def fit_shadow_basis(Y: torch.Tensor, rank: int, device=None, val_frac: float = 0.2):
    """Y [N, out]: stash of DENSE outputs of one site.
    Returns (U [out, r] orthonormal fp32 | None, held-out energy fraction captured, residual [N, out] fp32)."""
    Yd = Y.to(device).float() if device is not None else Y.float()
    N, out = Yd.shape
    n_val = max(1, int(round(val_frac * N))) if N >= 8 else 0
    n_fit = N - n_val
    r = int(max(0, min(int(rank), n_fit, out)))
    if r == 0:
        return None, 0.0, Yd
    _, _, Vh = torch.linalg.svd(Yd[:n_fit], full_matrices=False)
    U = Vh[:r].t().contiguous()                                   # [out, r]
    res = Yd - (Yd @ U) @ U.t()
    sl = slice(n_fit, N) if n_val else slice(0, N)
    cap = 1.0 - float(res[sl].pow(2).sum() / Yd[sl].pow(2).sum().clamp_min(1e-30))
    return U, cap, res


@torch.no_grad()
def ridge_router_init(feats: torch.Tensor, energy: torch.Tensor) -> torch.Tensor:
    """UNCENTERED ridge of log(block energy) on the mast features -> router weight [B, S].
    (The routers have no bias; a centered fit — v9 — drops every block's mean level, and the top-1 choice is then
    made on deviations only. Uncentered, the mean level rides on the mean feature direction.)"""
    X = feats.to(torch.float32)
    y = torch.log(energy.to(torch.float32).clamp_min(1e-12))
    lam = 1e-2 * (X * X).mean().clamp_min(1e-12) * X.shape[0]
    A = torch.linalg.solve(X.t() @ X + lam * torch.eye(X.shape[1], device=X.device), X.t() @ y)   # [S, B]
    return A.t().contiguous()


def _block_energy(res: torch.Tensor, block_idx: torch.Tensor) -> torch.Tensor:
    """res [N, rows] residual (or raw) outputs, block_idx [B, r] -> per-token block energy [N, B]."""
    B = block_idx.shape[0]
    cols = block_idx.reshape(-1).to(res.device)
    return res.index_select(1, cols).float().pow(2).view(res.shape[0], B, -1).sum(-1)


class ResonantLinear(nn.Module):
    """Executes y = W x with row-mode resonance:

        y = W[carrier] x  +  sum_{b in topk(phi)} c_b * W[block_b] x

    * ENTRY-EXACT PARTITION (Theorem 4): carrier ∪ ⋃blocks = every row exactly
      once, so evaluating all blocks reproduces y = W x EXACTLY (fp
      reassociation aside). The dense weight is ADOPTED (zero copy).
    * TAIL CERTIFICATE (Theorem 5): per-token error of a selected set S obeys
        || W x - y_S ||  <=  sqrt( sum_{b not in S} ||W[b] x||^2 )
      — measurable during calibration (one dense matmul yields all row dots).
    * Router: A [B, S] head on the shared AntennaMast (phi computed once per
      token per layer and shared by all sites of that layer).
    * Per-block gains c_b >= 0 (calibrated at init; fused into W at export).
    """

    def __init__(self, weight: torch.Tensor, plan: dict, mast: AntennaMast,
                 calib_feats: torch.Tensor | None = None,
                 calib_energy: torch.Tensor | None = None,
                 topk: int = 1):
        """weight: [out, in] (HF Linear .weight, adopted zero-copy, frozen).
        calib_feats: [T, S] mast features on calibration tokens (optional).
        calib_energy: [T, B] per-token per-block output energy (optional).
        """
        super().__init__()
        if not isinstance(weight, nn.Parameter):
            weight = nn.Parameter(weight, requires_grad=False)
        else:
            weight.requires_grad_(False)
        self.register_parameter("weight", weight)
        self.register_buffer("carrier_idx", plan["carrier_idx"].clone())
        self.register_buffer("block_idx", plan["block_idx"].clone())
        self.out_f, self.in_f = weight.shape
        self.B, self.r = plan["block_idx"].shape
        self.topk = min(topk, self.B)
        self.mast = mast                                   # shared, not owned
        self.router = nn.Linear(mast.S, self.B, bias=False)
        if calib_feats is not None and calib_energy is not None:
            self._ridge_init_router(calib_feats, calib_energy)
        else:
            nn.init.zeros_(self.router.weight)
        self.gains = nn.Parameter(torch.ones(self.B))
        self.default_mode = "topk"
        self.last_cert: dict | None = None
        # v10 shadow: U [out, r] (orthonormal output basis) and M = U^T W [r, in]; None = v9 behaviour
        self.register_buffer("sh_U", None)
        self.register_buffer("sh_M", None)
        self.router_st = True          # straight-through router gradient on the served path
        # module self-checkpointing switch (same contract as PrismMoeMLP):
        # in grad mode the whole row-mode computation is recomputed at
        # backward from the [T, in] input, so the soft/topk saved set
        # collapses to the [T, in] boundary BY CONSTRUCTION.
        self.self_ckpt = True

    def _ridge_init_router(self, feats: torch.Tensor, energy: torch.Tensor):
        """Ridge-regress log(energy + eps) on mast features -> warm router."""
        with torch.no_grad():
            X = feats.to(torch.float32)
            y = torch.log(energy.to(torch.float32).clamp_min(1e-12))
            Xc = X - X.mean(0, keepdim=True)
            yc = y - y.mean(0, keepdim=True)
            lam = 1e-2 * (Xc * Xc).mean().clamp_min(1e-12)
            eye = torch.eye(X.shape[1], device=X.device)
            A = torch.linalg.solve(Xc.t() @ Xc + lam * eye, Xc.t() @ yc)  # [S, B]
            self.router.weight.data.copy_(A.t().to(self.router.weight.dtype))

    # ---- v10 shadow ----
    @torch.no_grad()
    def attach_shadow(self, U: torch.Tensor):
        """U [out, r]: orthonormal basis of the principal OUTPUT directions (fit_shadow_basis)."""
        U = U.to(self.weight.device, torch.float32)
        M = torch.cat([U[:, i:i + 64].t() @ self.weight.float() for i in range(0, U.shape[1], 64)], 0)
        self.sh_U = U.to(self.weight.dtype).contiguous()
        self.sh_M = M.to(self.weight.dtype).contiguous()

    def _shadow(self, x: torch.Tensor) -> torch.Tensor:
        return (x @ self.sh_M.t()) @ self.sh_U.t()

    # ---- execution ----
    def _carrier_out(self, x: torch.Tensor) -> torch.Tensor:
        if self.carrier_idx.numel() == 0:
            return x.new_zeros(x.shape[0], self.out_f)
        return F.linear(x, self.weight[self.carrier_idx])

    def block_outputs(self, x: torch.Tensor) -> torch.Tensor:
        """ALL block outputs stacked [T, B, r] (calibration / full mode).
        Column-gather of ONE whole-module GEMM: the weight operand is the
        parameter VIEW (no [B*r, d] weight-row copy is formed); the output
        rows are the same either way, so values and certificates are
        unchanged."""
        y_all = F.linear(x, self.weight)                   # [T, out_f]
        return y_all[:, self.block_idx.reshape(-1)] \
            .view(x.shape[0], self.B, self.r)              # [T, B, r]

    def forward(self, x: torch.Tensor, mode: str | None = None,
                return_cert: bool = False):
        mode = mode or self.default_mode
        lead = x.shape[:-1]                                # support [..., in]
        T = int(x.numel() // self.in_f)
        x2 = x.reshape(T, self.in_f)
        # Module self-checkpointing: in grad mode (training) the module's
        # backward footprint is its [T, in] input BY CONSTRUCTION. The
        # dense and no-grad/serving paths are untouched (honest active
        # accounting). use_reentrant=False keeps the side-channel cert
        # differentiable and is correct with frozen inputs + trainable
        # params (gradients reach params through the recompute).
        if (self.training and torch.is_grad_enabled()
                and mode != "dense"
                and getattr(self, "self_ckpt", True)):
            y = _torch_checkpoint(self._forward_2d, x2, mode,
                                  use_reentrant=False)
        else:
            y = self._forward_2d(x2, mode)
        y = y.reshape(*lead, self.out_f) if len(lead) != 1 else y
        return (y, self.last_cert) if return_cert else y

    def _forward_2d(self, x: torch.Tensor, mode: str) -> torch.Tensor:
        T = x.shape[0]
        if mode == "dense":                                # literal dense path
            y = F.linear(x, self.weight)
            self.last_cert = {"rel": 0.0}
            return y
        if mode == "soft":                                 # differentiable all-blocks
            # w = softmax(logits)*B*gains: at init (zero router, gains=1) this is
            # EXACTLY dense (each disjoint block weighted 1.0) — training starts
            # at the Theorem-7 anchor and deforms continuously.
            logits = self.router(self.mast(x))             # [T, B]
            p = torch.softmax(logits, dim=-1)              # [T, B]
            yb = self.block_outputs(x)                     # [T, B, r]
            w = p * (float(self.B) * self.gains[None, :])
            # blocks are DISJOINT row-groups: scale each block by its own weight
            # and CONCATENATE (an einsum over b would wrongly sum the blocks)
            rest_out = (w.unsqueeze(-1) * yb).reshape(T, self.B * self.r)
            y = x.new_zeros(T, self.out_f)
            if self.carrier_idx.numel() > 0:
                y.index_copy_(1, self.carrier_idx, self._carrier_out(x))
            rest_all = self.block_idx.reshape(-1)
            y.index_copy_(1, rest_all, rest_out.to(y.dtype))
            # diagnostics only: store detached values so the stored
            # copy can never pin a (re)computed graph; the differentiable
            # p/logits flow through the RETURNED y, exactly as before.
            self.last_cert = {"p": p.detach(), "logits": logits.detach()}
            return y
        if mode == "full":                                 # EXACT resonant path
            rest_out = self.block_outputs(x).reshape(T, -1)  # [T, B*r] concat
            y = x.new_zeros(T, self.out_f)
            if self.carrier_idx.numel() > 0:
                y.index_copy_(1, self.carrier_idx, self._carrier_out(x))
            rest_all = self.block_idx.reshape(-1)
            y.index_copy_(1, rest_all, rest_out.to(y.dtype))
            self.last_cert = {"rel": 0.0}
            return y
        # ---- sparse resonant path (top-k) ----
        if torch.is_grad_enabled() or (globals().get("_wal_armed") is not None and _wal_armed(self)):
            # Grad-mode backprop supergraph: ONE whole-module GEMM whose
            # weight operand is the parameter VIEW, with the exact sparse
            # block weights applied AFTER the GEMM. A skipped block
            # contributes exactly zero to forward AND backward, so this
            # branch and the true-sparse chunked gather below are the SAME
            # function with the SAME gradients — while the [T*k, r, in]
            # weight copies the gather would save for grad_x never exist.
            # Serving (no-grad) keeps the honest sparse gather.
            phi = self.mast(x)
            logits = self.router(phi)
            topv, topi = logits.topk(self.topk, dim=-1)
            g = self.gains[topi]
            y_all = F.linear(x, self.weight)               # [T, out_f]
            w = torch.zeros(T, self.B, dtype=y_all.dtype, device=x.device)
            w = w.scatter(1, topi, g.to(y_all.dtype))      # differentiable
            factor = torch.zeros(T, self.out_f, dtype=y_all.dtype,
                                 device=x.device)
            if self.carrier_idx.numel() > 0:
                factor[:, self.carrier_idx] = 1.0
            factor[:, self.block_idx.reshape(-1)] = w.unsqueeze(-1) \
                .expand(T, self.B, self.r).reshape(T, self.B * self.r)
            self.last_cert = {"topi": topi.detach(), "logits": logits.detach()}
            if self.sh_U is None:
                return y_all * factor
            # v10 shadow: y = S + sel * (g * exact - S); carrier rows exact. `sel` is the hard 0/1 block
            # choice plus a straight-through soft part, so the router receives
            #   dL/dsel_b = <dL/dy_b, g_b exact_b - S_b>   for EVERY block (the value is unchanged).
            sel = torch.zeros(T, self.B, dtype=y_all.dtype, device=x.device).scatter(
                1, topi, torch.ones_like(topi, dtype=y_all.dtype))
            if self.training and getattr(self, "router_st", True):
                ps = torch.softmax(logits.float(), dim=-1).to(y_all.dtype)
                sel = sel + (ps - ps.detach())
            _mkb = globals().get("_mk_blocks")               # v16: the wallet (or, in quota mode, the market's probes)
            if _mkb is not None:
                sel = _mkb(self, sel, logits, topi, locals().get("tag", ""))
            rest = self.block_idx.reshape(-1)
            gate = torch.zeros(T, self.out_f, dtype=y_all.dtype, device=x.device)
            if self.carrier_idx.numel() > 0:
                gate[:, self.carrier_idx] = 1.0
            gate[:, rest] = sel.unsqueeze(-1).expand(T, self.B, self.r).reshape(T, self.B * self.r)
            gfull = torch.ones(self.out_f, dtype=y_all.dtype, device=x.device).scatter(
                0, rest, self.gains.to(y_all.dtype).repeat_interleave(self.r))
            S = self._shadow(x)
            return S + gate * (y_all * gfull[None, :] - S)
        phi = self.mast(x)                                 # [T, S]
        logits = self.router(phi)                          # [T, B]
        topv, topi = logits.topk(self.topk, dim=-1)        # [T, k]
        g = self.gains[topi]                               # [T, k]
        # Chunked over T: the Wsel gather is [T*k, r, in] — unbounded, OOMs at
        # 27B scale. Scoring is per-token, so chunking is mathematically identical.
        c = _gather_chunk(self.topk * self.r * self.in_f * self.weight.element_size())
        ys_parts, ti_parts = [], []
        for i in range(0, T, c):
            ti = topi[i:i + c]                             # [t, k]
            Ws = self.weight[self.block_idx[ti.reshape(-1)]].reshape(-1, self.r, self.in_f)
            xr = x[i:i + c].repeat_interleave(self.topk, 0).unsqueeze(-1)  # [t*k, in, 1]
            ys_parts.append(torch.bmm(Ws, xr).squeeze(-1).reshape(-1, self.topk, self.r))
            ti_parts.append(ti)
        topi = torch.cat(ti_parts, 0)
        ys = torch.cat(ys_parts, 0) * g.unsqueeze(-1)
        shadow = self.sh_U is not None
        y = self._shadow(x) if shadow else x.new_zeros(T, self.out_f)     # v10: unselected rows = shadow
        if self.carrier_idx.numel() > 0:
            y.index_copy_(1, self.carrier_idx, self._carrier_out(x))
        rows = self.block_idx[topi]                        # [T, k, r]
        if shadow:                                         # exact rows REPLACE the shadow values
            y.scatter_(1, rows.reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        else:
            y.scatter_add_(1, rows.reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        self.last_cert = {"topi": topi.detach(), "logits": logits.detach()}
        return y

    def materialize_dense(self) -> torch.Tensor:
        """Recover the EXACT original weight (entry partition inverse, with
        per-block gains folded in — gains==1 recovers the original bitwise)."""
        W = torch.zeros_like(self.weight.data)
        if self.carrier_idx.numel() > 0:
            W[self.carrier_idx] = self.weight.data[self.carrier_idx]
        rows = self.block_idx.reshape(-1)
        g = self.gains.detach()[:, None].expand(self.B, self.r).reshape(-1, 1)
        W[rows] = self.weight.data[rows] * g.to(self.weight.dtype)
        return W

    def active_params(self) -> int:
        c = self.carrier_idx.numel()
        sh = (self.sh_U.numel() + self.sh_M.numel()) if self.sh_U is not None else 0
        return c * self.in_f + min(self.topk, self.B) * self.r * self.in_f + sh


# ----------------------------------------------------------------------------
# 10. MoRM — Mixture of Recurrent Memories (write-sparse value heads)
# ----------------------------------------------------------------------------
class MoRMWritePolicy(nn.Module):
    """Per-token top-m write mask over H recurrent memory slabs.

    Ideal write energy of head h at token t: e_h = beta_h * ||k_h|| * ||v_h||.
    Computing it requires v — the projection we want to SKIP. So the policy
    predicts log e_h from cheap signals only: mast features phi(h) (trained
    head A [H, S]) plus the observable beta_h * ||k_h|| (k is needed anyway
    for the decay path; beta comes from the tiny in_proj_ba).

    Gradients: straight-through top-m (hard forward, soft backward), so the
    self-teacher (dense writes) teaches the policy WHICH heads deserved the
    write. m == H reduces to dense writes (exact).
    """

    def __init__(self, n_heads: int, mast: AntennaMast, m_write: int):
        super().__init__()
        self.H = n_heads
        self.m = min(m_write, n_heads)
        self.router = nn.Linear(mast.S, n_heads, bias=False)
        nn.init.zeros_(self.router.weight)
        self.last_mask: torch.Tensor | None = None

    def forward(self, phi: torch.Tensor,
                obs: torch.Tensor | None = None,
                m: int | None = None) -> torch.Tensor:
        """phi [T, S]; obs [T, H] = log(beta*||k|| + eps). Returns [T, H] mask.

        Forward value is the HARD top-m mask; when training, a soft softmax
        over the selected heads is added in straight-through fashion so
        gradients reach the router."""
        m = m or self.m
        s = self.router(phi)
        if obs is not None:
            s = s + obs
        topv, topi = s.topk(m, dim=-1)
        hard = torch.zeros_like(s).scatter_(-1, topi, 1.0)
        if self.training:
            # bf16-autocast trap: under autocast,
            # softmax ALWAYS runs in fp32 while zeros_like(s) follows s (bf16);
            # scatter_'s TENSOR overload demands an exact dtype match.
            # Explicit fp32 softmax + cast back: identical math on every device
            # (softmax over m<=32 logits costs nothing in fp32).
            soft = torch.zeros_like(s).scatter_(
                -1, topi, torch.softmax(topv.float(), dim=-1).to(s.dtype))
            hard = hard + (soft - soft.detach())          # straight-through
        # store the VALUE only — the straight-through gradient flows
        # through the RETURNED mask.
        self.last_mask = hard.detach()
        return hard


def morm_state_update(S_prev: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
                      beta: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Math-canonical write-sparse recurrent update.

    S_prev: [T, H, dk, dv]; k [T, H, dk]; v [T, H, dv]; beta [T, H];
    mask [T, H] in {0,1}. Returns S_prev + mask_h * beta_h * k_h v_h^T.

    (The real GatedDeltaNet interleaves decay/normalizer bookkeeping; the
    masked rank-1 write term is identical — masking v_h or beta_h zeroes the
    write exactly, which is what SectionedProjector exploits in-place.)"""
    outer = torch.einsum("thd,the->thde", k, v) * beta[:, :, None, None]
    return S_prev + outer * mask[:, :, None, None]


def morm_error_bound(S_dense: torch.Tensor, S_sparse: torch.Tensor,
                     k: torch.Tensor, v: torch.Tensor, beta: torch.Tensor,
                     mask: torch.Tensor):
    """Per-token certificate for skipped writes:
        ||S_dense - S_sparse||_F  <=  sum_{h skipped} beta_h ||k_h|| ||v_h||.
    Returns (gap, bound), each [T]."""
    gap = (S_dense - S_sparse).flatten(1).norm(dim=1)
    skipped = (k.norm(dim=-1) * v.norm(dim=-1) * beta.abs()) \
        .masked_fill(mask.bool(), 0.0).sum(dim=1)
    return gap, skipped


# ----------------------------------------------------------------------------
# 11. SectionedProjector — fused DeltaNet in_proj_qkvz under RESONA execution
# ----------------------------------------------------------------------------
class SectionedProjector(nn.Module):
    """Zero-copy split of a fused [q|k|v|z] weight (contiguous row sections):
         q -> routed rows (carrier + top-k blocks; per-token)
         k -> dense rows   (every position's decay/state path needs them)
         v -> MoRM head-routed gather (ONLY selected heads' rows evaluated)
         z -> routed rows  (output gate; per-token)

    The fused weight stays registered ONCE here (sections are views).
    mode='dense' reproduces the stock fused projection exactly; mode='full'
    reproduces it through the resonant path (Theorem 4); mode='topk' is the
    sparse serving path (v zero-padded to full width so the downstream
    DeltaNet code is UNCHANGED). Under autograd every sparse family
    executes as its masked-dense SUPERGRAPH (exact values, exact gradients,
    zero weight materialization); under no-grad it executes the true sparse
    path (honest active-parameter accounting)."""

    def __init__(self, weight: torch.Tensor, section_sizes, v_head_rows: int,
                 plan_q=None, plan_z=None, mast: AntennaMast | None = None,
                 morm: MoRMWritePolicy | None = None, topk: int = 1):
        super().__init__()
        if not isinstance(weight, nn.Parameter):
            weight = nn.Parameter(weight, requires_grad=False)
        else:
            weight.requires_grad_(False)
        self.register_parameter("weight", weight)
        q_r, k_r, v_r, z_r = section_sizes
        assert v_r % v_head_rows == 0, "v rows must divide into heads"
        self.q_rows, self.k_rows, self.v_rows, self.z_rows = q_r, k_r, v_r, z_r
        self.n_v_heads = v_r // v_head_rows
        self.v_head_rows = v_head_rows
        bounds = [0, q_r, q_r + k_r, q_r + k_r + v_r, q_r + k_r + v_r + z_r]
        self.register_buffer("sec_bounds", torch.tensor(bounds))
        self.mast = mast
        self.morm = morm
        self.topk = topk
        self.default_mode = "topk"
        self.register_parameter("v_gains", nn.Parameter(torch.ones(self.n_v_heads)))
        # routed-row machinery for q and z sections (inline, zero-copy views)
        for tag, n_rows, plan in (("q", q_r, plan_q), ("z", z_r, plan_z)):
            if plan is None:
                continue
            self.register_buffer(f"{tag}_carrier", plan["carrier_idx"].clone())
            self.register_buffer(f"{tag}_blocks", plan["block_idx"].clone())
            B, r = plan["block_idx"].shape
            setattr(self, f"{tag}_B", B)
            setattr(self, f"{tag}_r", r)
            setattr(self, f"{tag}_router",
                    nn.Linear(mast.S, B, bias=False) if mast is not None else None)
            self.register_parameter(f"{tag}_gains",
                                    nn.Parameter(torch.ones(B)))
            if mast is not None:
                getattr(self, f"{tag}_router").weight.data.zero_()
        self.last_mask: torch.Tensor | None = None
        # telemetry switch: default True stores last_out (probes read it);
        # the trainer sets it False for the training phase only (the dict
        # pins [T, q+k+v+z] tensors per GDN layer across fwd+bwd) and
        # restores it after.
        self.telemetry = True
        # v10 shadows of the q / v / z sections (None = v9 behaviour). With a v shadow EVERY head writes:
        # the MoRM critic no longer decides which writes exist, only which writes are EXACT.
        for _t in ("q", "v", "z"):
            self.register_buffer(f"{_t}_sh_U", None)
            self.register_buffer(f"{_t}_sh_M", None)
        self.router_st = True
        # module self-checkpointing switch (bounds the fused soft-mode
        # saved set to the [T, d] input; see ResonantLinear).
        self.self_ckpt = True

    # ---- v10 shadow ----
    def _sec_bounds(self, tag: str):
        b = [int(v) for v in self.sec_bounds]
        return {"q": (b[0], b[1]), "k": (b[1], b[2]), "v": (b[2], b[3]), "z": (b[3], b[4])}[tag]

    @torch.no_grad()
    def attach_shadow(self, tag: str, U: torch.Tensor):
        lo, hi = self._sec_bounds(tag)
        U = U.to(self.weight.device, torch.float32)
        W = self.weight[lo:hi].float()
        M = torch.cat([U[:, i:i + 64].t() @ W for i in range(0, U.shape[1], 64)], 0)
        setattr(self, f"{tag}_sh_U", U.to(self.weight.dtype).contiguous())
        setattr(self, f"{tag}_sh_M", M.to(self.weight.dtype).contiguous())

    def _shadow(self, x: torch.Tensor, tag: str) -> torch.Tensor:
        return (x @ getattr(self, f"{tag}_sh_M").t()) @ getattr(self, f"{tag}_sh_U").t()

    # ---- routed-rows helper (shared logic for q / z sections) ----
    def _routed_section(self, x: torch.Tensor, tag: str, lo: int, hi: int,
                        mode: str) -> torch.Tensor:
        W = self.weight[lo:hi]                             # section view
        T = x.shape[0]
        if mode == "dense":
            return F.linear(x, W)
        carrier = getattr(self, f"{tag}_carrier")
        blocks = getattr(self, f"{tag}_blocks")
        B, r = getattr(self, f"{tag}_B"), getattr(self, f"{tag}_r")
        if mode == "full":
            y = x.new_zeros(T, hi - lo)
            if carrier.numel() > 0:
                y.index_copy_(1, carrier, F.linear(x, W[carrier]))
            # post-GEMM column gather of the ONE section GEMM (weight VIEW
            # — no [B*r, d] weight copy).
            y.index_copy_(1, blocks.reshape(-1),
                          F.linear(x, W)[:, blocks.reshape(-1)])
            return y
        if mode == "soft":
            # softmax(logits)*B*gains: exact dense at init (see ResonantLinear).
            # Disjoint row-groups: per-block scale + concat (NOT a b-summing einsum).
            logits = getattr(self, f"{tag}_router")(self.mast(x))
            p = torch.softmax(logits, dim=-1)                     # [T, B]
            # post-GEMM column gather — no [B*r, d] weight copy.
            yb = F.linear(x, W)[:, blocks.reshape(-1)].view(T, B, r)
            w = p * (float(B) * getattr(self, f"{tag}_gains")[None, :])
            rest_out = (w.unsqueeze(-1) * yb).reshape(T, B * r)
            y = x.new_zeros(T, hi - lo)
            if carrier.numel() > 0:
                y.index_copy_(1, carrier, F.linear(x, W[carrier]))
            y.index_copy_(1, blocks.reshape(-1), rest_out.to(y.dtype))
            return y
        if torch.is_grad_enabled() or (globals().get("_wal_armed") is not None and _wal_armed(self)):
            # Grad-mode backprop supergraph: ONE whole-module GEMM whose
            # weight operand is the parameter VIEW, with the exact sparse
            # block weights applied AFTER the GEMM. A skipped block
            # contributes exactly zero to forward AND backward, so this
            # branch and the true-sparse chunked gather below are the SAME
            # function with the SAME gradients — while the [T*k, r, in]
            # weight copies the gather would save for grad_x never exist.
            # Serving (no-grad) keeps the honest sparse gather.
            logits = getattr(self, f"{tag}_router")(self.mast(x))
            topv, topi = logits.topk(min(self.topk, B), dim=-1)
            g = getattr(self, f"{tag}_gains")[topi]              # [T, k]
            y_all = F.linear(x, W)                               # [T, rows]
            w = torch.zeros(T, B, dtype=y_all.dtype, device=x.device)
            w = w.scatter(1, topi, g.to(y_all.dtype))            # differentiable
            factor = torch.zeros(T, hi - lo, dtype=y_all.dtype, device=x.device)
            if carrier.numel() > 0:
                factor[:, carrier] = 1.0
            factor[:, blocks.reshape(-1)] = w.unsqueeze(-1) \
                .expand(T, B, r).reshape(T, B * r)
            if getattr(self, f"{tag}_sh_U") is None:
                return y_all * factor
            # v10 shadow (see ResonantLinear): y = S + sel * (g * exact - S), straight-through router gradient
            sel = torch.zeros(T, B, dtype=y_all.dtype, device=x.device).scatter(
                1, topi, torch.ones_like(topi, dtype=y_all.dtype))
            if self.training and getattr(self, "router_st", True):
                ps = torch.softmax(logits.float(), dim=-1).to(y_all.dtype)
                sel = sel + (ps - ps.detach())
            _mkb = globals().get("_mk_blocks")               # v16: the wallet (or, in quota mode, the market's probes)
            if _mkb is not None:
                sel = _mkb(self, sel, logits, topi, locals().get("tag", ""))
            rest = blocks.reshape(-1)
            gate = torch.zeros(T, hi - lo, dtype=y_all.dtype, device=x.device)
            if carrier.numel() > 0:
                gate[:, carrier] = 1.0
            gate[:, rest] = sel.unsqueeze(-1).expand(T, B, r).reshape(T, B * r)
            gfull = torch.ones(hi - lo, dtype=y_all.dtype, device=x.device).scatter(
                0, rest, getattr(self, f"{tag}_gains").to(y_all.dtype).repeat_interleave(r))
            S = self._shadow(x, tag)
            return S + gate * (y_all * gfull[None, :] - S)
        logits = getattr(self, f"{tag}_router")(self.mast(x))     # [T, B]
        topv, topi = logits.topk(min(self.topk, B), dim=-1)
        g = getattr(self, f"{tag}_gains")[topi]
        # Chunked over T: the Wsel gather is [T*k, r, d] — unbounded, OOMs at
        # 27B scale. Scoring is per-token, so chunking is mathematically identical.
        k_sel = topi.shape[1]
        c = _gather_chunk(k_sel * r * self.weight.shape[1] * self.weight.element_size())
        ys_parts, ti_parts = [], []
        for i in range(0, T, c):
            ti = topi[i:i + c]                             # [t, k]
            Ws = W[blocks[ti.reshape(-1)]].reshape(-1, r, self.weight.shape[1])
            xr = x[i:i + c].repeat_interleave(k_sel, 0).unsqueeze(-1)      # [t*k, d, 1]
            ys_parts.append(torch.bmm(Ws, xr).squeeze(-1).reshape(-1, k_sel, r))
            ti_parts.append(ti)
        topi = torch.cat(ti_parts, 0)
        ys = torch.cat(ys_parts, 0) * g.unsqueeze(-1)
        shadow = getattr(self, f"{tag}_sh_U") is not None
        y = self._shadow(x, tag) if shadow else x.new_zeros(T, hi - lo)
        if carrier.numel() > 0:
            y.index_copy_(1, carrier, F.linear(x, W[carrier]))
        if shadow:
            y.scatter_(1, blocks[topi].reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        else:
            y.scatter_add_(1, blocks[topi].reshape(T, -1), ys.reshape(T, -1).to(y.dtype))
        return y

    def _sp_core(self, x: torch.Tensor, mode: str,
                 force_mask: torch.Tensor | None) -> torch.Tensor:
        """The 2-D fused [q|k|v|z] computation (checkpoint-wrapped by
        forward())."""
        T = x.shape[0]
        b0, b1, b2, b3, b4 = (int(self.sec_bounds[0]), int(self.sec_bounds[1]),
                              int(self.sec_bounds[2]), int(self.sec_bounds[3]),
                              int(self.sec_bounds[4]))
        if mode == "dense":
            self.last_out = None
            return F.linear(x, self.weight)
        # q / k / z sections
        q = self._routed_section(x, "q", b0, b1, mode) if hasattr(self, "q_B") \
            else F.linear(x, self.weight[b0:b1])
        k = F.linear(x, self.weight[b1:b2])                # dense by design
        z = self._routed_section(x, "z", b3, b4, mode) if hasattr(self, "z_B") \
            else F.linear(x, self.weight[b3:b4])
        # ---- MoRM: pick heads, evaluate ONLY their rows ----
        if self.morm is not None and mode != "full":
            mask = self.morm(self.mast(x))                 # [T, H] in (0,1]
        else:
            mask = torch.ones(T, self.n_v_heads, device=x.device, dtype=x.dtype)
        if force_mask is not None:
            mask = mask * force_mask
        self.last_mask = mask.detach()
        Wv = self.weight[b2:b3].reshape(self.n_v_heads, self.v_head_rows,
                                        self.weight.shape[1])
        sel = (mask > 0.5).nonzero(as_tuple=False)         # [N, 2] (t, h)
        v_shadow = self.v_sh_U is not None and mode != "full"
        v = self._shadow(x, "v") if v_shadow else x.new_zeros(T, self.v_rows)
        if sel.shape[0] > 0:
            # ZERO-MATERIALIZATION MoRM execution. A skipped (t, h) write
            # contributes EXACTLY zero downstream (the zero-padded v IS
            # the skip), so the sparse v equals the DENSE v masked by the
            # hard selection:
            #   grad ON  (training):  v = (x @ W_v^T) * hard_mask * v_gains
            #     — identical values and gradients to the sparse gather
            #       (skipped entries carry exactly zero gradient through
            #       the 0-mask product); nothing is saved for backward
            #       beyond [T, v_rows] tensors (~32 GiB -> ~48 MiB at 27B).
            #   grad OFF (serving):  segmented per-head GEMM — tokens
            #       grouped by head, ONE bmm over weight VIEWS — reads ONLY
            #       the selected heads' rows (the honest m x hr active
            #       accounting), never forms [N, hr, d], and moves ~60x
            #       less weight traffic than a chunked gather loop.
            # Both branches are verified against the reference oracle in
            # the suite.
            if torch.is_grad_enabled():
                v_dense = F.linear(x, self.weight[b2:b3])          # [T, v_rows]
                # v8: use the (STE) mask DIRECTLY — forward values are
                # exactly the hard 0/1 set, and the straight-through
                # gradient reaches the MoRM policy. v7 thresholded the
                # mask here ((mask > 0.5) is non-differentiable), which
                # silently severed the policy's training signal.
                m2 = mask.to(v_dense.dtype)                       # [T, H]
                gains_row = self.v_gains.to(v_dense.dtype) \
                    .repeat_interleave(self.v_head_rows)           # [v_rows]
                if v_shadow:       # v10: shadow write for every head, exact (x gain) where the critic says so
                    v = v + m2.repeat_interleave(self.v_head_rows, dim=1) \
                        * (v_dense * gains_row[None, :] - v)
                else:
                    v = v_dense * (m2.repeat_interleave(self.v_head_rows, dim=1)
                                   * gains_row[None, :])
            else:
                hidx, tidx = sel[:, 1], sel[:, 0]
                order = torch.argsort(hidx, stable=True)           # group by head
                h_s, t_s = hidx[order], tidx[order]
                counts = torch.bincount(h_s, minlength=self.n_v_heads)
                starts = torch.cumsum(counts, 0) - counts
                pos = torch.arange(sel.shape[0], device=x.device) - starts[h_s]
                max_n = int(counts.max())
                pad_t = torch.zeros(self.n_v_heads, max_n, dtype=torch.long,
                                    device=x.device)
                pad_t[h_s, pos] = t_s
                X_pad = x[pad_t]                                   # [H, max_n, d]
                V_pad = torch.bmm(X_pad, Wv.transpose(1, 2))       # [H, max_n, hr]
                v_sel = V_pad[h_s, pos]                            # [N, hr]
                v_sel = v_sel * self.v_gains.to(v_sel.dtype)[h_s][:, None]
                hr = self.v_head_rows
                rows = (h_s[:, None] * hr
                        + torch.arange(hr, device=x.device)[None, :])
                v[t_s[:, None].expand(-1, hr).reshape(-1),
                  rows.reshape(-1)] = v_sel.reshape(-1).to(v.dtype)
        # Telemetry: `telemetry=False` (set by the trainer for the training
        # phase) skips the dict — it pins [T, q+k+v+z] per GDN layer across
        # fwd+bwd and the trainer never reads it.
        if getattr(self, "telemetry", True):
            self.last_out = {"q": q, "k": k, "v": v, "z": z,
                             "beta_mask": mask}
        else:
            self.last_out = None
        return torch.cat([q, k, v, z], dim=-1)

    def forward(self, x: torch.Tensor, mode: str | None = None,
                force_mask: torch.Tensor | None = None):
        """Returns the FUSED output tensor [..., q+k+v+z] — the exact interface
        of the stock in_proj_qkvz Linear, so the downstream DeltaNet code runs
        UNCHANGED (it splits sections itself). Telemetry in self.last_out.
        Accepts any lead dims ([T,d] or [B,L,d], as HF decoders pass).

        Module self-checkpointing: in grad mode the whole fused computation
        is recomputed at backward from the [T, d] input, so the soft-mode
        saved set ([T, rows] outputs per section, ~0.44 GiB per DeltaNet
        layer at T=2048 — ~21 GiB across 48 layers) collapses to the
        [T, d] boundary BY CONSTRUCTION. MoRM semantics are untouched (the
        straight-through mask recomputes identically). The dense and
        no-grad/serving paths stay untouched (honest sparse accounting)."""
        lead = x.shape[:-1]
        x = x.reshape(-1, self.weight.shape[1])            # flatten to [T, d]
        mode = mode or self.default_mode
        # no input-requires-grad gate: non-reentrant checkpoint is correct
        # with frozen inputs + trainable params (the VL wrapper feeds
        # inputs_embeds and the embedding-side hook can silently miss).
        if (self.training and torch.is_grad_enabled()
                and mode != "dense"
                and getattr(self, "self_ckpt", True)):
            y = _torch_checkpoint(self._sp_core, x, mode, force_mask,
                                  use_reentrant=False)
        else:
            y = self._sp_core(x, mode, force_mask)
        return y.reshape(*lead, -1) if len(lead) != 1 else y


def _shortlist_chunk(shortlist: int, d: int, esize: int,
                     budget: int = 512 << 20) -> int:
    """Tokens per shortlist-scoring chunk so the gathered rows weight[sl]
    ([t, V_s, d]) stay under `budget` bytes of transient memory.

    27B head (d=5120, V_s=8192, bf16): 84 MB PER TOKEN -> 6 tokens = 503 MB.
    Unchunked, a 256-token batch tries to allocate 20 GiB and OOMs an A100.
    """
    return max(1, budget // max(1, shortlist * max(1, d) * max(1, esize)))


def _gather_chunk(per_row_bytes: int, budget: int = 256 << 20) -> int:
    """Rows per chunked-gather step so the gathered block stays under
    `budget` bytes of transient memory (256 MiB keeps the routed-linear
    Wsel and MoRM v-head gathers alive even in degraded-headroom runs;
    scoring is per-token, so chunking is mathematically exact)."""
    return max(1, budget // max(1, per_row_bytes))


# ----------------------------------------------------------------------------
# 12. HeadFunnel — Spectral Head Funnel (rank-r proxy + residual-aware ranking + exact shortlist)
# ----------------------------------------------------------------------------
class HeadFunnel(nn.Module):
    """Two-stage lm_head execution:

        stage 1 (proxy):  z~ = (h P_r) U_r^T          U_r = W P_r  (rank-r projection of the head)
                          score = z~ + kappa * s(h) * rho
        stage 2 (exact):  shortlist = top-V_s rows of the score;
                          logits[sl] = lm_head[sl] h          (EXACT rows)

    RESIDUAL-AWARE RANKING. With Q = I - P_r P_r^T, the proxy error of row i is
        z_i - z~_i = <Q w_i, Q h>,   |z_i - z~_i| <= rho_i * s,
        rho_i = ||Q w_i|| (stored, V floats),  s = ||Q h|| (one norm per token)
    (Cauchy-Schwarz), so z~ + s*rho is a certified upper bound on EVERY logit.
    The worst case is far looser than the typical alignment, therefore
    kappa in [0, 1] interpolates between the plain proxy (0) and the certified
    upper-confidence ranking (1) and is FITTED on calibration states
    (fit_kappa; kappa = 0 is always a candidate, so the fit never loses recall).
    (v8 additionally multiplied U_r by the singular values — a second sigma that
    ranked rows by sigma^2-weighted scores; the proxy is now the exact rank-r
    projection.)

    Returns (values [T, V_s], indices [T, V_s]). mode='dense' gives the full
    exact head. Active params: V*r + d*r + V (rho) + V_s*d  (vs V*d dense). The dense
    head is KEPT as the shortlist source — total parameters preserved exactly.
    """

    def __init__(self, weight: torch.Tensor, rank: int = 256,
                 shortlist: int = 8192, use_rho: bool = True):
        super().__init__()
        if not isinstance(weight, nn.Parameter):
            weight = nn.Parameter(weight, requires_grad=False)
        else:
            weight.requires_grad_(False)
        self.register_parameter("weight", weight)          # [V, d]
        self.V, self.d = weight.shape
        self.rank = max(8, min(rank, min(self.V, self.d) - 1))
        self.shortlist = min(shortlist, self.V)
        # Chunked spectral factors: torch.svd_lowrank would cast the FULL
        # [248320, 5120] head to fp32 (5.1 GiB on top of the resident
        # model). Instead eigendecompose the Gram matrix G = W^T W =
        # sum_chunks W_c^T W_c (fp32, [d, d] = 105 MiB), take P_r = top-r
        # eigenvectors, and build U_r = W @ P_r chunk-wise — the exact
        # rank-r projection, peak transient < 1 GiB, deterministic. The
        # per-row residual norm rho_i = sqrt(||w_i||^2 - ||U_r[i]||^2) falls out
        # of the same pass.
        with torch.no_grad():
            d = self.d
            G = torch.zeros(d, d, device=weight.device, dtype=torch.float32)
            for i in range(0, self.V, 32768):              # Gram accumulation
                Wc = weight[i:i + 32768].to(torch.float32)
                G.add_(Wc.t() @ Wc)
            evals, evecs = torch.linalg.eigh(G)            # ascending
            P_r = evecs[:, -self.rank:].flip(1)                      # [d, r]
            U_r = torch.empty(self.V, self.rank, device=weight.device,
                              dtype=weight.dtype)
            rho = torch.empty(self.V, device=weight.device, dtype=torch.float32)
            for i in range(0, self.V, 32768):              # U = W @ P_r
                Wc = weight[i:i + 32768].to(torch.float32)
                Uc = Wc @ P_r
                U_r[i:i + 32768] = Uc.to(weight.dtype)
                rho[i:i + 32768] = (Wc.pow(2).sum(1) - Uc.pow(2).sum(1)).clamp_min(0).sqrt()
        self.register_buffer("U_r", U_r.to(weight.dtype))
        self.register_buffer("P_r", P_r.to(weight.dtype))
        self.register_buffer("rho", rho if use_rho else None)
        self.kappa = 0.0                                   # fitted by fit_kappa()
        self.default_mode = "funnel"
        self.last_recall: float | None = None

    def _proxy_scores(self, hc: torch.Tensor, kappa: float | None = None) -> torch.Tensor:
        """[t, V] shortlist scores of tokens hc [t, d] (already in the head dtype)."""
        kappa = self.kappa if kappa is None else kappa
        hp = hc @ self.P_r                                 # [t, r]
        pc = hp @ self.U_r.t()                             # [t, V]  rank-r proxy
        if kappa and self.rho is not None:
            s = (hc.float().pow(2).sum(-1) - hp.float().pow(2).sum(-1)).clamp_min(0).sqrt()
            pc = pc + (float(kappa) * s)[:, None].to(pc.dtype) * self.rho[None, :].to(pc.dtype)
        return pc

    def forward(self, h: torch.Tensor, mode: str | None = None,
                ref_logits: torch.Tensor | None = None):
        """h [T, d] -> (values [T, V_s], indices [T, V_s]) | full logits."""
        mode = mode or self.default_mode
        if mode == "dense":
            return h @ self.weight.t(), None
        # Chunked over T (scoring is per-token, so chunking is exact):
        # bounds the [t, V_s, d] gather to ~512 MiB.
        c = _shortlist_chunk(self.shortlist, self.d, self.weight.element_size())
        sl_parts, ex_parts = [], []
        for i in range(0, h.shape[0], c):
            hc = h[i:i + c].to(self.weight.dtype)          # dtype-boundary guard
            slc = self._proxy_scores(hc).topk(self.shortlist, dim=-1).indices  # [t, V_s]
            rows = self.weight[slc]                        # [t, V_s, d] bounded
            ex_parts.append(torch.bmm(rows, hc.unsqueeze(-1))
                            .squeeze(-1).to(h.dtype))      # [t, V_s] exact
            sl_parts.append(slc)
        exact = (torch.cat(ex_parts, 0) if ex_parts
                 else h.new_empty((0, self.shortlist)))
        sl = (torch.cat(sl_parts, 0) if sl_parts
              else h.new_empty((0, self.shortlist), dtype=torch.long))
        if ref_logits is not None:
            true_top = ref_logits.topk(min(64, self.V), dim=-1).indices
            hit = (true_top.unsqueeze(-1) == sl.unsqueeze(1)).any(-1).float()
            self.last_recall = float(hit.mean())
        return exact, sl

    @torch.no_grad()
    def shortlist_recall(self, h: torch.Tensor, true_top: torch.Tensor,
                         kappa: float | None = None) -> float:
        """Mean fraction of each token's true top-n ids inside the shortlist.
        h [N, d] head inputs, true_top [N, n] ids of the exact top-n logits."""
        hit, tot = 0.0, 0
        for i in range(0, h.shape[0], 64):
            hc = h[i:i + 64].to(self.weight.dtype)
            sl = self._proxy_scores(hc, kappa).topk(self.shortlist, dim=-1).indices
            tt = true_top[i:i + 64]
            hit += float((tt.unsqueeze(-1) == sl.unsqueeze(1)).any(-1).float().sum())
            tot += tt.numel()
        return hit / max(tot, 1)

    @torch.no_grad()
    def fit_kappa(self, h: torch.Tensor, true_top: torch.Tensor,
                  grid=(0.0, 0.1, 0.2, 0.35, 0.5, 0.75, 1.0)) -> dict:
        """Pick kappa by shortlist recall on calibration states (ties -> smaller
        kappa). Returns {kappa: recall}; sets self.kappa."""
        if self.rho is None:
            self.kappa = 0.0
            return {0.0: self.shortlist_recall(h, true_top, 0.0)}
        res = {float(k_): self.shortlist_recall(h, true_top, k_) for k_ in grid}
        self.kappa = float(max(res, key=lambda k_: (round(res[k_], 6), -k_)))
        return res

    def active_params(self) -> int:
        """V*r (proxy rows) + d*r (the projection h P_r) + V_s*d (exact rows) + V (rho reads)."""
        return (self.V * self.rank + self.d * self.rank + self.shortlist * self.d
                + (self.V if self.rho is not None else 0))


# ----------------------------------------------------------------------------
# 13. active_waterfall / solve_budget_v2   (v9: mast evaluations charged per layer, trim ladder,
#     plan consumed by the pipeline)
# ----------------------------------------------------------------------------
def _rows_active(n_rows: int, n_blocks: int, topk: int,
                 carrier_frac: float) -> int:
    n_carrier0 = max(0, min(int(round(carrier_frac * n_rows)), n_rows // 4))
    n_blocks = max(1, min(n_blocks, n_rows - n_carrier0))
    r = (n_rows - n_carrier0) // n_blocks
    n_carrier = n_rows - n_blocks * r
    return n_carrier + min(topk, n_blocks) * r


def morm_nheads_probe(v_rows: int, head_rows: int) -> int:
    return v_rows // head_rows


def mast_reads(d: int, n_layers: int, n_linear_attn: int, n_full_attn: int,
               ss: dict, mast_S: int, mode: str = "applications") -> int:
    """Weight elements the AntennaMast sketches touch per token.

    phi(h) = silu(R_w h) is evaluated once per (layer, input width): the attention-side
    input of every layer (width d: q / gate / z / b-gate / critic of that layer share it)
    and the output-projection input of every layer (o_proj: o_in, DeltaNet out_proj:
    dn_out_in; a SIBLING sketch of that width).
      mode='applications' (strict, default): every evaluation is charged — what the
            compute / bandwidth of a '3B-active' model really pays.
      mode='distinct': every distinct sketch MATRIX is charged once (R and its siblings are
            shared by all layers) — the v8 convention, which additionally forgot the sibling."""
    if mode == "distinct":
        widths = {d}
        if n_full_attn and ss.get("o_in"):
            widths.add(ss["o_in"])
        if n_linear_attn and ss.get("dn_out_in"):
            widths.add(ss["dn_out_in"])
        return mast_S * sum(widths)
    return mast_S * (n_layers * d + n_full_attn * ss.get("o_in", 0)
                     + n_linear_attn * ss.get("dn_out_in", 0))


def active_waterfall(d: int, D_ffn: int, n_layers: int, n_linear_attn: int,
                     n_full_attn: int, vocab: int, ss: dict,
                     E_ffn: int, k_ffn: int, shared_ffn: int, rank_zssi: int,
                     mast_S: int, lin_blocks: int, lin_topk: int,
                     carrier_frac: float, m_write: int, funnel_rank: int,
                     funnel_shortlist: int, funnel_rho: bool = True,
                     mast_mode: str = "applications", shadow_lin: int = 0,
                     shadow_ffn: int = 0, shadow_v: bool = False) -> dict:
    """STRICT touched-weights accounting for one text token under RESONA.

    ss (section sizes): q_rows, k_rows, v_rows, gate_rows (full-attn fused
    v+gate => v half dense / gate half routed), o_in, dn_q_rows, dn_k_rows,
    dn_v_rows, dn_z_rows, dn_ba_rows, dn_v_head_rows, dn_out_in.
    v9 corrections vs v8 (all small except the first):
      * the mast sketches are charged per (layer, width) evaluation, see mast_reads
      * FFN: the shared core's ZSSI rows P[s] and the alpha/beta vectors are read too; the
        'E*mast_S' term of v8 was a phantom (the FFN router reads h directly: E*d)
      * DeltaNet conv: the taps of the selected v channels are read as well
      * head funnel: the d x r projection h @ P_r and (optionally) the V residual norms rho
    """
    a = {}
    a["embedding_lookup"] = d
    a["antenna_mast"] = mast_reads(d, n_layers, n_linear_attn, n_full_attn, ss, mast_S,
                                   mast_mode)
    m_e = max(1, (D_ffn - shared_ffn) // E_ffn)
    # v10 shadows: a rank-r shadow of an [n_out, n_in] site reads U [n_out, r] and M [r, n_in]
    def _shadow_cost(n_out, n_in):
        return int(shadow_lin) * (n_out + n_in) if (shadow_lin and n_out > 0) else 0
    a["ffn"] = n_layers * (k_ffn * 3 * d * m_e + 3 * d * shared_ffn
                           + E_ffn * d + d * rank_zssi
                           + (k_ffn * m_e + shared_ffn) * rank_zssi
                           + 2 * (k_ffn + 1) * rank_zssi
                           + ((2 * d * int(shadow_ffn) + d) if shadow_ffn else 0))
    fa = (_rows_active(ss["q_rows"], lin_blocks, lin_topk, carrier_frac) * d
          + ss["k_rows"] * d                        # dense (KV cache)
          + ss["v_rows"] * d                        # dense (KV cache)
          + _rows_active(ss["gate_rows"], lin_blocks, lin_topk, carrier_frac) * d
          + _rows_active(d, lin_blocks, lin_topk, carrier_frac) * ss["o_in"]
          + 3 * lin_blocks * mast_S                 # q/gate/o probe heads
          + _shadow_cost(ss["q_rows"], d) + _shadow_cost(ss["gate_rows"], d)
          + (_shadow_cost(d, ss["o_in"]) if ss["o_in"] else 0))
    a["full_attention"] = fa * n_full_attn
    dn = (_rows_active(ss["dn_q_rows"], lin_blocks, lin_topk, carrier_frac) * d
          + ss["dn_k_rows"] * d                     # dense
          + m_write * ss["dn_v_head_rows"] * d      # MoRM heads evaluated EXACTLY
          + (_shadow_cost(ss["dn_v_rows"], d) if shadow_v else 0)        # v10: shadow writes for all heads
          + _shadow_cost(ss["dn_q_rows"], d) + _shadow_cost(ss["dn_z_rows"], d)
          + (_shadow_cost(d, ss["dn_out_in"]) if ss["dn_out_in"] else 0)
          + _rows_active(ss["dn_z_rows"], lin_blocks, lin_topk, carrier_frac) * d
          + ss["dn_ba_rows"] * d
          + (ss["dn_q_rows"] + ss["dn_k_rows"]
             + (ss["dn_v_rows"] if (shadow_v and shadow_lin and ss["dn_q_rows"])
                else m_write * ss["dn_v_head_rows"])) * 4   # conv1d taps: q, k and the non-zero v channels
          + _rows_active(d, lin_blocks, lin_topk, carrier_frac) * ss["dn_out_in"]
          + (3 * lin_blocks + morm_nheads_probe(ss["dn_v_rows"],
                                                ss["dn_v_head_rows"])) * mast_S
          + morm_nheads_probe(ss["dn_v_rows"], ss["dn_v_head_rows"]))   # v9: the critic's per-head intercept
    a["linear_attention"] = dn * n_linear_attn
    a["lm_head_funnel"] = (vocab * funnel_rank + funnel_shortlist * d
                           + d * funnel_rank + (vocab if funnel_rho else 0))
    a["norms_misc"] = n_layers * 2 * d + d
    a["TOTAL_ACTIVE"] = sum(a.values())
    return a


def solve_budget_v2(d: int, D_ffn: int, n_layers: int, n_linear_attn: int,
                    n_full_attn: int, vocab: int, ss: dict,
                    target_active: float, mast_S: int = 512,
                    rank_zssi: int = 64, verbose: bool = True,
                    shared_ffn: int | None = None, e_ffn: int = 32,
                    min_shared: int = 64, funnel_rank: int = 256,
                    m_write_max: int | None = None,
                    shortlist_max: int | None = None,
                    mast_S_min: int = 256, mast_mode: str = "applications",
                    funnel_rho: bool = True, k_ffn0: int = 1, lin_blocks0: int = 16,
                    carrier0: float = 0.04, m_write0: int | None = None,
                    shadow_lin: int = 0, shadow_ffn: int = 0, shadow_v: bool = False,
                    shadow_lin_min: int = 96) -> dict:
    """Honest budget solver (v9).

    1. FLOOR plan: every optional quality knob at its minimum, the FFN shared core and
       the mast sketch dimension at the REQUESTED size (v8 silently planned a 192-neuron
       core while the pipeline built 512 — a ~0.2B overshoot hidden behind a x1.10 band).
    2. TRIM ladder, in this order, each lever only as far as the budget demands:
         mast_S (down to mast_S_min) -> m_write (down to 12) -> shared core (down to 256,
         steps of E keep (D - S) divisible by E) -> head shortlist (2048) -> m_write (8)
         -> shared core (down to min_shared).
    3. SPEND the slack in quality-per-parameter order (head shortlist -> m_write ->
       lin_topk -> carrier_frac -> k_ffn), exactly as v8; mast_S / shared core are never
       raised above the request.
    `m_write_max` / `shortlist_max` are the user's REQUESTS (config caps): the spend step
    never exceeds them. The returned plan carries `feasible` and the list of `trims`; the
    pipeline CONSUMES the plan (v8 only advertised it).
    """
    n_v_heads = max(1, ss["dn_v_rows"] // max(1, ss["dn_v_head_rows"]))
    S0 = 192 if shared_ffn is None else int(shared_ffn)
    E = int(e_ffn)
    S0 = S0 - ((S0 - D_ffn) % E)              # largest S <= S0 with (D - S) % E == 0
    m_cap = n_v_heads if m_write_max is None else max(1, min(int(m_write_max), n_v_heads))
    sl_cap = 16384 if shortlist_max is None else max(256, int(shortlist_max))
    _m0 = min(max(8, n_v_heads // 4), m_cap) if m_write0 is None else max(1, min(int(m_write0), m_cap))
    plan = dict(E_ffn=E, k_ffn=int(k_ffn0), shared_ffn=S0, funnel_rank=funnel_rank,
                funnel_shortlist=min(4096, sl_cap),
                m_write=_m0,
                lin_blocks=int(lin_blocks0), lin_topk=1, carrier_frac=float(carrier0), mast_S=int(mast_S),
                shadow_lin=int(shadow_lin), shadow_ffn=int(shadow_ffn))

    def eval_plan(p):
        return active_waterfall(d, D_ffn, n_layers, n_linear_attn, n_full_attn,
                                vocab, ss, E_ffn=p["E_ffn"], k_ffn=p["k_ffn"],
                                shared_ffn=p["shared_ffn"], rank_zssi=rank_zssi,
                                mast_S=p["mast_S"], lin_blocks=p["lin_blocks"],
                                lin_topk=p["lin_topk"],
                                carrier_frac=p["carrier_frac"],
                                m_write=p["m_write"], funnel_rank=p["funnel_rank"],
                                funnel_shortlist=p["funnel_shortlist"],
                                funnel_rho=funnel_rho, mast_mode=mast_mode,
                                shadow_lin=p["shadow_lin"], shadow_ffn=p["shadow_ffn"],
                                shadow_v=shadow_v)

    trims = []
    w = eval_plan(plan)
    requested = dict(plan)

    def descend(name, values):
        nonlocal w
        for v in values:
            if w["TOTAL_ACTIVE"] <= target_active:
                return
            if v >= plan[name]:
                continue
            plan[name] = v
            w = eval_plan(plan)
    mast_opts = [v for v in (384, 256, 192, 128) if mast_S_min <= v < plan["mast_S"]]
    m_lo = min(12, m_cap)
    sh_opts = list(range(plan["shadow_lin"] - 32, max(int(shadow_lin_min), 1) - 1, -32)) if plan["shadow_lin"] else []
    ladder = [("mast_S", mast_opts),
              ("shadow_lin", sh_opts),
              ("m_write", list(range(plan["m_write"] - 2, m_lo - 1, -2))),
              ("shared_ffn", list(range(S0 - E, 255, -E))),
              ("funnel_shortlist", [min(2048, sl_cap)]),
              ("m_write", [min(8, m_cap)]),
              ("shared_ffn", list(range(min(S0, 256) - E, min_shared - 1, -E)))]
    for name, vals in ladder:
        before = plan[name]
        descend(name, vals)
        if plan[name] != before:
            trims.append(f"{name} {before} -> {plan[name]}")
    m_vals = list(range(plan["m_write"], m_cap + 1, 2))
    if not m_vals or m_vals[-1] != m_cap:
        m_vals.append(m_cap)
    grid = [("funnel_shortlist", [v for v in (4096, 8192, 12288, 16384) if v <= sl_cap]),
            ("m_write", m_vals),
            ("lin_topk", [1, 2]),
            ("carrier_frac", [c_ for c_ in (0.01, 0.02, 0.04, 0.06, 0.08, 0.10) if c_ >= plan["carrier_frac"]]),
            ("k_ffn", [k_ for k_ in (1, 2, 4, 8, 16) if plan["k_ffn"] <= k_ <= 2 * plan["k_ffn"]])]
    for name, vals in grid:
        for val in vals:
            if val == plan[name]:
                continue
            trial = dict(plan)
            trial[name] = val
            tw = eval_plan(trial)
            if tw["TOTAL_ACTIVE"] <= target_active:
                plan, w = trial, tw
            else:
                break
    plan["feasible"] = bool(w["TOTAL_ACTIVE"] <= target_active)
    plan["trims"] = trims
    plan["requested"] = {k_: requested[k_] for k_ in ("mast_S", "shared_ffn", "m_write",
                                                      "funnel_shortlist", "shadow_lin")}
    plan["shadow_v"] = bool(shadow_v)
    plan["mast_mode"] = mast_mode
    if verbose:
        print(f"{'family':<18}{'active':>14}")
        for kk, vv in w.items():
            print(f"{kk:<18}{vv:>14,}")
        print(f"\nsolved plan: { {k: v for k, v in plan.items() if k not in ('trims', 'requested')} }")
        for t in trims:
            print(f"  [budget] trimmed: {t}")
        print(f"TOTAL ACTIVE = {w['TOTAL_ACTIVE']/1e9:.3f}B "
              f"(target {target_active/1e9:.2f}B, mast accounting: {mast_mode}) -> "
              f"{'OK' if plan['feasible'] else 'OVER BUDGET'}")
    return {"plan": plan, "waterfall": w}

RESONA_LINEARS: list = []      # (name, ResonantLinear) — whole-Linears swapped
RESONA_SECTIONED: list = []    # (name, SectionedProjector) — DeltaNet in_proj
RESONA_MORM: list = []         # (name, MoRMWritePolicy)
RESONA_FUNNEL: list = []       # (name, HeadFunnel) — the lm_head
MAST = None                    # the shared AntennaMast

# ---- global mode presets (one call re-aims the whole model) ----------------
def set_world(mode: str):
    """mode: 'dense_teacher'  = exact original dense model (teacher / oracle)
             'exact_resonant' = full resonant path, exact by Theorem 7
             'soft_train'     = differentiable all-expert paths (training)
             'compat_parity'  = FFN sparse, attention+head dense (COMPAT export
                                semantics — matches the stock-vLLM artifact)
             'serving'        = everything sparse (NATIVE artifact semantics)"""
    if mode == "dense_teacher":
        for _, m in PRISM_MODULES: m.default_mode = "uniform"
        for _, m in RESONA_LINEARS: m.default_mode = "dense"
        for _, m in RESONA_SECTIONED: m.default_mode = "dense"
        for _, m in RESONA_FUNNEL: m.default_mode = "dense"
    elif mode == "exact_resonant":
        for _, m in PRISM_MODULES: m.default_mode = "uniform"
        for _, m in RESONA_LINEARS: m.default_mode = "full"
        for _, m in RESONA_SECTIONED: m.default_mode = "full"
        for _, m in RESONA_FUNNEL: m.default_mode = "dense"
    elif mode == "soft_train":
        for _, m in PRISM_MODULES: m.default_mode = "soft"
        for _, m in RESONA_LINEARS: m.default_mode = "soft"
        for _, m in RESONA_SECTIONED: m.default_mode = "soft"
        for _, m in RESONA_FUNNEL: m.default_mode = "dense"
    elif mode == "compat_parity":
        for _, m in PRISM_MODULES: m.default_mode = "topk"
        for _, m in RESONA_LINEARS: m.default_mode = "dense"
        for _, m in RESONA_SECTIONED: m.default_mode = "dense"
        for _, m in RESONA_FUNNEL: m.default_mode = "dense"
    elif mode == "serving":
        for _, m in PRISM_MODULES: m.default_mode = "topk"
        for _, m in RESONA_LINEARS: m.default_mode = "topk"
        for _, m in RESONA_SECTIONED: m.default_mode = "topk"
        for _, m in RESONA_FUNNEL: m.default_mode = "funnel"
    else:
        raise ValueError(mode)
    # v8.0.1 [F4]: the BetaGate write levers (RESONA_MORM_GATES, filled by
    # the surgery cell) follow the world mode: dense_teacher / exact_resonant
    # / compat_parity need the stock BITWISE in_proj_ba (Theorem-7 anchor,
    # the free dense teacher, the COMPAT parity export), while soft_train /
    # serving run the write-sparsity. Pre-surgery the list is absent ->
    # this is a no-op.
    _gates = globals().get("RESONA_MORM_GATES") or []
    _gate_on = mode in ("soft_train", "serving")
    for _, _g in _gates:
        # v10: with a v shadow every head writes (the gate would delete the shadow writes)
        _g.enabled = _gate_on and not getattr(_g, "shadow_writes", False)
    for _, _m in PRISM_MODULES:            # the FFN shadow belongs to the NATIVE serving path only
        _m.use_shadow = (mode == "serving")
    return mode

print("RESONA core loaded (zero-materialization execution, module "
      "self-checkpointing, leak-free telemetry).")


In [ ]:
# ============================================================================
# MoRM 2.0 CORE (v8) — the standalone module, embedded verbatim
# (morm2.py ships alongside this notebook for local use; the cell and the
# file are generated from the same source and are byte-identical minus
# this header). Pure PyTorch, self-contained, CPU-verifiable.
# ============================================================================
"""
morm2.py — MoRM 2.0: Rate-Distortion Scheduling of Recurrent Memory Writes
===========================================================================

Upgrades MoRM 1.0 (Mixture of Recurrent Memories, PRISM/RESONA <= v7) from a
reactive per-token top-m write mask into a certified, anticipatory,
recoverable WRITE SCHEDULER for the value-head memories of Gated DeltaNet
(GDN) layers. Pure PyTorch, self-contained, CPU-verifiable.

--------------------------------------------------------------------------
MoRM 1.0 (v7) recap — and its two structural limits
--------------------------------------------------------------------------
MoRM 1.0 made the H per-head memory slabs of each GDN layer write-sparse:
a per-token policy picks m of H heads whose rank-1 state writes are actually
committed; the rest are skipped. Two structural limits:

  (L1) MYOPIC, ENERGY-GUIDED, IRREVERSIBLE WRITES. The policy scores heads
       by write ENERGY  e_h = beta_h * ||k_h|| * ||v_h||  — an immediate-
       magnitude proxy. But the value of a memory is its FUTURE RETRIEVAL:
       a write that is never read again is worthless at any energy, and a
       small write that a query 500 tokens later depends on is precious.
       And the skip is irreversible: information skipped at token t never
       enters the state, and nothing downstream can recover it.

  (L2) A FIXED WRITE RATE. Every token writes exactly m heads: whitespace,
       boilerplate and function words spend the same memory bandwidth as
       entity-dense tokens. The rate is a hyperparameter, never a
       controlled resource.

--------------------------------------------------------------------------
MoRM 2.0 — the four mechanisms (and their certificates)
--------------------------------------------------------------------------
  (A) WRITE INFLUENCE KERNEL (WIK) — exact counterfactual write values.
      The GDN recurrence is LINEAR in its writes (Theorem A):
          S_t = D_t S_{t-1} + w_t,      w_s := beta_s k_s delta_s^T,
          D_t = diag(lam_t),  delta_s = v_s                    (plain form)
                                 or v_s - S~_{s-1}^T k_s  (delta-corrected)
      so the perturbation of ANY future read from removing ONE write is a
      CLOSED-FORM quantity (Theorem B):
          ||delta o_t||_2 = scale * beta_s * |q_t^T (p(s,t) (*) k_s)|
                            * ||delta_s||_2,
          p(s,t) = prod_{tau=s+1..t} lam_tau   (per key dim).
      `write_influence_labels` computes, from ONE pass of trajectories
      (q, k, delta, beta, lam), the exact first-order counterfactual value
      of EVERY (token, head) write — zero ablation runs. MoRM 1.0's energy
      score is the degenerate special case with all read-overlaps set to 1.

  (B) ELASTIC WRITE RATE — token-adaptive budgets with E[m] = m_bar.
      A multiplicative rate controller (rate-distortion threshold): write
      head h at token t iff predicted influence L_hat_h(t) > lambda_t,
      with lambda_t adapting so the AVERAGE write count converges to the
      budget m_bar (Theorem D). Filler tokens write nothing; salient
      tokens write up to m_max heads. Same average cost as MoRM 1.0,
      strictly better allocation.

  (C) DEFERRED WRITE-BACK — skipped != lost.
      A write that loses the slot race is not dropped: it is PARKED in a
      one-slot per-head buffer and may COMMIT later with EXACT decay
      compensation (Theorem C): committing at t' the operands
          k~ = (prod_{tau=s+1..t'} lam_tau) (*) k_s,   v_s, beta_s
      gives, for every read at tau >= t', EXACTLY the state contribution
      the on-time write would have made (writes commute — the recurrence
      is linear in writes; verified numerically against the oracle). The
      deferral distortion is exactly the windowed influence
      I_h(s, [s, t')) — measurable, and precisely what the scheduler
      minimizes.

  (D) CHARGE LEDGER + STALENESS BOUND.
      Every dropped write charges its predicted remaining influence to a
      per-head ledger that AGES WITH THE HEAD'S OWN DECAY (the debt is
      forgiven exactly as fast as the information would have faded). The
      ledger is a policy input (the policy sees its own debt) and forces
      resolution: no pending write waits longer than K tokens (Theorem E)
      — it commits (pre-empting the fresh write, which inherits the slot
      and the buffer) or is dropped with its remaining value charged and
      certified.

  Plus a kernel-level CORRECTNESS FIX (Proposition F): the only write-
  masking lever that is exact in BOTH the plain outer-product form and the
  delta-corrected WY form of the stock FLA kernel is BETA-masking. FLA's
  `recompute_w_u_fwd_kernel` computes u = A^{-1} (beta (*) v) and
  w = A^{-1} (beta (*) k): beta = 0 turns row s of the unit-triangular
  factor into the identity row, hence u_s = w_s = 0 EXACTLY, while v = 0
  (MoRM 1.0's lever) leaks intra-chunk corrections — a *deletion write*
  -beta k k^T S~ instead of no write. MoRM 2.0 masks beta (and keeps the
  v-row gather purely as a compute saving, semantically inert once
  beta = 0).

--------------------------------------------------------------------------
Theorems (stated here; proven in MoRM2_THEORY.md; verified by verify_morm2)
--------------------------------------------------------------------------
  Theorem A (write superposition / commutation).  The state of the plain
    recurrence is an affine function of the multiset of writes; per-write
    contributions commute and never interact.
  Theorem B (influence exactness).  For a single removed write the output
    perturbation norm above is EXACT (no inequality); for a SET of removed
    writes the triangle inequality gives a certified bound, tight in the
    single-write case.
  Theorem C (deferred-commit exactness).  Plain form: the compensated
    commit reproduces the on-time state trajectory from the commit point
    onward EXACTLY. Delta-corrected form: exact for the carried write
    term; the correction term re-evaluates against the current state and
    is bounded by the same windowed-influence certificate.
  Theorem D (rate convergence).  lambda_{t+1} = lambda_t *
    exp(eta * (m_t - m_bar)) (under-budget -> lower the bar) converges in
    the stochastic-approximation sense to E[m_t] = m_bar for stationary
    score streams (eta -> 0 schedule supported).
  Theorem E (staleness bound).  With force-commit age K, every deferred
    write enters memory within K tokens of its origin or is dropped with
    its remaining influence charged (bounded by its WIK label).
  Proposition F (beta-mask exactness, kernel-level).  As above — verified
    against FLA's WY kernel algebra and numerically against a
    delta-corrected reference oracle.

Conventions: "influence" is an output-space L2 quantity (optionally
weighted per read); all internals compute in fp32 regardless of substrate
dtype; every function is batched over heads [H] and sequential over tokens
[T]. Shapes: q, k [T,H,dk]; v, delta [T,H,dv]; beta [T,H]; lam [T,H] or
[T,H,dk] in (0,1].
"""

from __future__ import annotations

import math
from dataclasses import dataclass, field
from typing import Callable, Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# Section 0 — Canonical GDN math + reference oracle
# ============================================================================
# Recurrence (per head h), S in R^{dk x dv}, o the read:
#   plain : S_t = diag(lam_t) S_{t-1} + beta_t k_t v_t^T
#   delta : S_t = diag(lam_t) S_{t-1} + beta_t k_t (v_t - S~^T k_t)^T,
#           S~ = diag(lam_t) S_{t-1}
#   read  : o_t = scale * S_t^T q_t        (write-then-read within a token)


def lam_to_full(lam: torch.Tensor, T: int, H: int, dk: int) -> torch.Tensor:
    """Broadcast lam (scalar / [H] / [T,H] / [T,H,dk]) to [T,H,dk] fp32."""
    if not torch.is_tensor(lam):
        return torch.full((T, H, dk), float(lam))
    if lam.dim() == 0:
        return lam.reshape(1, 1, 1).expand(T, H, dk).float()
    if lam.dim() == 1:
        assert lam.shape[0] == H
        return lam[None, :, None].expand(T, H, dk).float()
    if lam.dim() == 2:
        assert lam.shape[0] == T and lam.shape[1] == H
        return lam[:, :, None].expand(T, H, dk).float()
    if lam.dim() == 3:
        assert lam.shape == (T, H, dk)
        return lam.float()
    raise ValueError(f"lam must be 0/1/2/3-D, got {tuple(lam.shape)}")


def gdn_sequential(
    q: torch.Tensor, k: torch.Tensor, v: torch.Tensor,
    beta: torch.Tensor, lam: torch.Tensor,
    form: str = "delta", scale: float = 1.0,
    S0: Optional[torch.Tensor] = None,
    return_deltas: bool = False,
):
    """Pure-torch GDN reference oracle (sequential over T, batched over H).

    Returns (o [T,H,dv], S_final [H,dk,dv]) or, with return_deltas=True,
    (o, S_final, delta [T,H,dv]) where delta is the ACTUAL deposited value
    of each (token, head) rank-1 update — `v` for the plain form, the
    delta-corrected value for the delta form. delta is exactly the quantity
    the influence kernel needs: it is what a skipped write would actually
    have deposited.
    """
    assert form in ("plain", "delta"), form
    T, H, dk = q.shape
    dv = v.shape[-1]
    qf, kf, vf, bf = q.float(), k.float(), v.float(), beta.float()
    lamf = lam_to_full(lam, T, H, dk)
    S = (S0.float().clone() if S0 is not None
         else torch.zeros(H, dk, dv, device=q.device))
    o = torch.empty(T, H, dv, device=q.device, dtype=torch.float32)
    deltas = (torch.empty(T, H, dv, device=q.device, dtype=torch.float32)
              if return_deltas else None)
    for t in range(T):
        S = S * lamf[t][:, :, None]                      # diag(lam) S
        d = vf[t]
        if form == "delta":
            d = d - torch.einsum("hij,hi->hj", S, kf[t])  # v - S^T k
        if return_deltas:
            deltas[t] = d
        S = S + bf[t][:, None, None] * kf[t][:, :, None] * d[:, None, :]
        o[t] = torch.einsum("hi,hij->hj", qf[t], S) * scale
    if return_deltas:
        return o, S, deltas
    return o, S


def gdn_beta_masked_gap(
    q, k, v, beta, lam, mask, form="delta", scale=1.0, S0=None
) -> float:
    """|| o_dense - o_masked ||_F under the beta-mask lever (Prop F):
    masked heads write beta*mask = 0 — exact in both kernel forms."""
    o_dense, _ = gdn_sequential(q, k, v, beta, lam, form, scale, S0)
    o_sparse, _ = gdn_sequential(q, k, v, beta * mask, lam, form, scale, S0)
    return float((o_dense - o_sparse).norm())


# ============================================================================
# Section 1 — Write Influence Kernel (WIK): exact per-write labels
# ============================================================================
def write_influence_labels(
    q: torch.Tensor, k: torch.Tensor, delta: torch.Tensor,
    beta: torch.Tensor, lam: Union[torch.Tensor, float],
    scale: float = 1.0,
    read_weight: Optional[torch.Tensor] = None,
    window: int = 512,
    chunk: int = 64,
    coverage_horizon: Optional[int] = None,
) -> dict:
    """Exact first-order counterfactual value of every (token, head) write.

    For each write (h, s) with actual deposited value delta_s (Theorem B):

        I_h(s) = sum_{t=s}^{s+W-1}  w_t * scale * beta_s
                 * | q_t^T (p(s,t) (*) k_s) | * || delta_s ||_2

    with p(s,t) = prod_{tau=s+1..t} lam_tau (per key dim). This is the
    perturbation norm of read t when write (h, s) ALONE is removed — exact,
    with zero ablation runs (the read side is fully observed offline).

    Streaming windowed algorithm — memory O(H * W * dk), fp32 throughout:
      * a rolling buffer of decayed keys kt_s = p(s, b) (*) k_s for the
        window ending at block start b; each block ages the buffer by its
        decay product (the exponent is <= 0 everywhere: underflow-safe,
        never overflow);
      * block reads score against the buffer by one batched matmul
        (q (*) block_prefix_decay) @ kt^T;
      * intra-block (s in the same block) pairs score through the masked
        relative-decay attention exp(Gc_t - Gc_s), t >= s (exponent <= 0).

    Args:
      q, k  : [T,H,dk] kernel-space q, k (apply the kernel's L2 norm here
              if it normalizes; fold its attn-scale into `scale`).
      delta : [T,H,dv] actual written values (gdn_sequential with
              return_deltas=True; or v for the plain form).
      beta  : [T,H];  lam : [T,H] / [T,H,dk] / scalar.
      read_weight : [T] or [T,H] optional per-read weights (e.g. |dL/do_t|).
      window : influence horizon W (labels exact within it).
      chunk  : processing block C (memory knob; math is chunk-invariant).
      coverage_horizon : if set (default 2*window), also accumulate labels
              to this longer horizon and report the windowed coverage.

    Returns dict:
      labels       [T,H]  windowed influence I_h(s)
      labels_long  [T,H]  labels to `coverage_horizon` (if requested)
      coverage     float  mean fraction of the long-horizon influence
                          captured inside `window` (1.0 => window covers
                          essentially everything)
    """
    T, H, dk = q.shape
    W = max(1, min(window, T))
    C = max(1, min(chunk, W))
    qf, kf = q.float(), k.float()
    df, bf = delta.float(), beta.float()
    lamf = lam_to_full(lam, T, H, dk)
    loglam = lamf.clamp_min(1e-12).log()                 # <= 0

    rw = None
    if read_weight is not None:
        rw = read_weight.float()
        rw = rw[:, None].expand(T, H) if rw.dim() == 1 else rw

    dnorm = df.norm(dim=-1)                              # [T,H]
    acc = torch.zeros(T, H, device=q.device)             # short-horizon
    want_long = coverage_horizon is not None
    Wl = max(1, min(coverage_horizon or W, T))
    acc_l = torch.zeros(T, H, device=q.device) if want_long else None
    Wb = Wl if want_long else W                          # buffer length

    kt_buf = torch.zeros(H, Wb, dk, device=q.device)     # decayed keys
    idx_buf = torch.full((Wb,), -1, dtype=torch.long, device=q.device)

    for b in range(0, T, C):
        e = min(b + C, T)
        c = e - b
        g_blk = loglam[b:e]                              # [c,H,dk]
        Gc = torch.cumsum(g_blk, dim=0)                  # inclusive prefix
        qb = qf[b:e]                                     # [c,H,dk]

        # ---- (1) intra-block pairs: s, t in [b, e), t >= s ---------------
        # score(b+i, b+j) = q^T ( exp(Gc[i] - Gc[j]) (*) k_j ),  i >= j
        RC = max(1, (1 << 22) // max(1, H * dk * c))     # row sub-chunk
        rows = torch.arange(c, device=q.device)
        w_blk = rw[b:e] if rw is not None else None      # [c,H] or None
        GcT = Gc.permute(1, 0, 2)                         # [H, c, dk]
        kT = kf[b:e].permute(1, 0, 2)                    # [H, c, dk]
        for r0 in range(0, c, RC):
            r1 = min(r0 + RC, c)
            # rel[i,h,j,d] = exp(Gc[r0+i,h,d] - Gc[b+j,h,d]).
            # v8.0.2 [G8]: the exponent is <= 0 ONLY on the causal side
            # (i >= j); the masked-out FUTURE side goes positive and, with
            # real GDN decays (loglam down to ~-11 per token), a 64-token
            # block overflows exp to inf — then inf * 0 from the causal
            # mask is NaN and poisons every label downstream (ridge warm
            # start, critic, certificates). This never fired before
            # because the WIK capture itself was broken and the label
            # kernel never saw real data. Clamp the exponent to <= 0
            # BEFORE exp: a no-op on the causal side (exact), and the
            # masked side becomes finite garbage the mask zeroes out.
            rel = (Gc[r0:r1, :, None, :] - GcT[None, :, :, :]).clamp(
                max=0.0).exp()
            sc = (qb[r0:r1, :, None, :] * kT[None, :, :, :]
                  * rel).sum(-1)                         # [r,H,c]
            val = sc.abs().float()
            val = val * (rows[None, None, :] <=
                         (r0 + torch.arange(r1 - r0, device=q.device))[
                             :, None, None]).float()
            if w_blk is not None:
                val = val * w_blk[r0:r1, :, None]
            add = val.sum(0).t()                         # [c,H] over reads
            acc.index_add_(0, torch.arange(b, e, device=q.device), add)
            if want_long:
                acc_l.index_add_(0, torch.arange(b, e, device=q.device), add)

        # ---- (2) cross-buffer pairs: s < b, t in [b, e) -------------------
        valid = idx_buf >= 0
        n_val = int(valid.sum())
        if n_val > 0:
            # score(b+i, s_j) = (q_{b+i} (*) exp(Gc[i]))^T kt_j
            qP = qb * Gc.exp()                           # [c,H,dk]
            sc = torch.einsum("chd,hwd->chw", qP, kt_buf[:, :n_val])
            val = sc.abs()
            if w_blk is not None:
                val = val * w_blk[:, :, None]
            add_long = val.sum(0).t()                    # [n_val,H]
            s_ids = idx_buf[:n_val]
            # short-window mask: only reads with t - s < W count for `acc`;
            # every pair counts for the long accumulator.
            ages = (b - s_ids).float()                   # s_j distance to b
            tpos = torch.arange(c, device=q.device).float()
            in_win = (ages[None, :] + tpos[:, None]) < W  # [c, n_val]
            add_short = (val * in_win[:, None, :]).sum(0).t()
            acc.index_add_(0, s_ids, add_short)
            if want_long:
                acc_l.index_add_(0, s_ids, add_long)

        # ---- (3) age buffer by the block decay; append this block --------
        if n_val > 0:
            kt_buf[:, :n_val] = kt_buf[:, :n_val] * Gc[-1].exp()[:, None, :]
        new_idx = torch.arange(b, e, device=q.device)
        # decay each new token to the END of the block (so it is consistent
        # with the aged old entries at the next block boundary)
        tail = (Gc[-1][None, :, :] - Gc).exp()           # [c,H,dk], <= 1
        new_kt = kf[b:e] * tail                          # [c,H,dk]
        all_idx = torch.cat([idx_buf[:n_val], new_idx])
        all_kt = torch.cat([kt_buf[:, :n_val], new_kt.permute(1, 0, 2)],
                           dim=1)                        # [H, n_val+c, dk]
        keep = max(0, all_idx.shape[0] - Wb)
        m = all_idx.shape[0] - keep
        idx_buf = torch.full((Wb,), -1, dtype=torch.long, device=q.device)
        idx_buf[:m] = all_idx[keep:]
        kt_buf = torch.zeros(H, Wb, dk, device=q.device)
        kt_buf[:, :m] = all_kt[:, keep:]

    labels = acc * bf * dnorm * scale                    # [T,H]
    out = {"labels": labels}
    if want_long:
        labels_l = acc_l * bf * dnorm * scale
        # coverage measurable only where the LONG horizon is fully realized:
        # s <= T - Wl
        n_cov = T - Wl
        if n_cov > 8:
            sl = labels[:n_cov].sum()
            ll = labels_l[:n_cov].sum().clamp_min(1e-30)
            out["labels_long"] = labels_l
            out["coverage"] = float((sl / ll).clamp(0, 1))
        else:
            out["labels_long"] = labels_l
            out["coverage"] = 1.0
    return out


# ============================================================================
# Section 2 — Elastic write rate: the rate-distortion controller (Theorem D)
# ============================================================================
class ElasticBudgetController:
    """Multiplicative rate controller for token-adaptive write budgets.

    Write rule (per token): head h writes iff its content score exceeds the
    running threshold lambda (capped at m_max heads per token). After each
    token the threshold moves toward the budget:

        lambda <- lambda * exp(eta * (m_t - m_bar))

    (too few writes this token -> lower the bar, and vice versa) — a
    stochastic-approximation update whose equilibrium is E[m_t] = m_bar
    for stationary score streams (Theorem D). Supports the Robbins-Monro
    eta -> 0 schedule (`lr_decay_tokens`) for exact convergence, and the
    exact-mode switch m_bar = H (threshold driven below every score ->
    every head writes -> bit-exact dense behavior).
    """

    def __init__(self, m_bar: int, m_max: Optional[int] = None,
                 lr: float = 0.08, lr_decay_tokens: Optional[int] = None,
                 init_threshold: Optional[float] = None):
        self.m_bar = int(m_bar)
        self.m_max = None if m_max is None else int(max(m_max, m_bar))
        self.lr = float(lr)
        self.lr_decay_tokens = lr_decay_tokens
        self._steps = 0
        # positive seed: the update is MULTIPLICATIVE, so 0.0 would be a
        # fixed point — callers warm-start from a score quantile (see
        # schedule_writes / MoRM2Policy.forward) for fast convergence
        self.threshold = (float(init_threshold) if init_threshold is not None
                          else 1e-6)

    @property
    def exact_mode(self) -> bool:
        """True when every head always writes (bit-exact dense behavior)."""
        return self.m_max is None and self.m_bar >= 10**9

    def update(self, m_t: int) -> float:
        """One controller step; returns the new threshold.

        m_t < m_bar (under-budget) -> threshold decreases (admit more);
        m_t > m_bar -> threshold increases (admit fewer)."""
        eta = self.lr
        if self.lr_decay_tokens is not None:
            eta = self.lr / (1.0 + self._steps / self.lr_decay_tokens)
        self.threshold *= math.exp(eta * (m_t - self.m_bar))
        self.threshold = min(self.threshold, 1e30)      # numeric guard
        self._steps += 1
        return self.threshold

    def state_dict(self) -> dict:
        return {"threshold": self.threshold, "steps": self._steps,
                "m_bar": self.m_bar, "m_max": self.m_max}

    def load_state_dict(self, d: dict):
        self.threshold = float(d["threshold"])
        self._steps = int(d["steps"])


# ============================================================================
# Section 3 — The write scheduler: deferral, charge ledger, staleness
# ============================================================================
@dataclass
class ScheduleStats:
    """Telemetry of one scheduling run (all counts over T tokens)."""
    m_trace: torch.Tensor            # [T] writes per token
    writes: int = 0                  # fresh writes executed
    commits: int = 0                 # deferred writes committed
    defers: int = 0                  # fresh writes parked in the buffer
    drops: int = 0                   # writes dropped (charged to the ledger)
    max_staleness: int = 0           # max age at commit (<= K by Theorem E)
    mean_charge: float = 0.0
    mean_m: float = 0.0

    def summary(self) -> str:
        return (f"mean m = {self.mean_m:.2f} | fresh {self.writes} | "
                f"commits {self.commits} | defers {self.defers} | "
                f"drops {self.drops} | max staleness {self.max_staleness}")


class WriteScheduler:
    """The MoRM 2.0 causal write scheduler (one instance per GDN layer).

    Per token, for every head (vectorized over [H]):

      1. AGE      pendings decay by the CURRENT token's lam — so a pending
                  committed at t carries exactly the Theorem-C compensation
                  prod_{tau=s+1..t} lam_tau; the charge ledger ages the same
                  way (debt is forgiven as fast as the information would
                  have faded).
      2. DECIDE   content per head = pending (forced if age >= K, else the
                  larger of remaining value R and fresh score F); the head
                  writes iff content score > lambda_rate, capped at m_max.
      3. RESOLVE  writers emit fresh writes or compensated commits; losers
                  follow the deferral ladder:
                    fresh above threshold, slot lost -> buffer (one slot per
                      head; a weaker incumbent is evicted, dropped, charged)
                    fresh below threshold             -> dropped, charged
                    pending decayed below threshold   -> dropped, charged
                  a forced commit pre-empts a fresh write: the pending takes
                  the slot and the fresh write inherits the buffer.
      4. ADAPT    the rate controller sees m_t.

    All decisions are causal (scores/observables at or before t). The
    emitted (k_eff, v_eff, beta_eff) feed the STOCK kernel unchanged
    (input rewriting only — zero kernel surgery). Skipped heads carry
    beta_eff = 0: the Proposition-F exact mask.
    """

    def __init__(self, n_heads: int, dk: int, dv: int,
                 m_bar: int, staleness_k: int = 64,
                 m_max: Optional[int] = None,
                 controller_lr: float = 0.08,
                 init_threshold: Optional[float] = None,
                 v_provider: Optional[Callable] = None):
        self.H, self.dk, self.dv = n_heads, dk, dv
        self.K = int(staleness_k)
        self.v_provider = v_provider
        self.controller = ElasticBudgetController(
            m_bar=m_bar, m_max=m_max, lr=controller_lr,
            init_threshold=init_threshold)
        self.reset()

    def reset(self):
        self.pend_k = torch.zeros(self.H, self.dk)
        self.pend_v = torch.zeros(self.H, self.dv)
        self.pend_beta = torch.zeros(self.H)
        self.pend_R = torch.zeros(self.H)               # remaining value
        self.pend_age = torch.zeros(self.H, dtype=torch.long)
        self.pend_valid = torch.zeros(self.H, dtype=torch.bool)
        self.pend_t = torch.zeros(self.H, dtype=torch.long)  # origin token
        self.charge = torch.zeros(self.H)
        self.stats = ScheduleStats(m_trace=torch.zeros(0, dtype=torch.long))

    def to(self, device):
        for name in ("pend_k", "pend_v", "pend_beta", "pend_R",
                     "pend_age", "pend_valid", "pend_t", "charge"):
            setattr(self, name, getattr(self, name).to(device))
        return self

    # ------------------------------------------------------------------ step
    def step(self, scores_t: torch.Tensor, k_t: torch.Tensor,
             v_t: Optional[torch.Tensor], beta_t: torch.Tensor,
             lam_t: torch.Tensor, t: int = 0) -> dict:
        """Advance one token.

        scores_t [H] predicted influence (>= 0); k_t [H,dk]; v_t [H,dv] or
        None (then `v_provider(t_orig, heads)` must be set on the scheduler
        to materialize deferred values at commit time); beta_t [H];
        lam_t [H] or [H,dk] — this token's decay.
        """
        H, dev = self.H, scores_t.device
        if self.pend_k.device != dev:
            self.to(dev)
        # control plane: the scheduler is non-differentiable by design (the
        # policy's STE handles training) — detach everything it stores
        F = scores_t.detach().float().to(dev).clamp_min(0.0)   # fresh scores
        # per-key-dim decay of THIS token, [H, dk]
        if lam_t.dim() == 2:
            lam_full = lam_t.detach().float().to(dev)
        elif lam_t.dim() == 1:
            lam_full = lam_t.detach().float().to(dev)[:, None].expand(H, self.dk)
        else:
            lam_full = lam_t.detach().float().to(dev).reshape(1, 1).expand(H, self.dk)
        lam_scalar = lam_full.mean(-1).clamp(1e-6, 1.0)  # [H] aging rate

        # ---- 1. AGE (compensation clock + debt forgiveness) ---------------
        self.pend_k = self.pend_k * lam_full
        self.pend_R = self.pend_R * lam_scalar
        self.charge = self.charge * lam_scalar
        self.pend_age = self.pend_age + self.pend_valid.long()

        # ---- 2. DECIDE ----------------------------------------------------
        thr = self.controller.threshold
        R = torch.where(self.pend_valid, self.pend_R,
                        torch.full_like(self.pend_R, -1.0))
        force = self.pend_valid & (self.pend_age >= self.K)  # Theorem E
        commit_content = self.pend_valid & (force | (R >= F))
        content = torch.where(commit_content, R, F)
        wants = content > thr
        # Staleness guarantee (Theorem E): a pending that reached age K and
        # is still worth a slot (R above the rate line) commits NOW,
        # pre-empting the threshold/cap (its m_t is still counted, so the
        # controller adapts). Below the line it is dropped + charged (iv).
        forced_commit = force & (self.pend_R > thr)
        if self.controller.m_max is not None and self.controller.m_max < H:
            over = int((wants & ~forced_commit).sum()) - self.controller.m_max
            if over > 0:
                sc = torch.where(wants & ~forced_commit, content,
                                 torch.full_like(content, -1.0))
                drop_idx = sc.topk(over, largest=False).indices
                wants[drop_idx] = False
        wants = wants | forced_commit
        m_t = int(wants.sum())

        write = wants
        commit = write & commit_content
        fresh = write & ~commit_content

        # ---- 3a. EMIT (fresh writes + compensated commits) ----------------
        k_eff = k_t.detach().float().to(dev).clone()
        beta_eff = torch.where(write, beta_t.detach().float().to(dev),
                               torch.zeros(H, device=dev))
        v_eff = (v_t.detach().float().to(dev).clone() if v_t is not None
                 else torch.zeros(H, self.dv, device=dev))
        commit_heads = commit.nonzero(as_tuple=False).squeeze(-1)
        if commit_heads.numel() > 0:
            k_eff[commit_heads] = self.pend_k[commit_heads]
            beta_eff[commit_heads] = self.pend_beta[commit_heads]
            if v_t is not None:
                v_eff[commit_heads] = self.pend_v[commit_heads]
            elif self.v_provider is not None:
                v_eff[commit_heads] = self.v_provider(
                    self.pend_t[commit_heads], commit_heads)
            else:
                raise RuntimeError(
                    "deferred commit with neither v nor v_provider — "
                    "a commit must carry its buffered value")
            self.stats.max_staleness = max(
                self.stats.max_staleness,
                int(self.pend_age[commit_heads].max()))
        v_eff = torch.where(write[:, None], v_eff, torch.zeros_like(v_eff))

        # ---- 3b. RESOLVE the losers ---------------------------------------
        # (i) fresh above threshold but no slot: park into the buffer;
        #     evict (drop + charge) a weaker incumbent if occupied
        fresh_park = (~write) & (F > thr)
        incumbent = torch.where(self.pend_valid, self.pend_R,
                                torch.full_like(self.pend_R, -1.0))
        evict = fresh_park & self.pend_valid & (F > incumbent)
        park = fresh_park & (~self.pend_valid | evict)
        evicted_R = torch.where(evict, self.pend_R,
                                torch.zeros_like(self.pend_R))
        self.charge += evicted_R                           # evictions charged
        # (ii) a forced commit pre-empts a fresh write above threshold:
        #      the fresh write inherits the freed buffer slot
        preempt_park = commit & (F > thr)
        # (iii) fresh below threshold (or lost to a stronger incumbent):
        #      dropped, charged
        fresh_drop = ((~write) & (F <= thr)) | \
                     (fresh_park & self.pend_valid & ~evict)
        # (iv) pending decayed below the threshold: dropped, charged
        pend_drop = self.pend_valid & (~write) & (self.pend_R <= thr)

        # writers' pending bookkeeping: commits free their slot
        if commit_heads.numel() > 0:
            self.pend_valid.index_fill_(0, commit_heads, False)

        # park deferred fresh writes (store the RAW operands — never the
        # masked emit values, and never the commit-overridden k_eff of a
        # pre-empting head)
        park_idx = (park | preempt_park).nonzero(as_tuple=False).squeeze(-1)
        if park_idx.numel() > 0:
            self.pend_k[park_idx] = k_t.detach().float().to(dev)[park_idx]
            if v_t is not None:
                self.pend_v[park_idx] = v_t.detach().float().to(dev)[park_idx]
            self.pend_beta[park_idx] = beta_t.detach().float().to(dev)[park_idx]
            self.pend_R[park_idx] = F[park_idx]
            self.pend_t[park_idx] = t
            self.pend_age[park_idx] = 0
            self.pend_valid[park_idx] = True
            self.stats.defers += int(park_idx.numel())

        # charged drops
        fdi = fresh_drop.nonzero(as_tuple=False).squeeze(-1)
        if fdi.numel() > 0:
            self.charge[fdi] += F[fdi]
        pdi = pend_drop.nonzero(as_tuple=False).squeeze(-1)
        if pdi.numel() > 0:
            self.charge[pdi] += self.pend_R[pdi]
            self.pend_valid.index_fill_(0, pdi, False)
        self.stats.drops += int(fdi.numel() + pdi.numel() + evict.sum())

        # ---- 4. TELEMETRY + controller ------------------------------------
        self.stats.writes += int(fresh.sum())
        self.stats.commits += int(commit.sum())
        self.stats.m_trace = torch.cat(
            [self.stats.m_trace, torch.tensor([m_t])])
        self.stats.mean_charge = float(self.charge.detach().mean())
        self.stats.mean_m = float(self.stats.m_trace.float().mean())
        self.controller.update(m_t)

        return {"mask": write, "commit": commit, "k_eff": k_eff,
                "v_eff": v_eff, "beta_eff": beta_eff, "m_t": m_t}


def schedule_writes(
    scores: torch.Tensor, k: torch.Tensor,
    v: Optional[torch.Tensor], beta: torch.Tensor, lam: torch.Tensor,
    m_bar: int, staleness_k: int = 64, m_max: Optional[int] = None,
    controller_lr: float = 0.08,
    controller_lr_decay_tokens: Optional[int] = None,
    init_threshold: Optional[float] = None,
) -> dict:
    """Offline (prefill-style) causal scheduling over a full sequence.

    Args: scores [T,H] predicted influences (causal); k [T,H,dk]; v
      [T,H,dv] or None (deferral without v buffers (k, beta) and requires
      the caller's own value source at commit — pass v in tests/eval);
      beta [T,H]; lam [T,H] or [T,H,dk].
    Returns dict: mask [T,H] (1 = write), commit [T,H] (1 = deferred
      commit), k_eff / beta_eff [T,...] (beta_eff == 0 exactly on skipped
      heads — the Proposition-F lever; commits carry the Theorem-C
      compensated key), v_eff (or None), stats (ScheduleStats), charge [H].
    """
    T, H, dk = k.shape
    # threshold init: explicit > exact mode > warm start from the first
    # tokens' score distribution (the warm start OVER-ADMITS deliberately —
    # half the target quantile — so the controller only ever tightens)
    init_thr = init_threshold
    w0 = min(64, T)
    if init_thr is None:
        if m_bar >= H:
            init_thr = -1.0
        elif w0 > 0:
            q = max(0.0, 1.0 - m_bar / H)
            init_thr = 0.5 * float(scores[:w0].flatten().float()
                                   .quantile(q).clamp_min(1e-9))
    sched = WriteScheduler(H, dk, v.shape[-1] if v is not None else 1,
                           m_bar=m_bar, staleness_k=staleness_k,
                           m_max=m_max, controller_lr=controller_lr,
                           init_threshold=init_thr)
    if controller_lr_decay_tokens is not None:
        sched.controller.lr_decay_tokens = controller_lr_decay_tokens
    sched.to(scores.device)
    masks = torch.zeros(T, H, device=scores.device)
    commits = torch.zeros(T, H, device=scores.device)
    k_eff = k.float().clone()
    beta_eff = beta.float().clone()
    v_eff = (v.float().clone() if v is not None else None)
    lam_full = lam_to_full(lam, T, H, dk)
    for t in range(T):
        out = sched.step(scores[t], k[t], v[t] if v is not None else None,
                         beta[t], lam_full[t], t=t)
        masks[t] = out["mask"]
        commits[t] = out["commit"]
        k_eff[t] = out["k_eff"]
        beta_eff[t] = out["beta_eff"]
        if v is not None:
            v_eff[t] = out["v_eff"]
    return {"mask": masks, "commit": commits, "k_eff": k_eff,
            "v_eff": v_eff, "beta_eff": beta_eff,
            "stats": sched.stats, "charge": sched.charge}


# ============================================================================
# Section 4 — MoRM2Policy: the trainable, drop-in write policy
# ============================================================================
class MoRM2Policy(nn.Module):
    """Influence-predicting write policy with an elastic budget.

    Drop-in replacement for MoRM 1.0's `MoRMWritePolicy` (same forward
    contract: `policy(mast(x)) -> [T, H] mask`), upgraded to:

      * an INFLUENCE CRITIC — per-head linear heads over the shared
        AntennaMast features phi(h) PLUS per-head observable features
        (log write energy, log key norm, log(1+charge), pending-age/K),
        predicting the WIK influence label in log space;
      * an ELASTIC budget — the write set is threshold(L_hat > lambda_t)
        with the rate controller adapting lambda_t so E[m_t] = m_bar,
        instead of a fixed top-m;
      * a straight-through soft threshold (sigmoid((L_hat - lambda)/tau))
        so distillation gradients reach the critic;
      * the EXACT-MODE safety switch: m_bar = n_heads => mask == 1
        identically (bit-exact dense writes, one config away — inherited
        from MoRM 1.0).

    Charge/pending features are decode-time scheduler state; during
    training (batched, stateless across batches) they are zero and the
    critic still sees phi + energy observables.
    """

    N_OBS = 4   # [log(beta*||k||+eps), log(1+||k||), log(1+charge), age/K]

    def __init__(self, n_heads: int, mast, m_bar: int,
                 m_max: Optional[int] = None,
                 controller_lr: float = 0.08,
                 ste_tau: float = 1.0,
                 use_obs: bool = False,
                 critic_bias: bool = True):
        """use_obs=False (default): the critic consumes phi only — the
        convention the notebook's mask sites run (beta/knorm observables are
        not uniformly available at every call site). use_obs=True: the
        caller guarantees per-head obs at every scores()/forward() call
        (e.g. the decode scheduler, which has beta, k-norm, charge and
        pending-age) — then ridge_init(obs=...) fits both critic blocks."""
        super().__init__()
        self.H = n_heads
        self.mast = mast                                # shared, not owned
        self.m_bar = int(m_bar)
        self.m_max = m_max
        self.ste_tau = ste_tau
        self.use_obs = bool(use_obs)
        # v9: the critic carries a per-head INTERCEPT (the log-influence LEVEL of each head). v8's bias-free
        # critic + mean-removed ridge fit kept the slope and lost every head's mean level: exp(score) was right
        # only up to an arbitrary per-head factor, and the elastic threshold is shared by all heads
        # (pooled log-R^2 -5..-8 on the calibration labels; ~7.5 points less influence mass at an equal budget).
        self.critic_phi = nn.Linear(mast.S, n_heads, bias=bool(critic_bias))
        self.critic_obs = nn.Linear(self.N_OBS, n_heads, bias=False)
        nn.init.zeros_(self.critic_phi.weight)
        if self.critic_phi.bias is not None:
            nn.init.zeros_(self.critic_phi.bias)
        nn.init.zeros_(self.critic_obs.weight)
        # the live rate threshold (controller state, NOT gradient-trained)
        self.register_buffer("rate_threshold", torch.tensor(0.0))
        self.controller = ElasticBudgetController(
            m_bar=m_bar, m_max=m_max, lr=controller_lr)
        self.last_mask: Optional[torch.Tensor] = None
        self.last_m: Optional[torch.Tensor] = None
        self.last_scores: Optional[torch.Tensor] = None

    # -- exact mode -----------------------------------------------------------
    @property
    def exact_mode(self) -> bool:
        return self.m_bar >= self.H

    def set_budget(self, m_bar: int, m_max: Optional[int] = None):
        """Retarget the elastic budget (used by the training annealer)."""
        self.m_bar = int(m_bar)
        self.controller.m_bar = int(m_bar)
        self.controller.m_max = None if m_max is None else \
            int(max(m_max, m_bar))

    # -- critic ----------------------------------------------------------------
    def scores(self, phi: torch.Tensor,
               obs: Optional[torch.Tensor] = None) -> torch.Tensor:
        """phi [T,S], obs [T,H,N_OBS] -> predicted influence [T,H] (>=0)."""
        s = self.critic_phi(phi)
        if obs is not None and self.use_obs:
            s = s + torch.einsum("thf,hf->th", obs.float(),
                                 self.critic_obs.weight.float())
        return s.float().exp()

    def ridge_init(self, phi: torch.Tensor, obs: torch.Tensor,
                   labels: torch.Tensor, ridge: float = 1e-2):
        """Per-head ridge fit of log-influence on [phi || obs] features (+ v9: the per-head intercept).

        phi [N,S]; obs [N,H,N_OBS]; labels [N,H]. Solves H independent
        small least-squares systems (one per head) — the exact joint
        supervised warm start for the critic (MoRM 1.0 warm-started on
        v-ENERGY; MoRM 2.0 warm-starts on the exact influence labels).
        """
        with torch.no_grad():
            X_phi = phi.float()
            y = labels.float().clamp_min(1e-12).log()    # [N,H]
            Xo = obs.float() if (obs is not None and self.use_obs) \
                else torch.zeros(phi.shape[0], self.H, self.N_OBS,
                                 device=phi.device)      # [N,H,N_OBS]
            mu = X_phi.mean(0, keepdim=True)
            Xc = X_phi - mu
            for h in range(self.H):
                Xh = torch.cat([Xc, Xo[:, h, :]], dim=1)  # [N, S+N_OBS]
                yh = y[:, h] - y[:, h].mean()
                Xc2 = Xh - Xh.mean(0, keepdim=True)
                lam = ridge * Xc2.pow(2).mean().clamp_min(1e-12)
                A = torch.linalg.solve(
                    Xc2.t() @ Xc2 + lam * torch.eye(Xc2.shape[1],
                                                    device=Xc2.device),
                    Xc2.t() @ yh)
                self.critic_phi.weight.data[h] = A[:self.mast.S].to(
                    self.critic_phi.weight.dtype)
                self.critic_obs.weight.data[h] = A[self.mast.S:].to(
                    self.critic_obs.weight.dtype)
                if self.critic_phi.bias is not None:
                    # v9 INTERCEPT. The fit is on mean-removed y and features, i.e. the prediction at a raw
                    # input is  ybar_h + (phi - mu).A_phi + (obs - obsbar_h).A_obs ; the constant part
                    #   b_h = ybar_h - mu.A_phi - obsbar_h.A_obs
                    # has to live in the critic (v8 dropped it).
                    b_h = (y[:, h].mean() - mu[0] @ A[:self.mast.S]
                           - Xo[:, h, :].mean(0) @ A[self.mast.S:])
                    self.critic_phi.bias.data[h] = b_h.to(self.critic_phi.bias.dtype)

    # -- v1-compatible forward ---------------------------------------------------
    def forward(self, phi: torch.Tensor,
                obs: Optional[torch.Tensor] = None,
                m: Optional[int] = None) -> torch.Tensor:
        """phi [T,S] (obs [T,H,N_OBS] optional) -> hard write mask [T,H].

        Elastic mode (default): threshold(L_hat > rate_threshold), capped
        at m_max, straight-through-smoothed in training. If `m` is given,
        falls back to MoRM 1.0 fixed top-m semantics (compat hook for the
        v1 verification contracts).
        """
        T = phi.shape[0]
        if self.exact_mode and m is None:
            mask = phi.new_ones(T, self.H)
            self.last_mask, self.last_m = mask.detach(), None
            self.last_scores = None
            return mask
        L = self.scores(phi, obs)                        # [T,H] predicted
        if m is not None:                                # v1 compat: top-m
            mm = min(m, self.H)
            tv, ti = L.topk(mm, dim=-1)
            hard = torch.zeros_like(L).scatter_(-1, ti, 1.0)
            if self.training:
                soft = torch.zeros_like(L).scatter_(
                    -1, ti, torch.softmax(tv.float(), -1).to(L.dtype))
                hard = hard + (soft - soft.detach())
            self.last_mask, self.last_m = hard.detach(), hard.sum(-1).detach()
            self.last_scores = L.detach()
            return hard
        # lazy warm start: multiplicative control cannot leave 0.0, so the
        # first forward seeds the threshold from this batch's score
        # distribution (deliberately over-admitting: half the target
        # quantile — the controller only ever tightens from here)
        if float(self.rate_threshold) <= 0.0:
            with torch.no_grad():
                q0 = max(0.0, 1.0 - self.m_bar / self.H)
                thr0 = 0.5 * float(L.detach().flatten().quantile(q0)
                                   .clamp_min(1e-12))
                self.rate_threshold.fill_(thr0)
                self.controller.threshold = thr0
        thr = float(self.rate_threshold)
        hard = (L > thr).to(L.dtype)
        if (self.controller.m_max is not None
                and self.controller.m_max < self.H):
            over = hard.sum(-1, keepdim=True) > self.controller.m_max
            if bool(over.any()):
                ti = L.topk(self.controller.m_max, dim=-1).indices
                capped = torch.zeros_like(L).scatter_(-1, ti, 1.0)
                hard = torch.where(over, capped * (L > thr).to(L.dtype),
                                   hard)
        if self.training:                                # smooth-threshold STE
            soft = torch.sigmoid((L - thr) / max(self.ste_tau, 1e-6))
            hard = hard + (soft - soft.detach())
        self.last_mask = hard.detach()
        self.last_m = (hard.detach() > 0.5).sum(-1)
        self.last_scores = L.detach()
        return hard

    def controller_update(self, m_mean: float) -> float:
        """Feed the observed mean write count to the rate controller."""
        if not self.exact_mode:
            self.controller.update(m_mean)
            self.rate_threshold.fill_(self.controller.threshold)
        return float(self.rate_threshold)

    def obs_features(self, beta: torch.Tensor, knorm: torch.Tensor,
                     charge: Optional[torch.Tensor] = None,
                     age: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Build the per-head observable features [T,H,N_OBS].

        beta/knorm: [T,H]; charge [H] or [T,H]; age [H] or [T,H] (in
        units of K). Charge/age default to 0 (training / prefill).
        """
        T = beta.shape[0]
        z = beta.new_zeros(T, self.H)
        ch = charge if (charge is not None and charge.dim() == 2) \
            else (charge[None, :].expand(T, self.H)
                  if charge is not None else z)
        ag = age if (age is not None and age.dim() == 2) \
            else (age[None, :].expand(T, self.H)
                  if age is not None else z)
        return torch.stack([
            (beta.float() * knorm.float()).clamp_min(1e-12).log(),
            knorm.float().clamp_min(1e-12).log1p(),
            ch.float().clamp_min(0).log1p(),
            ag.float().clamp(0, 1.5)], dim=-1)


# ============================================================================
# Section 5 — Certificates + baselines
# ============================================================================
def energy_scores(k: torch.Tensor, v: torch.Tensor,
                  beta: torch.Tensor) -> torch.Tensor:
    """MoRM 1.0's heuristic write score: beta * ||k|| * ||v||  [T,H]."""
    return beta.float() * k.float().norm(dim=-1) * v.float().norm(dim=-1)


def morm2_skip_certificate(
    q, k, delta, beta, lam, mask, scale: float = 1.0,
    read_weight=None, window: int = 512,
) -> dict:
    """Certified output deviation of a write mask (Theorem B + triangle).

    The bound is the sum over SKIPPED writes of their exact per-write
    influence labels:

        || o_dense - o_masked ||_{1,weighted} <= sum_{skipped} I_h(s)

    (triangle over exact single-write perturbations). MoRM 1.0's
    Proposition-6 energy bound is the special case with read overlaps
    replaced by 1 — always looser. Returns the bound, the labels, and the
    measured true gap when the caller passes `true_gap`.
    """
    out = write_influence_labels(q, k, delta, beta, lam, scale=scale,
                                 read_weight=read_weight, window=window,
                                 coverage_horizon=None)
    labels = out["labels"]
    skipped = (mask.float() <= 0.5)
    bound = float((labels * skipped).sum())
    return {"bound": bound, "labels": labels,
            "n_skipped": int(skipped.sum())}


def influence_of_writes(q, k, delta, beta, lam, s_idx, h_idx, horizons,
                        scale: float = 1.0, read_weight=None) -> torch.Tensor:
    """Exact Theorem-B influence of specific writes over custom horizons.

    For write list (s_i, h_i) with horizon W_i: I_i = sum_{t=s_i}^{s_i+W_i-1}
    w_t * scale * beta * |q_t^T (p(s,t) (*) k_s)| * ||delta_s||. This is the
    deferral-distortion certificate: a write deferred from s and committed
    at c loses exactly I_h(s, [s, c)) (Theorem C's measurable twin).
    """
    T, H, dk = q.shape
    qf, kf, df = q.float(), k.float(), delta.float()
    loglam = lam_to_full(lam, T, H, dk).clamp_min(1e-12).log()
    rw = read_weight.float() if read_weight is not None else None
    res = torch.zeros(len(s_idx), device=q.device)
    for i, (s, h, W) in enumerate(zip(s_idx, h_idx, horizons)):
        e = min(s + W, T)
        if e <= s:
            continue
        # p(s, t) for t in [s, e): p[s]=1 (empty product), then the
        # inclusive cumulative products of loglam[s+1..t]
        gl = loglam[s + 1:e, h, :]                       # [L-1, dk]
        p = torch.cat([torch.ones(1, dk, device=q.device),
                       torch.cumsum(gl, dim=0).exp()], 0)  # [L, dk]
        sc = (qf[s:e, h, :] * kf[s, h, :] * p).sum(-1)   # [L]
        w = rw[s:e] if rw is not None else torch.ones(e - s,
                                                      device=q.device)
        if w.dim() == 2:
            w = w[:, h]
        res[i] = (w * sc.abs()).sum() * scale * float(beta[s, h]) * float(
            df[s, h, :].norm())
    return res

# ============================================================================
# Section 4b — BetaGate: the write-enable lever on the fused [b|a] projection
# ============================================================================
class BetaGate(nn.Module):
    """MoRM 2.0's write-enable mask on a stock fused [b|a] projection
    (Proposition F), plus the per-head decay the scheduler needs.

    Wraps (ADOPTS, never copies) the model's `in_proj_ba` Linear. The stock
    forward computes `out = Linear(x)` with the b-slice (per-head write
    gates, H values) first and the a-slice (per-head log-decay) second
    (the qwen3_5 convention; `b_first=False` flips it). The gate rewrites
    the b-slice in LOGIT space:

        b_masked = where(mask, b, logit_floor)     (floor -60 -> sigma ~ 9e-27)

    so that whatever downstream transform the stock code applies (sigmoid
    in-kernel or outside), skipped heads write ~0 — bit-exact when the
    kernel consumes post-sigmoid beta, and 26 orders of magnitude below
    signal otherwise (asserted < 1e-6 by the notebook's runtime probe).
    With mask == 1 (exact mode) the output is BITWISE the stock tensor.

    `lam_of(a_slice)` computes the per-head decay for the decode scheduler:
    the GDN formula exp(-exp(A_log) * softplus(a + dt_bias)) when the
    parent module carries A_log/dt_bias (Qwen3-Next-style), else the GLA
    formula sigmoid(a) — the notebook's calibration probe selects and
    validates the formula against the stock kernel's outputs.
    """

    def __init__(self, inner: nn.Module, n_heads: int,
                 policy: "MoRM2Policy",
                 b_first: bool = True, logit_floor: float = -60.0,
                 A_log: Optional[torch.Tensor] = None,
                 dt_bias: Optional[torch.Tensor] = None,
                 gate_formula: str = "auto",
                 b_only: Optional[bool] = None):
        super().__init__()
        self.inner = inner                        # adopted stock Linear
        # v8.0.1 [F5a]: re-register the adopted weight at the STOCK key too.
        # Wrapping must NOT rename the state-dict entry: '...in_proj_ba.weight'
        # is what the COMPAT export reader and the checkpoints load; the
        # inner submodule's duplicate '.inner.weight' is harmless (same
        # object, zero-copy — VGateShim follows the same convention).
        self.register_parameter("weight", inner.weight)
        # v8.0.1 [F5b]: world-mode lever — set_world() flips this with the
        # execution family (dense_teacher / exact_resonant / compat_parity
        # need the bitwise stock tensor; soft_train / serving run the gate).
        self.enabled = True
        # v8.0.2 [P3]: qwen3_5 stores b and a as SEPARATE Linears
        # (in_proj_b [H,d] / in_proj_a [H,d]) — there is no fused [2H]
        # module to slice. When the adopted module emits exactly H rows it
        # IS the b-projection alone: gate the whole output (no a-slice to
        # preserve). b_only can also be forced explicitly.
        self.b_only = bool(b_only) if b_only is not None else             bool(getattr(inner, "out_features", 0) == n_heads)
        self.H = int(n_heads)
        self.policy = policy
        self.b_first = bool(b_first)
        self.logit_floor = float(logit_floor)
        self.gate_formula = gate_formula
        # gate parameters are owned by the parent attention module — keep
        # references (not copies) so training moves stay in sync
        if A_log is not None:
            self.register_buffer("A_log", A_log.detach().clone(),
                                 persistent=False)
        else:
            self.A_log = None
        if dt_bias is not None:
            self.register_buffer("dt_bias", dt_bias.detach().clone(),
                                 persistent=False)
        else:
            self.dt_bias = None
        self.last_mask: Optional[torch.Tensor] = None

    # -- stock-compatible forward --------------------------------------------
    def forward(self, x: torch.Tensor):
        out = self.inner(x)                       # [..., 2H] (or [..., H])
        if not getattr(self, "enabled", True):
            return out        # v8.0.1 [F5b]: stock bitwise passthrough
        if getattr(self, "b_only", False):
            # v8.0.2 [P3]: SPLIT layout — inner IS in_proj_b; gate the b
            # logits alone (the a sibling stays dense outside this wrap).
            # Same logit-space blend as the fused path: hard mask -> the
            # values are exactly {b, floor}; mask == 1 -> bitwise stock.
            lead = out.shape[:-1]
            b = out.reshape(-1, out.shape[-1])
            phi = self.policy.mast(x.reshape(-1, x.shape[-1]))
            mask = self.policy(phi)               # [T, H] (STE in training)
            self.last_mask = mask.detach() if hasattr(mask, "detach") \
                else mask
            mf = mask.to(b.dtype).reshape(-1, self.H) if mask.dim() == 2 \
                else mask.to(b.dtype)
            b = (mf * b + (1.0 - mf) * self.logit_floor).to(out.dtype)
            return b.reshape(*lead, -1)
        lead = out.shape[:-1]
        out2 = out.reshape(-1, out.shape[-1])
        H = self.H
        if self.b_first:
            b, a = out2[:, :H], out2[:, H:]
        else:
            a, b = out2[:, :H], out2[:, H:]
        phi = self.policy.mast(x.reshape(-1, x.shape[-1]))
        mask = self.policy(phi)                   # [T, H] (STE in training)
        self.last_mask = mask.detach() if hasattr(mask, "detach") \
            else mask
        # blend (NOT torch.where): forward values are exactly {b, floor}
        # for the hard 0/1 mask, and the straight-through gradient from
        # the policy reaches this site — a `where(mask > 0.5, ...)` here
        # would silently sever the policy's training signal (the v7 bug,
        # fixed at both mask sites in v8)
        mf = mask.to(b.dtype).reshape(-1, H) if mask.dim() == 2 else \
            mask.to(b.dtype)
        b = mf * b + (1.0 - mf) * self.logit_floor
        b = b.to(out2.dtype)
        out2 = torch.cat([b, a], dim=-1) if self.b_first \
            else torch.cat([a, b], dim=-1)
        return out2.reshape(*lead, -1)

    # -- per-head decay for the scheduler -------------------------------------
    def lam_of(self, a_slice: torch.Tensor) -> torch.Tensor:
        """a_slice [..., H] (the a-part of the fused output) -> lam in (0,1).

        formula "gdn":  lam = exp(-exp(A_log) * softplus(a + dt_bias))
        formula "gla":  lam = sigmoid(a)
        formula "auto": gdn when A_log/dt_bias are wired, else gla
        """
        f = self.gate_formula
        if f == "auto":
            f = "gdn" if (self.A_log is not None
                          and self.dt_bias is not None) else "gla"
        if f == "gdn":
            A = self.A_log if self.A_log is not None \
                else torch.zeros_like(a_slice)
            db = self.dt_bias if self.dt_bias is not None \
                else torch.zeros_like(a_slice)
            return torch.exp(-torch.exp(A) * F.softplus(a_slice + db))
        return torch.sigmoid(a_slice)



# ============================================================================
# Section 6 — MoRM2Decoder: stateful decode-time execution (NATIVE serving)
# ============================================================================
class MoRM2Decoder:
    """Decode-time MoRM 2.0 execution for one GDN layer (one token per call).

    Owns the per-layer scheduler (deferral buffer, charge ledger, rate
    controller) and drives the trained policy one token at a time:

        step(phi_t [S], k_t [H,dk], beta_t [H], lam_t, v_t [H,dv] or None)
            -> dict(mask [H], k_eff, beta_eff, v_eff, m_t)

    The caller (the NATIVE serving loop) feeds the emitted (k_eff, v_eff,
    beta_eff) into the stock kernel for the generated token — input
    rewriting only. Charge/age features are live scheduler state, so at
    decode time the critic sees the debt it owes each head (during
    training those features are zero: batches are stateless).
    """

    def __init__(self, policy: MoRM2Policy, dk: int, dv: int,
                 staleness_k: int = 64, m_max: Optional[int] = None,
                 v_provider: Optional[Callable] = None):
        self.policy = policy
        self.dk, self.dv = dk, dv
        self.v_provider = v_provider
        self.sched = WriteScheduler(policy.H, dk, dv,
                                    m_bar=policy.m_bar,
                                    staleness_k=staleness_k,
                                    m_max=m_max or policy.m_max,
                                    init_threshold=-1.0
                                    if policy.exact_mode else None,
                                    v_provider=v_provider)
        self.t = 0

    def reset(self):
        self.sched = WriteScheduler(self.policy.H, self.dk, self.dv,
                                    m_bar=self.policy.m_bar,
                                    staleness_k=self.sched.K,
                                    m_max=self.policy.m_max,
                                    init_threshold=float(
                                        self.policy.rate_threshold),
                                    v_provider=self.v_provider)
        self.t = 0

    def step(self, phi_t: torch.Tensor, k_t: torch.Tensor,
             beta_t: torch.Tensor, lam_t: torch.Tensor,
             v_t: Optional[torch.Tensor] = None) -> dict:
        age_feat = (self.sched.pend_age.float() / max(1, self.sched.K)
                    ).clamp(0, 1.5)
        obs = self.policy.obs_features(
            beta_t.reshape(1, -1), k_t.float().norm(dim=-1).reshape(1, -1),
            charge=self.sched.charge.reshape(1, -1),
            age=age_feat.reshape(1, -1))[0]              # [H, N_OBS]
        scores = self.policy.scores(phi_t.reshape(1, -1),
                                    obs.reshape(1, *obs.shape))[0]
        out = self.sched.step(scores, k_t, v_t, beta_t, lam_t, t=self.t)
        self.t += 1
        return out


# ============================================================================
# Section 7 — The verification suite (every claim above is executable)
# ============================================================================
def verify_morm2(verbose: bool = True, seed: int = 0,
                 device: str = "cpu") -> dict:
    """Numerically verify the MoRM 2.0 contracts M1-M10. Returns a report
    dict; raises AssertionError on any violated contract.

      M1  oracle correctness (both forms vs a naive per-head loop)
      M2  WIK single-write labels are EXACT vs brute-force ablation
          (plain form; delta-form agreement reported — frozen correction)
      M2c label chunk-invariance + cross-implementation agreement
      M3  multi-skip certified bound holds (and its tightness)
      M4  Proposition F: beta-mask is the exact lever in BOTH kernel forms;
          v-masking (MoRM 1.0) leaks deletion writes in the delta form
      M5  Theorem C: deferred commit reproduces the on-time trajectory
          from the commit point onward (plain form EXACT; delta reported)
      M6  Theorem D: the elastic controller converges to E[m] = m_bar
      M7  Theorem E: staleness <= K with exact decay compensation at every
          commit (deterministic pre-emption scenario)
      M8  the rate-distortion win: at EQUAL average budget, influence
          scheduling beats energy top-m (MoRM 1.0) on a synthetic task
          whose energy/influence structure is exactly anti-correlated
      M9  exact mode is bit-exact dense (all writes, no rewrites)
      M10 charge-ledger invariants (drops charge, debt ages with decay)
    """
    torch.manual_seed(seed)
    rep = {}

    def log(*a):
        if verbose:
            print(*a)

    # ---- shared synthetic stream -------------------------------------------
    T, H, dk, dv = 192, 6, 16, 12
    scale = dk ** -0.5
    q = F.normalize(torch.randn(T, H, dk, device=device), dim=-1)
    k = F.normalize(torch.randn(T, H, dk, device=device), dim=-1)
    v = torch.randn(T, H, dv, device=device)
    beta = torch.sigmoid(torch.randn(T, H, device=device) * 0.8 + 1.0)
    lam_s = 1 - 0.06 * torch.rand(H, device=device)      # [H] scalar decay
    lam = lam_s[None, :].expand(T, H)

    # ---- M1: oracle vs naive loops ------------------------------------------
    def naive(q_, k_, v_, beta_, lam_full, form):
        Tn, Hn, dkn = q_.shape
        dvn = v_.shape[-1]
        o = torch.zeros(Tn, Hn, dvn)
        for h in range(Hn):
            S = torch.zeros(dkn, dvn)
            for t in range(Tn):
                S = torch.diag(lam_full[t, h, :]) @ S
                d = v_[t, h]
                if form == "delta":
                    d = d - S.t() @ k_[t, h]
                S = S + beta_[t, h] * torch.outer(k_[t, h], d)
                o[t, h] = q_[t, h] @ S
        return o * (q_.shape[-1] ** -0.5)

    lam_full = lam_to_full(lam, T, H, dk)
    o_p, _, deltas_p = gdn_sequential(q, k, v, beta, lam, "plain", scale,
                                      return_deltas=True)
    o_d, _, deltas_d = gdn_sequential(q, k, v, beta, lam, "delta", scale,
                                      return_deltas=True)
    for form in ("plain", "delta"):
        o_ref = o_p if form == "plain" else o_d
        o_nai = naive(q, k, v, beta, lam_full, form)
        err = float((o_ref - o_nai).norm() / o_nai.norm())
        assert err < 1e-5, f"M1 {form}: oracle != naive ({err:.2e})"
        rep[f"M1_{form}"] = err
    log(f"M1  oracle == naive reference (plain {rep['M1_plain']:.1e}, "
        f"delta {rep['M1_delta']:.1e})")

    # ---- M2: WIK exactness vs brute-force single-write ablation -------------
    labels = write_influence_labels(q, k, deltas_p, beta, lam, scale=scale,
                                    window=T, chunk=64)["labels"]
    smp = [(s, h) for s in range(8, T - 8, 17) for h in range(H)][:24]
    errs_plain = []
    o_dense = o_p
    for (s, h) in smp:
        b2 = beta.clone()
        b2[s, h] = 0.0
        o_abl, _ = gdn_sequential(q, k, v, b2, lam, "plain", scale)
        true_total = float((o_dense - o_abl).norm(dim=-1).sum())
        errs_plain.append(abs(true_total - float(labels[s, h]))
                          / max(float(labels[s, h]), 1e-9))
    rep["M2_plain_max_rel_err"] = max(errs_plain)
    assert max(errs_plain) < 1e-4, \
        f"M2: WIK labels not exact ({max(errs_plain):.2e})"
    labels_d = write_influence_labels(q, k, deltas_d, beta, lam, scale=scale,
                                      window=T, chunk=64)["labels"]
    errs_delta = []
    o_dense_d = o_d
    for (s, h) in smp[:12]:
        b2 = beta.clone()
        b2[s, h] = 0.0
        o_abl, _ = gdn_sequential(q, k, v, b2, lam, "delta", scale)
        true_total = float((o_dense_d - o_abl).norm(dim=-1).sum())
        errs_delta.append(abs(true_total - float(labels_d[s, h]))
                          / max(true_total, 1e-9))
    rep["M2_delta_med_rel_err"] = sorted(errs_delta)[len(errs_delta) // 2]
    log(f"M2  WIK single-write labels EXACT vs ablation "
        f"(plain max rel {max(errs_plain):.2e}; delta-form median agreement "
        f"{rep['M2_delta_med_rel_err']:.1%} — frozen-correction response)")

    # ---- M2c: chunk-invariance + cross-implementation -----------------------
    l16 = write_influence_labels(q, k, deltas_p, beta, lam, scale=scale,
                                 window=T, chunk=16)["labels"]
    inv = float((l16 - labels).norm() / labels.norm())
    assert inv < 1e-5, f"M2c: labels chunk-dependent ({inv:.2e})"
    rep["M2c_chunk_invariance"] = inv
    xval = []
    for (s, h) in smp[:8]:
        got = influence_of_writes(q, k, deltas_p, beta, lam, [s], [h],
                                  [T - s], scale=scale)
        xval.append(abs(float(got[0]) - float(labels[s, h]))
                    / max(float(labels[s, h]), 1e-9))
    rep["M2c_cross_impl"] = max(xval)
    assert max(xval) < 1e-4, \
        f"M2c: implementations disagree ({max(xval):.2e})"
    log(f"M2c labels chunk-invariant ({inv:.1e}) with cross-implementation "
        f"agreement ({max(xval):.1e})")

    # ---- M3: multi-skip certified bound --------------------------------------
    mask = (torch.rand(T, H, device=device) > 0.6).float()
    gap = gdn_beta_masked_gap(q, k, v, beta, lam, mask, "plain", scale)
    cert = morm2_skip_certificate(q, k, deltas_p, beta, lam, mask,
                                  scale=scale, window=T)
    assert gap <= cert["bound"] + 1e-4, \
        f"M3: true gap {gap:.3e} exceeds certified bound {cert['bound']:.3e}"
    rep["M3_tightness"] = gap / max(cert["bound"], 1e-9)
    log(f"M3  multi-skip bound holds: gap {gap:.3f} <= bound "
        f"{cert['bound']:.3f} (tightness {rep['M3_tightness']:.2f})")

    # ---- M4: Proposition F — the exact masking lever -------------------------
    # beta-masking IS skip semantics (the write is exactly absent). In the
    # plain form the v-mask coincides with it; in the delta-corrected form
    # (the stock FLA WY kernel) v-masking deposits -beta k k^T S~ deletion
    # writes instead — a different, UNCERTIFIED operation, not a skip.
    o_bm_d, _ = gdn_sequential(q, k, v, beta * mask, lam, "delta", scale)
    o_vm_d, _ = gdn_sequential(q, k, v * mask[..., None], beta, lam, "delta",
                               scale)
    leak = float((o_vm_d - o_bm_d).norm())
    assert leak > 1e-2, (
        f"M4: in the delta form v-masking must diverge from skip semantics "
        f"(leak {leak:.4f}) — it deposits deletion writes")
    o_bm_p, _ = gdn_sequential(q, k, v, beta * mask, lam, "plain", scale)
    o_vm_p, _ = gdn_sequential(q, k, v * mask[..., None], beta, lam, "plain",
                               scale)
    coincide = float((o_vm_p - o_bm_p).norm())
    assert coincide < 1e-4, (
        f"M4: in the plain form v-mask and beta-mask must coincide exactly "
        f"({coincide:.2e})")
    skip_gap_d = float((o_d - o_bm_d).norm())
    rep["M4_leak_vs_skip"] = leak / max(skip_gap_d, 1e-9)
    rep["M4_plain_coincide"] = coincide
    log(f"M4  beta-mask == skip semantics in both forms; the MoRM 1.0 "
        f"v-mask lever diverges from skip semantics by {leak:.3f} in the "
        f"delta form ({rep['M4_leak_vs_skip']:.2f}x the certified skip gap; "
        f"plain-form coincidence {coincide:.1e})")

    # ---- M5: Theorem C — deferred-commit exactness ---------------------------
    # Contention-free construction: head 0 has NO fresh write of its own at
    # the commit token (beta[c0, 0] = 0 in BOTH runs), so the commit slot
    # carries exactly the deferred write and nothing else changes.
    s0, c0 = 20, 60
    beta5 = beta.clone()
    beta5[c0, 0] = 0.0
    comp = lam[s0 + 1:c0 + 1, 0].prod()                  # scalar-per-head lam
    k_def = k.clone()
    k_def[c0, 0] = k[s0, 0] * comp
    b_def = beta5.clone()
    b_def[s0, 0] = 0.0
    b_def[c0, 0] = beta5[s0, 0]
    v_def = v.clone()
    v_def[c0, 0] = v[s0, 0]
    o_on, _ = gdn_sequential(q, k, v, beta5, lam, "plain", scale)
    o_df, _ = gdn_sequential(q, k_def, v_def, b_def, lam, "plain", scale)
    after = slice(c0, T)
    err_after = float((o_on[after] - o_df[after]).norm()
                      / o_on[after].norm())
    before = slice(s0, c0)
    loss_before = float((o_on[before] - o_df[before]).norm())
    assert err_after < 1e-5 and loss_before > 1e-3, \
        (f"M5: deferred commit must be exact from the commit point "
         f"(after: {err_after:.2e}, window loss: {loss_before:.3f})")
    rep["M5_exact_after_commit"] = err_after
    rep["M5_window_loss"] = loss_before
    # window loss must equal the windowed influence certificate
    win_cert = float(influence_of_writes(q, k, deltas_p, beta, lam, [s0],
                                         [0], [c0 - s0], scale=scale)[0])
    rep["M5_window_cert_ratio"] = loss_before / max(win_cert, 1e-9)
    o_df_d, _ = gdn_sequential(q, k_def, v_def, b_def, lam, "delta", scale)
    rep["M5_delta_after"] = float((o_d[after] - o_df_d[after]).norm()
                                  / o_d[after].norm())
    log(f"M5  deferred commit EXACT after commit (rel {err_after:.1e}); "
        f"window loss {loss_before:.3f} vs influence certificate "
        f"{win_cert:.3f}; delta-form after-commit rel "
        f"{rep['M5_delta_after']:.1e}")

    # ---- M6: Theorem D — rate convergence ------------------------------------
    T2, H2 = 4000, 8
    sc2 = torch.rand(T2, H2, device=device) ** 2
    m_bar = 3
    out6 = schedule_writes(sc2, torch.randn(T2, H2, 4, device=device),
                           torch.randn(T2, H2, 3, device=device),
                           torch.ones(T2, H2, device=device) * 0.5,
                           torch.full((T2, H2), 0.98, device=device),
                           m_bar=m_bar, staleness_k=10 ** 9, m_max=H2)
    burn = 500
    mm = float(out6["stats"].m_trace[burn:].float().mean())
    assert abs(mm - m_bar) < 0.25 * m_bar + 0.15, \
        f"M6: E[m]={mm:.2f} did not converge to m_bar={m_bar}"
    rep["M6_mean_m"] = mm
    log(f"M6  elastic rate converges: E[m] = {mm:.2f} (target {m_bar})")

    # ---- M7: Theorem E — staleness + compensation (deterministic) ------------
    # head 0 wins the single slot every token (F=2); heads 1,2 park at t=0
    # and are capped out until the staleness deadline forces their commit.
    K = 12
    sc3 = torch.zeros(T, H, device=device)
    sc3[:, 0] = 2.0
    sc3[0, 1:] = 1.0
    lam3 = torch.full((T, H), 0.97, device=device)
    out7 = schedule_writes(sc3, k, v, beta, lam3, m_bar=1, staleness_k=K,
                           m_max=1, init_threshold=0.5)
    st = out7["stats"]
    assert st.max_staleness <= K, \
        f"M7: staleness {st.max_staleness} exceeded K={K}"
    assert st.defers >= 2 and st.commits >= 2, \
        f"M7: scenario must exercise deferral+commit ({st.summary()})"
    commits = out7["commit"].nonzero(as_tuple=False)
    for t_cm, h_cm in commits.tolist():
        assert t_cm == K, \
            f"M7: forced commit must fire exactly at age K ({t_cm} != {K})"
        comp7 = lam3[1:K + 1, h_cm].prod()
        assert torch.allclose(out7["k_eff"][t_cm, h_cm],
                              k[0, h_cm] * comp7, atol=1e-5), \
            "M7: commit key is not the exact decay compensation"
        assert abs(float(out7["beta_eff"][t_cm, h_cm])
                   - float(beta[0, h_cm])) < 1e-6, \
            "M7: commit beta must be the parked write's beta"
        assert torch.allclose(out7["v_eff"][t_cm, h_cm], v[0, h_cm]), \
            "M7: commit value must be the parked write's value"
        assert float(out7["mask"][0, h_cm]) == 0.0, \
            "M7: the parked write must have been skipped at its origin"
    rep["M7_max_staleness"] = st.max_staleness
    log(f"M7  staleness bound respected (max age {st.max_staleness} <= "
        f"K={K}); commits carry exact compensation | {st.summary()}")

    # ---- M8: the rate-distortion win -----------------------------------------
    # Anti-correlated construction: LOUD writes (||v|| x6, keys orthogonal
    # to every query — provably zero influence) vs QUIET writes (||v|| x1,
    # each key is exactly what one later query retrieves). Energy
    # scheduling buys loud; influence scheduling buys quiet.
    T3, H3 = 256, 4
    gap_ahead = 24
    q3 = torch.zeros(T3, H3, dk, device=device)
    Qr = F.normalize(torch.randn(H3, dk, device=device), dim=-1)
    ret_pos = torch.arange(H3, T3, 8, device=device)     # retrieval tokens
    # each retrieval token r reads head (r % H3) with Qr[r % H3]
    q3[ret_pos, ret_pos % H3] = Qr[ret_pos % H3]
    role = (torch.rand(T3, H3, device=device) < 0.5)     # loud per (t, h)
    k3 = torch.randn(T3, H3, dk, device=device)
    # loud keys: projected into the orthogonal complement of the H3 query
    # directions of their head -> EXACTLY zero influence
    for h in range(H3):
        basis = Qr[h:h + 1]                              # [1, dk]
        P = torch.eye(dk, device=device) - basis.t() @ basis
        cols = P @ torch.randn(dk, T3, device=device)
        cols = F.normalize(cols, dim=0)
        m_h = role[:, h]
        k3[m_h, h] = cols.t()[m_h]
        # quiet keys that are due for retrieval: key == future query
        for r in range(h, T3 - gap_ahead):
            if not bool(role[r, h]) and (r % H3) == h and \
                    (r + gap_ahead) in ret_pos.tolist():
                k3[r, h] = Qr[(r + gap_ahead) % H3]
    k3 = F.normalize(k3, dim=-1)
    v3 = torch.randn(T3, H3, dv, device=device) \
        * torch.where(role, 6.0, 1.0)[..., None]
    beta3 = torch.sigmoid(torch.randn(T3, H3, device=device) + 1.0)
    lam3b = torch.full((T3, H3), 0.97, device=device)
    o3, _, d3 = gdn_sequential(q3, k3, v3, beta3, lam3b, "plain",
                               dk ** -0.5, return_deltas=True)
    lab3 = write_influence_labels(q3, k3, d3, beta3, lam3b,
                                  scale=dk ** -0.5, window=T3)["labels"]
    m_budget = 2
    en = energy_scores(k3, v3, beta3)
    top_e = en.topk(m_budget, dim=-1).indices
    mask_e = torch.zeros_like(en).scatter_(-1, top_e, 1.0)
    gap_e = gdn_beta_masked_gap(q3, k3, v3, beta3, lam3b, mask_e, "plain",
                                dk ** -0.5)
    top_i = lab3.topk(m_budget, dim=-1).indices
    mask_i = torch.zeros_like(lab3).scatter_(-1, top_i, 1.0)
    gap_i = gdn_beta_masked_gap(q3, k3, v3, beta3, lam3b, mask_i, "plain",
                                dk ** -0.5)
    noisy = lab3 * torch.exp(torch.randn_like(lab3) * 0.35)
    out8 = schedule_writes(noisy, k3, v3, beta3, lam3b, m_bar=m_budget,
                           staleness_k=16, m_max=H3)
    mm8 = float(out8["stats"].m_trace.float().mean())
    gap_2 = gdn_beta_masked_gap(q3, k3, v3, beta3, lam3b, out8["mask"],
                                "plain", dk ** -0.5)
    assert gap_2 < 0.85 * gap_e, \
        (f"M8: MoRM 2.0 ({gap_2:.3f}) must beat energy top-m ({gap_e:.3f}) "
         f"at the same budget (mean m {mm8:.2f} vs {m_budget})")
    rep["M8_gap_energy"] = gap_e
    rep["M8_gap_morm2"] = gap_2
    rep["M8_gap_oracle"] = gap_i
    log(f"M8  equal-budget output deviation: energy top-m {gap_e:.3f} -> "
        f"MoRM 2.0 {gap_2:.2e} (oracle {gap_i:.2e}) — "
        f"{gap_e / max(gap_2, 1e-9):.0f}x better")

    # ---- M9: exact mode is bit-exact -----------------------------------------
    out9 = schedule_writes(torch.rand(T, H, device=device), k, v, beta, lam,
                           m_bar=H, staleness_k=8, m_max=None)
    assert bool((out9["mask"] == 1).all()) and out9["stats"].commits == 0
    o9, _ = gdn_sequential(q, out9["k_eff"], v, out9["beta_eff"], lam,
                           "plain", scale)
    err9 = float((o9 - o_p).norm() / o_p.norm())
    assert err9 < 1e-6, f"M9: exact mode not bit-exact ({err9:.2e})"
    rep["M9_exact_rel"] = err9
    log(f"M9  exact mode (m_bar=H) is bit-exact dense (rel {err9:.1e})")

    # ---- M10: charge-ledger invariants ---------------------------------------
    sched = WriteScheduler(H, dk, dv, m_bar=1, staleness_k=64,
                           init_threshold=1e9)   # nothing is ever eligible
    sched.step(torch.full((H,), 0.1), k[0], v[0], beta[0], lam[0], t=0)
    c1 = float(sched.charge.sum())
    for t in range(1, 40):
        sched.step(torch.zeros(H), k[t], v[t], beta[t], lam[t], t=t)
    c40 = float(sched.charge.sum())
    assert c1 > 0 and c40 < c1, (
        f"M10: charge must accumulate then age with decay ({c1:.3f} -> "
        f"{c40:.3f})")
    assert (sched.charge >= 0).all(), "M10: negative charge"
    rep["M10_charge_aged"] = (c1, c40)
    log(f"M10 charge ledger: drops charge the ledger ({c1:.3f}), then the "
        f"debt ages with the head decay ({c40:.3f})")

    log("verify_morm2: ALL CONTRACTS PASS")
    return rep


# ============================================================================
# 14. census_active — touched-weights census of a LIVE module graph (v9)
# ============================================================================
# v8's measured_active() summed a hand-picked list of registries and so never saw
# the dense k_proj / v half / in_proj_a / conv taps / norms, nor a surgery site that
# was skipped (kept dense). This walk visits EVERY module of the text decoder once
# (a module hanging under two parents — the MoRM policy — is counted once) and
# charges what one serving token touches; any 2-D+ parameter it cannot classify is
# charged in full (never silently free). It mirrors active_waterfall() family by
# family, so a plan can be compared with the graph that was actually built.
import re as _re_census

_CENSUS_SKIP = _re_census.compile(
    r"(visual|vision|merger|mtp|embed|lm_head|patch_embed|multi_modal)", _re_census.I)
_LAYER_RX = _re_census.compile(r"^(.*?\blayers\.\d+)\b")


def _prism_active(m) -> int:
    k = int(m.k_topk or 1)
    S = int(m.shared_idx.numel())
    sh = (int(m.sh_U.numel() + m.sh_M.numel() + m.sh_b.numel())
          if getattr(m, "sh_U", None) is not None else 0)            # v10 FFN shadow
    return (k * 3 * m.d_model * m.m_e + 3 * m.d_model * S + m.E * m.d_model
            + m.d_model * m.rank + (k * m.m_e + S) * m.rank + 2 * (k + 1) * m.rank + sh)


def _morm_heads(sp) -> int:
    pol = sp.morm
    mb = None if pol is None else getattr(pol, "m_bar", getattr(pol, "m", None))
    return sp.n_v_heads if mb is None else min(int(mb), sp.n_v_heads)


def _sectioned_active(m) -> int:
    din = int(m.weight.shape[1])
    qr, zr = getattr(m, "q_router", None), getattr(m, "z_router", None)
    q = ((m.q_carrier.numel() + min(m.topk, m.q_B) * m.q_r) * din
         + (qr.weight.numel() if qr is not None else 0)) if hasattr(m, "q_B") \
        else m.q_rows * din
    z = ((m.z_carrier.numel() + min(m.topk, m.z_B) * m.z_r) * din
         + (zr.weight.numel() if zr is not None else 0)) if hasattr(m, "z_B") \
        else m.z_rows * din
    sh = sum(int(getattr(m, f"{t}_sh_U").numel() + getattr(m, f"{t}_sh_M").numel())
             for t in ("q", "v", "z") if getattr(m, f"{t}_sh_U", None) is not None)   # v10 shadows
    return q + z + m.k_rows * din + _morm_heads(m) * m.v_head_rows * din + sh


def _morm_active(p) -> int:
    if hasattr(p, "critic_phi"):
        return int(p.critic_phi.weight.numel()
                   + (p.critic_phi.bias.numel() if p.critic_phi.bias is not None else 0)
                   + (p.critic_obs.weight.numel() if getattr(p, "use_obs", False) else 0))
    return int(p.router.weight.numel())


def census_active(model, hid: int, n_layers: int, funnel=None, mast=None,
                  mast_mode: str = "applications") -> dict:
    """Touched weights per serving text token, by family (see active_waterfall).
    Side results: census_active.last = {unclassified, mast_applications, mast_distinct}."""
    fam = {"embedding_lookup": int(hid), "antenna_mast": 0, "ffn": 0,
           "full_attention": 0, "linear_attention": 0, "lm_head_funnel": 0,
           "norms_misc": int(n_layers) * 2 * int(hid) + int(hid), "other": 0}
    seen, phi_evals, unclassified = set(), set(), []

    def family_of(path):
        if "linear_attn" in path:
            return "linear_attention"
        if "self_attn" in path:
            return "full_attention"
        if "mlp" in path:
            return "ffn"
        return "other"

    def need_phi(path, width):
        mm = _LAYER_RX.match(path)
        phi_evals.add((mm.group(1) if mm else path, int(width)))

    def walk(mod, path):
        if id(mod) in seen:
            return
        seen.add(id(mod))
        if path and _CENSUS_SKIP.search(path):
            return
        fam_n = family_of(path)
        if isinstance(mod, PrismMoeMLP):
            fam["ffn"] += _prism_active(mod)
            return
        if isinstance(mod, ResonantLinear):
            fam[fam_n] += mod.active_params() + mod.router.weight.numel()
            need_phi(path, mod.in_f)
            return
        if isinstance(mod, SectionedProjector):
            fam[fam_n] += _sectioned_active(mod)
            if hasattr(mod, "q_B") or hasattr(mod, "z_B") or mod.morm is not None:
                need_phi(path, mod.weight.shape[1])
            if mod.morm is not None:
                walk(mod.morm, path + ".morm")
            return
        if isinstance(mod, BetaGate):
            fam[fam_n] += int(mod.inner.weight.numel())      # b (and a) logits: dense reads
            need_phi(path, mod.inner.weight.shape[1])
            walk(mod.policy, path + ".policy")               # shared child: counted once
            return
        if isinstance(mod, (MoRM2Policy, MoRMWritePolicy)):
            fam[fam_n] += _morm_active(mod)
            return
        if hasattr(mod, "rl_gate") and hasattr(mod, "v_w"):  # VGateShim: dense v half + routed gate
            fam[fam_n] += int(mod.v_w.numel())
            walk(mod.rl_gate, path + ".rl_gate")
            return
        if isinstance(mod, nn.Linear):
            fam[fam_n] += int(mod.weight.numel())
            return
        conv = getattr(mod, "conv1d", None)
        if isinstance(conv, nn.Conv1d) and id(conv) not in seen:
            sp = next((c for c in mod.children() if isinstance(c, SectionedProjector)), None)
            n_ch = (sp.q_rows + sp.k_rows
                    + (sp.v_rows if getattr(sp, "v_sh_U", None) is not None      # v10: every v channel is non-zero
                       else _morm_heads(sp) * sp.v_head_rows)
                    if sp is not None else int(conv.weight.shape[0]))
            fam["linear_attention"] += n_ch * int(conv.weight.shape[-1])
            seen.add(id(conv))
        kids = list(mod.named_children())
        if not kids:
            for n_, p_ in mod.named_parameters(recurse=False):
                if p_.ndim >= 2:
                    fam[fam_n] += int(p_.numel())
                    unclassified.append(f"{path}.{n_}")
            return
        for n_, c_ in kids:
            walk(c_, f"{path}.{n_}" if path else n_)

    walk(model, "")
    S = int(mast.S) if mast is not None else 0
    app = S * sum(w_ for _, w_ in phi_evals)
    dis = S * sum({w_ for _, w_ in phi_evals})
    fam["antenna_mast"] = app if mast_mode == "applications" else dis
    fam["lm_head_funnel"] = int(funnel.active_params()) if funnel is not None else 0
    fam["TOTAL_ACTIVE"] = sum(fam.values())
    census_active.last = {"unclassified": unclassified, "mast_applications": app,
                          "mast_distinct": dis}
    return fam


census_active.last = {}


## Load the dense model, introspect it, re-derive the plan

In [ ]:
# Load the dense model — waterfall, section introspection, 3B budget solver
from transformers import AutoConfig, AutoTokenizer, AutoProcessor
import inspect, re

cfg = AutoConfig.from_pretrained(MODEL_ID)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
proc = None
try:
    proc = AutoProcessor.from_pretrained(MODEL_ID)   # multimodal processor
except Exception as e:
    print("Processor load skipped:", e)

tc = getattr(cfg, "text_config", cfg)
D_INT   = tc.intermediate_size
N_LAY   = tc.num_hidden_layers
HID     = tc.hidden_size
VOCAB   = tc.vocab_size
N_LIN   = sum(1 for t in getattr(tc, "layer_types", []) if t == "linear_attention")
N_FULL  = sum(1 for t in getattr(tc, "layer_types", []) if t == "full_attention")
print(f"hidden={HID} | layers={N_LAY} ({N_LIN} DeltaNet + {N_FULL} full-attn) "
      f"| intermediate={D_INT} | vocab={VOCAB} "
      f"| ctx={getattr(tc,'max_position_embeddings','?')}")

# ----------------------- load the dense model -------------------------------
from transformers import AutoModelForImageTextToText, AutoModelForCausalLM
import torch

# ---- v8.0.3 OOM guard: re-run safety + fail-fast capacity check -----------
release_previous_run()
if RUN_MODE == "full" and torch.cuda.is_available() and not globals().get("FORCE_LOAD", False):
    _vfree, _vtot = vram_gib()
    _NEED_FREE_GIB = 60.0        # 27B bf16 (~52 GiB) + calibration/training headroom
    if _vfree < _NEED_FREE_GIB:
        raise RuntimeError(
            f"Not enough free GPU memory for RUN_MODE='full': {_vfree:.0f} GiB free of "
            f"{_vtot:.0f} GiB (need >= {_NEED_FREE_GIB:.0f} GiB free; the bf16 weights alone are ~52 GiB).\n"
            "  - Colab: Runtime > Change runtime type > A100 with the 'High-RAM' option (80 GB), "
            "or an H100 / G4.   (40GB A100이면 High-RAM(80GB)으로 바꾸세요)\n"
            "  - A 40 GB A100 / L4 / T4 cannot hold this model: use RUN_MODE='demo' there.\n"
            "  - If a previous run still holds the GPU: Runtime > Restart session, run from the top.\n"
            "  - To skip this check anyway: set FORCE_LOAD = True before this cell.")
LOAD_KW = dict(dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32)
if RUN_MODE == "full":
    LOAD_KW.update(device_map="auto", low_cpu_mem_usage=True)
else:
    LOAD_KW.update(device_map={"": "cuda" if torch.cuda.is_available() else "cpu"})

_vl_err = None
try:
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **LOAD_KW)
except Exception as e:
    if is_oom(e):
        raise                      # a second full load can only OOM harder
    _vl_err = str(e)
if _vl_err is not None:            # handled OUTSIDE the except block: the half-loaded
    print(f"VL loader failed ({_vl_err}); falling back to text-only CausalLM")   # VL attempt is
    free_cuda("before text-only fallback")                                      # releasable now
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **LOAD_KW)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
print("Loaded", MODEL_ID, f"({sum(p.numel() for p in model.parameters())/1e9:.2f}B params)")
free_cuda(f"after {MODEL_ID} load")   # baseline census before calibration

# --------------------------- param waterfall --------------------------------
from collections import defaultdict
def waterfall(m):
    buckets = defaultdict(int)
    def bucketize(name):
        ln = name.lower()
        if "visual" in ln or "vision" in ln or "merger" in ln: return "vision_tower"
        if "mtp" in ln: return "mtp_block"
        if "embed" in ln: return "embeddings"
        if "lm_head" in ln: return "lm_head"
        if ".mlp." in ln: return "ffn"
        if "attn" in ln or "linear_attn" in ln or "delta" in ln or "conv" in ln: return "attention"
        if "norm" in ln: return "norms"
        return "other"
    for name, p in m.named_parameters():
        buckets[bucketize(name)] += p.numel()
    return dict(buckets)

wf = waterfall(model)
total_dense = sum(wf.values())
print(f"\n{'bucket':<14}{'params':>14}{'%':>8}")
for k, v in sorted(wf.items(), key=lambda x: -x[1]):
    print(f"{k:<14}{v:>14,}{100*v/total_dense:>7.1f}%")
print(f"{'TOTAL':<14}{total_dense:>14,}")

# ============================================================================
# SECTION INTROSPECTION — multi-signal, never-fatal
# ============================================================================
#   R1 DISCOVER DeltaNet modules by multiple independent signals (class-name
#      regex, attribute fingerprints, fused-shape signature) — never one.
#   R2 RESOLVE sections dynamically: attribute-name match first, then a
#      config-consistent shape search; every size tagged measured/inferred.
#   R3 CROSS-CHECK every claimed split (v|gate, fused qkvz) against a second
#      shape; inconsistent splits are DISABLED — never guessed.
#   R4 SURGERY ONLY on name-resolved sites; shape-only resolutions still
#      feed the budget waterfall (their savings are honestly removed).
#   R5 NO DEAD-END ASSERTS: anything unresolved degrades gracefully to its
#      exact dense cost, with a diagnostic dump of the offending module.

_EXCLUDE_PAT = re.compile(
    r"(visual|vision|image|patch_embed|mtp|multi_modal|mm_|proj_encoder)")
_DN_CLASS_PAT = re.compile(
    r"(deltanet|delta_?net|gated_?delta|\bgdn\b|linear_?attention|linear_?attn"
    r"|linearized|recurrent|retent|rwkv|mamba|state_?space)", re.I)
_DN_ATTRS = ("in_proj_qkvz", "in_proj_qkv", "in_proj_ba", "in_proj_z",
             "in_proj", "qkvz", "conv1d", "short_conv", "A_log", "a_log",
             "dt_bias", "l2norm", "decay", "gdn")
_FUSED_ATTR_NAMES = ("in_proj_qkvz", "in_proj_qkv", "qkvz_proj", "in_proj",
                     "Wqkvz", "qkvz", "in_proj_qkvy")
_BA_ATTR_NAMES = ("in_proj_ba", "ba_proj", "in_proj_ba_g", "ba")
_OUT_ATTR_NAMES = ("out_proj", "o_proj", "wo", "out")
_CONV_ATTR_NAMES = ("conv1d", "short_conv", "conv", "depthwise_conv")


def _excluded(name):
    return bool(_EXCLUDE_PAT.search(name.lower()))


def _own_2d(mod):
    out = []
    for n, p in mod.named_parameters(recurse=False):
        if p.ndim == 2:
            out.append((n, tuple(p.shape)))
    return out


def _iter_2d(mod):
    """2D weights at this granularity: own params + each direct child
    Linear's weight (fused projections are child Linears in practice)."""
    for n, p in _own_2d(mod):
        yield (mod, n, dict(mod.named_parameters(recurse=False))[n])
    for cn, child in mod.named_children():
        W = getattr(child, "weight", None)
        if isinstance(W, torch.Tensor) and W.ndim == 2:
            yield (child, f"{cn}.weight", W)


def find_delta_nets_v3(model, hints=None):
    """R1 — multi-signal DeltaNet discovery (never a single attribute)."""
    hints = hints or {}
    n_k, n_v = hints.get("n_k"), hints.get("n_v")
    fused_rows = (2 * n_k + 2 * n_v) if (n_k and n_v) else None
    qkv_rows = (2 * n_k + n_v) if (n_k and n_v) else None   # split layout
    found = []
    for name, mod in model.named_modules():
        if _excluded(name):
            continue
        signals = []
        if _DN_CLASS_PAT.search(type(mod).__name__):
            signals.append("class:" + type(mod).__name__)
        probe = {n for n, _ in mod.named_parameters(recurse=False)}
        probe |= {n for n, _ in mod.named_children()}
        hits = sorted({a for a in _DN_ATTRS if a in probe})
        inproj_hit = any(a.startswith("in_proj") or a == "qkvz" for a in hits)
        if hits:
            signals.append("attrs:" + ",".join(hits[:4]))
        for _, _, W in _iter_2d(mod):
            if fused_rows is not None and W.shape[0] == fused_rows:
                signals.append(f"fused-shape:{tuple(W.shape)}")
                break
            if qkv_rows is not None and W.shape[0] == qkv_rows:
                signals.append(f"qkv-shape:{tuple(W.shape)}")
                break
        strong = bool(signals) and (
            any(s.startswith("class:") for s in signals)
            or len(signals) >= 2 or inproj_hit)
        if strong:
            found.append({"name": name, "mod": mod, "signals": signals})
    found.sort(key=lambda r: r["name"].count("."))       # deepest wins
    kept, kept_paths = [], []
    for r in found:
        if any(r["name"].startswith(p + ".") for p in kept_paths):
            continue
        kept.append(r)
        kept_paths.append(r["name"])
    return kept


def _resolve_by_name(mod, names):
    for nm in names:
        child = getattr(mod, nm, None)
        if child is None:
            continue
        W = getattr(child, "weight", None)
        if isinstance(W, torch.Tensor) and W.ndim == 2:
            return (mod, nm, W, "name")
        if isinstance(child, torch.Tensor) and child.ndim == 2:
            return (mod, nm, child, "name")
    return None


def _resolve_by_shape(mod, rows, exclude_names=()):
    hits = [(o, n, t) for o, n, t in _iter_2d(mod)
            if t.shape[0] == rows and n not in exclude_names]
    if len(hits) == 1:
        o, n, t = hits[0]
        return (o, n, t, "shape")
    return None


def resolve_dn_sections(mod, hints):
    """R2 — resolve fused [q|k|v|z] (+ ba/out/conv) of one DeltaNet module."""
    notes = []
    sec = {"notes": notes}
    d = hints.get("d")
    n_k, n_v = hints.get("n_k"), hints.get("n_v")
    v_head_rows = hints.get("v_head_rows")

    ref = _resolve_by_name(mod, _FUSED_ATTR_NAMES)
    if ref is None:
        rows_hint = (2 * n_k + 2 * n_v) if (n_k and n_v) else None
        if rows_hint:
            ref = _resolve_by_shape(mod, rows_hint)
            if ref:
                notes.append("fused proj resolved by SHAPE only — surgery "
                             "disabled for this site (R4)")
    if ref:
        owner, attr, W, prov = ref
        sec["qkvz"] = (owner, attr, W, prov)
        R = W.shape[0]
        sec["split"] = False
        sec["z"] = None
        if n_k and n_v and R == 2 * n_k + 2 * n_v:
            # fused [q|k|v|z] projection (qwen3_next-style GDN)
            q_rows = k_rows = n_k
            v_rows = z_rows = n_v
            sec["provenance_sections"] = "measured(config-consistent)"
        elif n_k and n_v and R == 2 * n_k + n_v:
            # The qwen3_5-family GDN (Qwen3.5/3.6/3.8, which Qwen3.8-27B
            # belongs to) stores SPLIT projections: in_proj_qkv [q|k|v] +
            # in_proj_z [z] (+ in_proj_b / in_proj_a). Resolve the z
            # sibling by name/shape and record the SPLIT layout: sections
            # stay MEASURED, surgery wires in_proj_qkv (q routed, k dense,
            # v MoRM) and in_proj_z (routed) separately, each preserving
            # the stock module interface and state-dict key.
            q_rows = k_rows = n_k
            v_rows = n_v
            zref = _resolve_by_name(mod, ("in_proj_z",))
            if zref is None:
                zref = _resolve_by_shape(mod, n_v,
                                         exclude_names=("in_proj_qkv",
                                                        "in_proj_qkvz"))
            if zref is not None and tuple(zref[2].shape) == (n_v, W.shape[1]):
                z_rows = n_v
                sec["split"] = True
                sec["z"] = zref
                sec["provenance_sections"] = \
                    "measured(split q|k|v + separate z; qwen3_5 family)"
            else:
                z_rows = 0                    # no z sibling: 3-section site
                sec["provenance_sections"] = "measured(split q|k|v, z absent)"
                notes.append("in_proj_qkv split layout without in_proj_z — "
                             "z routed rows disabled for this site (R5)")
        elif R % 4 == 0:
            q_rows = k_rows = v_rows = z_rows = R // 4
            sec["provenance_sections"] = "inferred(quarter-split)"
            notes.append(f"fused rows {R} != 2*n_k+2*n_v — assumed q=k=v=z=R/4")
        else:
            q_rows = k_rows = R // 4
            v_rows = z_rows = R - 2 * (R // 4)
            sec["provenance_sections"] = "inferred(partial-split)"
        sec.update(q_rows=q_rows, k_rows=k_rows, v_rows=v_rows, z_rows=z_rows)
    else:
        sec["qkvz"] = None
        sec["split"] = False
        sec["z"] = None
        notes.append("fused qkvz projection NOT resolved")

    ref_ba = _resolve_by_name(mod, _BA_ATTR_NAMES)
    if ref_ba is None and sec.get("v_rows") and v_head_rows:
        ref_ba = _resolve_by_shape(mod, 2 * (sec["v_rows"] // v_head_rows))
    sec["ba"] = ref_ba
    if ref_ba is not None:
        sec["ba_rows"] = ref_ba[2].shape[0]
        # v8 (MoRM 2.0): fused [b|a] layout for the write-enable gate.
        # qwen3_5 convention: b (per-head write gates, H values) FIRST,
        # a (per-head log-decay) SECOND — BetaGate wraps in_proj_ba with
        # b_first=True; the Theorem-7 anchor test fails loudly if a target
        # model ever deviates from this layout.
        if sec.get("v_head_rows"):
            sec["b_rows"] = sec["v_rows"] // sec["v_head_rows"]
            sec["a_rows"] = max(0, int(ref_ba[2].shape[0]) - sec["b_rows"])
    else:
        # split b/a siblings (qwen3_5: in_proj_b [H,d] + in_proj_a [H,d])
        _nvh = (sec.get("v_rows") or 0) // v_head_rows if v_head_rows else 0
        _bref = _resolve_by_name(mod, ("in_proj_b",))
        _aref = _resolve_by_name(mod, ("in_proj_a",))
        _found = [nm for nm, r in (("in_proj_b", _bref),
                                   ("in_proj_a", _aref)) if r is not None]
        sec["ba_rows"] = _nvh * len(_found) if _found else None
        # v8.0.2 [P2]: record the SPLIT b-projection reference. qwen3_5 has
        # NO fused in_proj_ba, so sec["ba"] stays None here — and without
        # this reference the surgery cell would silently skip the BetaGate
        # install, leaving MoRM 2.0 INERT (critic fitted, the mask never
        # reaching a write). in_proj_a (the decay pre-activation) is left
        # dense on purpose: a drives decay, not writes — the exact skip
        # lever is beta alone (Proposition F).
        if _bref is not None and _nvh and _bref[2].shape[0] == _nvh:
            sec["b"] = _bref
            sec["b_rows"] = _nvh
            if _aref is not None and _aref[2].shape[0] == _nvh:
                sec["a"] = _aref
                sec["a_rows"] = _nvh

    ref_out = _resolve_by_name(mod, _OUT_ATTR_NAMES)
    if ref_out is None and d and sec.get("v_rows"):
        ref_out = _resolve_by_shape(mod, d)
    sec["out"] = ref_out
    sec["out_in"] = ref_out[2].shape[1] if ref_out else None
    ref_conv = _resolve_by_name(mod, _CONV_ATTR_NAMES)
    sec["conv"] = ref_conv

    if v_head_rows and sec.get("v_rows") and sec["v_rows"] % v_head_rows == 0:
        sec["v_head_rows"] = v_head_rows
    elif sec.get("v_rows"):
        sec["v_head_rows"] = next((c for c in (128, 64)
                                   if sec["v_rows"] % c == 0), None)
    else:
        sec["v_head_rows"] = None

    sec["name_resolved"] = bool(sec.get("qkvz")) and sec["qkvz"][3] == "name"
    if sec.get("split"):
        # z rows live in the SIBLING weight (sec["z"]): q+k+v must fill the
        # qkv projection exactly and z must fill the sibling exactly (R3).
        ok_shape = (sec.get("q_rows") and sec.get("v_rows")
                    and (sec["q_rows"] + sec["k_rows"] + sec["v_rows"])
                    == sec["qkvz"][2].shape[0]
                    and sec["z_rows"] == sec["z"][2].shape[0])
    else:
        ok_shape = (sec.get("q_rows") and sec.get("v_rows")
                    and (sec["q_rows"] + sec["k_rows"] + sec["v_rows"]
                         + sec["z_rows"]) == sec["qkvz"][2].shape[0])
    if sec.get("qkvz") and not ok_shape:
        sec["name_resolved"] = False
        notes.append("section sum != fused rows — surgery disabled (R3)")
    sec["morm_ok"] = bool(sec.get("v_head_rows") and sec.get("v_rows"))
    return sec


def _diag_dump(model, max_layers=1):
    lines = ["--- DIAGNOSTIC DUMP (first decoder layer) ---"]
    layers = getattr(getattr(model, "model", model), "layers", None)
    if layers is None:
        layers = [m for _, m in model.named_children()]
    for i, layer in enumerate(list(layers)[:max_layers]):
        lines.append(f"[layer {i}] class={type(layer).__name__}")
        for cn, child in layer.named_children():
            shapes = ", ".join(f"{n}{tuple(p.shape)}"
                               for n, p in _own_2d(child)[:6])
            lines.append(f"    .{cn}: {type(child).__name__} {shapes}")
    return "\n".join(lines)


def build_section_table(model, tc, n_lin_expected=0):
    """tc may be a config object OR a dict. Returns (SS, DNS_REF, GATES,
    report_lines, FAS). Families that cannot be resolved degrade, never
    crash (R5)."""
    rep = []
    _get = (lambda k, dv=None: tc.get(k, dv)) if isinstance(tc, dict) \
        else (lambda k, dv=None: getattr(tc, k, dv))
    hints = dict(
        d=_get("hidden_size"),
        n_k=((_get("linear_num_key_heads") or 0)
             * (_get("linear_key_head_dim") or 0) or None),
        n_v=((_get("linear_num_value_heads") or 0)
             * (_get("linear_value_head_dim") or 0) or None),
        v_head_rows=_get("linear_value_head_dim"),
    )
    cands = find_delta_nets_v3(model, hints)
    rep.append(f"DeltaNet candidates discovered: {len(cands)} "
               f"(config says {n_lin_expected})")
    for c in cands[:3]:
        rep.append(f"  {c['name']}: {c['signals']}")

    DNS_REF, SS, unresolved = [], {}, []
    for c in cands:
        sec = resolve_dn_sections(c["mod"], hints)
        if sec.get("qkvz") and sec["name_resolved"] and sec.get("q_rows"):
            DNS_REF.append((c["name"], c["mod"], sec))
        elif sec.get("qkvz"):
            unresolved.append((c["name"], sec))
    if cands and not DNS_REF:
        rep.append("!! DeltaNet modules exist but NO surgery-capable site — "
                   "their exact dense cost stays in the waterfall (R5).")
        rep.extend(_diag_dump(model).splitlines())
    elif len(cands) < n_lin_expected:
        rep.append(f"!! resolved {len(DNS_REF)}/{n_lin_expected} DeltaNet "
                   "sites — unresolved ones keep dense cost (R5).")
        for nm, sec in unresolved[:2]:
            rep.append(f"  unresolved {nm}: {sec['notes']}")

    if DNS_REF:
        sec0 = DNS_REF[0][2]
        SS.update(dn_q_rows=sec0["q_rows"], dn_k_rows=sec0["k_rows"],
                  dn_v_rows=sec0["v_rows"], dn_z_rows=sec0["z_rows"],
                  dn_ba_rows=sec0.get("ba_rows") or 0,
                  dn_v_head_rows=sec0.get("v_head_rows") or 0,
                  dn_out_in=sec0.get("out_in") or 0)
        rep.append(f"DN sections: q={SS['dn_q_rows']} k={SS['dn_k_rows']} "
                   f"v={SS['dn_v_rows']} z={SS['dn_z_rows']} "
                   f"vhead={SS['dn_v_head_rows']} "
                   f"({sec0.get('provenance_sections', '?')})")
    else:
        SS.update(dn_q_rows=0, dn_k_rows=0, dn_v_rows=0, dn_z_rows=0,
                  dn_ba_rows=0, dn_v_head_rows=0, dn_out_in=0)

    FAS = [(n, m) for n, m in model.named_modules()
           if hasattr(m, "q_proj") and hasattr(m, "o_proj")
           and not _excluded(n)]
    rep.append(f"full-attention layers discovered: {len(FAS)}")
    VGATE_OK = True
    if FAS:
        fa0 = FAS[0][1]
        q_rows = fa0.q_proj.weight.shape[0]
        k_rows = fa0.k_proj.weight.shape[0]
        v_raw = fa0.v_proj.weight.shape[0]
        o_in = fa0.o_proj.weight.shape[1]
        gate_fused = bool(_get("attn_output_gate", False))
        v_rows, gate_rows = (v_raw // 2, v_raw // 2) if gate_fused \
            else (v_raw, 0)
        # R3 cross-check: the v|gate split is only trusted when o_proj agrees
        if gate_fused and o_in != v_rows:
            VGATE_OK = False
            rep.append(f"!! v|gate split INCONSISTENT: o_in={o_in} != "
                       f"v_rows={v_rows} — gate surgery disabled, v_proj "
                       f"kept dense at {v_raw} rows (R3).")
            gate_rows, v_rows = 0, v_raw
        SS.update(q_rows=q_rows, k_rows=k_rows, v_rows=v_rows,
                  gate_rows=gate_rows, o_in=o_in)
        rep.append("full-attn sizes: " + str({k: SS[k] for k in
                   ("q_rows", "k_rows", "v_rows", "gate_rows", "o_in")}))
    else:
        SS.update(q_rows=0, k_rows=0, v_rows=0, gate_rows=0, o_in=0)

    GATES = {"dn_surgery": len(DNS_REF) > 0, "vgate_surgery": VGATE_OK}
    return SS, DNS_REF, GATES, rep, FAS


def surgery_ss(SS, GATES):
    """Solver-facing SS: disabled families contribute their exact dense cost."""
    if GATES.get("dn_surgery", False):
        return dict(SS)
    out = dict(SS)
    for k in ("dn_q_rows", "dn_k_rows", "dn_v_rows", "dn_z_rows"):
        out[k] = 0
    out["dn_v_head_rows"] = 1
    return out


SS, DNS_REF, GATES, INTRO_REP, FAS = build_section_table(model, tc, N_LIN)
for _ln in INTRO_REP:
    print(_ln)

# dynamic need_keys: only what the ENABLED surgery paths really require.
missing = []
if GATES["dn_surgery"]:
    _sec0 = DNS_REF[0][2]
    missing += ["dn_" + k for k in ("q_rows", "k_rows", "v_rows", "z_rows")
                if not _sec0.get(k)]
if GATES["vgate_surgery"]:
    missing += [k for k in ("q_rows", "k_rows", "v_rows", "gate_rows", "o_in")
                if not SS.get(k)]
if missing:
    print("!! section sizes incomplete:", missing,
          "— affected families keep their exact dense cost (R5)")
else:
    print("dynamic need_keys satisfied")

SS_SOLVER = surgery_ss(SS, GATES)

# ======================= THE 3B BUDGET SOLVER (v9: honest, and its plan is CONSUMED) ===========
# v8 solved for a 192-neuron shared core ... and then BUILT SHARED_SIZE=512 (+0.2B) with the
# config's own carrier / top-k / write cap: the model that was actually assembled drew ~3.15B
# while the log said 2.95B, hidden behind `assert act <= TARGET * 1.10`.  Now the solver is
# handed the REQUESTED geometry (shared core, write cap, head-shortlist cap, sketch width),
# trims only what the budget demands, and the pipeline RE-BINDS SHARED_SIZE / MAST_S /
# LIN_TOPK / CARRIER_FRAC / FUNNEL_SHORTLIST / M_WRITE from the plan it gets back.
_morm_site = bool(GATES["dn_surgery"] and DNS_REF[0][2].get("morm_ok"))
if _morm_site:
    _nh = DNS_REF[0][2]["v_rows"] // DNS_REF[0][2]["v_head_rows"]
    _m_cap = _nh if M_WRITE_FRAC is None else max(1, int(M_WRITE_FRAC * _nh))
else:
    _nh, _m_cap = 0, None
_solve_kw = dict(d=HID, D_ffn=D_INT, n_layers=N_LAY,
                 n_linear_attn=N_LIN, n_full_attn=N_FULL,
                 vocab=VOCAB, ss=SS_SOLVER, mast_S=MAST_S, rank_zssi=COMPLETION_RANK,
                 shared_ffn=SHARED_SIZE, e_ffn=FFN_EXPERTS, funnel_rank=FUNNEL_RANK,
                 m_write_max=_m_cap, shortlist_max=FUNNEL_SHORTLIST,
                 mast_S_min=MAST_S_MIN, mast_mode=MAST_ACCOUNTING,
                 funnel_rho=FUNNEL_USE_RHO,
                 # v10: the plan starts from the REQUESTED geometry and carries the shadows
                 k_ffn0=FFN_TOPK, lin_blocks0=LIN_BLOCKS, carrier0=CARRIER_FRAC,
                 m_write0=(M_WRITE_FLOOR if (SHADOW_V and SHADOW_RANK_LIN and _morm_site) else None),
                 shadow_lin=SHADOW_RANK_LIN, shadow_ffn=SHADOW_RANK_FFN,
                 shadow_v=bool(SHADOW_V and SHADOW_RANK_LIN and _morm_site))
if RUN_MODE == "full":
    sol = solve_budget_v2(target_active=TARGET_ACTIVE, **_solve_kw)
    PLAN, WF = sol["plan"], sol["waterfall"]
    if not PLAN["feasible"]:
        _msg = (f"3B mandate cannot be met: even the trimmed floor plan draws "
                f"{WF['TOTAL_ACTIVE']/1e9:.3f}B > {TARGET_ACTIVE/1e9:.2f}B "
                f"({MAST_ACCOUNTING} sketch accounting). Lower SHARED_SIZE / MAST_S_MIN, "
                f"or set MAST_ACCOUNTING='distinct' (each sketch matrix counted once), "
                f"or raise TARGET_ACTIVE.")
        if ENFORCE_BUDGET:
            raise RuntimeError(_msg)
        print("!! " + _msg + "  (ENFORCE_BUDGET=False: continuing)")
    # ---- the plan is CONSUMED: every knob the solver trimmed or spent is re-bound here ----
    SHARED_SIZE, MAST_S = PLAN["shared_ffn"], PLAN["mast_S"]
    LIN_BLOCKS, LIN_TOPK = PLAN["lin_blocks"], PLAN["lin_topk"]
    CARRIER_FRAC = PLAN["carrier_frac"]
    FUNNEL_SHORTLIST, FUNNEL_RANK = PLAN["funnel_shortlist"], PLAN["funnel_rank"]
    SHADOW_RANK_LIN = PLAN["shadow_lin"]
    for _t in PLAN["trims"]:
        print(f"  [budget] trimmed to fit {TARGET_ACTIVE/1e9:.2f}B: {_t}")
else:
    sol = solve_budget_v2(target_active=float("inf"), **_solve_kw)
    PLAN, WF = sol["plan"], sol["waterfall"]
    print(f"\ndemo minimal plan active: {WF['TOTAL_ACTIVE']/1e6:.1f}M "
          f"({100*WF['TOTAL_ACTIVE']/total_dense:.1f}% of dense)")

# reconcile FFN E with balanced-fission divisibility
feasible = [E for E, _ in CANDIDATES
            if (D_INT - SHARED_SIZE) % E == 0 and E <= PLAN["E_ffn"]]
E_CHOSEN = max(feasible) if feasible else max(E for E, _ in CANDIDATES
                                              if (D_INT - SHARED_SIZE) % E == 0)
K_CHOSEN = PLAN["k_ffn"]
M_E      = (D_INT - SHARED_SIZE) // E_CHOSEN
if _morm_site:
    M_WRITE = max(1, min(PLAN["m_write"], _nh)) if RUN_MODE == "full" else (
        _nh if M_WRITE_FRAC is None else max(1, int(M_WRITE_FRAC * _nh)))
else:
    M_WRITE = 1          # inert: MoRM is only installed on morm_ok sites
FUNNEL_ON = True


def _realized_waterfall(**ov):
    """Waterfall of the geometry that is ACTUALLY about to be built (not the solver's idea of it)."""
    kw = dict(E_ffn=E_CHOSEN, k_ffn=K_CHOSEN, shared_ffn=SHARED_SIZE,
              rank_zssi=COMPLETION_RANK, mast_S=MAST_S, lin_blocks=LIN_BLOCKS,
              lin_topk=LIN_TOPK, carrier_frac=CARRIER_FRAC,
              m_write=(M_WRITE if _morm_site else PLAN["m_write"]),
              funnel_rank=FUNNEL_RANK, funnel_shortlist=FUNNEL_SHORTLIST,
              funnel_rho=FUNNEL_USE_RHO, mast_mode=MAST_ACCOUNTING,
              shadow_lin=SHADOW_RANK_LIN, shadow_ffn=SHADOW_RANK_FFN,
              shadow_v=bool(SHADOW_V and SHADOW_RANK_LIN and _morm_site))
    kw.update(ov)
    return active_waterfall(HID, D_INT, N_LAY, N_LIN, N_FULL, VOCAB, SS_SOLVER, **kw)


WF = _realized_waterfall()           # what is about to be BUILT (full and demo alike)
if RUN_MODE != "full":
    print(f"demo realized geometry draws {WF['TOTAL_ACTIVE']/1e6:.1f}M active (no budget is enforced in demo)")
if RUN_MODE == "full":
    _alt = "distinct" if MAST_ACCOUNTING == "applications" else "applications"
    _WF_alt = _realized_waterfall(mast_mode=_alt)
    if WF["TOTAL_ACTIVE"] > TARGET_ACTIVE:
        _msg = (f"realized geometry draws {WF['TOTAL_ACTIVE']/1e9:.3f}B > "
                f"{TARGET_ACTIVE/1e9:.2f}B (E reconcile changed the plan)")
        if ENFORCE_BUDGET:
            raise RuntimeError(_msg)
        print("!! " + _msg)
    print(f"\n===> BUDGET (waterfall, '{MAST_ACCOUNTING}' sketch accounting): "
          f"{WF['TOTAL_ACTIVE']/1e9:.3f}B <= {TARGET_ACTIVE/1e9:.2f}B  "
          f"[same geometry, '{_alt}' convention: {_WF_alt['TOTAL_ACTIVE']/1e9:.3f}B] <===")
    print("     the census of the REAL module graph re-checks this after the surgery cell")
print(f"\nresolved: E={E_CHOSEN} k={K_CHOSEN} m_e={M_E} shared={SHARED_SIZE} "
      f"| dn_surgery={GATES['dn_surgery']} vgate={GATES['vgate_surgery']} "
      f"| MoRM m_write={M_WRITE} | mast S={MAST_S} lin_topk={LIN_TOPK} carrier={CARRIER_FRAC} "
      f"| funnel r={FUNNEL_RANK} V_s={FUNNEL_SHORTLIST} "
      f"| v10 shadows: lin rank {SHADOW_RANK_LIN}, ffn rank {SHADOW_RANK_FFN}, shadow-v {bool(SHADOW_V and SHADOW_RANK_LIN)} "
      f"| blocks {LIN_BLOCKS}")

# ------------- capture the ORIGINAL dense logits (the anchor) ----------------
PROBE_IDS = tok("The theory of resonant computation states that", return_tensors="pt").input_ids[:, :32]
DEV = next(model.parameters()).device
def _probe_logits():
    with torch.no_grad():
        return model(input_ids=PROBE_IDS.to(DEV)).logits.float().clone()


# v8.0.4: if the fla Triton kernel cannot run on this GPU / Triton build, the very first
# forward (this one) fails. Switch every Gated-DeltaNet layer to the PyTorch reference
# kernel and go on — slower, but the run survives. (Handled outside the except block.)
_fla_err = None
try:
    ORIG_LOGITS = _probe_logits()
except Exception as _e:
    if not globals().get("FLA_ACTIVE") or is_oom(_e):
        raise
    _fla_err = f"{type(_e).__name__}: {str(_e)[:200]}"
if _fla_err is not None:
    import inspect as _insp_f
    _gm = next((_insp_f.getmodule(type(m)) for m in model.modules()
                if "GatedDeltaNet" in type(m).__name__), None)
    for _fn in ("torch_chunk_gated_delta_rule", "torch_recurrent_gated_delta_rule"):
        _f = getattr(_gm, _fn, None)
        if callable(_f):
            setattr(_gm, _fn, _insp_f.unwrap(_f))
    FLA_ACTIVE = False
    free_cuda("fla fallback")
    print(f"[fla] Triton GDN kernel failed here ({_fla_err}) -> every Gated-DeltaNet layer now "
          "uses the PyTorch reference kernel (slower, same math). Set USE_FLA = False next time.")
    ORIG_LOGITS = _probe_logits()
print("anchor logits captured:", tuple(ORIG_LOGITS.shape))


## Corpus

In [ ]:
# corpus + FFN discovery (the calibration passes of v10 are NOT repeated)
from datasets import load_dataset

# ---- v8.0.4 [DATA]: a domain-balanced corpus ----------------------------------
# v8.0.2 calibrated AND distilled on 1% of wikitext only: encyclopedic English.
# Expert specialization (spectra, duty cycles, router prototypes) and the student's
# distillation then only ever "saw" that one register, and 16 epochs over 384
# blocks invited memorization. Now the same pipeline sees a mix of web / Korean /
# code / chat text (sources are STREAMED, a few MB each; any source that fails to
# load is skipped and wikitext fills the gap, so the worst case is the old corpus).
# Blocks are shuffled, so the calibration blocks (the first ones) and the held-out
# tail both carry the whole mixture. Set CORPUS_MIX = False to get wikitext only.
CORPUS_MIX = globals().get("CORPUS_MIX", RUN_MODE == "full")
MIX_TOTAL_BLOCKS = globals().get("MIX_TOTAL_BLOCKS", 1152)        # ~1.2M tokens
MIX_SHARES = globals().get("MIX_SHARES", {"fineweb-edu": 0.30, "ko-wiki": 0.15,
                                           "code": 0.15, "chat": 0.25})  # wikitext = the rest
MIX_DOC_CAP = 4 * SEQ_LEN        # one document contributes <= 4 blocks (breadth over depth)


def _wikitext_ids():
    try:
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="train[:1%]")
    except Exception:
        ds = load_dataset("wikitext", "wikitext-103-raw-v1", split="train[:1%]")
    text = "\n\n".join(t for t in ds["text"] if t.strip())
    return tok(text, return_tensors=None, add_special_tokens=False)["input_ids"]


def _stream_texts(name):
    """Lazily yield raw documents of one source."""
    if name == "fineweb-edu":
        ds = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT",
                          split="train", streaming=True)
        for r in ds:
            yield r["text"]
    elif name == "ko-wiki":
        ds = load_dataset("wikimedia/wikipedia", "20231101.ko", split="train", streaming=True)
        for r in ds:
            yield r["text"]
    elif name == "code":
        ds = load_dataset("codeparrot/codeparrot-clean", split="train", streaming=True)
        for r in ds:
            yield r["content"]
    elif name == "chat":
        ds = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft", streaming=True)
        for r in ds:
            msgs = r["messages"]
            try:
                yield tok.apply_chat_template(msgs, tokenize=False)
            except Exception:
                yield "\n".join(f"{m['role']}: {m['content']}" for m in msgs)
    else:
        raise ValueError(f"unknown corpus source {name!r}")


def _blocks_from(name, n_blocks, max_docs=20000):
    """Tokenize a stream, EOS-join documents, cut into SEQ_LEN blocks."""
    need_tok = n_blocks * SEQ_LEN
    eos = tok.eos_token_id if tok.eos_token_id is not None else 0
    ids, docs = [], 0
    for t in _stream_texts(name):
        if not t or not t.strip():
            continue
        ids.extend(tok(t, add_special_tokens=False)["input_ids"][:MIX_DOC_CAP])
        ids.append(eos)
        docs += 1
        if len(ids) >= need_tok or docs >= max_docs:
            break
    return [ids[i * SEQ_LEN:(i + 1) * SEQ_LEN] for i in range(len(ids) // SEQ_LEN)][:n_blocks]


CORPUS_COMPOSITION = {}
if CORPUS_MIX:
    _parts = {}
    for _src, _share in MIX_SHARES.items():
        _want = int(MIX_TOTAL_BLOCKS * _share)
        try:
            _parts[_src] = _blocks_from(_src, _want)
        except Exception as _e:
            print(f"  corpus source '{_src}' unavailable ({type(_e).__name__}: "
                  f"{str(_e)[:90]}) — skipped")
            _parts[_src] = []
        print(f"  corpus {_src:<12s}: {len(_parts[_src]):4d}/{_want} blocks")
    _wiki_ids = _wikitext_ids()
    _wiki_blocks = [_wiki_ids[i * SEQ_LEN:(i + 1) * SEQ_LEN]
                    for i in range(len(_wiki_ids) // SEQ_LEN)]
    _wiki_want = max(0, MIX_TOTAL_BLOCKS - sum(len(v) for v in _parts.values()))
    _parts["wikitext"] = _wiki_blocks[:_wiki_want]
    print(f"  corpus {'wikitext':<12s}: {len(_parts['wikitext']):4d}/{_wiki_want} blocks "
          "(fills whatever the other sources did not provide)")
    _all = [b for v in _parts.values() for b in v]
    random.Random(0).shuffle(_all)
    ids_all = [t for b in _all for t in b]
    CORPUS_COMPOSITION = {s: len(v) for s, v in _parts.items()}
    del _parts, _wiki_ids, _wiki_blocks, _all
else:
    ids_all = _wikitext_ids()

n_blocks = max(2, CAL_TOKENS // SEQ_LEN)
blocks = torch.tensor([ids_all[i*SEQ_LEN:(i+1)*SEQ_LEN]
                       for i in range(min(n_blocks, len(ids_all)//SEQ_LEN))])
print(f"calibration corpus: {blocks.shape[0]} blocks x {SEQ_LEN} tokens")

# deterministic subsamples across sessions (the hooks below draw
# randperm from the global RNG)
torch.manual_seed(0)

# ------------------------- discover every dense FFN -------------------------
def discover_ffns(m):
    found = []
    for name, mod in m.named_modules():
        if hasattr(mod, "gate_proj") and hasattr(mod, "up_proj") and hasattr(mod, "down_proj"):
            if "visual" in name.lower() or "vision" in name.lower():
                continue
            found.append((name, mod))
    return found

FFNS = discover_ffns(model)
FFNS_MAP = dict(FFNS)
print(f"discovered {len(FFNS)} dense SwiGLU FFNs")
STREAM_DTYPE = torch.float32


## Rebuild from the saved state (v16.2; a finished v17 if there is one)

In [ ]:
# v11 — REBUILD the finished v10 model from its saved state (no calibration, no surgery, no training)
# The frozen dense weights come from the Hub (loaded above); everything v10 made — expert assignment, row plans,
# shadows, routers, gains, completion fields, MoRM critics, thresholds — comes from v10_state.pt.
import json
_bases = [DRIVE_DIR] + ([os.environ["PRISM_STATE_ALT"]] if os.environ.get("PRISM_STATE_ALT") else [])
_cands = [os.path.join(b, d, d + ".pt") for d in ("v17_state", "v16_state", "v15_state", "v14_state", "v13_state", "v12_state", "v10_state")
          for b in _bases]                                    # newest state first
STATE_PATH = globals().get("STATE_PATH") or next((p for p in _cands if os.path.exists(p)), _cands[-1])
if not os.path.exists(STATE_PATH):
    _hint = ("" if os.path.ismount("/content/drive") else
             "  (Google Drive가 마운트되지 않았습니다 — 맨 위의 Drive 셀을 먼저 실행하세요)")
    raise AssertionError(f"저장본이 없습니다: {STATE_PATH}{_hint}")
_t0 = time.time()
STATE = torch.load(STATE_PATH, map_location="cpu", weights_only=False)
SB, ST, META, SCFG = STATE["buffers"], STATE["trainable"], STATE["meta"], STATE.get("config", {})
print(f"state: {STATE_PATH} ({os.path.getsize(STATE_PATH) / 2**30:.2f} GiB) | saved by {SCFG.get('__PRISM_VER__')} "
      f"at step {STATE.get('train', {}).get('abs_step')} | {len(ST)} trained tensors, {len(SB)} buffers")

# ---- the geometry is the SAVED one (never re-derived): every knob the modules depend on is re-bound here
_diff = []
for _k in ("E_CHOSEN", "K_CHOSEN", "M_E", "SHARED_SIZE", "COMPLETION_RANK", "MAST_S", "LIN_BLOCKS", "LIN_TOPK",
           "CARRIER_FRAC", "SHADOW_RANK_LIN", "SHADOW_RANK_FFN", "SHADOW_V", "M_WRITE", "FUNNEL_RANK",
           "FUNNEL_SHORTLIST", "FUNNEL_KAPPA", "FUNNEL_RECALL", "MAST_ACCOUNTING"):
    _v = SCFG.get(_k)
    if _v is None or (isinstance(_v, str) and _v.startswith("<unserializable")):
        continue
    if _k in globals() and globals()[_k] != _v:
        _diff.append(f"{_k}: {globals()[_k]!r} -> {_v!r}")
    globals()[_k] = _v
if _diff:
    print("  config re-bound to the saved geometry:", "; ".join(_diff))
    WF = _realized_waterfall()
DEV = next(model.parameters()).device

# ---- registries (same names the rest of the notebook uses)
_SURG_ORIG = {}
def _swap(parent, attr, new):
    key = (id(parent), attr)
    if key not in _SURG_ORIG:
        _SURG_ORIG[key] = (parent, attr, getattr(parent, attr, None))
    setattr(parent, attr, new)

for _lst in (PRISM_MODULES, RESONA_LINEARS, RESONA_SECTIONED, RESONA_MORM, RESONA_FUNNEL):
    _lst.clear()
RESONA_MORM_GATES = []
MORM2_STASH = {}

# ---- 1) FFNs: PrismMoeMLP from the saved neuron -> expert assignment
_ffn_meta = META["prism"]
for li, (name, mlp) in enumerate(FFNS):
    assert name in _ffn_meta, f"FFN {name} is not in the saved state"
    assign = SB[name + ".assign"].to(DEV)
    prism = PrismMoeMLP(mlp.gate_proj.weight.data.t(), mlp.up_proj.weight.data.t(), mlp.down_proj.weight.data,
                        assign, int(_ffn_meta[name]["E"]), rank=int(_ffn_meta[name]["rank"]), seed=li,
                        k_topk=int(_ffn_meta[name]["k_topk"])).to(DEV)
    prism.default_mode = "soft"
    _swap(model.get_submodule(name.rpartition(".")[0]), name.rpartition(".")[2], prism)
    PRISM_MODULES.append((name, prism))
print(f"  rebuilt {len(PRISM_MODULES)} FFNs (E={E_CHOSEN}, top-{K_CHOSEN})")

# ---- 2) the shared sketch
_mm = STATE["extra"]["mast"]
MAST = AntennaMast(int(_mm["d_model"]), sketch_dim=int(_mm["S"]), seed=int(_mm["seed"])).to(DEV)
with torch.no_grad():
    MAST.R.copy_(_mm["R"].to(device=MAST.R.device, dtype=MAST.R.dtype))

# ---- 3) routed whole-Linears
for name, a in META["linears"].items():
    assert "(gate)" not in name, "v-gate sites are not supported by the v11 rebuild"
    mod = model.get_submodule(name)
    plan = {"carrier_idx": SB[name + ".carrier_idx"].to(DEV), "block_idx": SB[name + ".block_idx"].to(DEV)}
    rl = ResonantLinear(mod.weight.data, plan, MAST, topk=int(a["topk"])).to(DEV)
    rl.router_st = bool(a.get("router_st", True))
    rl.default_mode = "dense"
    _swap(model.get_submodule(name.rpartition(".")[0]), name.rpartition(".")[2], rl)
    RESONA_LINEARS.append((name, rl))
print(f"  rebuilt {len(RESONA_LINEARS)} resonant linears")

# ---- 4) DeltaNet sectioned projectors + MoRM policies + write gates
_morm_by_dn = {}
for name, a in META["sectioned"].items():
    dn_name, attr = name.rpartition(".")[0], name.rpartition(".")[2]
    mod = model.get_submodule(name)
    b = [int(v) for v in SB[name + ".sec_bounds"]]
    secs = (b[1] - b[0], b[2] - b[1], b[3] - b[2], b[4] - b[3])
    plan_q = ({"carrier_idx": SB[name + ".q_carrier"].to(DEV), "block_idx": SB[name + ".q_blocks"].to(DEV)}
              if (name + ".q_blocks") in SB else None)
    plan_z = ({"carrier_idx": SB[name + ".z_carrier"].to(DEV), "block_idx": SB[name + ".z_blocks"].to(DEV)}
              if (name + ".z_blocks") in SB else None)
    morm = None
    pm = META["morm"].get(dn_name)
    if pm is not None:
        _has_bias = any(k.endswith(".critic_phi.bias") and k.startswith(dn_name + ".") for k in ST)
        morm = MoRM2Policy(n_heads=int(pm["H"]), mast=MAST, m_bar=int(pm["m_bar"]),
                           m_max=(int(pm["m_max"]) if pm.get("m_max") is not None else None),
                           controller_lr=MORM2_CTRL_LR, ste_tau=float(pm.get("ste_tau") or MORM2_STE_TAU),
                           critic_bias=_has_bias).to(DEV)
        morm._crit_fit = None
        morm.strict_m = (int(pm['strict_m']) if pm.get('strict_m') is not None else None)     # v16: a state saved under the law
        _morm_by_dn[dn_name] = (morm, pm)
    sp = SectionedProjector(mod.weight.data, secs, v_head_rows=int(a["v_head_rows"]), plan_q=plan_q, plan_z=plan_z,
                            mast=MAST, morm=morm, topk=int(a["topk"])).to(DEV)
    sp.router_st = bool(a.get("router_st", True))
    sp.default_mode = "dense"
    _swap(model.get_submodule(dn_name), attr, sp)
    RESONA_SECTIONED.append((name, sp))
    if morm is not None:
        RESONA_MORM.append((dn_name, morm))
for gname, a in META.get("gates", {}).items():
    dn_name, battr = gname.rpartition(".")[0], gname.rpartition(".")[2]
    dn_mod = model.get_submodule(dn_name)
    morm, _pm = _morm_by_dn[dn_name]
    gate = BetaGate(getattr(dn_mod, battr), int(a["H"]), morm, b_first=bool(a.get("b_first", True)),
                    b_only=bool(a.get("b_only")), logit_floor=float(a.get("logit_floor", MORM2_BETA_LOGIT_FLOOR)),
                    A_log=getattr(dn_mod, "A_log", None), dt_bias=getattr(dn_mod, "dt_bias", None)).to(DEV)
    gate.shadow_writes = bool(a.get("shadow_writes", False))
    _swap(dn_mod, battr, gate)
    RESONA_MORM_GATES.append((gname, gate))
print(f"  rebuilt {len(RESONA_SECTIONED)} sectioned projectors | MoRM policies {len(RESONA_MORM)} | "
      f"write gates {len(RESONA_MORM_GATES)}")

# ---- 5) head funnel (deterministic factors; the saved ones are written over them)
_fm = STATE["extra"]["funnel"]
funnel = HeadFunnel(model.get_parameter("lm_head.weight").detach(), rank=int(_fm["U_r"].shape[1]),
                    shortlist=int(_fm["shortlist"]), use_rho=_fm.get("rho") is not None)
with torch.no_grad():
    funnel.U_r.copy_(_fm["U_r"].to(device=funnel.U_r.device, dtype=funnel.U_r.dtype))
    funnel.P_r.copy_(_fm["P_r"].to(device=funnel.P_r.device, dtype=funnel.P_r.dtype))
    if _fm.get("rho") is not None and getattr(funnel, "rho", None) is not None:
        funnel.rho.copy_(_fm["rho"].to(device=funnel.rho.device, dtype=funnel.rho.dtype))
funnel.kappa = float(_fm["kappa"])
RESONA_FUNNEL.append(("lm_head(funnel)", funnel))
FUNNEL_RECALL_K0 = float("nan")
FUNNEL_KAPPA = float(funnel.kappa)
_fr = SCFG.get("FUNNEL_RECALL")
FUNNEL_RECALL = float(_fr) if isinstance(_fr, (int, float)) else float("nan")

# ---- 6) load every saved tensor into the rebuilt graph
_named = dict(model.named_parameters())
_ld, _miss, _bad = 0, [], []
with torch.no_grad():
    for _n, _t in ST.items():
        if _n.rpartition('.')[2].startswith('ns_'):                # lens / own-neuron / head tensors: their cells install them
            continue
        _p = _named.get(_n)
        if _p is None:
            _miss.append(_n)
        elif tuple(_p.shape) != tuple(_t.shape):
            _bad.append((_n, tuple(_t.shape), tuple(_p.shape)))
        else:
            _p.copy_(_t.to(device=_p.device, dtype=_p.dtype))
            _ld += 1
    _lb, _bskip = 0, []
    for _n, _t in SB.items():
        _path, _, _attr = _n.rpartition(".")
        if _attr.startswith('ns_'):                                # lens buffers: installed by the lens cell
            continue
        try:
            _mod = model.get_submodule(_path)
        except Exception:
            _bskip.append(_n)
            continue
        if _attr not in getattr(_mod, "_buffers", {}):
            _bskip.append(_n)
            continue
        _cur = _mod._buffers[_attr]
        if _cur is None or tuple(_cur.shape) != tuple(_t.shape):
            setattr(_mod, _attr, _t.to(device=DEV))                 # e.g. the shadows (registered as None)
        else:
            _cur.copy_(_t.to(device=_cur.device, dtype=_cur.dtype))
        _lb += 1
_fresh = [n for n, p in _named.items() if p.requires_grad and n not in ST]
print(f"  loaded {_ld}/{len(ST)} trained tensors and {_lb}/{len(SB)} buffers | missing {len(_miss)} | "
      f"shape mismatch {len(_bad)} | buffers without a home {len(_bskip)} | trainables left fresh {len(_fresh)}")
for _x in (_miss[:4] + _bad[:4] + _bskip[:4] + _fresh[:4]):
    print("      ", _x)
assert not _miss and not _bad and not _fresh, "the rebuilt graph does not match the saved state (see the list above)"

# ---- 7) policy / gate / world state
for dn_name, (morm, pm) in _morm_by_dn.items():
    c = pm.get("controller") or {}
    morm.set_budget(int(pm["m_bar"]), m_max=(int(c["m_max"]) if c.get("m_max") is not None else None))
    morm.controller.threshold = float(morm.rate_threshold)
for _, _m in PRISM_MODULES:
    _m.last_aux = None


def measured_active(detail: bool = False):
    c = census_active(model, HID, N_LAY, funnel=funnel, mast=MAST, mast_mode=MAST_ACCOUNTING)
    return c if detail else c["TOTAL_ACTIVE"]


set_world("soft_train")
CENSUS = measured_active(detail=True)
print(f"  census of the rebuilt graph: {CENSUS['TOTAL_ACTIVE'] / 1e9:.3f}B active "
      f"(plan {WF['TOTAL_ACTIVE'] / 1e9:.3f}B) | rebuilt in {time.time() - _t0:.0f}s")
free_cuda("v10 state rebuilt")


## Teacher cache (reused, built only if missing) + held-out perplexity

In [ ]:
# Teacher cache (the free in-place teacher) + before-training PPL
# v8.0.4 [DATA]: teach on (almost) the whole corpus. v8.0.2 cached only 6x the calibration
# blocks (384) and then trained 6400 micro-batches on them (~17 epochs: memorization).
# The held-out tail stays 1/16 of the teaching set and never overlaps it.
TEACH_BLOCKS_MAX = globals().get("TEACH_BLOCKS_MAX", 1024)
if RUN_MODE == "full":
    TEACH_BLOCKS_N = max(blocks.shape[0] * 6,
                         min(TEACH_BLOCKS_MAX, int((len(ids_all) // SEQ_LEN) / (1 + 1 / 16))))
else:
    TEACH_BLOCKS_N = 32
HOLD_N = max(2, TEACH_BLOCKS_N // 16)
need = TEACH_BLOCKS_N + HOLD_N
if len(ids_all) < need * SEQ_LEN:
    TEACH_BLOCKS_N = max(4, len(ids_all) // SEQ_LEN - HOLD_N)
    print(f"corpus smaller than requested; teaching on {TEACH_BLOCKS_N} blocks")
teach_blocks = torch.tensor([ids_all[i*SEQ_LEN:(i+1)*SEQ_LEN]
                             for i in range(TEACH_BLOCKS_N)])
tail0 = TEACH_BLOCKS_N * SEQ_LEN
heldout = torch.tensor([ids_all[(tail0 + j*SEQ_LEN):(tail0 + (j+1)*SEQ_LEN)]
                        for j in range(HOLD_N)])
if heldout.numel() == 0:      # degenerate corpora: fall back to calibration tail
    heldout = blocks[-2:].clone()
    print("WARNING: held-out overlaps calibration (tiny corpus)")

DEV = next(model.parameters()).device
V = model.config.text_config.vocab_size if hasattr(model.config, "text_config") \
    else model.config.vocab_size

def set_mode(mode):
    """Shim mapping v1 mode names onto the set_world presets."""
    if mode == "uniform":
        set_world("dense_teacher")          # == the original dense model (Thm 7)
    elif mode == "topk":
        set_world("serving")
    elif mode == "soft":
        set_world("soft_train")
    else:
        raise ValueError(mode)

@torch.no_grad()
def cache_teacher(shard_path):
    set_mode("uniform")                       # == the original dense model (Thm 1)

    def _run(bs):                             # bs = pure batching knob
        shards = []
        for i0 in range(0, teach_blocks.shape[0], bs):
            batch = teach_blocks[i0:i0+bs].to(DEV)
            logits = model(input_ids=batch).logits                # [B, L, V]
            B, L, _ = logits.shape
            ids_k = torch.empty(B, L, TEACHER_TOPK, dtype=torch.long)
            lp_k  = torch.empty(B, L, TEACHER_TOPK, dtype=torch.float16)
            for t0 in range(0, L, 512):                           # chunked fp32 softmax
                lg = logits[:, t0:t0+512].float()
                lsm = lg.log_softmax(-1)
                tv, ti = lsm.topk(TEACHER_TOPK, dim=-1)
                ids_k[:, t0:t0+512] = ti.cpu()
                lp_k[:, t0:t0+512]  = tv.cpu().half()
                del lg, lsm
            shards.append({"input_ids": batch.cpu(),
                           "teacher_ids": ids_k, "teacher_logprobs": lp_k})
            del logits
        return shards

    # v8.0.3: on OOM halve the micro-batch (identical output — bs only batches)
    shards = oom_backoff(_run, [2, 1] if RUN_MODE == "full" else [8, 4, 2, 1],
                         tag="teacher cache")
    torch.save(shards, shard_path)
    set_mode("soft")
    return shard_path

# v8.0.4: the cache file is keyed by WHAT it contains (model, sequence length, top-k and the
# exact token blocks). The old fixed name was silently reused from Drive even after the corpus
# changed, which would have trained on stale data while evaluating on new data.
import hashlib
_sig = hashlib.sha1(f"{MODEL_ID}|{SEQ_LEN}|{TEACHER_TOPK}|{TEACH_BLOCKS_N}".encode()
                    + teach_blocks.numpy().tobytes()).hexdigest()[:10]
TEACHER_SHARD = os.path.join(CACHE_DIR, f"teacher_top{TEACHER_TOPK}_{_sig}.pt")
print(f"teacher set: {TEACH_BLOCKS_N} blocks x {SEQ_LEN} tokens, held-out {HOLD_N} blocks "
      f"-> cache {os.path.basename(TEACHER_SHARD)}")
if os.path.exists(TEACHER_SHARD) and not os.environ.get("PRISM_RECACHE"):
    print("teacher cache found on Drive — reusing")
else:
    t0 = time.time(); cache_teacher(TEACHER_SHARD)
    print(f"teacher cached -> {TEACHER_SHARD} ({time.time()-t0:.0f}s)")

# --------------------- before-training perplexities -------------------------
@torch.no_grad()
def ppl(blocks_, mode):
    # Chunked NLL: CE is a per-token sum, so chunking over the sequence
    # axis is EXACT and bounds the fp32 transient to one [B, 256, V]
    # slice (~250 MiB).
    set_mode(mode)

    def _run(bs):                             # bs = pure batching knob
        nll, ntok = 0.0, 0
        for i0 in range(0, blocks_.shape[0], bs):
            b = blocks_[i0:i0+bs].to(DEV)
            out = model(input_ids=b).logits
            lg = out[:, :-1]; tg = b[:, 1:]
            V = lg.shape[-1]
            for c0 in range(0, lg.shape[1], 256):
                c1 = min(c0 + 256, lg.shape[1])
                lgc = lg[:, c0:c1].float()
                nll += F.cross_entropy(lgc.reshape(-1, V), tg[:, c0:c1].reshape(-1),
                                       reduction="sum").item()
                del lgc
            ntok += tg.numel()
            del out, lg
        return nll, ntok

    # v8.0.3: on OOM halve the micro-batch (identical output)
    nll, ntok = oom_backoff(_run, [2, 1] if RUN_MODE == "full" else [8, 4, 2, 1],
                            tag=f"ppl[{mode}]")
    set_mode("soft")
    return math.exp(nll / ntok)

PPL = {}
PPL["teacher_dense"]     = ppl(heldout, "uniform")
PPL["student_topk@init"] = ppl(heldout, "topk")
PPL["student_soft@init"] = ppl(heldout, "soft")     # v9.1: decides whether the soft phase is worth running
torch.save(PPL, os.path.join(CACHE_DIR, "ppl_before.pt"))
free_cuda("teacher cache + pre-training PPL")   # phase boundary
print({k: round(v, 3) for k, v in PPL.items()})
print("NOTE: all numbers share the same substrate. dense -> served top-k is the sparsity damage that training "
      "must close;\n      ln(PPL) is the cross-entropy: a served PPL in the thousands means the calibrated "
      "student starts far from the teacher\n      (expect a long recovery; raise STEPS / lower the sparsity).")


## Neuron lens — restored from the state (v16: own neurons, market probes)

In [ ]:
# NEURON LENS (v12)  +  SECOND SIGHT, GHOST NEURONS, CONSEQUENCE HEAD (v15)  +  OWN NEURONS, MARKET PROBES (v16)
#
# What v16 (MOLT) adds to the served FFN — both are exact no-ops until a later cell switches them on:
#
#   OWN NEURONS       the m neurons through which most of a layer's served output flows get a trainable COPY of their
#                     three weight vectors (gate row, up row, down column) that belongs to the student. When such a
#                     neuron is read, the copy is read instead of the dense rows (same 3*d parameters per token); the
#                     dense rows stay untouched for the teacher. At the start the copy equals the original.
#   WALLET            (the market of v16, see the MOLT cells) every token walks through the network with a wallet. A
#                     site no longer reads "its k best items": it posts a PRICE (a threshold on its score) and a token
#                     buys what clears it — more here, fewer there, a different mix for every token. What a token saves
#                     below a site's quota it may spend above another's; the balance can never end below zero, so every
#                     single token stays inside the 3.0B. The helpers below are shared by every kind of site.
#   MARKET PROBES     (quota mode, when the wallet is off) two zero scalars per layer sit on the exact-neuron mask:
#                         M1 + p_plus * [the next DK neurons below the cut] - p_minus * [the last DK neurons above it]
#                     their gradients are, to first order, what the loss would gain by reading DK more neurons in this
#                     layer and what it would lose by reading DK fewer. The market cell moves the budget on them.
#
# The v12 text further down describes the base mechanism. What v15 changes in the served FFN:
#
#   SECOND SIGHT      v12 looks at ALL D neurons with the full rank-r lens: 2*D*r parameters per token and layer — on
#                     the 27B that is 28% of the FFN budget, spent mostly on neurons that are never read. Now the lens
#                     looks twice: a COARSE look at every neuron (its r1 strongest directions — they are stored
#                     strongest-last, so this is a slice, nothing new is fitted), then the full-rank look only at a
#                     SHORTLIST of n_s neurons.   lens rows read:  2*D*r1 + 2*n_s*(r - r1)   instead of   2*D*r
#   GHOST NEURONS     a neuron read exactly costs 3*d (gate row, up row, down column). The lens already holds an
#                     estimate a^_j of its activation, so the next k2 neurons below the exact ones are VOICED:
#                         contribution_j = gain_j * (a^_j - mu_j) * w_down_j          cost: d (its real down column)
#                     a third of the price, with the real output direction. gain_j starts at the measured regression
#                     of the true activation on the estimate (an unreliable estimate speaks softly) and is trained.
#                     Per token: k exact neurons, k2 ghosts, the rest mean + linear shadow.
#   CONSEQUENCE HEAD  score_j = |a^_j - mu_j| * ||w_down_j|| * exp(w_j + q_j . (z Pc)),  z = x B (the lens code).
#                     w and the low-rank context start at 0 (= the size-only rule) and are trained by COUNTERFACTUAL
#                     CREDIT: the forward pass is the hard choice that is served; the backward pass sends every
#                     neuron near a threshold what the answer would gain or lose if it moved up a tier:
#                         mean -> ghost:   gain_j (a^_j - mu_j) <dL/dy, w_down_j>
#                         ghost -> exact:  (a_j - mu_j - gain_j (a^_j - mu_j)) <dL/dy, w_down_j>
#   SELF-REFIT        every training forward computes the exact activations anyway: the predictor (B, Ag, Au) is
#                     regressed on them (full rank and coarse rank) on the stream the model has NOW. The task loss
#                     reaches the predictor too, through the ghosts.
#   BUDGET            everything read per token is charged (_ns_cost). Second sight FREES budget; ghosts, the head
#                     and the ghost gains are paid from it; a layer never costs more than it did.
#
# v12 — NEURON LENS: per-token, per-NEURON FFN selection through a predictive lens (zero gradient steps)
#
# v10 serves each FFN as  shared core + top-k of 128 FIXED neuron groups + a rank-128 shadow. After training that
# FFN still misses ~87% of its dense twin's output on the same input (ltd 0.87): a group is a blunt instrument.
# Here the SAME touched-parameter budget of every layer is spent differently, and nothing is trained:
#
#   lens     g^ = (x B) Ag^T , u^ = (x B) Au^T      rank-r picture of the gate / up pre-activations — the best
#                                                   rank-r one in the metric of the inputs this layer really sees
#   select   the k neurons with the largest  |silu(g^) u^ - mu| * ||W_down column||   for THIS token
#            (mu = the neuron's mean activation: an unselected neuron contributes its mean, not zero)
#   exact    those k neurons are computed with the ORIGINAL dense weights (3 * d parameters each)
#   shadow   b + U (M x): closed-form reduced-rank regression of everything the selection left behind
#
# Touched per token and layer:  3*d*k (exact neurons) + lens + shadow  <=  what v10 spends on that layer.
# Order of work:  (1) LAB on three layers — v10 vs. lens vs. the oracle selection, a small grid of (lens rank,
# shadow rank); it stops here if the lens is not clearly better.  (2) every layer, bottom-up, each one fitted on
# the stream produced by the layers already switched.  (3) VERDICT on held-out text by KL to the dense teacher:
# all prefixes / suffixes of the depth are tried and the best one is kept — "none" is a legal answer (rollback).
NS_BLOCKS   = globals().get("NS_BLOCKS", 48 if RUN_MODE == "full" else 6)      # calibration documents
NS_TOK      = globals().get("NS_TOK", 256 if RUN_MODE == "full" else 24)       # tokens sampled per document
NS_PRED_RANKS   = tuple(globals().get("NS_PRED_RANKS", (32, 64, 128) if RUN_MODE == "full" else (4, 8)))
NS_SHADOW_RANKS = tuple(globals().get("NS_SHADOW_RANKS", (SHADOW_RANK_FFN, 4 * SHADOW_RANK_FFN)))
NS_MIN_GAIN = globals().get("NS_MIN_GAIN", 0.90)     # the lab must cut the local FFN error to <= 90% of v10's
NS_FORCE    = globals().get("NS_FORCE", False)       # True: go on even if the lab says no
NS_CHUNK    = globals().get("NS_CHUNK", 2048)        # tokens per chunk on the served path
NS_VERDICT_BLOCKS = globals().get("NS_VERDICT_BLOCKS", 8)
NS_VAL_FRAC = 0.25
V15_ST_TEMP = globals().get("V15_ST_TEMP", 0.5)     # width (in log-score units) of the band around the threshold that gets credit
V15_W_CLAMP = globals().get("V15_W_CLAMP", 3.0)     # |log importance| is clamped here (a factor e^3 = 20 either way)
import contextlib


_MK = globals().get("_MK") or {"on": False, "probes": {}, "width": {}}     # the budget market (filled by the MOLT cells)


def _mk_width(mod):
    """Probe width (in items) of a site while the market listens; 0 = no probe."""
    return int(_MK["width"].get(id(mod), 0)) if _MK.get("on") else 0


# ---- WALLET (v16): per-token budgets that are strict by construction ------------------------------------------------
# State of ONE forward pass (reset by a hook on the token embedding):
#   bal [T]     what each token may still spend above the quotas, in parameters. Starts at b0 = ceiling - census(quotas).
#   store       the number of items every site sold to every token. A site that is executed AGAIN inside the same forward
#               (gradient checkpointing recomputes layers during the backward pass, in reverse order) replays its entry:
#               the choice of a token is made once, in depth order, and never re-made.
# A site s with quota n0, price th (a threshold on its log-score u), per-item cost c and window w sells to token t
#       n(t) = clamp( #{u >= th - pace(t)},  n0 - w,  min(n0 + w, n0 + floor((bal(t) + rho * R_after(s)) / c)) )
#   pace(t) = beta * sigma_s * clip(bal(t) / b_ref, -1, 1)   a token with savings bids lower, a token in debt higher
#   R_after(s) = what the sites AFTER s can still refund (c * (n0 - lo) summed): a token may run a debt of at most rho
#                times that, and the cap on n makes every later site collect it. After the last site bal >= 0, i.e.
#                   parameters touched by token t = census(quotas) + b0 - bal(t)  <=  ceiling      for EVERY token.
_WAL = globals().get("_WAL") or {"on": False, "sites": {}, "T": 0, "B": 0, "L": 0, "bal": None, "store": {}, "b0": 0.0,
                                 "omega": 0.0, "beta": 1.0, "rho": 0.5, "bref": 1.0, "calib": None, "fid": 0}


def _wal_begin(B, L, device):
    """A new forward pass: fresh wallets. (v17: first, what the tokens of the last TRAINING forward left unspent.)"""
    if _WAL.get("bal") is not None and _WAL.get("train_fwd"):
        _WAL["unspent_sum"] = float(_WAL.get("unspent_sum", 0.0)) + float(_WAL["bal"].mean())
        _WAL["unspent_n"] = int(_WAL.get("unspent_n", 0)) + 1
    _WAL["train_fwd"] = False
    _WAL["fid"] += 1
    _WAL["B"], _WAL["L"], _WAL["T"] = int(B), int(L), int(B) * int(L)
    _WAL["store"] = {}
    _WAL["bal"] = torch.full((int(B) * int(L),), float(_WAL["b0"]), dtype=torch.float64, device=device) if _WAL["on"] else None


def _wal_site(mod, tag=""):
    st = _WAL["sites"].get((id(mod), tag)) if (_WAL["on"] or _WAL["calib"] is not None) else None
    return st


def _wal_armed(mod):
    """True when this module must run its masked (any-count) path even without gradients: the wallet sells a
    different number of items to every token (or the calibration is reading where its quota cuts the scores)."""
    return bool(_WAL["on"] or _WAL["calib"] is not None) and getattr(mod, "ns_wal", False)


def _wal_open(omega):
    """Set the window (0 = every token buys the quota ... 1 = the full range) and, for every site, what the sites
    AFTER it can still refund — the collateral of the credit line."""
    _WAL["omega"] = float(max(0.0, min(1.0, omega)))
    tot, vol = 0.0, 0.0
    for st in reversed(_WAL.get("order", [])):
        st["after"] = tot
        n0, w = int(st["n0"]()), int(_WAL["omega"] * st["span"])
        lo, hi = max(int(st["lo"]), n0 - w), min(int(st["hi"]), n0 + w)
        if w > 0 and hi > lo:
            tot += float(st["cost"]) * (n0 - lo)
            vol += float(st["cost"]) * (hi - lo)
    _WAL["vol"] = vol
    _WAL["bref"] = max(1.0, float(_WAL.get("bref_frac", 0.1)) * vol)


def _wal_receipt():
    """What the tokens of the LAST forward spent above the quotas: (min, mean, max) in parameters, and the lowest
    balance — which must not be negative: that is the per-token guarantee."""
    bal = _WAL.get("bal")
    if not _WAL["on"] or bal is None or bal.numel() == 0:
        return None
    sp = float(_WAL["b0"]) - bal
    if float(bal.min()) < -0.5:                           # cannot happen while every site is executed; if it does,
        _WAL["violations"] = int(_WAL.get("violations", 0)) + 1       # the credit line is closed (then it cannot happen at all)
        _WAL["rho"] = 0.0
    return float(sp.min()), float(sp.mean()), float(sp.max()), float(bal.min())


def _wal_emb_hook(mod, inp, out):
    ids = inp[0] if isinstance(inp, (tuple, list)) else inp
    if torch.is_tensor(ids) and ids.dim() == 2:
        _wal_begin(ids.shape[0], ids.shape[1], ids.device)
    return None


def _wal_build(span_ffn=0.5, more_blocks=3, more_heads=4):
    """Register every site that can sell a per-token number of items. Nothing changes until the window opens."""
    _WAL["sites"], _WAL["order"] = {}, []
    kind_of = lambda n: ".".join(n.split(".")[-2:])

    def add(key, kind, mod, tag, n0, cost, lo, hi, span):
        if hi <= lo or span <= 0:
            return
        _WAL["sites"][(id(mod), tag)] = {
            "key": key, "kind": kind, "mod": mod, "tag": tag, "n0": n0, "cost": float(cost), "lo": int(lo), "hi": int(hi),
            "span": int(span), "th": 0.0, "sig": 1.0, "after": 0.0, "probe": torch.nn.Parameter(torch.zeros((), device=DEV)),
            "acc": {"g": 0.0, "steps": 0, "mass": 0.0, "dem": 0.0, "n": 0.0, "clip": 0.0, "fwd": 0}}

    for name, m in PRISM_MODULES:
        if not (hasattr(m, "ns_B") and getattr(m, "ns_on", False)) or int(getattr(m, "ns_k2", 0) or 0) > 0 or _ns_prog(m)[0]:
            continue
        k0 = int(m.ns_k)
        sp = int(span_ffn * k0)
        add("ffn:" + name, "FFN neurons", m, "", (lambda m=m: int(m.ns_k)), 3 * m.d_model, max(1, k0 - sp), min(m.D - 1, k0 + sp), sp)
    for name, m in RESONA_LINEARS:
        if getattr(m, "sh_U", None) is None:
            continue
        m.ns_wal = True
        t0 = int(m.topk)
        add("rl:" + name, kind_of(name) + " row-blocks", m, "", (lambda m=m: int(m.topk)), int(m.r) * int(m.in_f), 0,
            min(int(m.B) - 1, t0 + int(more_blocks)), int(more_blocks))
    for name, m in RESONA_SECTIONED:
        for t in ("q", "z"):
            if hasattr(m, f"{t}_B") and getattr(m, f"{t}_sh_U", None) is not None:
                m.ns_wal = True
                t0 = int(m.topk)
                add("sp:" + name + ":" + t, f"in_proj_qkv {t} row-blocks", m, t, (lambda m=m: int(m.topk)),
                    int(getattr(m, f"{t}_r")) * int(m.weight.shape[1]), 0, min(int(getattr(m, f"{t}_B")) - 1, t0 + int(more_blocks)), int(more_blocks))
        pol = getattr(m, "morm", None)
        if pol is not None and getattr(pol, "strict_m", None) is not None and (id(pol), "") not in _WAL["sites"]:
            m0 = int(pol.strict_m)
            add("morm:" + name, "memory heads", pol, "", (lambda p=pol: int(p.strict_m)), int(m.v_head_rows) * int(m.weight.shape[1]), 0,
                min(int(pol.H) - 1, m0 + int(more_heads)), int(more_heads))
    emb = model.get_input_embeddings()
    if getattr(emb, "_wal_hooked", None) is not None:
        try:
            emb._wal_hooked.remove()
        except Exception:
            pass
    emb._wal_hooked = emb.register_forward_hook(_wal_emb_hook)
    return list(_WAL["sites"].values())


def _wal_state():
    """What a saved state needs to serve the same per-token purchases again."""
    return {"on": bool(_WAL["on"]), "omega": float(_WAL["omega"]), "b0": float(_WAL["b0"]), "beta": float(_WAL["beta"]),
            "rho": float(_WAL["rho"]), "tau": float(_WAL.get("tau", 0.5)), "bref_frac": float(_WAL.get("bref_frac", 0.1)),
            "order": [st["key"] for st in _WAL.get("order", [])],
            "sites": {st["key"]: {"th": float(st["th"]), "sig": float(st["sig"]), "lo": int(st["lo"]), "hi": int(st["hi"]),
                                  "span": int(st["span"])} for st in _WAL.get("order", [])}}


def _wal_load(ws):
    """Put a saved wallet back (prices, windows, order). Returns the number of sites restored."""
    by_key = {st["key"]: st for st in _WAL["sites"].values()}
    _WAL["order"] = []
    for k_ in ws.get("order", []):
        st, sv = by_key.get(k_), ws["sites"].get(k_)
        if st is None or sv is None:
            continue
        st.update({"th": float(sv["th"]), "sig": float(sv["sig"]), "lo": int(sv["lo"]), "hi": int(sv["hi"]), "span": int(sv["span"])})
        _WAL["order"].append(st)
    keep = {id(st) for st in _WAL["order"]}
    _WAL["sites"] = {k_: st for k_, st in _WAL["sites"].items() if id(st) in keep}
    for k_ in ("b0", "beta", "rho", "tau", "bref_frac"):
        if k_ in ws:
            _WAL[k_] = float(ws[k_])
    _WAL["on"] = bool(ws.get("on", False)) and bool(_WAL["order"])
    _wal_open(float(ws.get("omega", 0.0)))
    return len(_WAL["order"])


def _wal_pick(mod, tag, topu):
    """topu [T, H]: the site's H best log-scores of every token, best first (H >= the site's upper bound).
    Returns None (the site sells its quota to everybody) or (keep [T, H] 0/1 — what each token buys, wm [T, H] —
    weights of the items AT the price, for the price probe: zero outside the window, thr [T] — the price each token saw)."""
    st = _wal_site(mod, tag)
    if st is None:
        return None
    T_, H = int(topu.shape[0]), int(topu.shape[1])
    n0 = int(st["n0"]())
    if _WAL["calib"] is not None:                         # calibration: where does this site's quota cut the scores?
        if 1 <= n0 < H:
            _WAL["calib"].setdefault(st["key"], []).append((0.5 * (topu[:, n0 - 1] + topu[:, n0])).detach().float().cpu())
        return None
    w = int(float(_WAL["omega"]) * st["span"])
    if not _WAL["on"] or w <= 0 or T_ != _WAL["T"] or _WAL["bal"] is None:
        return None
    lo, hi = max(int(st["lo"]), n0 - w), min(int(st["hi"]), n0 + w, H)
    if hi <= lo:
        return None
    key = (st["key"], _WAL["fid"])
    rank = torch.arange(H, device=topu.device)[None, :]
    got = _WAL["store"].get(key)
    if got is None:                                        # DECIDE (first execution of this site in this forward)
        bal = _WAL["bal"]
        thr = float(st["th"]) - float(_WAL["beta"]) * float(st["sig"]) * (bal / float(_WAL["bref"])).clamp(-1.0, 1.0).float()
        dem = (topu[:, :hi] >= thr[:, None]).sum(-1)                       # what the token wants at this price (<= hi)
        afford = n0 + torch.floor((bal + float(_WAL["rho"]) * float(st["after"])) / float(st["cost"]))
        n = torch.minimum(dem.clamp(min=lo).double(), afford.clamp(min=lo, max=hi)).long()
        _WAL["bal"] = bal - float(st["cost"]) * (n - n0).double()
        _WAL["store"][key] = (n, thr)
        if mod.training:                                   # prices are learned from training passes only
            _WAL["train_fwd"] = True
            a_ = st["acc"]
            a_["dem"] += float((dem.clamp(min=lo).float() - n0).mean()) * float(st["cost"])
            a_["n"] += float(n.float().mean())
            a_["clip"] += float((afford < dem.double()).float().mean())
            a_["fwd"] += 1
    else:                                                  # REPLAY (a recompute inside the same forward)
        n, thr = got
    keep = (rank < n[:, None]).to(topu.dtype)
    win = ((rank >= lo) & (rank < hi)).to(topu.dtype)
    tau = max(float(st["sig"]) * float(_WAL.get("tau", 0.5)), 1e-6)
    sg = torch.sigmoid((topu - thr[:, None]) / tau)
    wm = sg * (1.0 - sg) * win
    if got is None and mod.training:
        st["acc"]["mass"] += float(wm.sum(-1).mean())
    return keep, wm, thr


def _mk_blocks(mod, sel, logits, topi, tag=""):
    """Market probes on the block choice of a routed projection: sel [T, B] is the hard choice (+ straight-through).
    Value unchanged (both probes are zero); d loss / d probe = what one more / one less row-block would change."""
    st = _wal_site(mod, tag)
    if st is not None:                                     # WALLET: every token buys its own number of row-blocks
        H = min(int(logits.shape[1]), int(st["hi"]) + 1)
        topu, idx = logits.detach().float().topk(H, dim=-1)
        pick = _wal_pick(mod, tag, topu)
        if pick is not None:
            keep, wm, _ = pick
            hard0 = torch.zeros_like(sel).scatter(1, topi, torch.ones(topi.shape, dtype=sel.dtype, device=sel.device))
            new = torch.zeros_like(sel).scatter(1, idx, keep.to(sel.dtype)) + (sel - hard0)      # + the router's straight-through part
            if mod.training and torch.is_grad_enabled():
                new = new + st["probe"].to(sel.dtype) * torch.zeros_like(sel).scatter(1, idx, wm.to(sel.dtype))
            return new
        return sel
    if not _mk_width(mod) or not mod.training:
        return sel
    pr = _MK["probes"].get(id(mod))
    if pr is None:
        return sel
    k, B = int(topi.shape[1]), int(logits.shape[1])
    one = lambda ix: torch.zeros_like(sel).scatter(1, ix, torch.ones(ix.shape, dtype=sel.dtype, device=sel.device))
    if k < B:
        sel = sel + pr[0].to(sel.dtype) * one(logits.detach().topk(k + 1, dim=-1).indices[:, k:k + 1])
    if k >= 1:
        sel = sel - pr[1].to(sel.dtype) * one(topi[:, k - 1:k])
    return sel


# ---- strict memory writes (v16, the law): a policy with strict_m set writes EXACTLY strict_m heads per token
if not hasattr(MoRM2Policy, "_forward_elastic"):
    MoRM2Policy._forward_elastic = MoRM2Policy.forward


def _morm_forward_v16(self, phi, obs=None, m=None):
    """strict_m is set: EXACTLY strict_m heads per token write exactly (the best scores); every other head writes its
    shadow. In training the mask carries the straight-through path to the critic and the two market probes."""
    sm = getattr(self, "strict_m", None)
    if sm is None or m is not None:
        return MoRM2Policy._forward_elastic(self, phi, obs, m)
    T, H = phi.shape[0], int(self.H)
    mm = max(1, min(int(sm), H))
    if mm >= H:
        mask = phi.new_ones(T, H)
        self.last_mask, self.last_m, self.last_scores = mask.detach(), None, None
        return mask
    # v16.2: the straight-through mask lives in LOG-score space. In score space d soft / d s = sigmoid' x exp(s) / tau:
    # the m-th head always sits at the line (sigmoid' = 1/4), so a head with s ~ 30 sent a gradient x 1e12 into the
    # critic and the hidden states -> inf -> NaN in every layer below (v16.2, layers 45 and 29). In log space the factor
    # is at most 1/(4 tau). The choice itself (the order of the scores) is the same.
    lS = _morm_logscores(self, phi, obs)                   # [T, H] log predicted influence (capped)
    L = lS.exp()
    train = bool(self.training and torch.is_grad_enabled())
    st = _wal_site(self, "")
    if st is not None:                                     # WALLET: every token buys its own number of exact writes
        Hh = min(H, int(st["hi"]) + 1)
        topu, idx = lS.detach().float().topk(Hh, dim=-1)
        pick = _wal_pick(self, "", topu)
        if pick is not None:
            keep, wm, thr = pick
            hard = torch.zeros_like(L).scatter(-1, idx, keep.to(L.dtype))
            if train:
                soft = torch.sigmoid((lS - thr[:, None].to(lS.dtype)) / max(float(self.ste_tau), 1e-6))
                hard = hard + (soft - soft.detach()) + st["probe"] * torch.zeros_like(L).scatter(-1, idx, wm.to(L.dtype))
            self.last_mask = hard.detach()
            self.last_m = (hard.detach() > 0.5).sum(-1)
            self.last_scores = L.detach()
            return hard
    dk = min(_mk_width(self), H - mm, mm) if train else 0
    topv, ti = lS.topk(mm + dk, dim=-1)
    one = lambda ix: torch.zeros_like(L).scatter(-1, ix, torch.ones(ix.shape, dtype=L.dtype, device=L.device))
    hard = one(ti[:, :mm])
    if train:
        soft = torch.sigmoid((lS - topv[:, mm - 1:mm].detach()) / max(float(self.ste_tau), 1e-6))
        hard = hard + (soft - soft.detach())
        if dk:
            _pp, _pm = _MK["probes"][id(self)]
            hard = hard + _pp * one(ti[:, mm:mm + dk]) - _pm * one(ti[:, mm - dk:mm])
    self.last_mask = hard.detach()
    self.last_m = (hard.detach() > 0.5).sum(-1)
    self.last_scores = L.detach()
    return hard


MoRM2Policy.forward = _morm_forward_v16

V16_SCORE_CAP = float(globals().get("V16_SCORE_CAP", 80.0))


def _morm_logscores(self, phi, obs=None):
    s = self.critic_phi(phi)
    if obs is not None and self.use_obs:
        s = s + torch.einsum("thf,hf->th", obs.float(), self.critic_obs.weight.float())
    return s.float().clamp(max=V16_SCORE_CAP)


def _morm_scores_v16(self, phi, obs=None):
    """The critic's predicted influence exp(s), with s capped at V16_SCORE_CAP: exp(s) overflowed to inf on some tokens
    (s > 88.7 in fp32); the forward stayed finite (the straight-through term is 1 - 1) but its backward was 0 x inf = NaN,
    which then ran down every layer below (v16.1 / v16.2, layer 45)."""
    s = self.critic_phi(phi)
    if obs is not None and self.use_obs:
        s = s + torch.einsum("thf,hf->th", obs.float(), self.critic_obs.weight.float())
    return s.float().clamp(max=V16_SCORE_CAP).exp()


MoRM2Policy.scores = _morm_scores_v16

if "_morm_heads_v10" not in globals():
    _morm_heads_v10 = _morm_heads


def _morm_heads(sp) -> int:                               # the census asks this function
    pol = sp.morm
    sm = None if pol is None else getattr(pol, "strict_m", None)
    if sm is not None:
        return max(1, min(int(sm), int(sp.n_v_heads)))
    return _morm_heads_v10(sp)


def _ns_fp32(x):
    """Scores are computed in fp32 in training AND in serving (the Trainer's bf16 autocast is switched off here), so
    the neurons picked while training are the neurons picked when serving."""
    ac = getattr(torch, "autocast", None)
    if ac is None or x.device.type != "cuda":
        return contextlib.nullcontext()
    return ac(device_type="cuda", enabled=False)


class _StopForward(Exception):
    pass


def _eigh(Mx):
    """Symmetric eigendecomposition; fp32 first, fp64 if the fp32 solver does not converge."""
    try:
        return torch.linalg.eigh(Mx)
    except Exception as _e:
        if is_oom(_e):
            raise
        w, V = torch.linalg.eigh(Mx.double())
        return w.to(Mx.dtype), V.to(Mx.dtype)


def _first(t):
    return t[0] if isinstance(t, (tuple, list)) else t


# ------------------------------------------------------------------ served path (monkeypatch, v10 path untouched)
def _ns_down(prism, a):
    """a [T, D] fp32 -> a @ W_down^T, through the substrate dtype like every other served GEMM."""
    Wd = prism.W_down
    return (a.to(Wd.dtype) @ Wd.t()).float()


def _ns_scores(prism, x_f32, B, Ag, Au, mu, dn):
    z = x_f32 @ B
    return (F.silu(z @ Ag.t()) * (z @ Au.t()) - mu).abs() * dn


def _ns_hat(self, z, r1=0):
    """Lens estimate of every neuron's activation from the lens code z; r1 > 0: the coarse look (strongest r1 directions)."""
    if r1:
        return F.silu(z[:, -r1:] @ self.ns_Ag[:, -r1:].t()) * (z[:, -r1:] @ self.ns_Au[:, -r1:].t())
    return F.silu(z @ self.ns_Ag.t()) * (z @ self.ns_Au.t())


def _ns_prog(self):
    """(r1, n_s) when second sight is on for this layer, else (0, D)."""
    r1 = int(getattr(self, "ns_r1", 0) or 0)
    if 0 < r1 < int(self.ns_B.shape[1]):
        return r1, max(1, min(int(getattr(self, "ns_ns", self.D)), self.D))
    return 0, self.D


def _ns_select(self, xf, a, train):
    """The three-tier choice for one chunk. Returns (M1, G, fit):
    M1 [T, D]  hard 0/1 mask of the EXACT neurons (in training it also carries the straight-through credit path),
    G  [T, D]  what the GHOST neurons say, gain * (a^ - mu) on their positions (None when the layer has no ghosts),
    fit        (num, den) of the lens regression (None outside training / when the predictor is frozen)."""
    T_, D = xf.shape[0], self.D
    _zeros = lambda: torch.zeros(T_, D, dtype=torch.float32, device=xf.device)
    _put = lambda ix: _zeros().scatter(1, ix, torch.ones(ix.shape, dtype=torch.float32, device=xf.device))
    with _ns_fp32(xf):
        xd = xf.detach()                                   # the selection is a function of the stream, not a route for its gradient
        z = xd @ self.ns_B
        a_hat = _ns_hat(self, z)
        base = (a_hat.detach() - self.ns_mu).abs() * self.ns_dn
        head = bool(getattr(self, "ns_hd", False))
        lw, boost = None, None
        if head:
            lw = self.ns_w[None, :]
            if self.ns_Qc.shape[1] > 0:
                lw = lw + (z.detach() @ self.ns_Pc) @ self.ns_Qc.t()
            lw = lw.clamp(-V15_W_CLAMP, V15_W_CLAMP)
            boost = torch.exp(lw.detach())
        s = base * boost if head else base
        r1, n_s = _ns_prog(self)
        a1, short = None, None
        if r1:                                             # SECOND SIGHT: coarse look at everyone, sharp look at the shortlist
            a1 = _ns_hat(self, z, r1)
            s1 = (a1.detach() - self.ns_mu).abs() * self.ns_dn
            if head:
                s1 = s1 * boost
            short = _put(s1.topk(n_s, dim=-1).indices)
            s = s * short                                  # a neuron outside the shortlist was never looked at sharply
        k1 = int(self.ns_k)
        k2 = int(getattr(self, "ns_k2", 0) or 0)
        st = _wal_site(self, "") if (k2 == 0 and not r1) else None
        if st is not None:                                 # WALLET: every token buys its own number of exact neurons
            Hh = min(D, int(st["hi"]) + 1)
            topv, idx = s.topk(Hh, dim=-1)
            pick = _wal_pick(self, "", torch.log(topv.clamp_min(1e-30)))
            if pick is not None:
                keep, wm, thr = pick
                M1 = _zeros().scatter(1, idx, keep)
                fit = None
                if train:
                    if head and getattr(self, "ns_st", False):     # counterfactual credit around the price each token saw
                        logs = torch.log(base.clamp_min(1e-30)) + lw
                        soft1 = torch.sigmoid((logs - thr[:, None]) / float(V15_ST_TEMP))
                        M1 = M1 + (soft1 - soft1.detach())
                    M1 = M1 + st["probe"] * _zeros().scatter(1, idx, wm)
                    if getattr(self, "ns_fit", False) and a_hat.requires_grad:
                        w2 = self.ns_dn.pow(2)[None, :]
                        ad = a.detach()
                        fit = (((a_hat - ad).pow(2) * w2).sum(), ((ad - self.ns_mu).pow(2) * w2).sum())
                return M1, None, fit
        dk = min(_mk_width(self), max(0, D - k1), k1) if (train and k2 == 0) else 0      # market probes: layers without ghosts
        topv, idx = s.topk(min(k1 + k2 + dk, D), dim=-1)
        idx_next = idx[:, k1:k1 + dk] if dk else None
        if dk:
            topv, idx = topv[:, :k1], idx[:, :k1]
        M1 = _put(idx[:, :k1])
        M2 = _put(idx[:, k1:]) if k2 > 0 else None
        fit = None
        if train:
            if head and getattr(self, "ns_st", False):     # counterfactual credit (value unchanged, gradient added)
                logs = torch.log(base.clamp_min(1e-30)) + lw
                soft1 = torch.sigmoid((logs - torch.log(topv[:, k1 - 1:k1].clamp_min(1e-30))) / float(V15_ST_TEMP))
                if short is not None:
                    soft1 = soft1 * short
                st1 = soft1 - soft1.detach()
                M1 = M1 + st1
                if M2 is not None:
                    soft2 = torch.sigmoid((logs - torch.log(topv[:, -1:].clamp_min(1e-30))) / float(V15_ST_TEMP))
                    if short is not None:
                        soft2 = soft2 * short
                    M2 = M2 + (soft2 - soft2.detach()) - st1
            if dk:                                         # value unchanged (both probes are 0); their gradients are the prices
                _pp, _pm = _MK["probes"][id(self)]
                M1 = M1 + _pp * _put(idx_next) - _pm * _put(idx[:, k1 - dk:k1])
            if getattr(self, "ns_fit", False) and a_hat.requires_grad:
                w2 = self.ns_dn.pow(2)[None, :]
                ad = a.detach()
                den = ((ad - self.ns_mu).pow(2) * w2).sum()
                num = ((a_hat - ad).pow(2) * w2).sum()
                if a1 is not None:                         # the coarse look must stay a good shortlister while the predictor moves
                    num, den = num + ((a1 - ad).pow(2) * w2).sum(), den * 2.0
                fit = (num, den)
        G = None
        if M2 is not None:
            # in training the ghosts' estimate is recomputed WITH the gradient path to the stream (and to the predictor):
            # a voiced neuron is part of the function, not only of the choice
            ag = _ns_hat(self, xf @ self.ns_B) if train else a_hat
            G = self.ns_gc[None, :] * (ag - self.ns_mu) * M2
    return M1, G, fit


def _ns_forward(self, h):
    """Served neuron-lens FFN. The exact neurons are the only ones whose gate / up rows the result depends on, the
    ghosts the only further ones whose down column it depends on; the notebook computes all of it through masks
    over the dense GEMMs (same numbers as row gathers, simpler to verify — and it is what makes the counterfactual
    credit of the neurons that were NOT read available while training).
    Returns (y, fit) with fit = relative error of the lens predictor on this call (None outside training)."""
    outs, num, den = [], None, None
    train = bool(self.training and torch.is_grad_enabled())
    for c0 in range(0, h.shape[0], NS_CHUNK):
        x = h[c0:c0 + NS_CHUNK]
        xf = x.float()
        xw = _dm(x, self.W_gate)
        a = F.silu((xw @ self.W_gate).float()) * (xw @ self.W_up).float()
        own = getattr(self, "ns_oi", None)
        own = own if (own is not None and own.numel() > 0 and getattr(self, "ns_own", False)) else None
        if own is not None:                                # the student's own copies of these neurons' gate / up rows
            wd = self.W_gate.dtype
            a = a.index_copy(1, own, F.silu((xw @ self.ns_og.to(wd)).float()) * (xw @ self.ns_ou.to(wd)).float())
        if train:
            M1, G, fit = _ns_select(self, xf, a, True)
        else:
            with torch.no_grad():
                M1, G, fit = _ns_select(self, xf, a, False)
        if fit is not None:
            num = fit[0] if num is None else num + fit[0]
            den = fit[1] if den is None else den + fit[1]
        act = (a - self.ns_mu) * M1
        if G is not None:
            act = act + G
        _acc = getattr(self, "ns_acc", None)
        if _acc is not None and not train:                 # calibration: how much served output flows through each neuron
            _acc += (act.detach() * self.ns_dn).pow(2).sum(0)
        # differentiable in x (through the exact neurons and the ghosts), in the shadow (ns_U / ns_M / ns_b), in the
        # ghost gains and the predictor (through the ghosts) and — in training — in the masks (credit for the head)
        if own is None:
            y = _ns_down(self, act)
        else:                                              # ... and of their down columns
            wd = self.W_down.dtype
            y = _ns_down(self, act.index_fill(1, own, 0.0)) + (act.index_select(1, own).to(wd) @ self.ns_od.to(wd).t()).float()
        y = y + ((xf @ self.ns_M.t()) @ self.ns_U.t()).float() + self.ns_b
        outs.append(y.to(h.dtype))
    y = outs[0] if len(outs) == 1 else torch.cat(outs, 0)
    return y, (None if num is None else num / den.clamp_min(1e-30))


if not hasattr(PrismMoeMLP, "_forward_2d_v10"):
    PrismMoeMLP._forward_2d_v10 = PrismMoeMLP._forward_2d


def _forward_2d_ns(self, h, mode=None, k=8, use_completion=True, return_aux=False):
    m = mode if mode is not None else getattr(self, "default_mode", "soft")
    if m == "topk" and getattr(self, "ns_on", False):
        y, fit = _ns_forward(self, h)
        aux = {"ns": True}
        if fit is not None:
            aux["ns_fit"] = fit                    # the trainer adds it to the loss (it reaches B / Ag / Au only)
        if (not torch.is_grad_enabled()) or getattr(self, "_store_aux", False):
            self.last_aux = aux
        return y, aux
    return PrismMoeMLP._forward_2d_v10(self, h, mode, k, use_completion, return_aux)


PrismMoeMLP._forward_2d = _forward_2d_ns

if "_prism_active_v10" not in globals():
    _prism_active_v10 = _prism_active


def _prism_active(m) -> int:                      # the census asks this function: lens layers report the lens budget
    if getattr(m, "ns_on", False):
        return int(m.ns_active)
    return _prism_active_v10(m)


def _ns_fixed(d, D, r, R):
    return d * r + 2 * D * r + 2 * D + 2 * R * d + d          # lens (B, Ag, Au) + mu + norms + shadow (U, M, b)


def _ns_head_cost(D, r, rc):
    return D + rc * (r + D)                                  # w [D] + Pc [r, rc] + Qc [D, rc] — all read per token


def _ns_lens_rows(D, r, r1=0, n_s=0):
    """Parameters of Ag / Au read per token: all rows at full rank, or (second sight) all rows at rank r1 and the
    shortlist at the remaining r - r1."""
    if 0 < r1 < r:
        return 2 * D * r1 + 2 * min(int(n_s), D) * (r - r1)
    return 2 * D * r


def _ns_cost(m, k=None, k2=None):
    """Touched parameters per token of a lens layer: B + the lens rows actually read + mu, ||w_down|| + shadow (U, M, b)
    + 3*d per exact neuron + (d per ghost + the D ghost gains) + the consequence head when it is on."""
    d, D, r, R = m.d_model, m.D, int(m.ns_B.shape[1]), int(m.ns_U.shape[1])
    k = int(m.ns_k) if k is None else int(k)
    k2 = int(getattr(m, "ns_k2", 0) or 0) if k2 is None else int(k2)
    r1, n_s = _ns_prog(m)
    c = d * r + _ns_lens_rows(D, r, r1, n_s) + 2 * D + 2 * R * d + d + 3 * d * k
    if k2 > 0:
        c += d * k2 + D
    if getattr(m, "ns_hd", False):
        c += _ns_head_cost(D, r, int(m.ns_Qc.shape[1]))
    return int(c)


def _ns_set(m, nm, val):
    """(Re)place a lens tensor as an fp32 buffer, whatever it was before (buffer or trained parameter)."""
    if nm in getattr(m, "_parameters", {}):
        del m._parameters[nm]
    if nm in getattr(m, "_buffers", {}):
        m._buffers[nm] = val.detach().float().contiguous()
    else:
        m.register_buffer(nm, val.detach().float().contiguous())


def _ns_head_install(m, w, Pc, Qc, on=True):
    _ns_set(m, "ns_w", w)
    _ns_set(m, "ns_Pc", Pc)
    _ns_set(m, "ns_Qc", Qc)
    m.ns_hd = bool(on)


def _ns_own_install(m, oi, og=None, ou=None, od=None, on=True):
    """Give layer m its own copies of the neurons oi (LongTensor [n]). og / ou / od [d, n] default to the dense values."""
    oi = oi.to(m.W_gate.device).long()
    if "ns_oi" in getattr(m, "_buffers", {}):
        m._buffers["ns_oi"] = oi
    else:
        m.register_buffer("ns_oi", oi)
    _ns_set(m, "ns_og", m.W_gate.index_select(1, oi) if og is None else og)
    _ns_set(m, "ns_ou", m.W_up.index_select(1, oi) if ou is None else ou)
    _ns_set(m, "ns_od", m.W_down.index_select(1, oi) if od is None else od)
    m.ns_own = bool(on)


def _ns_install(prism, fit_):
    for nm in ("B", "Ag", "Au", "mu", "dn", "U", "M", "b"):
        _ns_set(prism, "ns_" + nm, fit_[nm])
    prism.ns_hd = False
    prism.ns_r1, prism.ns_ns, prism.ns_k2 = 0, int(prism.D), 0          # second sight / ghosts: off until a cell turns them on
    prism.ns_k, prism.ns_r, prism.ns_R, prism.ns_active = int(fit_["k"]), int(fit_["r"]), int(fit_["R"]), int(fit_["active"])


# ------------------------------------------------------------------ a state that already carries a lens (v12 state)
_ns_src = {**STATE.get("buffers", {}), **STATE.get("trainable", {})} if "STATE" in globals() else {}   # v13 trains the shadows
_ns_saved = [n for n, _ in PRISM_MODULES if f"{n}.ns_B" in _ns_src]
NS_LOADED = False
if _ns_saved and globals().get("NS_LOAD", True):
    for n, m in PRISM_MODULES:
        if f"{n}.ns_B" not in _ns_src:
            continue
        _a = STATE["meta"]["prism"].get(n, {})
        _ns_install(m, {**{nm: _ns_src[f"{n}.ns_{nm}"].to(DEV) for nm in ("B", "Ag", "Au", "mu", "dn", "U", "M", "b")},
                        "k": _a["ns_k"], "r": _a.get("ns_r", 0), "R": _a.get("ns_R", 0), "active": _a["ns_active"]})
        m.ns_on = bool(_a.get("ns_on", False))
        if f"{n}.ns_w" in _ns_src:                                   # a v15 state: the consequence head
            _ns_head_install(m, _ns_src[f"{n}.ns_w"].to(DEV), _ns_src[f"{n}.ns_Pc"].to(DEV),
                             _ns_src[f"{n}.ns_Qc"].to(DEV), on=bool(_a.get("ns_hd", True)))
        if f"{n}.ns_gc" in _ns_src:                                  # ... its ghost gains
            _ns_set(m, "ns_gc", _ns_src[f"{n}.ns_gc"].to(DEV))
        if f"{n}.ns_oi" in _ns_src:                                  # a v16 state: the student's own neurons
            _ns_own_install(m, _ns_src[f"{n}.ns_oi"], _ns_src[f"{n}.ns_og"].to(DEV), _ns_src[f"{n}.ns_ou"].to(DEV),
                            _ns_src[f"{n}.ns_od"].to(DEV), on=bool(_a.get("ns_own", True)))
        m.ns_r1, m.ns_ns = int(_a.get("ns_r1") or 0), int(_a.get("ns_ns") or m.D)     # ... and its second-sight plan
        m.ns_k2 = int(_a.get("ns_k2") or 0) if hasattr(m, "ns_gc") else 0
    NS_LOADED = True
    _ws = STATE.get("config", {}).get("V16_WALLET_STATE")
    if isinstance(_ws, dict) and _ws.get("order"):               # a state saved with the wallet open
        _wal_build()
        print(f"wallet restored from the state: {_wal_load(_ws)} sites, window {100 * _WAL['omega']:.0f}%, "
              f"{'on' if _WAL['on'] else 'off'}")
    del _ns_src
    NS_KEPT_RESTORED = any(bool(getattr(m, 'ns_on', False)) for _, m in PRISM_MODULES)
    print(f"neuron lens restored from the state: {sum(bool(getattr(m, 'ns_on', False)) for _, m in PRISM_MODULES)}"
          f"/{len(PRISM_MODULES)} layers on | second sight on {sum(1 for _, m in PRISM_MODULES if hasattr(m, 'ns_B') and _ns_prog(m)[0])}"
          f" | ghosts on {sum(1 for _, m in PRISM_MODULES if int(getattr(m, 'ns_k2', 0) or 0) > 0)} | consequence head on "
          f"{sum(bool(getattr(m, 'ns_hd', False)) for _, m in PRISM_MODULES)} | own neurons on "
          f"{sum(bool(getattr(m, 'ns_own', False)) for _, m in PRISM_MODULES)} — nothing is re-fitted (NS_LOAD=False to fit again)")


# ------------------------------------------------------------------ calibration rows
_calib = teach_blocks if "teach_blocks" in globals() else blocks
_nb = min(int(NS_BLOCKS), _calib.shape[0])
_bidx = sorted({int(round(v)) for v in torch.linspace(0, _calib.shape[0] - 1, _nb).tolist()})
_gen = torch.Generator().manual_seed(12)
_pos = [torch.randperm(_calib.shape[1], generator=_gen)[:min(int(NS_TOK), _calib.shape[1])].sort().values for _ in _bidx]
_n_tok = sum(int(p.numel()) for p in _pos)
_blk_of_row = torch.cat([torch.full((int(p.numel()),), i) for i, p in enumerate(_pos)])
_n_val_blk = max(1, int(round(NS_VAL_FRAC * len(_bidx))))
_fit = (_blk_of_row < len(_bidx) - _n_val_blk).nonzero().flatten().to(DEV)       # whole DOCUMENTS are held out
_val = (_blk_of_row >= len(_bidx) - _n_val_blk).nonzero().flatten().to(DEV)
_L = len(PRISM_MODULES)


@torch.no_grad()
def _ffn_inputs(li):
    """The FFN input of layer li on the calibration rows, with the model as it is served RIGHT NOW (fp32, GPU)."""
    _, prism = PRISM_MODULES[li]
    got = {}

    def pre(mod, args):
        got["x"] = _first(args)
        raise _StopForward()
    hk = prism.register_forward_pre_hook(pre)
    X = torch.empty(_n_tok, prism.d_model, device=DEV)
    row = 0
    try:
        for bi, p in zip(_bidx, _pos):
            try:
                model(input_ids=_calib[bi:bi + 1].to(DEV))
            except _StopForward:
                pass
            X[row:row + p.numel()] = got["x"].reshape(-1, prism.d_model).index_select(0, p.to(DEV)).float()
            row += int(p.numel())
    finally:
        hk.remove()
    return X


@torch.no_grad()
def _dense_acts(prism, X):
    """Dense neuron activations A [N, D] and the dense FFN output Y [N, d] on X (the layer's own dense twin)."""
    A = torch.empty(X.shape[0], prism.D, device=DEV)
    Y = torch.empty(X.shape[0], prism.W_down.shape[0], device=DEV)
    for c in range(0, X.shape[0], NS_CHUNK):
        x = X[c:c + NS_CHUNK].to(prism.W_gate.dtype)
        a = F.silu((x @ prism.W_gate).float()) * (x @ prism.W_up).float()
        A[c:c + NS_CHUNK] = a
        Y[c:c + NS_CHUNK] = _ns_down(prism, a)
    return A, Y


@torch.no_grad()
def _v10_out(prism, X):
    """What the v10 FFN (groups + shadow, as trained) answers on the same inputs."""
    was = getattr(prism, "ns_on", False)
    prism.ns_on = False
    try:
        return torch.cat([prism(X[c:c + NS_CHUNK].to(prism.W_gate.dtype), mode="topk").float()
                          for c in range(0, X.shape[0], NS_CHUNK)], 0)
    finally:
        prism.ns_on = was


@torch.no_grad()
def _lens(prism, Xf, r_max):
    """Best rank-r picture of [W_gate; W_up] in the metric of the inputs: returns B [d, r], Ag / Au [D, r] with the
    strongest direction LAST, so a smaller rank is simply the last columns."""
    C = (Xf.t() @ Xf) / Xf.shape[0]
    lam, Q = _eigh(C)
    lam = lam.clamp_min(float(lam.max()) * 1e-5)
    Ch = (Q * lam.sqrt()[None, :]) @ Q.t()
    Cih = (Q / lam.sqrt()[None, :]) @ Q.t()
    Wg, Wu = prism.W_gate.float(), prism.W_up.float()                 # [d, D]
    Gg, Gu = Ch @ (Wg @ Wg.t()) @ Ch, Ch @ (Wu @ Wu.t()) @ Ch
    Gm = Gg / torch.trace(Gg).clamp_min(1e-30) + Gu / torch.trace(Gu).clamp_min(1e-30)
    _, V = _eigh(Gm)
    Vr = V[:, -int(r_max):]
    Lm = Ch @ Vr
    return Cih @ Vr, (Lm.t() @ Wg).t().contiguous(), (Lm.t() @ Wu).t().contiguous()


@torch.no_grad()
def _gram(X):
    xm = X[_fit].mean(0)
    Xc = X[_fit] - xm
    la, Qa = _eigh(Xc.t() @ Xc)
    return xm, Xc, la.clamp_min(0), Qa


@torch.no_grad()
def _rrr(gram, X, Rres, R, lam_grid=(1e-4, 1e-3, 1e-2, 1e-1)):
    """Rres ~ b + U (M x): ridge in the eigenbasis of the inputs (lambda picked on the held-out documents), then
    the best rank-R output subspace of the fitted values."""
    xm, Xc, la, Qa = gram
    rm = Rres[_fit].mean(0)
    T = Qa.t() @ (Xc.t() @ (Rres[_fit] - rm))
    scale = float(la.mean().clamp_min(1e-12))
    Xv, Rv = (X[_val] - xm) @ Qa, Rres[_val]
    best = None
    for lam in lam_grid:
        Gt = T / (la + lam * scale)[:, None]
        e = float((Rv - (rm + Xv @ Gt)).pow(2).sum())
        if best is None or e < best[0]:
            best = (e, Gt)
    Gt = best[1]
    _, V = _eigh(Gt.t() @ (la[:, None] * Gt))
    U = V[:, -int(R):].contiguous()
    M = (Qa @ (Gt @ U)).t().contiguous()
    return U, M, rm - (xm @ M.t()) @ U.t()


@torch.no_grad()
def _fit_cfg(prism, X, A, Y, lens, gram, r, R, budget, oracle=False):
    """One (lens rank r, shadow rank R) split of the layer's budget. Returns the buffers + held-out local error."""
    d, D = prism.d_model, prism.D
    k = int((budget - _ns_fixed(d, D, r, R)) // (3 * d))
    if k < 4:
        return None
    k = min(k, D)
    B, Ag, Au = lens[0][:, -r:], lens[1][:, -r:], lens[2][:, -r:]
    mu = A[_fit].mean(0)
    dn = prism._dn2.float().sqrt()
    Am = torch.empty_like(A)
    for c in range(0, X.shape[0], NS_CHUNK):
        a = A[c:c + NS_CHUNK] - mu
        s = (a.abs() * dn) if oracle else _ns_scores(prism, X[c:c + NS_CHUNK], B, Ag, Au, mu, dn)
        idx = s.topk(k, dim=-1).indices
        Am[c:c + NS_CHUNK] = torch.zeros_like(a).scatter(1, idx, a.gather(1, idx))
    b0 = _ns_down(prism, mu[None])[0]
    Ysel = torch.cat([_ns_down(prism, Am[c:c + NS_CHUNK]) for c in range(0, X.shape[0], NS_CHUNK)], 0) + b0
    del Am
    U, M, b = _rrr(gram, X, Y - Ysel, R)
    Yh = Ysel + (X @ M.t()) @ U.t() + b
    err = float((Y[_val] - Yh[_val]).pow(2).sum()) / float(Y[_val].pow(2).sum().clamp_min(1e-30))
    return {"B": B, "Ag": Ag, "Au": Au, "mu": mu, "dn": dn, "U": U, "M": M, "b": b + b0, "k": k, "r": r, "R": R,
            "active": 3 * d * k + _ns_fixed(d, D, r, R), "err": err}


def _rel_err(Yh, Y):
    return float((Y[_val] - Yh[_val]).pow(2).sum()) / float(Y[_val].pow(2).sum().clamp_min(1e-30))


@torch.no_grad()
def teacher_kl(blocks_, n_max=8):
    """Mean full-vocabulary KL(teacher || served student) per token on held-out blocks — the distance that matters."""
    tot, n = 0.0, 0
    for i in range(min(int(n_max), blocks_.shape[0])):
        b = blocks_[i:i + 1].to(DEV)
        set_world("dense_teacher")
        lt = model(input_ids=b).logits[0]
        set_world("serving")
        ls = model(input_ids=b).logits[0]
        for c0 in range(0, lt.shape[0], 256):
            pt = lt[c0:c0 + 256].float().log_softmax(-1)
            ps = ls[c0:c0 + 256].float().log_softmax(-1)
            tot += float((pt.exp() * (pt - ps)).sum())
            n += int(pt.shape[0])
            del pt, ps
        del lt, ls
    set_world("soft_train")
    return tot / max(n, 1)


def _ns_apply(mask):
    for li, (_, m) in enumerate(PRISM_MODULES):
        m.ns_on = bool(mask[li]) and hasattr(m, "ns_B")


def _empty_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


NS_REPORT, NS_LAB = [], []
NS_KEPT, NS_CONFIG = (True, "restored") if (NS_LOADED and NS_KEPT_RESTORED) else (False, "none")
_t_all = time.time()
if not NS_LOADED:
    _ns_apply([False] * _L)
PPL_V10 = ppl(heldout, "topk") if not NS_LOADED else float("nan")
KL_V10 = teacher_kl(heldout, NS_VERDICT_BLOCKS) if not NS_LOADED else float("nan")
if not NS_LOADED:
    print(f"neuron lens: {len(_bidx)} documents x {int(_pos[0].numel())} tokens = {_n_tok} rows ({int(_fit.numel())} fit / "
          f"{int(_val.numel())} held-out) | baseline v10: held-out PPL {PPL_V10:.2f}, KL to the teacher {KL_V10:.4f}")

    # ================================================================ (1) LAB
    set_world("serving")
    _probe = sorted({_L // 8, _L // 2, _L - 1 - _L // 8})
    _grid = [(r, R) for r in NS_PRED_RANKS for R in NS_SHADOW_RANKS]
    _acc = {g: [] for g in _grid}
    print(f"\n(1) LAB — layers {_probe}: relative error of the FFN output against its own dense twin, same inputs "
          f"(0 = the dense FFN, 1 = no FFN at all)")
    print(f"    {'layer':>5} {'v10':>7} | " + " ".join(f"r{r:>3}/R{R:<4}" for r, R in _grid) + " | oracle selection (best split)")
    for li in _probe:
        _, prism = PRISM_MODULES[li]
        X = _ffn_inputs(li)
        A, Y = _dense_acts(prism, X)
        e10 = _rel_err(_v10_out(prism, X), Y)
        lens = _lens(prism, X[_fit], max(NS_PRED_RANKS))
        gram = _gram(X)
        budget = _prism_active_v10(prism)
        row = {}
        for g in _grid:
            f_ = _fit_cfg(prism, X, A, Y, lens, gram, g[0], g[1], budget)
            row[g] = None if f_ is None else (f_["err"], f_["k"])
            if f_ is not None:
                _acc[g].append(f_["err"] / max(e10, 1e-12))
            del f_
        gbest = min((g for g in _grid if row[g] is not None), key=lambda g: row[g][0], default=None)
        orc = _fit_cfg(prism, X, A, Y, lens, gram, gbest[0], gbest[1], budget, oracle=True) if gbest else None
        print(f"    {li:>5} {e10:7.3f} | " + " ".join(f"{(row[g][0] if row[g] else float('nan')):9.3f}" for g in _grid)
              + (f" | {orc['err']:.3f}  (k={orc['k']} of {prism.D} neurons)" if orc else " | -"))
        NS_LAB.append({"layer": li, "v10": e10, "grid": {f"r{g[0]}/R{g[1]}": row[g] for g in _grid},
                       "oracle": None if orc is None else orc["err"]})
        del X, A, Y, lens, gram, orc
        _empty_cache()
    _valid = [g for g in _grid if len(_acc[g]) == len(_probe)]
    assert _valid, "no (lens rank, shadow rank) split fits inside the layer budget — lower NS_PRED_RANKS / NS_SHADOW_RANKS"
    NS_SPLIT = min(_valid, key=lambda g: sum(_acc[g]) / len(_acc[g]))
    _gain = sum(_acc[NS_SPLIT]) / len(_acc[NS_SPLIT])
    print(f"    chosen split: lens rank {NS_SPLIT[0]}, shadow rank {NS_SPLIT[1]} -> local error is {_gain:.2f}x v10's "
          f"(needs <= {NS_MIN_GAIN:.2f} to go on) | {time.time() - _t_all:.0f}s")
    _go = (_gain <= NS_MIN_GAIN) or bool(NS_FORCE)
    if not _go:
        print("\nLAB VERDICT: the lens is NOT clearly closer to the dense FFN than v10 is -> nothing is installed, "
              "v10 stays as it is. (NS_FORCE=True overrides.)")

    # ================================================================ (2) every layer, bottom-up
    if _go:
        print(f"\n(2) fitting all {_L} layers, bottom-up, each on the stream of the layers already switched")
        _t1 = time.time()
        for li, (name, prism) in enumerate(PRISM_MODULES):
            X = _ffn_inputs(li)
            A, Y = _dense_acts(prism, X)
            e10 = _rel_err(_v10_out(prism, X), Y)
            try:
                f_ = _fit_cfg(prism, X, A, Y, _lens(prism, X[_fit], NS_SPLIT[0]), _gram(X), NS_SPLIT[0], NS_SPLIT[1],
                              _prism_active_v10(prism))
            except Exception as _e:
                if is_oom(_e):
                    raise
                print(f"    layer {li}: fit failed ({type(_e).__name__}: {str(_e)[:100]}) -> v10 FFN kept")
                f_ = None
            took = f_ is not None and f_["err"] < e10
            if took:
                _ns_install(prism, f_)
                prism.ns_on = True
            NS_REPORT.append({"layer": li, "v10": e10, "lens": None if f_ is None else f_["err"],
                              "k": None if f_ is None else f_["k"], "on": bool(took)})
            if li % max(1, _L // 8) == 0 or li == _L - 1:
                print(f"    layer {li:2d}: local FFN error {e10:.3f} -> {(f_['err'] if f_ else float('nan')):.3f}"
                      f"{'' if took else '  (v10 kept)'} | k={f_['k'] if f_ else '-'} | {time.time() - _t1:.0f}s")
            del X, A, Y, f_
            _empty_cache()
        _fitted = [bool(r["on"]) for r in NS_REPORT]
        print(f"    {sum(_fitted)}/{_L} layers have a lens ({time.time() - _t1:.0f}s)")

        # ============================================================ (3) verdict by KL to the teacher, held-out text
        print(f"\n(3) VERDICT — KL to the dense teacher on held-out text ({NS_VERDICT_BLOCKS} blocks); v10 = {KL_V10:.4f}")
        _step = max(1, _L // 8)
        _cands = [("none", [False] * _L)]
        for p in range(_step, _L + 1, _step):
            _cands.append((f"layers 0..{p - 1}", [(i < p) and _fitted[i] for i in range(_L)]))
        for p in range(_step, _L, _step):
            _cands.append((f"layers {p}..{_L - 1}", [(i >= p) and _fitted[i] for i in range(_L)]))
        _best = ("none", [False] * _L, KL_V10)
        for tag, mask in _cands[1:]:
            if not any(mask):
                continue
            _ns_apply(mask)
            kl = teacher_kl(heldout, NS_VERDICT_BLOCKS)
            print(f"    lens on {tag:<16}: KL {kl:.4f}" + ("   <- best so far" if kl < _best[2] else ""))
            if kl < _best[2]:
                _best = (tag, mask, kl)
        NS_CONFIG = _best[0]
        _ns_apply(_best[1])
        NS_KEPT = NS_CONFIG != "none"

_active = measured_active()
if NS_KEPT and RUN_MODE == "full" and TARGET_ACTIVE and _active > TARGET_ACTIVE:
    print(f"!! census {_active / 1e9:.3f}B is over the {TARGET_ACTIVE / 1e9:.1f}B mandate -> lens switched off")
    _ns_apply([False] * _L)
    NS_KEPT, NS_CONFIG = False, "none"
    _active = measured_active()
PPL_V12 = ppl(heldout, "topk")
KL_V12 = teacher_kl(heldout, NS_VERDICT_BLOCKS)
PPL["student_topk@v12"] = PPL_V12
if NS_LOADED:
    print(f"\nNEURON LENS (restored): held-out PPL {PPL_V12:.2f} | KL to the teacher {KL_V12:.4f} | active {_active / 1e9:.3f}B")
elif NS_KEPT:
    print(f"\nNEURON LENS KEPT on {NS_CONFIG}: KL to the teacher {KL_V10:.4f} -> {KL_V12:.4f} nats/token | held-out PPL "
          f"{PPL_V10:.2f} -> {PPL_V12:.2f} (dense teacher {PPL.get('teacher_dense', float('nan')):.2f}) | active "
          f"{_active / 1e9:.3f}B | zero gradient steps | {time.time() - _t_all:.0f}s")
else:
    print(f"\nNEURON LENS NOT KEPT: no depth range lowered the KL to the teacher (v10 {KL_V10:.4f}) — the model is "
          f"exactly v10 again | held-out PPL {PPL_V12:.2f} | {time.time() - _t_all:.0f}s")
set_world("soft_train")
free_cuda("neuron lens")


## Restoration check (the model as it was saved, against the number saved with it)

In [ ]:
# is the rebuilt model the SAME model the state file was saved from?
# (v14 compared the lens-OFF model with a number measured before the healing and raised a false alarm; the check
#  now compares the model exactly as it is served — lens on where the state says so — with the saved number.)
_pj = os.path.join(os.path.dirname(STATE_PATH), "ppl_after.json")
RESTORE_REL, PPL_SAVED = None, None
if os.path.exists(_pj):
    with open(_pj) as _f:
        _d = json.load(_f)
    if NS_LOADED:
        PPL_SAVED, _now, _what = _d.get("student_topk"), float(PPL_V12), "served model (lens as saved)"
        _dh = _d.get("dense_head_check")
        if isinstance(_dh, (int, float)) and _dh == _dh:              # a v16 state: at this point of the notebook the strict head
            PPL_SAVED, _what = float(_dh), "served model, head not yet strict (the law cell below installs it)"   # is not installed yet
    else:
        PPL_SAVED, _now, _what = (_d.get("v10_restored") or _d.get("student_topk")), float(PPL["student_topk@init"]), "v10 student"
    if PPL_SAVED:
        RESTORE_REL = abs(_now - PPL_SAVED) / PPL_SAVED
        _tag = "EXACT RESTORE" if RESTORE_REL < 0.02 else ("CLOSE (different GPU / kernels?)" if RESTORE_REL < 0.05 else "DIFFERENT")
        print(f"restoration check — {_what}: saved {PPL_SAVED:.2f}, rebuilt {_now:.2f} ({100 * RESTORE_REL:.2f}% apart) -> {_tag}"
              f" | dense teacher {PPL['teacher_dense']:.2f}")
        if RESTORE_REL >= 0.05 and not globals().get("ALLOW_RESTORE_MISMATCH", False):
            raise RuntimeError(
                f"복원한 모델이 저장할 때의 모델과 다릅니다 (PPL {PPL_SAVED:.2f} -> {_now:.2f}). 이 상태로 몇 시간을 학습하면 "
                "크레딧만 씁니다. 이 셀까지의 출력을 그대로 보내 주세요. (그래도 진행하려면 ALLOW_RESTORE_MISMATCH=True)")
if RESTORE_REL is None:
    print("(no ppl_after.json next to the state file: nothing to compare with)")


## X-ray before (second sight is off in v16: v15 measured it and rejected it)

In [ ]:
# v15 — SECOND SIGHT + GHOST NEURONS: the same FFN budget, spent on twice as many neurons (zero gradient steps)
#
# Where the FFN budget of the 27B goes today (per layer, 18.3M parameters touched per token):
#     exact neurons   517 x 3d            7.9M   43%
#     linear shadow   rank 512            5.2M   29%
#     the lens        2 x D x 128 + B     5.1M   28%   <- a full-rank look at 17,408 neurons to pick 517 of them
# This cell (1) measures, layer by layer on the real stream, how coarse the first look can be without changing which
# neurons get picked, (2) measures how well the lens predicts the activation of the neurons just below the cut — that
# is what a ghost neuron would say — and (3) re-spends what the coarse look frees. The options are tried on held-out
# text with NO training; the best one is kept only if a separate set of held-out blocks agrees. Nothing here can
# raise a layer's budget, and any error in this cell leaves the model exactly as it was restored.
V15_SS_ON      = globals().get("V15_SS_ON", False)     # v15 measured it on the 27B and rejected it
V15_SS_BLOCKS  = globals().get("V15_SS_BLOCKS", 8 if RUN_MODE == "full" else 4)       # calibration blocks (teaching text)
V15_SS_KEEP    = globals().get("V15_SS_KEEP", 0.985)     # the two-look choice must keep this share of the one-look choice's score
V15_SS_R1      = tuple(globals().get("V15_SS_R1", (16, 32, 64) if RUN_MODE == "full" else (2, 4)))
V15_SS_MULT    = tuple(globals().get("V15_SS_MULT", (4, 8) if RUN_MODE == "full" else (2, 3)))   # shortlist = mult x k
V15_SS_TUNE    = globals().get("V15_SS_TUNE", 4 if RUN_MODE == "full" else 2)          # held-out blocks the options are compared on
V15_SS_SHIFT   = globals().get("V15_SS_SHIFT", 0.25)     # option "shift": this share of the exact neurons becomes 3x as many ghosts
V15_GHOST_BIAS = globals().get("V15_GHOST_BIAS", 0.01)   # an option WITH ghosts wins a tie this wide: its gains and the predictor
#                                                           behind it are trained afterwards, exact neurons cannot improve
V15_SS_FORCE   = globals().get("V15_SS_FORCE", None)     # testing: "look" | "exact" | "ghost" | "mix" | "shift" keeps that option whatever the KL
import re as _re_ss, traceback as _tb_ss

_LRX_SS = _re_ss.compile(r"layers\.(\d+)(?:\.|$)")


@torch.no_grad()
def v15_eval(blocks_, n_max):
    """(KL(teacher || served student) per token, share of positions where both put the same token first)."""
    tot, agree, n = 0.0, 0, 0
    for i in range(min(int(n_max), blocks_.shape[0])):
        b = blocks_[i:i + 1].to(DEV)
        set_world("dense_teacher")
        lt = model(input_ids=b).logits[0]
        set_world("serving")
        ls = model(input_ids=b).logits[0]
        for c0 in range(0, lt.shape[0], 256):
            pt = lt[c0:c0 + 256].float().log_softmax(-1)
            ps = ls[c0:c0 + 256].float().log_softmax(-1)
            tot += float((pt.exp() * (pt - ps)).sum())
            agree += int((pt.argmax(-1) == ps.argmax(-1)).sum())
            n += int(pt.shape[0])
            del pt, ps
        del lt, ls
    set_world("serving")
    return tot / max(n, 1), agree / max(n, 1)


@torch.no_grad()
def v15_xray(n_blocks=2, n_tok=384, quiet=False):
    """X-RAY: for every kind of sparse site, the relative error of what it serves against its OWN dense computation
    on the inputs it really gets (0 = dense, 1 = as wrong as outputting nothing). Three probe layers. This is the
    local number that pointed at the FFNs in v12; it is read here for the attention side, which never had one."""
    model.eval()
    set_world("serving")
    L_ = len(PRISM_MODULES)
    probe = sorted({L_ // 8, L_ // 2, L_ - 1 - L_ // 8})
    lay = lambda p: int(_LRX_SS.search(p).group(1)) if _LRX_SS.search(p) else -1
    sites = ([(n, m, "FFN (lens)") for n, m in PRISM_MODULES if lay(n) in probe and getattr(m, "ns_on", False)]
             + [(n, m, ".".join(n.split(".")[-2:])) for n, m in RESONA_LINEARS if lay(n) in probe]
             + [(n, m, ".".join(n.split(".")[-2:])) for n, m in RESONA_SECTIONED if lay(n) in probe])
    got, hooks = {}, []
    for n, m, _ in sites:
        def pre(mod, args, __n=n):
            x = args[0] if isinstance(args, (tuple, list)) else args
            got.setdefault(__n, []).append(x.reshape(-1, x.shape[-1])[:n_tok].detach())
        hooks.append(m.register_forward_pre_hook(pre))
    try:
        for i in range(min(n_blocks, heldout.shape[0])):
            model(input_ids=heldout[i:i + 1].to(DEV))
    finally:
        for h in hooks:
            h.remove()
    acc = {}

    def add(kind, served, dense):
        e = acc.setdefault(kind, [0.0, 0.0])
        e[0] += float((served.float() - dense.float()).pow(2).sum())
        e[1] += float(dense.float().pow(2).sum())

    for n, m, kind in sites:
        if n not in got:
            continue
        x = torch.cat(got[n], 0)
        if isinstance(m, PrismMoeMLP):
            add(kind, m(x, mode="topk"), m(x, mode="uniform"))
        elif isinstance(m, SectionedProjector):
            ys, yd = m(x, mode="topk"), m(x, mode="dense")
            b = [int(v) for v in m.sec_bounds]
            for t, lo, hi in (("q", b[0], b[1]), ("v", b[2], b[3]), ("z", b[3], b[4])):
                if hi > lo:
                    add(f"{kind} [{t}]", ys[:, lo:hi], yd[:, lo:hi])
        else:
            add(kind, m(x, mode="topk"), m(x, mode="dense"))
    set_world("serving")
    out = {k: (v[0] / max(v[1], 1e-30)) for k, v in sorted(acc.items())}
    if not quiet:
        print(f"    X-RAY (layers {probe}) — served vs. the site's own dense output, same inputs:")
        for k, v in out.items():
            print(f"        {k:<34} relative error {v:.3f}")
    return out


SSIGHT = {"kept": False, "choice": "none", "options": {}, "layers": [], "ghost_corr": None, "error": None}
_ss_lens = [(li, m) for li, (_, m) in enumerate(PRISM_MODULES) if hasattr(m, "ns_B") and getattr(m, "ns_on", False)]
_ss_done = any(_ns_prog(m)[0] or int(getattr(m, "ns_k2", 0) or 0) > 0 for _, m in _ss_lens)
KL_RESTORED, PPL_RESTORED = float(KL_V12), float(PPL_V12)
B_RESTORED = measured_active()                 # the budget of the restored model: nothing below may exceed it
_SS_BASE = {li: (int(m.ns_k), int(m.ns_active)) for li, m in _ss_lens}


def _ss_reset():
    for li, m in _ss_lens:
        m.ns_r1, m.ns_ns, m.ns_k2 = 0, int(m.D), 0
        m.ns_k, m.ns_active = _SS_BASE[li]


model.eval()
print("(0) where the model stands locally, before anything is changed:")
try:
    XRAY_BEFORE = v15_xray()
except Exception as _e:
    XRAY_BEFORE = {}
    print(f"    (x-ray skipped: {type(_e).__name__}: {str(_e)[:120]})")

if not V15_SS_ON:
    print("\nsecond sight is off (V15_SS_ON = False)")
elif _ss_done:
    SSIGHT["kept"], SSIGHT["choice"] = True, "restored with the state"
    print("\nsecond sight / ghost neurons are already part of the restored state: nothing is measured again")
elif not _ss_lens:
    print("\nno lens layer is switched on in this state: second sight has nothing to work on")
else:
    _t0 = time.time()
    try:
        # ============================================================ (1) one pass, hooks on every lens layer
        _K2CAP = 2
        _ss_stat = {}
        for li, m in _ss_lens:
            r_ = int(m.ns_B.shape[1])
            grid = [(r1, min(int(m.D), mult * int(m.ns_k))) for r1 in V15_SS_R1 if 0 < r1 < r_ for mult in V15_SS_MULT]
            grid = sorted({g for g in grid if g[1] >= int(m.ns_k) + 1 and _ns_lens_rows(m.D, r_, g[0], g[1]) < _ns_lens_rows(m.D, r_)})
            _ss_stat[li] = {"grid": grid, "den": 0.0, "num": {g: 0.0 for g in grid},
                         "sxy": torch.zeros(m.D, device=DEV), "sxx": torch.zeros(m.D, device=DEV),
                         "syy": torch.zeros(m.D, device=DEV), "exy": 0.0, "exx": 0.0, "eyy": 0.0}

        def _mk_hook(li, m):
            st = _ss_stat[li]

            def pre(mod, args):
                x = (args[0] if isinstance(args, (tuple, list)) else args)
                x = x.reshape(-1, x.shape[-1])
                for c0 in range(0, x.shape[0], NS_CHUNK):
                    xc = x[c0:c0 + NS_CHUNK]
                    xf = xc.float()
                    with _ns_fp32(xf):
                        z = xf @ mod.ns_B
                        dev = _ns_hat(mod, z) - mod.ns_mu
                        s = dev.abs() * mod.ns_dn
                        k = int(mod.ns_k)
                        kk = min(mod.D, k * (1 + _K2CAP))
                        topv, idx = s.topk(kk, dim=-1)
                        st["den"] += float(topv[:, :k].sum())
                        for g in st["grid"]:
                            s1 = (_ns_hat(mod, z, g[0]) - mod.ns_mu).abs() * mod.ns_dn
                            short = torch.zeros_like(s).scatter(1, s1.topk(g[1], dim=-1).indices,
                                                                torch.ones(s.shape[0], g[1], device=s.device))
                            st["num"][g] += float((s * short).topk(k, dim=-1).values.sum())
                            del s1, short
                        xw = _dm(xc, mod.W_gate)
                        da = F.silu((xw @ mod.W_gate).float()) * (xw @ mod.W_up).float() - mod.ns_mu
                        one = torch.ones(idx.shape, device=s.device)
                        zone = torch.zeros_like(s).scatter(1, idx[:, k:], one[:, k:])       # the neurons just below the cut
                        st["sxy"] += (dev * da * zone).sum(0)
                        st["sxx"] += (dev * dev * zone).sum(0)
                        st["syy"] += (da * da * zone).sum(0)
                        top = torch.zeros_like(s).scatter(1, idx[:, :k], one[:, :k])        # ... and the ones that are read exactly
                        st["exy"] += float((dev * da * top).sum())
                        st["exx"] += float((dev * dev * top).sum())
                        st["eyy"] += float((da * da * top).sum())
                        del zone, top, da, dev, s
            return pre

        _hooks = [m.register_forward_pre_hook(_mk_hook(li, m)) for li, m in _ss_lens]
        set_world("serving")
        _cal = teach_blocks if "teach_blocks" in globals() else blocks
        _cidx = sorted({int(round(v)) for v in torch.linspace(0, _cal.shape[0] - 1, min(int(V15_SS_BLOCKS), _cal.shape[0])).tolist()})
        try:
            with torch.no_grad():
                for bi in _cidx:
                    model(input_ids=_cal[bi:bi + 1].to(DEV))
        finally:
            for h in _hooks:
                h.remove()

        # ============================================================ (2) per layer: the coarsest look that keeps the choice
        _ss_cfg, _ss_gain, _free_tot, _lens_tot = {}, {}, 0, 0
        for li, m in _ss_lens:
            st, r_ = _ss_stat[li], int(m.ns_B.shape[1])
            full = _ns_lens_rows(m.D, r_)
            ok = [g for g in st["grid"] if st["num"][g] / max(st["den"], 1e-30) >= V15_SS_KEEP]
            best = min(ok, key=lambda g: _ns_lens_rows(m.D, r_, g[0], g[1])) if ok else None
            _ss_cfg[li] = best
            _lens_tot += full
            _free_tot += (full - _ns_lens_rows(m.D, r_, best[0], best[1])) if best else 0
            # ghost gain: regression of the true deviation on the estimated one, in the zone where ghosts live; a
            # neuron with few visits leans on the layer's own average slope
            cg = float(st["sxy"].sum() / st["sxx"].sum().clamp_min(1e-30))
            prior = st["sxx"].mean().clamp_min(1e-30)
            _ss_gain[li] = ((st["sxy"] + prior * cg) / (st["sxx"] + prior)).clamp(0.0, 1.5)
            SSIGHT["layers"].append({"layer": li, "k": int(m.ns_k), "look": best,
                                 "kept_score": (st["num"][best] / max(st["den"], 1e-30)) if best else None,
                                 "lens_corr_ghost_zone": float(st["sxy"].sum() / (st["sxx"].sum() * st["syy"].sum()).sqrt().clamp_min(1e-30)),
                                 "lens_corr_exact_zone": st["exy"] / max(math.sqrt(st["exx"] * st["eyy"]), 1e-30),
                                 "ghost_slope": cg})
        _n_on = sum(1 for v in _ss_cfg.values() if v)
        _cz = [r["lens_corr_ghost_zone"] for r in SSIGHT["layers"]]
        _ce = [r["lens_corr_exact_zone"] for r in SSIGHT["layers"]]
        SSIGHT["ghost_corr"] = sum(_cz) / len(_cz)
        print(f"\n(1) SECOND SIGHT — measured on {len(_cidx)} blocks ({time.time() - _t0:.0f}s): a coarse first look keeps >= "
              f"{100 * V15_SS_KEEP:.1f}% of the one-look choice on {_n_on}/{len(_ss_lens)} layers and frees "
              f"{_free_tot / 1e6:.1f}M of the {_lens_tot / 1e6:.1f}M lens parameters read per token "
              f"({100 * _free_tot / max(1, _lens_tot):.0f}%)")
        _step = max(1, len(SSIGHT["layers"]) // 8)
        for r in SSIGHT["layers"][::_step]:
            print(f"    layer {r['layer']:2d}: k={r['k']:4d} | look {('rank ' + str(r['look'][0]) + ' -> shortlist ' + str(r['look'][1])) if r['look'] else 'one look (kept)':<28}"
                  f" | lens vs. truth: correlation {r['lens_corr_exact_zone']:.2f} on the exact neurons, "
                  f"{r['lens_corr_ghost_zone']:.2f} just below the cut (ghost slope {r['ghost_slope']:.2f})")
        print(f"    what a ghost would say correlates {SSIGHT['ghost_corr']:.2f} with the truth on average "
              f"(1 = a ghost is as good as reading the neuron; 0 = ghosts are noise)")

        # ============================================================ (3) the options, on held-out text, no training
        def _ss_apply(kind, gain=1.0):
            n_exact, n_ghost = 0, 0
            for li, m in _ss_lens:
                k0, act0 = _SS_BASE[li]
                m.ns_r1, m.ns_ns, m.ns_k2, m.ns_k = 0, int(m.D), 0, k0
                d = m.d_model
                if kind != "base" and _ss_cfg[li]:
                    m.ns_r1, m.ns_ns = _ss_cfg[li]
                    free = act0 - _ns_cost(m, k0, 0)
                    k2 = 0
                    if kind == "exact":
                        m.ns_k = k0 + free // (3 * d)
                    elif kind == "ghost":
                        k2 = (free - m.D) // d
                    elif kind == "mix":
                        m.ns_k = k0 + (free // 2) // (3 * d)
                        k2 = (act0 - _ns_cost(m, m.ns_k, 0) - m.D) // d
                    elif kind == "shift":
                        m.ns_k = max(4, int(round(k0 * (1.0 - V15_SS_SHIFT))))
                        k2 = (act0 - _ns_cost(m, m.ns_k, 0) - m.D) // d
                    k2 = int(max(0, min(k2, m.ns_ns - m.ns_k, m.D - m.ns_k)))
                    if k2 > 0:
                        _ns_set(m, "ns_gc", gain * _ss_gain[li])
                        m.ns_k2 = k2
                m.ns_active = _ns_cost(m)
                assert m.ns_active <= act0, f"layer {li}: option {kind} costs more than the layer did"
                n_exact += int(m.ns_k)
                n_ghost += int(m.ns_k2)
            return n_exact / len(_ss_lens), n_ghost / len(_ss_lens)

        _tune = (heldout[NS_VERDICT_BLOCKS:NS_VERDICT_BLOCKS + V15_SS_TUNE] if heldout.shape[0] >= NS_VERDICT_BLOCKS + 2
                 else _cal[-int(V15_SS_TUNE):])
        _opts = [("base", 1.0)]
        if _n_on:
            _opts += [("look", 1.0), ("exact", 1.0), ("ghost", 1.0), ("ghost", 0.5), ("mix", 1.0), ("shift", 1.0)]
        print(f"\n(2) HOW TO SPEND IT — KL to the teacher on {_tune.shape[0]} held-out tuning blocks, no training "
              f"(neurons per token and layer, averaged over the layers):")
        _res = []
        for kind, gain in _opts:
            ne, ng = _ss_apply(kind, gain)
            kl = v15_eval(_tune, _tune.shape[0])[0]
            act = measured_active()
            tag = {"base": "as restored (one look)", "look": "two looks, nothing re-spent",
                   "exact": "two looks -> more exact neurons", "ghost": f"two looks -> ghosts (gain x{gain:g})",
                   "mix": "two looks -> half exact, half ghosts", "shift": f"... and {100 * V15_SS_SHIFT:.0f}% of the exact ones become ghosts"}[kind]
            _res.append({"kind": kind, "gain": gain, "kl": kl, "exact": ne, "ghosts": ng, "active": act, "tag": tag})
            print(f"    {tag:<52} exact {ne:7.1f} | ghosts {ng:7.1f} | KL {kl:.4f} | census {act / 1e9:.3f}B")
        SSIGHT["options"] = [{k: v for k, v in r.items()} for r in _res]
        _base_kl = _res[0]["kl"]
        _cand = [r for r in _res[1:] if r["kind"] != "look"]
        _pick = None
        if V15_SS_FORCE:
            _pick = next((r for r in _res if r["kind"] == V15_SS_FORCE), None)
        elif _cand:
            _best = min(_cand, key=lambda r: r["kl"])
            _gh = [r for r in _cand if r["ghosts"] > 0 and r["kl"] <= _best["kl"] * (1.0 + V15_GHOST_BIAS)]
            _pick = min(_gh, key=lambda r: r["kl"]) if _gh else _best
            if not (_pick["kl"] < _base_kl):
                _pick = None
        if _pick is None:
            _ss_reset()
            print(f"    -> nothing beats the restored reading on the tuning blocks ({_base_kl:.4f}): the FFNs stay as they were")
        else:
            _ss_apply(_pick["kind"], _pick["gain"])
            _klv = v15_eval(heldout, NS_VERDICT_BLOCKS)[0]
            _act = measured_active()
            _ok = (_klv < KL_RESTORED or bool(V15_SS_FORCE)) and not (RUN_MODE == "full" and TARGET_ACTIVE and _act > TARGET_ACTIVE)
            print(f"    -> {_pick['tag']}: on the {NS_VERDICT_BLOCKS} verdict blocks KL {KL_RESTORED:.4f} -> {_klv:.4f} "
                  f"| census {_act / 1e9:.3f}B" + ("" if _ok else "  — NOT confirmed, rolled back"))
            if _ok:
                SSIGHT["kept"], SSIGHT["choice"] = True, _pick["tag"]
                KL_V12, PPL_V12 = float(_klv), float(ppl(heldout, "topk"))
                PPL["student_topk@second_sight"] = PPL_V12
                print(f"\nSECOND SIGHT KEPT ({_pick['tag']}): {_pick['exact']:.0f} exact + {_pick['ghosts']:.0f} ghost neurons per token and "
                      f"layer | KL {KL_RESTORED:.4f} -> {KL_V12:.4f} | held-out PPL {PPL_RESTORED:.2f} -> {PPL_V12:.2f} | zero gradient "
                      f"steps | {time.time() - _t0:.0f}s")
            else:
                _ss_reset()
    except Exception as _e:
        if is_oom(_e):
            free_cuda("second sight (OOM)")
        _ss_reset()
        SSIGHT["kept"], SSIGHT["choice"], SSIGHT["error"] = False, "none", f"{type(_e).__name__}: {str(_e)[:300]}"
        print(f"\n!! second sight stopped with an error and was undone — the model is exactly the restored one, the run goes on.\n"
              f"   {SSIGHT['error']}\n{_tb_ss.format_exc()[-1500:]}")
    if not SSIGHT["kept"]:
        for li, m in _ss_lens:
            assert (int(m.ns_k), int(m.ns_active)) == _SS_BASE[li] and not _ns_prog(m)[0] and not int(m.ns_k2)
set_world("serving")
free_cuda("second sight")


## Data — text, problems, 3x the reasoning traces, fresh text (v17: labelled in the step, no cache)

In [ ]:
# v16 — DATA: what the student is taught on, and what it is measured on
#
# v10..v14 taught on plain web / wiki text only. v15 added 384 blocks of chat-format arithmetic and GSM8K-train problems:
# in 300 updates the KL on reasoning text fell 1.45 -> 0.60 while the KL on plain text fell 2.01 -> 1.76. The domain of
# the text decides what is learned. v16 adds a third source: REASONING TRACES written by a current model of the
# teacher's own family (Qwen3.8-Max; a public distillation set, chat format, with the thinking kept). The dense
# Qwen3.8-27B still labels every token of every block itself — a dataset only supplies the text the teacher reads.
#
#   text       the v10 teaching blocks (teacher cache reused)                       measured on: heldout
#   problems   arithmetic + GSM8K-train word problems, thinking off (as in v15)     measured on: heldout_problems
#   traces     questions + thinking + answers of Qwen3.8-Max / GLM-5.2 / Kimi-K3    measured on: heldout_traces
#
# Never trained on: the math probe's own problems, the GSM8K TEST split (traces whose prompt contains a GSM8K test
# question are dropped), and the held-out blocks.
V15_MATH_BLOCKS  = globals().get("V15_MATH_BLOCKS", 384 if RUN_MODE == "full" else 4)
V15_MATH_HOLD    = globals().get("V15_MATH_HOLD", 6 if RUN_MODE == "full" else 2)
V15_MATH_REPEAT  = globals().get("V15_MATH_REPEAT", 1)        # a reasoning block is drawn this many times as often as a text block
V15_MATH_W       = globals().get("V15_MATH_W", 0.5)           # weight of the reasoning KL in the score that picks checkpoints
V16_TRACE_BLOCKS = globals().get("V16_TRACE_BLOCKS", 1536 if RUN_MODE == "full" else 4)    # v17: 3x (no cache to build)
V16_TRACE_HOLD   = globals().get("V16_TRACE_HOLD", 6 if RUN_MODE == "full" else 2)
V16_TRACE_REPOS  = tuple(globals().get("V16_TRACE_REPOS", ("inferenceport-ai/qwen3.8-max-glm5.2-kimi-k3-distillation",
                                                           "r0b0tlab/qwen3.8-max-glm5.2-kimi-k3-distillation",
                                                           "p-research/qwen3.8-max-glm5.2-kimi-k3-distillation")))
V16_TRACE_CONFIG = globals().get("V16_TRACE_CONFIG", "sft_balanced")
V16_TRACE_SHARE  = dict(globals().get("V16_TRACE_SHARE", {"math": 0.30, "reasoning": 0.25, "science": 0.15, "code": 0.15, "other": 0.15}))
V16_TRACE_MAXTOK = globals().get("V16_TRACE_MAXTOK", 1536)   # longer exchanges are skipped (a block is 1024 tokens)
V16_TRACE_SCAN   = globals().get("V16_TRACE_SCAN", 150000 if RUN_MODE == "full" else 400)    # rows looked at, at most
# ---- v17: the teacher labels every batch inside the training step, so the data no longer has to be cached
V17_INSTEP       = globals().get("V17_INSTEP", True)
V17_FRESH_BLOCKS = globals().get("V17_FRESH_BLOCKS", 1024 if RUN_MODE == "full" else 4)    # NEW text, never seen by any version
V17_FRESH_SHARE  = dict(globals().get("V17_FRESH_SHARE", {"fineweb-edu": 0.40, "code": 0.25, "ko-wiki": 0.20, "chat": 0.15}))
V17_FRESH_SKIP   = globals().get("V17_FRESH_SKIP", 6000 if RUN_MODE == "full" else 0)     # documents skipped per source (v10 read the first ones)

import hashlib, random as _random, re as _re15

_L = len(PRISM_MODULES)
_LRX = _re15.compile(r"layers\.(\d+)(?:\.|$)")


def _lay(path):
    mm = _LRX.search(path)
    return int(mm.group(1)) if mm else 0


def _tok_ids(s):
    return list(tok(s, add_special_tokens=False)["input_ids"])


_EOT = "<|im_end|>"
try:
    _i = tok.convert_tokens_to_ids(_EOT)
    if not (isinstance(_i, int) and _i >= 0 and _i != getattr(tok, "unk_token_id", None)):
        _EOT = ""
except Exception:
    _EOT = ""


def _chat(user, assistant):
    """One finished exchange in the SAME prompt format the math probe uses (thinking off)."""
    try:
        s = tok.apply_chat_template([{"role": "user", "content": user}], tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
    except Exception:
        s = user + "\n"
    return s + assistant + _EOT + "\n"


def _chat_think(msgs, thinking, answer):
    """A finished exchange WITH the thinking: the prompt the tokenizer builds for thinking mode, then the trace."""
    try:
        s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    except Exception:
        try:
            s = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        except Exception:
            s = "\n".join(str(m.get("content", "")) for m in msgs) + "\n"
    thinking, answer = (thinking or "").strip(), (answer or "").strip()
    if not thinking:
        return s + answer + _EOT + "\n"
    if s.rstrip().endswith("<think>"):                    # the template already opened the thinking block
        return s.rstrip() + "\n" + thinking + "\n</think>\n\n" + answer + _EOT + "\n"
    return s + "<think>\n" + thinking + "\n</think>\n\n" + answer + _EOT + "\n"


_P_A = "Compute the value. Reply with exactly one line of the form 'Answer: <number>'.\n\n"
_P_S = "Compute the value. Write one short line per step, then finish with a final line of the form 'Answer: <number>'.\n\n"
_P_B = ("Solve the problem. Write at most 5 short lines of reasoning, then finish with a final "
        "line of the form 'Answer: <number>'.\n\nProblem: ")

# the probe's own one-step problems are generated exactly as the probe does and are never trained on
_pr, _PROBE_Q = _random.Random(1234), set()
for _i in range(400):
    _k = _i % 4
    if _k == 0:
        _a, _b = _pr.randint(11, 99), _pr.randint(11, 99); _PROBE_Q.add(f"{_a} + {_b}")
    elif _k == 1:
        _a, _b = _pr.randint(50, 99), _pr.randint(11, 49); _PROBE_Q.add(f"{_a} - {_b}")
    elif _k == 2:
        _a, _b = _pr.randint(2, 9), _pr.randint(11, 30); _PROBE_Q.add(f"{_a} * {_b}")
    else:
        _b = _pr.randint(2, 9); _g = _pr.randint(3, 20); _PROBE_Q.add(f"{_b * _g} / {_b}")


def _one_step(rng):
    while True:
        k = rng.randrange(4)
        if k == 0:
            a, b = rng.randint(11, 999), rng.randint(11, 999); q, g = f"{a} + {b}", a + b
        elif k == 1:
            a = rng.randint(50, 999); b = rng.randint(11, a - 1); q, g = f"{a} - {b}", a - b
        elif k == 2:
            a, b = rng.randint(2, 12), rng.randint(11, 99); q, g = f"{a} * {b}", a * b
        else:
            b = rng.randint(2, 12); g = rng.randint(3, 99); q = f"{b * g} / {b}"
        if q not in _PROBE_Q:
            return _chat(_P_A + q, f"Answer: {g}")


def _multi_step(rng):
    k = rng.randrange(4)
    a, b, c = rng.randint(11, 99), rng.randint(11, 99), rng.randint(2, 9)
    if k == 0:
        return _chat(_P_S + f"({a} + {b}) * {c}", f"{a} + {b} = {a + b}\n{a + b} * {c} = {(a + b) * c}\nAnswer: {(a + b) * c}")
    if k == 1:
        return _chat(_P_S + f"{a} * {c} - {b}", f"{a} * {c} = {a * c}\n{a * c} - {b} = {a * c - b}\nAnswer: {a * c - b}")
    if k == 2:
        return _chat(_P_S + f"{a} + {b} + {c * 10}", f"{a} + {b} = {a + b}\n{a + b} + {c * 10} = {a + b + c * 10}\nAnswer: {a + b + c * 10}")
    return _chat(f"What is {a} plus {b}? Finish with a line of the form 'Answer: <number>'.", f"{a} + {b} = {a + b}\nAnswer: {a + b}")


_CALC = _re15.compile(r"<<[^>]*>>")
_WS = _re15.compile(r"[^a-z0-9]+")
_norm_words = lambda s: [w for w in _WS.split(str(s).lower()) if w]
_GSM, _GSM_TEST_KEYS = [], set()
try:
    from datasets import load_dataset as _ld15
    _ds = _ld15("openai/gsm8k", "main", split="train")          # the TRAIN split; the probe reads the TEST split
    _order = list(range(len(_ds)))
    _random.Random(15).shuffle(_order)
    for _j in _order[:4000]:
        _r = _ds[_j]
        _body, _, _gold = _r["answer"].partition("####")
        _lines = [l.strip() for l in _CALC.sub("", _body).strip().split("\n") if l.strip()]
        _gold = _gold.strip().replace(",", "")
        if _gold and 1 <= len(_lines) <= 6:
            _GSM.append(_chat(_P_B + _r["question"].strip(), "\n".join(_lines) + f"\nAnswer: {_gold}"))
    del _ds
except Exception as _e:
    print(f"GSM8K train split not available ({type(_e).__name__}: {str(_e)[:100]}) -> the problems are arithmetic only")
try:                                                            # the TEST questions, only to keep them OUT of the traces
    _dst = _ld15("openai/gsm8k", "main", split="test")
    for _r in _dst:
        _w = _norm_words(_r.get("question", ""))
        if len(_w) >= 10:
            _GSM_TEST_KEYS.add(" ".join(_w[:10]))
    del _dst
except Exception as _e:
    print(f"(GSM8K test questions not available for the contamination filter: {type(_e).__name__})")


def _contaminated(user_text):
    """True when the prompt contains the opening of a GSM8K TEST question or one of the probe's arithmetic problems."""
    if len(user_text) < 400 and any(q in user_text for q in _PROBE_Q):
        return True
    if _GSM_TEST_KEYS:
        w = _norm_words(user_text)
        for i in range(0, max(0, len(w) - 9)):
            if " ".join(w[i:i + 10]) in _GSM_TEST_KEYS:
                return True
    return False


def _math_stream(n_tok, seed, gsm):
    """Token stream of whole exchanges, kinds interleaved so that ~55% of the tokens are word problems (when there
    are any), ~25% one-step and ~20% multi-step arithmetic. Deterministic."""
    rng = _random.Random(seed)
    share = {"gsm": 0.55 if gsm else 0.0, "one": 0.25, "multi": 0.20}
    tot = sum(share.values())
    got, ids, gi, counts = {k: 0 for k in share}, [], 0, {k: 0 for k in share}
    while len(ids) < n_tok:
        kind = max(share, key=lambda k: share[k] / tot - got[k] / max(1, len(ids)))
        if kind == "gsm":
            if gi >= len(gsm):
                share["gsm"] = 0.0
                tot = sum(share.values())
                continue
            s = gsm[gi]; gi += 1
        else:
            s = _one_step(rng) if kind == "one" else _multi_step(rng)
        t = _tok_ids(s)
        ids.extend(t); got[kind] += len(t); counts[kind] += 1
    return ids, counts


# ------------------------------------------------------------------ reasoning traces of the teacher's family
def _trace_domain(row):
    d = str(row.get("domain") or "").lower()
    for key in ("math", "reasoning", "science", "code"):
        if key in d:
            return key
    return "other"


def _trace_text(row):
    """One row of the distillation set -> (domain, finished exchange as text) or None when the row is not a plain
    question -> (thinking) -> answer exchange."""
    msgs = row.get("messages")
    if isinstance(msgs, str):                             # some exports keep the conversation as a JSON string
        try:
            msgs = json.loads(msgs)
        except Exception:
            return None
    tools = row.get("tools")
    if isinstance(tools, str):
        tools = tools.strip() not in ("", "[]", "null", "None")
    if not isinstance(msgs, (list, tuple)) or len(msgs) < 2 or tools:
        return None
    msgs = [m for m in msgs if isinstance(m, dict)]
    if not msgs or msgs[-1].get("role") != "assistant" or msgs[-1].get("tool_calls"):
        return None
    if any(m.get("role") not in ("system", "user", "assistant") or m.get("tool_calls") for m in msgs[:-1]):
        return None
    last = msgs[-1]
    answer, thinking = last.get("content"), last.get("reasoning_content")
    if not isinstance(answer, str) or not answer.strip():
        return None
    prompt = [{"role": m["role"], "content": str(m.get("content") or "")} for m in msgs[:-1]]
    if not any(m["role"] == "user" for m in prompt):
        return None
    if _contaminated(" ".join(m["content"] for m in prompt if m["role"] == "user")):
        return "contaminated"
    return _trace_domain(row), _chat_think(prompt, thinking if isinstance(thinking, str) else "", answer)


def _load_traces(n_tok_need):
    """Whole exchanges per domain, enough tokens for the requested blocks (streamed: nothing big is downloaded)."""
    want = {k: v * n_tok_need * 1.15 for k, v in V16_TRACE_SHARE.items()}
    got = {k: [] for k in V16_TRACE_SHARE}
    have = {k: 0 for k in V16_TRACE_SHARE}
    info = {"repo": None, "seen": 0, "dropped_contaminated": 0, "skipped_long": 0, "error": None}
    from datasets import load_dataset as _ld
    for repo in V16_TRACE_REPOS:
        try:
            try:
                ds = _ld(repo, V16_TRACE_CONFIG, split="train", streaming=True)
            except Exception:
                ds = _ld(repo, split="train", streaming=True)
            for row in ds:
                info["seen"] += 1
                if info["seen"] > V16_TRACE_SCAN or all(have[k] >= want[k] for k in want):
                    break
                if not isinstance(row, dict):
                    continue
                r = _trace_text(row)
                if r is None:
                    continue
                if r == "contaminated":
                    info["dropped_contaminated"] += 1
                    continue
                dom, text = r
                if have[dom] >= want[dom]:
                    continue
                ids = _tok_ids(text)
                if len(ids) > V16_TRACE_MAXTOK or len(ids) < 16:
                    info["skipped_long"] += int(len(ids) > V16_TRACE_MAXTOK)
                    if len(ids) > len(info.get("long_ids") or []):         # v17: the long-context curve reads the longest one
                        info["long_ids"], info["long_dom"] = ids, dom
                    continue
                got[dom].append(ids)
                have[dom] += len(ids)
            if sum(have.values()) > 0:
                info["repo"] = repo
                break
        except Exception as _e:
            info["error"] = f"{repo}: {type(_e).__name__}: {str(_e)[:120]}"
    return got, have, info


def _trace_stream(pool, n_tok, seed, start):
    """Token stream of whole exchanges, the domains interleaved by their shares. `start` = how many exchanges of each
    domain an earlier stream already used (training and held-out never share an exchange)."""
    share = {k: v for k, v in V16_TRACE_SHARE.items() if len(pool.get(k, [])) > start.get(k, 0)}
    tot = sum(share.values()) or 1.0
    ids, pos, got, counts = [], dict(start), {k: 0 for k in share}, {k: 0 for k in share}
    while len(ids) < n_tok and share:
        kind = max(share, key=lambda k: share[k] / tot - got[k] / max(1, len(ids)))
        if pos.get(kind, 0) >= len(pool[kind]):
            del share[kind]
            tot = sum(share.values()) or 1.0
            continue
        t = pool[kind][pos.get(kind, 0)]
        pos[kind] = pos.get(kind, 0) + 1
        ids.extend(t); got[kind] += len(t); counts[kind] += 1
    return ids, counts, pos


@torch.no_grad()
def _teacher_cache(blocks_t, path, tag="reasoning"):
    """Top-K log-probabilities of the DENSE teacher on blocks_t (same file format as the text cache)."""
    set_world("dense_teacher")

    def _run(bs):
        shards = []
        for i0 in range(0, blocks_t.shape[0], bs):
            batch = blocks_t[i0:i0 + bs].to(DEV)
            logits = model(input_ids=batch).logits
            B_, L_, _ = logits.shape
            ids_k = torch.empty(B_, L_, TEACHER_TOPK, dtype=torch.long)
            lp_k = torch.empty(B_, L_, TEACHER_TOPK, dtype=torch.float16)
            for t0 in range(0, L_, 512):
                lsm = logits[:, t0:t0 + 512].float().log_softmax(-1)
                tv, ti = lsm.topk(TEACHER_TOPK, dim=-1)
                ids_k[:, t0:t0 + 512] = ti.cpu()
                lp_k[:, t0:t0 + 512] = tv.cpu().half()
                del lsm
            shards.append({"input_ids": batch.cpu(), "teacher_ids": ids_k, "teacher_logprobs": lp_k})
            del logits
        return shards

    shards = oom_backoff(_run, [2, 1] if RUN_MODE == "full" else [8, 4, 2, 1], tag=f"teacher cache ({tag})")
    torch.save(shards, path + ".tmp")
    os.replace(path + ".tmp", path)
    set_world("serving")
    return shards


def _plain(blocks_t, src, rows=64):
    """v17: blocks without cached targets (the dense teacher labels them inside the training step)."""
    return [{"input_ids": blocks_t[i:i + rows].clone(), "src": src} for i in range(0, blocks_t.shape[0], rows)]


def _labelled(blocks_t, tag):
    if V17_INSTEP:
        return _plain(blocks_t, {"math": "problems", "trace": "traces"}.get(tag, tag)), "labelled in the step"
    sig = hashlib.sha1(f"{MODEL_ID}|{SEQ_LEN}|{TEACHER_TOPK}|{tag}".encode() + blocks_t.numpy().tobytes()).hexdigest()[:10]
    path = os.path.join(CACHE_DIR, f"teacher_{tag}_top{TEACHER_TOPK}_{sig}.pt")
    if os.path.exists(path) and not os.environ.get("PRISM_RECACHE"):
        return torch.load(path, weights_only=False), "reused"
    return _teacher_cache(blocks_t, path, tag), "built"


model.eval()
V15_EXTRA_SHARDS, V15_MATH_INFO, V16_TRACE_INFO = [], {}, {}
heldout_problems, heldout_traces, heldout_math = None, None, None
_blk = lambda ids, n: torch.tensor([ids[i * SEQ_LEN:(i + 1) * SEQ_LEN] for i in range(n)])
if V15_MATH_BLOCKS > 0:
    _t0 = time.time()
    _n_hold_gsm = min(200, len(_GSM) // 5)
    _ids, _cnt = _math_stream(V15_MATH_BLOCKS * SEQ_LEN, 2026, _GSM[:len(_GSM) - _n_hold_gsm])
    math_blocks = _blk(_ids, V15_MATH_BLOCKS)
    _idh, _cnth = _math_stream(V15_MATH_HOLD * SEQ_LEN, 4051, _GSM[len(_GSM) - _n_hold_gsm:])    # other problems
    heldout_problems = _blk(_idh, V15_MATH_HOLD)
    _ms, _how = _labelled(math_blocks, "math")
    V15_EXTRA_SHARDS += list(_ms) * max(1, int(V15_MATH_REPEAT))
    V15_MATH_INFO = {"blocks": int(V15_MATH_BLOCKS), "exchanges": _cnt, "heldout_blocks": int(V15_MATH_HOLD),
                     "heldout_exchanges": _cnth, "repeat": int(V15_MATH_REPEAT)}
    print(f"(1) PROBLEMS — {V15_MATH_BLOCKS} blocks x {SEQ_LEN} tokens: {_cnt['gsm']} GSM8K-train word problems, "
          f"{_cnt['one']} one-step and {_cnt['multi']} multi-step arithmetic exchanges, labelled by the dense teacher "
          f"({_how}, {time.time() - _t0:.0f}s) | held-out: {V15_MATH_HOLD} blocks of OTHER problems")
    del _ids, _idh, _ms
else:
    print("(1) problems are off (V15_MATH_BLOCKS = 0)")

trace_blocks = None
if V16_TRACE_BLOCKS > 0:
    _t0 = time.time()
    try:
        _pool, _have, _ti = _load_traces((V16_TRACE_BLOCKS + V16_TRACE_HOLD) * SEQ_LEN)
    except Exception as _e:
        _pool, _have, _ti = {}, {}, {"repo": None, "error": f"{type(_e).__name__}: {str(_e)[:160]}", "seen": 0}
    _tot = sum(_have.values()) if _have else 0
    _nh = int(V16_TRACE_HOLD) if _tot >= (V16_TRACE_HOLD + 2) * SEQ_LEN else 0
    _nb = min(int(V16_TRACE_BLOCKS), max(0, _tot // SEQ_LEN - _nh))
    if _nb >= 2 and _nh >= 1:
        # the held-out exchanges are taken FIRST (so a shorter pool shortens the training set, never the test)
        _idh, _cnth, _pos = _trace_stream(_pool, _nh * SEQ_LEN, 4052, {})
        _ids, _cnt, _ = _trace_stream(_pool, _nb * SEQ_LEN, 2027, _pos)
        _nb = min(_nb, len(_ids) // SEQ_LEN)
        _nh = min(_nh, len(_idh) // SEQ_LEN)
    if _nb >= 2 and _nh >= 1:
        trace_blocks, heldout_traces = _blk(_ids, _nb), _blk(_idh, _nh)
        _ts, _how = _labelled(trace_blocks, "trace")
        V15_EXTRA_SHARDS += list(_ts)
        V16_TRACE_INFO = {"repo": _ti["repo"], "blocks": int(_nb), "exchanges": _cnt, "heldout_blocks": int(_nh),
                          "heldout_exchanges": _cnth, "rows_seen": int(_ti["seen"]),
                          "dropped_contaminated": int(_ti.get("dropped_contaminated", 0)),
                          "skipped_long": int(_ti.get("skipped_long", 0))}
        print(f"(2) TRACES — {_nb} blocks x {SEQ_LEN} tokens from {_ti['repo']}: "
              + ", ".join(f"{v} {k}" for k, v in _cnt.items()) + f" exchanges with their thinking, labelled by the dense teacher "
              f"({_how}, {time.time() - _t0:.0f}s) | held-out: {_nh} blocks of OTHER exchanges | {_ti.get('dropped_contaminated', 0)} "
              f"rows dropped by the contamination filter, {_ti.get('skipped_long', 0)} too long, {_ti['seen']} rows read")
        del _ids, _idh, _ts
        if _ti.get("long_ids") is not None:
            V17_LONG_IDS = list(_ti["long_ids"])
            V17_LONG_SRC = f"the longest {_ti.get('long_dom')} exchange of {_ti['repo']} (skipped from training for its length)"
    else:
        V16_TRACE_INFO = {"repo": None, "blocks": 0, "error": _ti.get("error"), "rows_seen": int(_ti.get("seen", 0))}
        print(f"(2) TRACES — not available ({_ti.get('error') or 'no usable rows'}; {_ti.get('seen', 0)} rows read) -> the run "
              "goes on with text + problems, exactly the v15 data")
    del _pool
else:
    print("(2) traces are off (V16_TRACE_BLOCKS = 0)")

# (4) FRESH TEXT (v17): documents no earlier version read (the first V17_FRESH_SKIP of every source are skipped)
if V17_FRESH_BLOCKS > 0 and V17_INSTEP:
    _t0 = time.time()
    _fresh, _fc = [], {}
    _eos = tok.eos_token_id if tok.eos_token_id is not None else 0
    for _src, _sh in V17_FRESH_SHARE.items():
        _need = int(V17_FRESH_BLOCKS * _sh) * SEQ_LEN
        _ids, _seen = [], 0
        try:
            for _t in _stream_texts(_src):
                _seen += 1
                if _seen <= int(V17_FRESH_SKIP) or not _t or not _t.strip():
                    continue
                if _contaminated(_t):
                    continue
                _ids.extend(_tok_ids(_t)[:MIX_DOC_CAP])
                _ids.append(_eos)
                if len(_ids) >= _need:
                    break
        except Exception as _e:
            print(f"  fresh {_src}: unavailable ({type(_e).__name__}: {str(_e)[:90]})")
        _nb = len(_ids) // SEQ_LEN
        _fc[_src] = _nb
        _fresh += [_ids[i * SEQ_LEN:(i + 1) * SEQ_LEN] for i in range(min(_nb, int(V17_FRESH_BLOCKS * _sh)))]
    if _fresh:
        _random.Random(17).shuffle(_fresh)
        V15_EXTRA_SHARDS += _plain(torch.tensor(_fresh), "fresh")
    print(f"(4) FRESH TEXT — {len(_fresh)} blocks no version has read: " + ", ".join(f"{k} {v}" for k, v in _fc.items())
          + f" ({time.time() - _t0:.0f}s)")
    V17_FRESH_INFO = {"blocks": len(_fresh), **_fc}
    del _fresh

# the reasoning set the score reads: problems and traces interleaved (so "the first n blocks" always holds both)
_parts = [h for h in (heldout_problems, heldout_traces) if h is not None]
if _parts:
    _rows = []
    for _i in range(max(int(h.shape[0]) for h in _parts)):
        _rows += [h[_i] for h in _parts if _i < h.shape[0]]
    heldout_math = torch.stack(_rows)
V15_MATH_HOLD = int(heldout_math.shape[0]) if heldout_math is not None else 0
print(f"(3) training set: the v10 text blocks + {len(V15_EXTRA_SHARDS)} shards of problems / traces | measured on: "
      f"{heldout.shape[0]} text blocks" + (f", {V15_MATH_HOLD} reasoning blocks "
      f"({0 if heldout_problems is None else heldout_problems.shape[0]} problems + "
      f"{0 if heldout_traces is None else heldout_traces.shape[0]} traces)" if heldout_math is not None else ""))


def v15_score(e_text, e_math):
    """The number that picks checkpoints and decides the verdict: text KL + V15_MATH_W x reasoning KL."""
    return float(e_text[0]) + (float(V15_MATH_W) * float(e_math[0]) if e_math is not None else 0.0)


free_cuda("v16 data")


## MOLT 1 — the law: strict head, strict writes, audit, dense swap

In [ ]:
# v16 — MOLT, step 1: THE LAW  (every token, without exception, touches <= 3.0B parameters — and is MEASURED that way)
#
# An audit of the code path that produced every number up to v15 found two places where "3B per token" was not what
# was executed:
#
#   OUTPUT HEAD     the census charges the head FUNNEL (rank-r proxy over the vocabulary + exact rows for a shortlist,
#                   0.086B on the 27B). But the funnel was never wired into the model: every KL, every perplexity and
#                   every probe answer so far was computed with the DENSE head — 1.27B parameters that were not counted.
#   MEMORY WRITES   the DeltaNet write policy (MoRM 2.0, v8) admits a head when its score passes a threshold that a
#                   controller steers so that the MEAN number of exact writes per token is m. One token may write up to
#                   75% of the heads, another none; the census charged the mean.
#
# This cell makes both strict and then measures, for the first time, the model that the census describes:
#
#   strict head     logits_i = h . W_i (exact) for the V_s tokens with the best proxy score, the proxy value
#                   (h P) . U_i + b_i for every other token. The shortlist is part of the forward pass of the served
#                   world — evaluation, generation and (in the training cell) the loss. P, U and the new bias b become
#                   trainable later: the head is the one part of the student that has never seen a gradient.
#   strict writes   exactly m heads per token: the m best scores, no threshold, no average.
#
# Then two tables, both under the law:
#   AUDIT       the same held-out blocks read four ways (as before / strict writes / strict head / both): what the two
#               loopholes were worth.
#   DENSE SWAP  one family of sites at a time is given its dense computation back while everything else stays as
#               served: how much of the remaining distance to the teacher each family is responsible for. (Not
#               additive — a ranking of where the damage is.)
# From here on "the model" always means the model under the law; the numbers of v10..v15 are kept for the record only.
V16_STRICT_HEAD  = globals().get("V16_STRICT_HEAD", True)
V16_STRICT_WRITE = globals().get("V16_STRICT_WRITE", True)
V16_AUDIT_TEXT   = globals().get("V16_AUDIT_TEXT", 4 if RUN_MODE == "full" else 2)
V16_AUDIT_MATH   = globals().get("V16_AUDIT_MATH", 4 if RUN_MODE == "full" else 2)
V16_HEAD_GROW    = globals().get("V16_HEAD_GROW", True)      # v16.2: every free parameter under the 3B goes to exact head rows
V16_LIFT_MARGIN  = globals().get("V16_LIFT_MARGIN", 2.0)     # v16.2 LIFT: logits the teacher's tokens must clear the shortlist line by
V16_DENSE_SWAP   = globals().get("V16_DENSE_SWAP", False)     # measured on 2026-10-07 (every family: worse when dense) — not repeated
V16_DEC_P        = globals().get("V16_DEC_P", 0.9)       # a position is DECISIVE when the teacher puts at least this much on one token
V16_RUN          = globals().get("V16_RUN", 12)          # ... and a "run" is this many decisive positions in a row
V16_TL_MAX       = globals().get("V16_TL_MAX", 20 if RUN_MODE == "full" else 8)      # teacher logits kept in host RAM (blocks)
import hashlib as _hl16

V16 = globals().get("V16") if isinstance(globals().get("V16"), dict) else {}
V16.update({"head_strict": False, "write_strict": False, "audit": {}, "swap": {}, "law": None})
model.eval()
_LMH = getattr(model, "lm_head", None)
if V16_HEAD_GROW and (globals().get("V16_HEAD_ADD") or (RUN_MODE == "full" and TARGET_ACTIVE)):
    _k0 = int(funnel.shortlist)
    _add = int(globals().get("V16_HEAD_ADD") or max(0, (int(TARGET_ACTIVE) - int(measured_active()) - 2_000_000) // int(funnel.d)))
    funnel.shortlist = min(int(funnel.V), _k0 + _add // 256 * 256 if _add >= 256 else _k0 + _add)
    print(f"[v16.2] HEAD GROW: exact head rows {_k0} -> {int(funnel.shortlist)} (+{(int(funnel.shortlist) - _k0) * int(funnel.d) / 1e6:.1f}M, the free parameters under the 3B) | census {measured_active() / 1e9:.4f}B")
assert _LMH is not None and getattr(_LMH, "weight", None) is not None, "the model has no lm_head with a weight: the strict head cannot be installed"


# ================================================================ strict memory writes (the rule itself lives in the lens cell: a state
#                                                                  saved under the law must be strict from the first forward on)
def _v16_writes(strict):
    """strict=True: every policy writes exactly its budget; False: the elastic rule of v8..v15."""
    for _, m in RESONA_MORM:
        if strict:
            if getattr(m, "strict_m", None) is None:
                m.strict_m = int(getattr(m, "_strict_keep", None) or m.m_bar)
        else:
            if getattr(m, "strict_m", None) is not None:
                m._strict_keep = int(m.strict_m)
            m.strict_m = None
    V16["write_strict"] = bool(strict) and bool(RESONA_MORM)


# ================================================================ strict output head
def _fh_set(name, val):
    """(Re)place a head tensor on lm_head as an fp32 buffer (a later cell may turn it into a trained parameter)."""
    if name in getattr(_LMH, "_parameters", {}):
        del _LMH._parameters[name]
    if name in getattr(_LMH, "_buffers", {}):
        _LMH._buffers[name] = val.detach().float().contiguous()
    else:
        _LMH.register_buffer(name, val.detach().float().contiguous(), persistent=False)


_hx = STATE.get("extra", {}).get("head") if "STATE" in globals() else None
if isinstance(_hx, dict) and "U" in _hx:                  # a v16 state: the trained head
    _fh_set("ns_fP", _hx["P"].to(DEV)); _fh_set("ns_fU", _hx["U"].to(DEV)); _fh_set("ns_fb", _hx["b"].to(DEV))
    V16["head_restored"] = True
else:                                                      # the v10 funnel exactly as the census has charged it since v10
    _fh_set("ns_fP", funnel.P_r.to(DEV)); _fh_set("ns_fU", funnel.U_r.to(DEV))
    _fh_set("ns_fb", torch.zeros(funnel.V, device=DEV))
    V16["head_restored"] = False
if float(getattr(funnel, "kappa", 0.0) or 0.0) != 0.0:
    print(f"    (the funnel's residual term kappa={funnel.kappa} is dropped: the strict head ranks by proxy + bias; its V reads "
          "are the bias's now)")
    funnel.kappa = 0.0
_FH_CACHE = {}


def _fh_factors():
    """P and U in the substrate dtype (the dtype every served GEMM runs in). Cached while the masters do not change."""
    P, U, wd = _LMH.ns_fP, _LMH.ns_fU, _LMH.weight.dtype
    key = (id(P), getattr(P, "_version", None), id(U), getattr(U, "_version", None), str(wd))
    if None not in key and _FH_CACHE.get("key") == key:
        return _FH_CACHE["P"], _FH_CACHE["U"]
    Pb, Ub = P.detach().to(wd), U.detach().to(wd)
    if None not in key:
        _FH_CACHE.update({"key": key, "P": Pb, "U": Ub})
    return Pb, Ub


def _fh_proxy(hc, Pb, Ub, b):
    """hc [n, d] (substrate dtype) -> proxy logits [n, V] fp32 = (h P) U^T + b. They are BOTH the ranking that picks the
    shortlist and the value served for every token outside it."""
    return ((hc @ Pb) @ Ub.t()).float() + b


def _fh_mix(hc, W, Pb, Ub, b, k, extra=0):
    """Served logits [n, V] fp32 of the strict head, the proxy logits, the exact logits and the top-(k + extra) proxy ids."""
    zt = _fh_proxy(hc, Pb, Ub, b)
    ze = (hc @ W.t()).float()
    idx = zt.topk(min(int(k) + int(extra), zt.shape[-1]), dim=-1).indices
    mask = torch.zeros_like(zt).scatter(1, idx[:, :int(k)], torch.ones(idx[:, :int(k)].shape, dtype=zt.dtype, device=zt.device))
    return zt + mask * (ze - zt), zt, ze, idx, mask


def _fh_hook(mod, inp, out):
    """lm_head forward hook: in the served world the dense logits are replaced by the strict head's. (The dense logits
    of the shortlisted tokens are the exact rows the strict head reads; everything else is overwritten by the proxy.)"""
    if not V16.get("head_strict") or getattr(funnel, "default_mode", "dense") != "funnel":
        return None
    h = inp[0] if isinstance(inp, (tuple, list)) else inp
    if not torch.is_tensor(out) or out.shape[-1] != _LMH.weight.shape[0]:
        return None
    lead, V_, wd = out.shape[:-1], out.shape[-1], _LMH.weight.dtype
    h2, o2 = h.reshape(-1, h.shape[-1]), out.reshape(-1, V_)
    Pb, Ub = _fh_factors()
    k = min(int(funnel.shortlist), V_)
    parts = []
    for c0 in range(0, h2.shape[0], 256):
        zt = _fh_proxy(h2[c0:c0 + 256].to(wd), Pb, Ub, _LMH.ns_fb)
        idx = zt.topk(k, dim=-1).indices
        mask = torch.zeros_like(zt).scatter(1, idx, torch.ones(idx.shape, dtype=zt.dtype, device=zt.device))
        parts.append((zt + mask * (o2[c0:c0 + 256].float() - zt)).to(out.dtype))
        del zt, idx, mask
    return (parts[0] if len(parts) == 1 else torch.cat(parts, 0)).reshape(*lead, V_)


if "_FH_HANDLE" in globals() and _FH_HANDLE is not None:
    try:
        _FH_HANDLE.remove()
    except Exception:
        pass
_FH_HANDLE = _LMH.register_forward_hook(_fh_hook)


def _fh_active():
    """Parameters the strict head touches per token: U (V x r), P (d x r), the bias (V), the shortlisted exact rows."""
    r_ = int(_LMH.ns_fU.shape[1])
    return int(funnel.V * r_ + funnel.d * r_ + funnel.V + int(funnel.shortlist) * funnel.d)


funnel.active_params = _fh_active                        # the census reads the head through this
funnel.rho = None                                        # its V reads are the bias's now (kappa is 0)


def _v16_head(strict):
    V16["head_strict"] = bool(strict)


# ================================================================ the strict head inside the training loss
class _StrictHeadLoss(torch.autograd.Function):
    """The strict head inside the loss (hidden states in; a [B, L, V] tensor never exists). Chunk by chunk:
        zt = (h P) U^T + b         proxy logits
        ze = h W^T                 exact logits (one dense GEMM = the supergraph; only the shortlist's entries are used)
        z  = ze on the shortlist (the k best zt), zt everywhere else          <- the logits that are served
    Outputs: nll [B, L] and slog [B, L, K] with the meaning they have in _ChunkedHeadLoss, and px (a scalar): how far
    the proxy is from the student's own full head, RELATIVE to how much that head has to say,
        px = sum_ti p_ti (zt_ti - ze_ti)^2 / sum_ti p_ti (ze_ti - mean_t)^2 ,   p = softmax(ze), mean_t = sum_i p_ti ze_ti
    (0 = the proxy is the full head; 1 = as wrong as a constant). px reaches P, U and b only: the hidden state, p and
    the denominator are constants for it.
    Backward: G = dL/dz. The exact part of G goes to the hidden state through W, the proxy part to the hidden state,
    P, U and b. On the way the market's two numbers for the shortlist are read off G: what the loss would change, to
    first order, with dk more / dk fewer exact rows."""

    @staticmethod
    def forward(ctx, hidden, W, P, U, b, labels, tid, chunk, k, dk, tlp=None, wrec=0.0, ht=None):
        B, L, d = hidden.shape
        K = tid.shape[-1]
        nll = torch.empty(B, L, dtype=torch.float32, device=hidden.device)
        slog = torch.empty(B, L, K, dtype=torch.float32, device=hidden.device)
        fkl = torch.zeros(B, L, dtype=torch.float32, device=hidden.device)   # v17 FULL KL (needs the teacher's final hidden ht)
        px, den, rec = 0.0, 0.0, 0.0
        wrec = float(wrec) if tlp is not None else 0.0
        with torch.no_grad():
            Pb, Ub, bf = P.to(W.dtype), U.to(W.dtype), b.float()
            for c0 in range(0, L, int(chunk)):
                c1 = min(c0 + int(chunk), L)
                hc = hidden[:, c0:c1].reshape(-1, d).to(W.dtype)
                z, zt, ze, _ix, _ = _fh_mix(hc, W, Pb, Ub, bf, k)
                if wrec:                                   # LIFT: the teacher's tokens over the shortlist line
                    _th = zt.gather(1, _ix[:, k - 1:k])
                    _pt = tlp[:, c0:c1].reshape(-1, K).float().exp()
                    rec += float((_pt * F.softplus(_th + float(V16_LIFT_MARGIN) - zt.gather(1, tid[:, c0:c1].reshape(-1, K)))).sum())
                    del _th, _pt
                pf = ze.softmax(-1)
                px += float((pf * (zt - ze).pow(2)).sum())
                den += float((pf * (ze - (pf * ze).sum(-1, keepdim=True)).pow(2)).sum())
                del pf
                lse = z.logsumexp(-1)
                lab = labels[:, c0:c1].reshape(-1)
                nll[:, c0:c1] = (lse - z.gather(-1, lab[:, None]).squeeze(-1)).reshape(B, c1 - c0)
                tidc = tid[:, c0:c1].reshape(-1, K)
                slog[:, c0:c1] = (z.gather(-1, tidc) - lse[:, None]).reshape(B, c1 - c0, K)
                if ht is not None:                         # KL(teacher || served student) over ALL 248k tokens
                    _lpt = (ht[:, c0:c1].reshape(-1, d).to(W.dtype) @ W.t()).float().log_softmax(-1)
                    fkl[:, c0:c1] = (_lpt.exp() * (_lpt - (z - lse[:, None]))).sum(-1).reshape(B, c1 - c0)
                    del _lpt
                del hc, z, zt, ze, lse, lab, tidc
        npos = float(B * L)
        ctx.save_for_backward(hidden, W, P, U, b, labels, tid, tlp if wrec else None, ht)
        ctx.cfg = (int(chunk), int(k), int(dk), max(float(den), 1e-12), wrec / max(npos, 1.0))
        V16["lift_last"] = rec / max(npos, 1.0)
        return nll, slog, torch.tensor(px / max(float(den), 1e-12) + wrec * rec / max(npos, 1.0), dtype=torch.float32, device=hidden.device), fkl

    @staticmethod
    def backward(ctx, g_nll, g_slog, g_px, g_fkl=None):
        hidden, W, P, U, b, labels, tid, tlp, ht = ctx.saved_tensors
        chunk, k, dk, den, wr = ctx.cfg
        B, L, d = hidden.shape
        K = tid.shape[-1]
        gh = torch.empty_like(hidden)
        gP = torch.zeros(P.shape, dtype=torch.float32, device=P.device)
        gU = torch.zeros(U.shape, dtype=torch.float32, device=U.device)
        gb = torch.zeros(b.shape, dtype=torch.float32, device=b.device)
        gpx = float(g_px) if g_px is not None else 0.0
        mk_p, mk_m = 0.0, 0.0
        with torch.no_grad():
            wd = W.dtype
            Pb, Ub, bf = P.to(wd), U.to(wd), b.float()
            for c0 in range(0, L, chunk):
                c1 = min(c0 + chunk, L)
                hc = hidden[:, c0:c1].reshape(-1, d).to(wd)
                z, zt, ze, idx, mask = _fh_mix(hc, W, Pb, Ub, bf, k, extra=dk)
                p = z.softmax(-1)
                del z
                lab = labels[:, c0:c1].reshape(-1)
                if g_nll is not None:
                    g = g_nll[:, c0:c1].reshape(-1, 1).to(p.dtype) * p
                    g.scatter_add_(-1, lab[:, None], (-g_nll[:, c0:c1].reshape(-1)).to(p.dtype)[:, None])
                else:
                    g = torch.zeros_like(p)
                if g_fkl is not None and ht is not None:     # d KL(p_t || q) / dz = (q - p_t), per position
                    _gf = g_fkl[:, c0:c1].reshape(-1, 1).to(p.dtype)
                    _pt = (ht[:, c0:c1].reshape(-1, d).to(wd) @ W.t()).float().softmax(-1).to(p.dtype)
                    g += _gf * (p - _pt)
                    del _gf, _pt
                if g_slog is not None:
                    tidc = tid[:, c0:c1].reshape(-1, K)
                    gs = g_slog[:, c0:c1].reshape(-1, K).to(p.dtype)
                    g.scatter_add_(-1, tidc, gs)
                    g -= gs.sum(-1, keepdim=True) * p
                del p
                diff = ze - zt
                if dk and idx.shape[1] >= k + dk and k - dk >= 0:
                    v = g * diff
                    mk_p += float(v.gather(1, idx[:, k:k + dk]).sum())        # loss change if the next dk rows were exact
                    mk_m -= float(v.gather(1, idx[:, k - dk:k]).sum())        # loss change if the last dk rows were proxy
                    del v
                ge = g * mask
                gt = g - ge                                                    # the part of G that sits on proxy logits
                del g, mask
                A = gt.to(wd) @ Ub                                             # [n, r]: task gradient w.r.t. (h P)
                gh[:, c0:c1] = ((ge.to(wd) @ W) + (A @ Pb.t())).reshape(B, c1 - c0, d).to(hidden.dtype)
                del ge
                if gpx != 0.0:                                                 # d px / d zt = 2 softmax(ze) (zt - ze) / den
                    gt = gt - (2.0 * gpx / den) * (ze.softmax(-1) * diff)
                    if wr and tlp is not None:                 # LIFT: d/dzt_j = -p_j sigmoid(line + margin - zt_j)
                        _tc = tid[:, c0:c1].reshape(-1, K)
                        _th = zt.gather(1, idx[:, k - 1:k])
                        _pt = tlp[:, c0:c1].reshape(-1, K).float().exp()
                        _gl = -_pt * torch.sigmoid(_th + float(V16_LIFT_MARGIN) - zt.gather(1, _tc))
                        gt = gt.scatter_add(1, _tc, (gpx * wr) * _gl.to(gt.dtype))
                        del _tc, _th, _pt, _gl
                    A = gt.to(wd) @ Ub
                gP += (hc.t() @ A).float()
                gU += (gt.to(wd).t() @ (hc @ Pb)).float()
                gb += gt.sum(0)
                del hc, zt, ze, idx, diff, gt, A
        if dk:
            _hg = _MK.setdefault("head_g", [0.0, 0.0])
            _hg[0] += mk_p
            _hg[1] += mk_m
        return gh, None, gP, gU, gb, None, None, None, None, None, None, None, None


def v16_fork(tlp, slog, alpha):
    """FORK LOSS, per position:  -(1 - q) log q ,  q = the student's probability on the teacher's ACCEPTABLE tokens
    (those the teacher gives at least alpha x the probability of its best token). Where the teacher is sure, q is the
    probability of the one right token; where the teacher is open, any of its good tokens will do. A position the
    student already gets right contributes nothing; a FORK — the student walking off where the teacher would not —
    contributes its full log-loss. tlp / slog [..., K]: teacher / student log-probabilities at the teacher's top-K ids
    (the teacher's are sorted, best first)."""
    tl = tlp.float()
    out_ = tl < (tl[..., :1] + math.log(float(alpha)))
    lq = torch.logsumexp(slog.float().masked_fill(out_, -1e30), -1).clamp(max=-1e-6)
    return -((1.0 - lq.exp()).detach() * lq)


# ================================================================ the dense-swap switch (measurement only)
V16_DENSE = set()                                        # families that run dense INSIDE the served world
_lin_kind = lambda n: ".".join(n.split(".")[-2:])
if "_set_world_v15" not in globals():
    _set_world_v15 = set_world


def set_world(mode: str):
    r = _set_world_v15(mode)
    if mode == "serving" and V16_DENSE:
        if "ffn" in V16_DENSE:
            for _, m in PRISM_MODULES:
                m.default_mode, m.use_shadow = "uniform", False
        for n, m in RESONA_LINEARS:
            if _lin_kind(n) in V16_DENSE:
                m.default_mode = "dense"
        if "sectioned" in V16_DENSE:
            for _, m in RESONA_SECTIONED:
                m.default_mode = "dense"
        if "head" in V16_DENSE:
            for _, m in RESONA_FUNNEL:
                m.default_mode = "dense"
    return r


# ================================================================ evaluation: KL, agreement, DECISIVE agreement, runs
_TL = {}                                                  # dense teacher logits of the blocks that are read again and again (host RAM)


def _teacher_logits(b):
    key = _hl16.sha1(b.cpu().numpy().tobytes()).hexdigest()
    if key in _TL:
        return _TL[key].to(DEV)
    set_world("dense_teacher")
    lt = model(input_ids=b).logits[0]
    if len(_TL) < int(V16_TL_MAX):
        _TL[key] = lt.detach().to("cpu")
    return lt


@torch.no_grad()
def v15_eval(blocks_, n_max):
    """(KL(teacher || served student) per token, share of positions where both put the same token first).
    v15_eval.last additionally holds what the gates depend on: the agreement on the DECISIVE positions (the teacher puts
    >= V16_DEC_P on one token: a digit of an answer, not a choice of words) and the share of runs of V16_RUN decisive
    positions in a row on which the student agrees every time."""
    tot, agree, n = 0.0, 0, 0
    d_n, d_ok, r_n, r_ok = 0, 0, 0, 0
    sp_sum, sp_max, sp_n = 0.0, None, 0
    lcut = math.log(float(V16_DEC_P))
    for i in range(min(int(n_max), blocks_.shape[0])):
        b = blocks_[i:i + 1].to(DEV)
        lt = _teacher_logits(b)
        set_world("serving")
        ls = model(input_ids=b).logits[0]
        _rc = _wal_receipt() if "_wal_receipt" in globals() else None      # the wallet's receipt of this forward
        if _rc is not None:
            sp_sum, sp_n = sp_sum + _rc[1], sp_n + 1
            sp_max = _rc[2] if sp_max is None else max(sp_max, _rc[2])
        oks = []
        for c0 in range(0, lt.shape[0], 256):
            pt = lt[c0:c0 + 256].float().log_softmax(-1)
            ps = ls[c0:c0 + 256].float().log_softmax(-1)
            tot += float((pt.exp() * (pt - ps)).sum())
            tv, ta = pt.max(-1)
            ok = (ta == ps.argmax(-1))
            agree += int(ok.sum())
            n += int(pt.shape[0])
            oks.append(ok[tv >= lcut])
            del pt, ps
        d = torch.cat(oks).float() if oks else torch.zeros(0)
        d_n += int(d.numel())
        d_ok += int(d.sum())
        nw = int(d.numel()) // int(V16_RUN)
        if nw:
            r_n += nw
            r_ok += int((d[:nw * int(V16_RUN)].reshape(nw, int(V16_RUN)).sum(-1) >= int(V16_RUN) - 0.5).sum())
        del lt, ls
    set_world("serving")
    v15_eval.last = {"decisive": d_ok / max(d_n, 1), "decisive_share": d_n / max(n, 1), "run": r_ok / max(r_n, 1),
                     "n_decisive": d_n, "n_runs": r_n,
                     "spent_mean": (sp_sum / sp_n if sp_n else None), "spent_max": sp_max}
    return tot / max(n, 1), agree / max(n, 1)


v15_eval.last = {}


def _v16_read(tag=None):
    """One audit line: text and reasoning KL / agreement / decisive agreement / runs on the audit blocks."""
    et = v15_eval(heldout, V16_AUDIT_TEXT)
    lt = dict(v15_eval.last)
    em, lm = (None, {})
    if globals().get("heldout_math") is not None:
        em = v15_eval(heldout_math, V16_AUDIT_MATH)
        lm = dict(v15_eval.last)
    out = {"text_kl": float(et[0]), "text_agree": float(et[1]), "text_decisive": float(lt.get("decisive", 0.0)),
           "reasoning_kl": (float(em[0]) if em else None), "reasoning_agree": (float(em[1]) if em else None),
           "reasoning_decisive": (float(lm.get("decisive", 0.0)) if em else None),
           "reasoning_run": (float(lm.get("run", 0.0)) if em else None), "census": int(measured_active())}
    if tag is not None:
        print(f"    {tag:<44} text KL {out['text_kl']:.4f} | reasoning KL "
              + (f"{out['reasoning_kl']:.4f}" if em else "   n/a")
              + f" | decisive agreement text {100 * out['text_decisive']:.1f}%"
              + (f", reasoning {100 * out['reasoning_decisive']:.1f}% (runs of {V16_RUN}: {100 * out['reasoning_run']:.1f}%)" if em else "")
              + f" | census {out['census'] / 1e9:.3f}B")
    return out


def _wal_measuring(flag):
    """A state saved with the wallet lets a token run a debt against what LATER sites will refund. In the comparison
    lines below some sites are not executed (dense, or the old write rule), so nobody would collect: while they are
    measured the credit line is closed (a token can then only spend what it has saved — still strict)."""
    if "_WAL" in globals() and _WAL.get("order"):
        if flag:
            _WAL.setdefault("_rho_keep", _WAL["rho"])
            _WAL["rho"] = 0.0
        elif "_rho_keep" in _WAL:
            _WAL["rho"] = _WAL.pop("_rho_keep")


# ================================================================ AUDIT
_t0 = time.time()
print(f"(1) AUDIT — {V16_AUDIT_TEXT} text + {V16_AUDIT_MATH if globals().get('heldout_math') is not None else 0} reasoning held-out blocks, "
      f"read four ways (decisive = the teacher puts >= {V16_DEC_P:g} on one token):")
_was_strict = any(getattr(m, "strict_m", None) is not None for _, m in RESONA_MORM)
_wal_measuring(True)
_v16_writes(False); _v16_head(False)
if _was_strict or V16["head_restored"]:
    print("    (this state was saved under the law: the first three lines show it with the loopholes re-opened, for comparison only)")
V16["audit"]["as_before"] = _v16_read("as measured until v15 (dense head, mean writes)")
if RESONA_MORM:
    _mm = [m.last_m.float() for _, m in RESONA_MORM if getattr(m, "last_m", None) is not None]
    if _mm:
        _mx = max(float(t.max()) for t in _mm)
        _mn = min(float(t.min()) for t in _mm)
        _H = max(int(m.H) for _, m in RESONA_MORM)
        V16["audit"]["writes_seen"] = {"min": _mn, "max": _mx, "heads": _H, "budget": int(RESONA_MORM[0][1].m_bar)}
        print(f"        exact writes per token under the mean rule (last block): between {_mn:.0f} and {_mx:.0f} of {_H} heads "
              f"(the census charged {RESONA_MORM[0][1].m_bar})")
_v16_writes(True)
V16["audit"]["strict_writes"] = _v16_read("+ strict writes (exactly m heads per token)")
_v16_writes(False); _v16_head(True)
V16["audit"]["strict_head"] = _v16_read("+ strict head (the funnel that was counted)")
_v16_writes(bool(V16_STRICT_WRITE)); _v16_head(bool(V16_STRICT_HEAD))
_wal_measuring(False)
V16["law"] = _v16_read("THE LAW (both)" if (V16_STRICT_WRITE and V16_STRICT_HEAD) else "the model as it will be trained")
V16["audit"]["law"] = V16["law"]
_a, _l = V16["audit"]["as_before"], V16["law"]
print(f"    => under the law the text KL is {_l['text_kl']:.4f} (it was reported as {_a['text_kl']:.4f}); that difference is what the two "
      f"loopholes were worth. Every number below is under the law. | {time.time() - _t0:.0f}s")
try:                                                       # the head by itself: how often the student's own best token survives the shortlist
    _hid = {}
    _hk = _LMH.register_forward_pre_hook(lambda mod, args: _hid.__setitem__("h", (args[0] if isinstance(args, (tuple, list)) else args).detach()))
    set_world("serving")
    with torch.no_grad():
        model(input_ids=heldout[0:1].to(DEV))
    _hk.remove()
    _h2 = _hid["h"].reshape(-1, _hid["h"].shape[-1])[:256].to(_LMH.weight.dtype)
    with torch.no_grad():
        _Pb, _Ub = _fh_factors()
        _z, _zt, _ze, _ix, _mk_ = _fh_mix(_h2, _LMH.weight, _Pb, _Ub, _LMH.ns_fb, min(int(funnel.shortlist), funnel.V))
        _pf = _ze.softmax(-1)
        V16["head_quality"] = {"top1_in_shortlist": float(_mk_.gather(1, _ze.argmax(-1, keepdim=True)).mean()),
                               "mass_in_shortlist": float((_pf * _mk_).sum(-1).mean())}
    print(f"    the head by itself (256 positions): the dense head's first token is inside the shortlist "
          f"{100 * V16['head_quality']['top1_in_shortlist']:.1f}% of the time; the shortlist holds "
          f"{100 * V16['head_quality']['mass_in_shortlist']:.1f}% of the dense head's probability")
    del _hid, _h2, _z, _zt, _ze, _ix, _mk_, _pf
except Exception as _e:
    print(f"    (head-quality probe skipped: {type(_e).__name__}: {str(_e)[:120]})")
B_LAW = int(measured_active())
if RUN_MODE == "full" and TARGET_ACTIVE:
    assert B_LAW <= TARGET_ACTIVE, f"census under the law {B_LAW / 1e9:.3f}B is over the {TARGET_ACTIVE / 1e9:.1f}B mandate"

# ================================================================ DENSE SWAP
if V16_DENSE_SWAP:
    _t0 = time.time()
    print(f"\n(2) DENSE SWAP — one family dense, everything else as served under the law (how much KL that family alone gives back):")
    _fams = [("FFN, all layers", {"ffn"}, None)]
    for _kind in sorted({_lin_kind(n) for n, _ in RESONA_LINEARS}):
        _fams.append((_kind, {_kind}, None))
    if RESONA_SECTIONED:
        _fams.append(("linear_attn in_proj (q, k, v, z)", {"sectioned"}, None))
    if RESONA_MORM and V16["write_strict"]:
        _fams.append(("memory writes only (all heads exact)", set(), "writes"))
    _fams.append(("output head (dense)", {"head"}, None))
    _wal_measuring(True)
    for _tag, _set, _special in _fams:
        try:
            V16_DENSE.clear()
            V16_DENSE.update(_set)
            _keep = None
            if _special == "writes":
                _keep = [int(m.strict_m) for _, m in RESONA_MORM]
                for _, m in RESONA_MORM:
                    m.strict_m = int(m.H)
            _r = _v16_read()
            _d_t = _l["text_kl"] - _r["text_kl"]
            _d_m = (_l["reasoning_kl"] - _r["reasoning_kl"]) if _r["reasoning_kl"] is not None else None
            V16["swap"][_tag] = {**_r, "text_gain": _d_t, "reasoning_gain": _d_m}
            print(f"    {_tag:<40} text KL {_r['text_kl']:.4f} ({-_d_t:+.4f})"
                  + (f" | reasoning KL {_r['reasoning_kl']:.4f} ({-_d_m:+.4f}) | decisive {100 * _r['reasoning_decisive']:.1f}%"
                     if _d_m is not None else ""))
        except Exception as _e:
            if is_oom(_e):
                free_cuda("dense swap OOM")
            print(f"    {_tag:<40} skipped ({type(_e).__name__}: {str(_e)[:100]})")
        finally:
            V16_DENSE.clear()
            if _special == "writes" and _keep is not None:
                for (_, m), _v in zip(RESONA_MORM, _keep):
                    m.strict_m = _v
    _wal_measuring(False)
    set_world("serving")
    if V16["swap"]:
        _top = max(V16["swap"], key=lambda k_: V16["swap"][k_]["text_gain"] + 0.5 * (V16["swap"][k_]["reasoning_gain"] or 0.0))
        print(f"    => the largest single share of the damage: {_top} | {time.time() - _t0:.0f}s")
else:
    print("\n(2) dense swap is off (V16_DENSE_SWAP = False)")
set_world("serving")
free_cuda("v16 law")


## What trains, the wallet, LOOP, the teacher in the step, pre-flight, model to beat, long curve before

In [ ]:
# v16 — MOLT, steps 2 and 3: THE MARKET and OWN NEURONS (preparation of the training run)
#
# Up to v15 the student was the teacher seen through a mask: frozen dense weights, a plan of how many neurons /
# row-blocks / memory heads each site may read (v10's budget solver; re-spent once by v14's training-free market, in
# ~30 coarse depth bands, for 1% of KL), and low-rank patches trained around it. The cell above put the model under the law. This cell lets the student take
# ownership of the three things it has only borrowed so far:
#
#   ITS HEAD      the strict head's proxy (P, U) and bias train — the one part of the model that never saw a gradient.
#   ITS BUDGET    THE WALLET. Up to here every site reads a fixed number of items and every token reads exactly that.
#                 Now a site posts a PRICE (a threshold on its score) and every token carries a wallet through the
#                 depth: it buys what clears the price — its own number at every site — pays above-quota purchases
#                 with what it saved below quota elsewhere (or with a debt that later sites must collect), and can
#                 never end below zero. So the ceiling of 3.0B holds for EVERY token by arithmetic (see the lens cell),
#                 while the mix of what a token reads is its own. Prices are learned by tatonnement on the marginal
#                 value that a zero-valued probe reads off the ordinary backward pass (_wal_prices below).
#                 The per-site QUOTA MARKET (C2) stays as the stand-in: it trades only if the wallet is switched off.
#   ITS NEURONS   per layer the V16_OWN_M neurons that carry most of the served FFN output get a trainable copy of their
#                 weights (see the lens cell). Not a low-rank patch: the student's own full-rank, nonlinear features.
#                 The teacher's weights are not touched; at the start the copies are equal to the originals.
#
# And the loss is aimed at what the gates measure:
#   FORK LOSS     -(1 - q) log q with q = the student's probability on the tokens the teacher finds acceptable (see the
#                 law cell). An answer written alone fails at its FIRST fork; a KL averaged over all positions does not
#                 know that. v15: 77.6% token agreement on reasoning text and 1/20 on one-step arithmetic — 0.776^12 = 5%.
#
# Kept from v15: consequence head, self-refitting lens, awakened attention shadows (as many as the memory holds), the
# stream anchor, pre-flight with fallbacks, governor, resume file, exact rollback.
V15_STEPS        = globals().get("V15_STEPS", 320 if RUN_MODE == "full" else 8)
V15_GRAD_ACCUM   = globals().get("V15_GRAD_ACCUM", 8 if RUN_MODE == "full" else GRAD_ACCUM)   # sequences per optimizer step (v10..v13: 16).
#                  Half the batch = twice the updates for the same GPU time; v13 made most of its gain in its first 100 updates.
TRAIN_BUDGET_MIN = globals().get("TRAIN_BUDGET_MIN", 200 if RUN_MODE == "full" else 30)   # wall-clock cap of THIS session's training
LAMBDA_ANCHOR    = globals().get("LAMBDA_ANCHOR", 0.25)        # stream anchor (v13); 0 = off (and no in-step teacher pass)
ANCHOR_STRIDE    = globals().get("ANCHOR_STRIDE", 4)
LR_LENS          = globals().get("LR_LENS", 2e-5)             # FFN lens shadows (as in v13)
# 1. consequence head
V15_HEAD         = globals().get("V15_HEAD", True)
V15_HEAD_RANK    = globals().get("V15_HEAD_RANK", 8 if RUN_MODE == "full" else 2)       # context rank (0 = one number per neuron only)
LR_HEAD          = globals().get("LR_HEAD", 3e-3)
LR_GHOST         = globals().get("LR_GHOST", 3e-3)            # ghost gains (numbers of order 1, one per neuron)
V15_PREFLIGHT    = globals().get("V15_PREFLIGHT", True)
# 2./3. what else trains: "auto" = as much as the free GPU memory holds, deepest layers first
V15_TRAIN_ATTN   = globals().get("V15_TRAIN_ATTN", "auto")    # attention shadows: "auto" | True | False
V15_TRAIN_PRED   = globals().get("V15_TRAIN_PRED", "auto")    # lens predictor:    "auto" | True | False
LR_ATTN_REL      = globals().get("LR_ATTN_REL", 2e-3)         # step size relative to each tensor's own RMS
LR_PRED_REL      = globals().get("LR_PRED_REL", 5e-4)
LAMBDA_LENS      = globals().get("LAMBDA_LENS", 0.05)         # the lens regression reaches the predictor only, and Adam ignores its scale;
#                  it is kept small so that it does not eat the shared gradient-norm clip of everything else
V15_VRAM_RESERVE_GIB = globals().get("V15_VRAM_RESERVE_GIB", 9.5)     # left free for the activations of one training step
#                  (v15 ran with 8.0 and ended 3.2 GiB short of the limit at its peak; the strict head loss needs ~1.5 GiB more)
# 4. MOLT
V16_HEAD_TRAIN   = globals().get("V16_HEAD_TRAIN", True)      # the strict head's proxy and bias train
LR_FH_REL        = globals().get("LR_FH_REL", 3e-3)           # P, U: step relative to each tensor's RMS
LR_FH_BIAS       = globals().get("LR_FH_BIAS", 1e-2)          # the bias (logit units; starts at 0)
LAMBDA_PX        = globals().get("LAMBDA_PX", 0.1)
LAMBDA_LIFT      = globals().get("LAMBDA_LIFT", 1.0)          # v16.2 LIFT (reaches P, U, b only)            # proxy -> the student's own full head (reaches P, U, b only)
V16_OWN          = globals().get("V16_OWN", True)             # own neurons
V16_OWN_M        = globals().get("V16_OWN_M", 256 if RUN_MODE == "full" else 8)       # per layer
LR_OWN_REL       = globals().get("LR_OWN_REL", 2e-3)
V16_OWN_TOL      = globals().get("V16_OWN_TOL", 0.01)         # installing the copies may move the KL by at most this share (they are copies)
LAMBDA_FORK      = globals().get("LAMBDA_FORK", 2.0)          # fork loss (0 = off)
V16_FORK_ALPHA   = globals().get("V16_FORK_ALPHA", 0.5)       # acceptable = at least this share of the teacher's best token
V16_MARKET       = globals().get("V16_MARKET", True)
MK_EVERY         = globals().get("MK_EVERY", 10 if RUN_MODE == "full" else 2)         # a trading round every this many updates
MK_START         = globals().get("MK_START", 20 if RUN_MODE == "full" else 2)         # ... from this update on (after the warm-up)
MK_STOP_FRAC     = globals().get("MK_STOP_FRAC", 0.85)        # ... until this share of the run (the end is for settling)
MK_MIN_OBS       = globals().get("MK_MIN_OBS", 6 if RUN_MODE == "full" else 2)        # updates a price must have been observed
MK_Z             = globals().get("MK_Z", 1.0)                 # confidence bounds: mean -/+ MK_Z standard errors
MK_RHO           = globals().get("MK_RHO", 2.0)               # a buyer must gain this many times what the sellers lose
MK_ROUND_FRAC    = globals().get("MK_ROUND_FRAC", 0.004)      # at most this share of the 3B changes hands in one round
MK_DRIFT         = globals().get("MK_DRIFT", 0.5)             # a site's holding stays within +/- this share of where it started
MK_DK_FFN        = globals().get("MK_DK_FFN", 16 if RUN_MODE == "full" else 2)        # unit of trade: neurons per FFN layer
MK_DK_HEAD       = globals().get("MK_DK_HEAD", 256 if RUN_MODE == "full" else 8)      # ... vocabulary rows of the head
MK_MARGIN        = globals().get("MK_MARGIN", 1_000_000 if RUN_MODE == "full" else 0) # the census stays this far under the mandate
V16_WALLET       = globals().get("V16_WALLET", True)          # per-token wallets (False: the quota market above trades instead)
WAL_START        = globals().get("WAL_START", 10 if RUN_MODE == "full" else 1)        # the window starts to open at this update ...
WAL_OPEN         = globals().get("WAL_OPEN", 110 if RUN_MODE == "full" else 4)        # ... and is fully open at this one
WAL_MAX          = globals().get("WAL_MAX", 1.0)              # how far the window opens (1 = the full range below)
WAL_SPAN_FFN     = globals().get("WAL_SPAN_FFN", 1.0)   # v17: 0.5 -> 1.0 (v16.2: the hungriest tokens sat exactly on the +50% cap)         # a token may read between (1 - this) and (1 + this) x the neuron quota of a layer
WAL_MORE_BLOCKS  = globals().get("WAL_MORE_BLOCKS", 6)   # v17: 3 -> 6        # ... between 0 and quota + this many row-blocks of a routed projection
WAL_MORE_HEADS   = globals().get("WAL_MORE_HEADS", 8)    # v17: 4 -> 8         # ... between 0 and quota + this many exact memory writes
WAL_RHO          = globals().get("WAL_RHO", 0.5)              # credit line: a token may owe this share of what later sites can refund
WAL_BETA         = globals().get("WAL_BETA", 1.0)             # pacing: savings lower / debts raise a token's prices by up to this many score spreads
WAL_BREF         = globals().get("WAL_BREF", 0.1)             # ... reached at this share of the tradable volume
WAL_TAU          = globals().get("WAL_TAU", 0.5)              # width of "at the price" (in score spreads) for the price probes
WAL_EVERY        = globals().get("WAL_EVERY", 5 if RUN_MODE == "full" else 2)         # prices move every this many updates
WAL_LR           = globals().get("WAL_LR", 0.15)              # ... by at most this many score spreads relative to each other
WAL_GLR          = globals().get("WAL_GLR", 0.25)             # ... and all together by at most this many
# ---- v17 knobs
V17_FULLKL       = globals().get("V17_FULLKL", True)           # FULL KL: the teacher's whole 248k distribution, not its top 64
V17_DRIFT        = globals().get("V17_DRIFT", 0.5)             # DRIFT: share of the student's own forks written into the batch
V17_DRIFT_MAX    = globals().get("V17_DRIFT_MAX", 0.06)        # ... at most this share of the positions of a sequence
V17_DRIFT_EVERY  = globals().get("V17_DRIFT_EVERY", 2 if RUN_MODE == "full" else 1)   # ... in one micro-batch of this many
V17_WAL_UNSPENT  = globals().get("V17_WAL_UNSPENT", True)      # price level follows the unspent budget (v16: the wants)
V17_WAL_TARGET   = globals().get("V17_WAL_TARGET", 0.002)      # ... aiming at this share of the ceiling left unspent per token
V17_INSTEP       = globals().get("V17_INSTEP", True)           # the dense teacher labels every batch inside the step (no cache)
V17_LOOP_ON      = globals().get("V17_LOOP_ON", True)          # LOOP: the student writes answers, the teacher grades every token
V17_LOOP_EVERY   = globals().get("V17_LOOP_EVERY", 24 if RUN_MODE == "full" else 2)   # one micro-batch row in this many is LOOP
V17_LOOP_N       = globals().get("V17_LOOP_N", 24 if RUN_MODE == "full" else 4)      # problems the student answers per round
V17_LOOP_BATCH   = globals().get("V17_LOOP_BATCH", 12 if RUN_MODE == "full" else 4)  # decoded together (equal prompt lengths)
V17_LOOP_NEW     = globals().get("V17_LOOP_NEW", 10 if RUN_MODE == "full" else 4)    # new tokens at most per answer
V17_WATCH_EVERY  = globals().get("V17_WATCH_EVERY", 25 if RUN_MODE == "full" else 4)
V17_LONG_MAX     = globals().get("V17_LONG_MAX", 32768 if RUN_MODE == "full" else 256)  # the long-context curve reads up to this many tokens
WAL_PRINT        = globals().get("WAL_PRINT", 4)
# 5. schedule / safety / persistence
V15_WARMUP       = globals().get("V15_WARMUP", 30 if RUN_MODE == "full" else 2)
V15_FLAT_FRAC    = globals().get("V15_FLAT_FRAC", 0.5)        # full learning rate until this fraction of the run (steps OR minutes, whichever is further) ...
V15_LR_FLOOR     = globals().get("V15_LR_FLOOR", 0.15)        # ... then a cosine down to this fraction (never to zero)
V15_EVAL_EVERY   = globals().get("V15_EVAL_EVERY", 50 if RUN_MODE == "full" else 4)
V15_EVAL_BLOCKS  = globals().get("V15_EVAL_BLOCKS", 4 if RUN_MODE == "full" else 2)
V15_EVAL_MATH    = globals().get("V15_EVAL_MATH", 3 if RUN_MODE == "full" else 2)
V15_GOV_TOL      = globals().get("V15_GOV_TOL", 1.06)         # an eval this much worse than the best one is a strike
V15_GOV_PATIENCE = globals().get("V15_GOV_PATIENCE", 2)       # strikes in a row -> back to the best state, learning rates halved
V15_GOV_MAX_CUTS = globals().get("V15_GOV_MAX_CUTS", 3)
V15_PERSIST_EVERY = globals().get("V15_PERSIST_EVERY", 100 if RUN_MODE == "full" else 4)
V15_RESUME_ON    = globals().get("V15_RESUME_ON", True)       # continue from v17_state/resume.pt when a session was cut
V15_FINALIZE_ONLY = globals().get("V15_FINALIZE_ONLY", False) # True: no more training — verdict, save and probe of the resumed best state
V15_PUSH_RESUME  = globals().get("V15_PUSH_RESUME", globals().get("PRISM_ENV") == "other")   # also upload the resume file to the Hub

from transformers import TrainerCallback
import hashlib, contextlib, random as _random, re as _re15, shutil as _shutil

model.eval()
_L = len(PRISM_MODULES)
_has = [hasattr(m, "ns_B") for _, m in PRISM_MODULES]
assert any(_has), "no lens buffers on any layer — the lens cell above must run (or restore) first"
V12_MASK = [bool(getattr(m, "ns_on", False)) for _, m in PRISM_MODULES]      # the lens exactly where the state had it
V15_DIR = os.path.join(DRIVE_DIR, "v17_state")
os.makedirs(V15_DIR, exist_ok=True)
LOCAL_DIR = globals().get("LOCAL_DIR") or ("/content/prism_local" if globals().get("PRISM_ENV") == "colab" else
                                           os.path.join(os.path.dirname(os.environ["HF_HOME"].rstrip("/")), "prism_local")
                                           if os.environ.get("HF_HOME") else CACHE_DIR)
os.makedirs(LOCAL_DIR, exist_ok=True)
_LRX = _re15.compile(r"layers\.(\d+)(?:\.|$)")


def _lay(path):
    mm = _LRX.search(path)
    return int(mm.group(1)) if mm else 0


assert "v15_score" in globals() and "V15_EXTRA_SHARDS" in globals(), "the data cell must run before this one"
assert isinstance(globals().get("V16"), dict) and V16.get("law") is not None, "the law cell must run before this one"

# ================================================================ (B) consequence head (paid for with exact neurons)
_ns_apply(V12_MASK)
B_START = measured_active()                                   # the budget of the model as it reached this cell
_ns_apply(_has)
V15_K0 = [(int(m.ns_k), int(m.ns_active), bool(getattr(m, "ns_hd", False))) if _has[li] else None
          for li, (_, m) in enumerate(PRISM_MODULES)]
_PF_PLAN = {li: (_ns_prog(m)[0], int(getattr(m, "ns_ns", m.D)), int(m.ns_k), int(getattr(m, "ns_k2", 0) or 0), int(m.ns_active),
                 bool(getattr(m, "ns_hd", False))) for li, (_, m) in enumerate(PRISM_MODULES) if _has[li]}
_n_new, _dk = 0, []
for li, (n, m) in enumerate(PRISM_MODULES):
    if not _has[li]:
        continue
    m.ns_r, m.ns_R = int(m.ns_B.shape[1]), int(m.ns_U.shape[1])
    if not V15_HEAD or getattr(m, "ns_hd", False):
        continue                                               # off, or a v15 state that already carries its head
    rc = int(V15_HEAD_RANK)
    act0 = int(m.ns_active)
    k_new = int(m.ns_k) - int(math.ceil(max(0, _ns_cost(m) + _ns_head_cost(m.D, m.ns_r, rc) - act0) / (3 * m.d_model)))
    if k_new < 4:
        continue
    g = torch.Generator().manual_seed(1500 + li)
    _ns_head_install(m, torch.zeros(m.D, device=DEV),
                     (torch.randn(m.ns_r, rc, generator=g) / math.sqrt(max(1, m.ns_r))).to(DEV),
                     torch.zeros(m.D, rc, device=DEV), on=True)
    _dk.append(int(m.ns_k) - k_new)
    m.ns_k = k_new
    m.ns_active = _ns_cost(m, k_new)
    assert m.ns_active <= act0, "the consequence head must not raise a layer's budget"
    _n_new += 1
_active = measured_active()
V15_K1 = [(int(m.ns_k), int(m.ns_active), bool(getattr(m, "ns_hd", False))) if _has[li] else None
          for li, (_, m) in enumerate(PRISM_MODULES)]
_n_old = sum(bool(getattr(m, 'ns_hd', False)) for _, m in PRISM_MODULES) - _n_new
if _n_new:
    print(f"\n(B) CONSEQUENCE HEAD — installed on {_n_new} layers: rank-{V15_HEAD_RANK} context + one number per neuron, all zero at "
          f"the start (= the size-only rule). Paid for by reading {(sum(_dk) / max(1, len(_dk))):.1f} fewer exact neurons per layer "
          f"| census {B_START / 1e9:.3f}B -> {_active / 1e9:.3f}B")
else:
    print(f"\n(B) CONSEQUENCE HEAD — " + (f"already in the restored state on {_n_old} layers: it keeps training" if _n_old else "off")
          + f" | census {_active / 1e9:.3f}B")
assert _active <= max(B_START, globals().get("B_RESTORED", 0)), "census grew"
if RUN_MODE == "full" and TARGET_ACTIVE:
    assert _active <= TARGET_ACTIVE, f"census {_active / 1e9:.3f}B is over the {TARGET_ACTIVE / 1e9:.1f}B mandate"


# ================================================================ (C0) own neurons: where the served FFN output flows
V16_OWN_INFO = {"layers": 0, "m": int(V16_OWN_M), "share": None, "kl_before": None, "kl_after": None, "on": False, "restored": 0}
_own_new = []
_own_old = [m for li, (_, m) in enumerate(PRISM_MODULES) if _has[li] and getattr(m, "ns_oi", None) is not None]
V16_OWN_INFO["restored"] = len(_own_old)
if V16_OWN and int(V16_OWN_M) > 0:
    _t0 = time.time()
    _todo = [(li, m) for li, (_, m) in enumerate(PRISM_MODULES)
             if _has[li] and getattr(m, "ns_on", False) and getattr(m, "ns_oi", None) is None]
    if _todo:
        for _, m in _todo:
            m.ns_acc = torch.zeros(m.D, device=DEV)
        set_world("serving")
        _cal = [teach_blocks[0:2]] + [b_[0:2] for b_ in (globals().get("math_blocks"), globals().get("trace_blocks")) if b_ is not None]
        try:
            with torch.no_grad():
                for _cb in _cal:
                    for _i in range(_cb.shape[0]):
                        model(input_ids=_cb[_i:_i + 1].to(DEV))
            _kl0 = float(v15_eval(heldout, 2)[0])
            _sh = []
            for li, m in _todo:
                _acc = m.ns_acc
                mo = max(1, min(int(V16_OWN_M), int(m.ns_k), int(m.D)))
                oi = _acc.topk(mo).indices.sort().values
                _sh.append(float(_acc.index_select(0, oi).sum() / _acc.sum().clamp_min(1e-30)))
                _ns_own_install(m, oi)
                _own_new.append(m)
        finally:
            for _, m in _todo:
                m.ns_acc = None
        _kl1 = float(v15_eval(heldout, 2)[0])
        V16_OWN_INFO.update({"share": sum(_sh) / max(1, len(_sh)), "kl_before": _kl0, "kl_after": _kl1})
        if abs(_kl1 - _kl0) > float(V16_OWN_TOL) * max(_kl0, 1e-9):
            for m in _own_new:
                m.ns_own = False
            print(f"(C0) OWN NEURONS — !! the copies changed the model (text KL {_kl0:.4f} -> {_kl1:.4f}, they must be equal to the "
                  "originals) -> switched OFF for this run; please send this line")
            _own_new = []
        else:
            print(f"(C0) OWN NEURONS — {len(_own_new)} layers x {int(V16_OWN_M)} neurons: the ones that carry "
                  f"{100 * V16_OWN_INFO['share']:.1f}% of the served FFN output (measured on {sum(int(c.shape[0]) for c in _cal)} blocks). "
                  f"Copies installed: text KL {_kl0:.4f} -> {_kl1:.4f} (equal up to rounding: nothing is trained yet) | {time.time() - _t0:.0f}s")
    elif _own_old:
        print(f"(C0) OWN NEURONS — already in the restored state on {len(_own_old)} layers: they keep training")
else:
    for m in _own_old:
        m.ns_own = False
    print("(C0) own neurons are off (V16_OWN = False)")
_owners = [(li, m) for li, (_, m) in enumerate(PRISM_MODULES)
           if _has[li] and getattr(m, "ns_oi", None) is not None and getattr(m, "ns_own", False)]
V16_OWN_INFO.update({"layers": len(_owners), "on": bool(_owners)})


# ================================================================ (C) what trains
def _to_param(mod, name):
    t = getattr(mod, name)
    if isinstance(t, torch.nn.Parameter):
        t.requires_grad_(True)
        return t
    val = t.detach().float().clone()
    try:
        delattr(mod, name)
    except Exception:
        mod._buffers.pop(name, None)
    mod.register_parameter(name, torch.nn.Parameter(val, requires_grad=True))
    return getattr(mod, name)


def _rl_shadow(self, x):                        # trainable fp32 masters, served in the substrate dtype (as before)
    M, U = self.sh_M, self.sh_U
    if M.dtype != x.dtype:
        M, U = M.to(x.dtype), U.to(x.dtype)
    return (x @ M.t()) @ U.t()


def _sp_shadow(self, x, tag):
    M, U = getattr(self, f"{tag}_sh_M"), getattr(self, f"{tag}_sh_U")
    if M.dtype != x.dtype:
        M, U = M.to(x.dtype), U.to(x.dtype)
    return (x @ M.t()) @ U.t()


ResonantLinear._shadow = _rl_shadow
SectionedProjector._shadow = _sp_shadow

# -- always: the FFN lens shadows (v13) + the head; v10's FFN-side fields stay frozen (unused under the lens)
V15_LENS_PARAMS, V15_HEAD_W, V15_HEAD_P, V15_GHOST_PARAMS = [], [], [], []
for li, (_, m) in enumerate(PRISM_MODULES):
    if not _has[li]:
        continue
    for nm in ("ns_U", "ns_M", "ns_b"):
        V15_LENS_PARAMS.append(_to_param(m, nm))
    if getattr(m, "ns_hd", False):
        V15_HEAD_W += [_to_param(m, "ns_w"), _to_param(m, "ns_Qc")]
        V15_HEAD_P.append(_to_param(m, "ns_Pc"))
        m.ns_st = True
    if int(getattr(m, "ns_k2", 0) or 0) > 0:
        V15_GHOST_PARAMS.append(_to_param(m, "ns_gc"))
    for p in (m.U, m.P, m.alpha, m.beta, m.alpha_shared, m.beta_shared, m.router.weight, m.gains, m.log_tau):
        p.requires_grad_(False)

# -- always (when the head is strict): the head's proxy and bias — the part of the student that never saw a gradient
V16_HEAD_PARAMS = []
if V16.get("head_strict") and V16_HEAD_TRAIN:
    V16_HEAD_PARAMS = [_to_param(_LMH, "ns_fP"), _to_param(_LMH, "ns_fU"), _to_param(_LMH, "ns_fb")]

# -- by measured free memory, in the order V16_TRAIN_ORDER: attention shadows (input side M), own neurons, attention
#    shadows (output side U), the lens predictor. Each kind goes deepest layer first; the first unit that does not fit
#    ends the list (no cherry-picking further down).
V16_TRAIN_ORDER = tuple(globals().get("V16_TRAIN_ORDER", ("attn_in", "own", "attn_out", "pred")))
_att = []                                        # (layer, module, attr_U, attr_M)
for n, m in RESONA_LINEARS:
    if getattr(m, "sh_U", None) is not None:
        _att.append((_lay(n), m, "sh_U", "sh_M"))
for n, m in RESONA_SECTIONED:
    for t in ("q", "v", "z"):
        if getattr(m, f"{t}_sh_U", None) is not None:
            _att.append((_lay(n), m, f"{t}_sh_U", f"{t}_sh_M"))
_layers_desc = sorted({l for l, *_ in _att}, reverse=True)
_by_kind = {"attn_in": [], "attn_out": [], "own": [], "pred": []}      # (tag, layer, [(module, attr)], bytes needed, forced)
if V15_TRAIN_ATTN:
    for key, side, col in (("attn_in", "attention shadows, input side", 3), ("attn_out", "attention shadows, output side", 2)):
        for l in _layers_desc:
            items = [(e[1], e[col]) for e in _att if e[0] == l]
            n_ = sum(int(getattr(mm, a).numel()) for mm, a in items)
            _by_kind[key].append((side, l, items, 14 * n_, V15_TRAIN_ATTN is True))      # +2 fp32 master, 4 grad, 8 Adam
for li, m in sorted(_owners, key=lambda e: -e[0]):
    items = [(m, "ns_og"), (m, "ns_ou"), (m, "ns_od")]
    _by_kind["own"].append(("own neurons", li, items, 12 * sum(int(getattr(mm, a).numel()) for mm, a in items), V16_OWN == "force"))
if V15_TRAIN_PRED:
    for li in range(_L - 1, -1, -1):
        m = PRISM_MODULES[li][1]
        if _has[li]:
            items = [(m, "ns_B"), (m, "ns_Ag"), (m, "ns_Au")]
            _by_kind["pred"].append(("lens predictor", li, items, 12 * sum(int(getattr(mm, a).numel()) for mm, a in items),
                                     V15_TRAIN_PRED is True))
_units = [u for key in V16_TRAIN_ORDER for u in _by_kind.get(key, [])]
_n_base = sum(int(p.numel()) for p in model.parameters() if p.requires_grad)
if globals().get("V15_VRAM_BUDGET_BYTES") is not None:
    _room = float(V15_VRAM_BUDGET_BYTES)
elif torch.cuda.is_available():
    free_cuda("before choosing what trains")
    _room = float(torch.cuda.mem_get_info()[0]) - V15_VRAM_RESERVE_GIB * 2**30 - 12.0 * _n_base
else:
    _room = float("inf")
V15_ATTN_PARAMS, V15_PRED_PARAMS, V16_OWN_PARAMS, V15_TRAINED = [], [], [], {}
_full = False
for tag, l, items, need, forced in _units:
    if not forced and (_full or need > _room):
        _full = True                             # stop at the first unit that does not fit: no cherry-picking further down
        V15_TRAINED.setdefault(tag, [0, 0])[1] += 1
        continue
    _room -= need
    for mm, a in items:
        (V15_PRED_PARAMS if tag == "lens predictor" else V16_OWN_PARAMS if tag == "own neurons" else V15_ATTN_PARAMS).append(_to_param(mm, a))
    if tag == "lens predictor":
        items[0][0].ns_fit = True
    V15_TRAINED.setdefault(tag, [0, 0])[0] += 1


def _rel_groups(params, lr_rel):
    """Adam moves every entry by ~lr per step whatever its scale, so the step is set RELATIVE to each tensor's RMS
    (tensors are bucketed by powers of two of their RMS)."""
    buckets = {}
    for p in params:
        rms = float(p.detach().float().pow(2).mean().sqrt())
        buckets.setdefault(int(round(math.log2(max(rms, 1e-12)))), []).append(p)
    return [{"params": ps, "lr": float(lr_rel) * 2.0 ** b} for b, ps in sorted(buckets.items())]


def _v16_groups():
    """The optimizer groups of everything this cell made trainable (built again after the pre-flight's cuts)."""
    g = [{"params": V15_LENS_PARAMS, "lr": LR_LENS}]
    if V15_HEAD_W:
        # eps far below the default: the credit of a neuron is a small number, and it must not be drowned by Adam's 1e-8
        g += [{"params": V15_HEAD_W, "lr": LR_HEAD, "eps": 1e-10}, {"params": V15_HEAD_P, "lr": LR_HEAD * 0.1, "eps": 1e-10}]
    if V15_GHOST_PARAMS:
        g += [{"params": V15_GHOST_PARAMS, "lr": LR_GHOST}]
    g += _rel_groups(V15_ATTN_PARAMS, LR_ATTN_REL) + _rel_groups(V15_PRED_PARAMS, LR_PRED_REL)
    g += _rel_groups(V16_OWN_PARAMS, LR_OWN_REL)
    if V16_HEAD_PARAMS:
        g += _rel_groups(V16_HEAD_PARAMS[:2], LR_FH_REL) + [{"params": [V16_HEAD_PARAMS[2]], "lr": LR_FH_BIAS}]
    return [x for x in g if x["params"]]


V15_EXTRA_GROUPS = _v16_groups()
_cnt_ = lambda ps: sum(int(p.numel()) for p in ps)
_n_all = sum(int(p.numel()) for p in model.parameters() if p.requires_grad)
print(f"\n(C) WHAT TRAINS — {_n_all / 1e6:.1f}M parameters ({100 * _n_all / max(1, sum(int(p.numel()) for p in model.parameters())):.2f}% "
      f"of the model; v15 trained 1194.3M on the 27B):")
print(f"    strict head (proxy + bias)  {_cnt_(V16_HEAD_PARAMS) / 1e6:9.1f}M   lr {LR_FH_REL:g} x RMS, bias {LR_FH_BIAS:g}   (never trained before)")
print(f"    own neurons                 {_cnt_(V16_OWN_PARAMS) / 1e6:9.1f}M   lr {LR_OWN_REL:g} x RMS   (new: the student's own weights)")
print(f"    FFN lens shadows            {_cnt_(V15_LENS_PARAMS) / 1e6:9.1f}M   lr {LR_LENS:g}")
print(f"    consequence head            {(_cnt_(V15_HEAD_W) + _cnt_(V15_HEAD_P)) / 1e6:9.1f}M   lr {LR_HEAD:g}")
print(f"    ghost gains                 {_cnt_(V15_GHOST_PARAMS) / 1e6:9.1f}M   lr {LR_GHOST:g}")
print(f"    attention shadows           {_cnt_(V15_ATTN_PARAMS) / 1e6:9.1f}M   lr {LR_ATTN_REL:g} x RMS")
print(f"    lens predictor              {_cnt_(V15_PRED_PARAMS) / 1e6:9.1f}M   lr {LR_PRED_REL:g} x RMS")
print(f"    v10 attention-side routers  {(_n_all - _cnt_(V15_LENS_PARAMS) - _cnt_(V15_HEAD_W) - _cnt_(V15_HEAD_P) - _cnt_(V15_GHOST_PARAMS) - _cnt_(V15_ATTN_PARAMS) - _cnt_(V15_PRED_PARAMS) - _cnt_(V16_OWN_PARAMS) - _cnt_(V16_HEAD_PARAMS)) / 1e6:9.1f}M   v10 learning rates")
for tag, (took, left) in V15_TRAINED.items():
    print(f"    {tag}: {took} layer-groups trainable" + (f", {left} left frozen (GPU memory — deepest layers were taken first)" if left else ""))
if torch.cuda.is_available():
    print(f"    estimated optimizer + gradient memory: {12 * _n_all / 2**30:.1f} GiB | reserve for one step: {V15_VRAM_RESERVE_GIB:g} GiB")


# ================================================================ (C2) THE MARKET: sites, probes, prices, trades
_MK.update({"on": False, "probes": {}, "width": {}, "acc": {}, "log": [], "head_g": [0.0, 0.0], "rounds": 0, "trades": 0,
            "cap": None, "closed": None})
MK_SITES = []


def _mk_site(key, kind, mod, get, set_, unit, cost, lo, hi):
    n0 = int(get())
    if hi - n0 < unit and n0 - lo < unit:
        return                                             # nothing to trade here
    MK_SITES.append({"key": key, "kind": kind, "mod": mod, "get": get, "set": set_, "unit": int(unit), "cost": int(cost),
                     "lo": int(lo), "hi": int(hi), "n0": n0})


def _mk_ffn_set(m, n):
    m.ns_k = int(n)
    m.ns_active = _ns_cost(m)


if V16_MARKET:
    for li, (name, m) in enumerate(PRISM_MODULES):         # FFN layers: unit = MK_DK_FFN neurons (3 d parameters each)
        if not (_has[li] and getattr(m, "ns_on", False)) or int(getattr(m, "ns_k2", 0) or 0) > 0 or _ns_prog(m)[0]:
            continue
        k0, u = int(m.ns_k), int(MK_DK_FFN)
        if k0 >= 4 * u:
            _mk_site("ffn:" + name, "FFN neurons", m, (lambda m=m: int(m.ns_k)), (lambda n, m=m: _mk_ffn_set(m, n)), u,
                     3 * m.d_model * u, max(2 * u, int(k0 * (1 - MK_DRIFT))), min(m.D // 2, int(k0 * (1 + MK_DRIFT))))
    for name, m in RESONA_LINEARS:                         # routed projections: unit = one row-block
        if getattr(m, "sh_U", None) is None:
            continue
        _mk_site("rl:" + name, _lin_kind(name) + " row-blocks", m, (lambda m=m: int(m.topk)),
                 (lambda n, m=m: setattr(m, "topk", int(n))), 1, int(m.r) * int(m.in_f), 1, min(int(m.B), int(m.topk) + 4))
    for name, m in RESONA_SECTIONED:                       # DeltaNet q / z sections: unit = one q block + one z block
        tags = [t for t in ("q", "z") if hasattr(m, f"{t}_B") and getattr(m, f"{t}_sh_U", None) is not None]
        if len(tags) != len([t for t in ("q", "z") if hasattr(m, f"{t}_B")]) or not tags:
            continue
        _mk_site("sp:" + name, "in_proj_qkv " + "/".join(tags) + " row-blocks", m, (lambda m=m: int(m.topk)),
                 (lambda n, m=m: setattr(m, "topk", int(n))), 1,
                 sum(int(getattr(m, f"{t}_r")) for t in tags) * int(m.weight.shape[1]), 1,
                 min(min(int(getattr(m, f"{t}_B")) for t in tags), int(m.topk) + 4))
    for dn_name, pol in RESONA_MORM:                       # memory writes: unit = one exact head
        sp = next((m for _, m in RESONA_SECTIONED if getattr(m, "morm", None) is pol), None)
        if sp is None or getattr(pol, "strict_m", None) is None:
            continue
        m0 = int(pol.strict_m)
        _mk_site("morm:" + dn_name, "memory heads", pol, (lambda p=pol: int(p.strict_m)),
                 (lambda n, p=pol: setattr(p, "strict_m", int(n))), 1, int(sp.v_head_rows) * int(sp.weight.shape[1]),
                 max(1, m0 - 2), min(int(pol.H), m0 + 4))
    if V16.get("head_strict"):                             # the head: unit = MK_DK_HEAD exact vocabulary rows
        u, s0 = int(MK_DK_HEAD), int(funnel.shortlist)
        if s0 >= 4 * u:
            _mk_site("head", "head rows", funnel, (lambda: int(funnel.shortlist)),
                     (lambda n: setattr(funnel, "shortlist", int(n))), u, u * int(funnel.d),
                     max(2 * u, int(s0 * (1 - MK_DRIFT))), min(int(funnel.V), int(s0 * (1 + MK_DRIFT))))
    for s_ in MK_SITES:
        _MK["acc"][s_["key"]] = []
        if s_["key"] != "head":
            _MK["probes"][id(s_["mod"])] = (torch.nn.Parameter(torch.zeros((), device=DEV)), torch.nn.Parameter(torch.zeros((), device=DEV)))
            _MK["width"][id(s_["mod"])] = int(s_["unit"])
    _MK["on"] = bool(MK_SITES)
_c0 = int(measured_active())
_MK["cap"] = max(_c0, int((TARGET_ACTIVE if (RUN_MODE == "full" and TARGET_ACTIVE) else _c0) - MK_MARGIN))
_kinds = {}
for s_ in MK_SITES:
    e = _kinds.setdefault(s_["kind"], [0, 0, 0])
    e[0] += 1; e[1] += s_["n0"] * s_["cost"] // s_["unit"]; e[2] = s_["cost"]
if MK_SITES:
    print(f"\n(C2) THE MARKET — {len(MK_SITES)} sites trade; census {_c0 / 1e9:.4f}B, ceiling {_MK['cap'] / 1e9:.4f}B "
          f"(free to spend at once: {(_MK['cap'] - _c0) / 1e6:.1f}M). A round every {MK_EVERY} updates from update {MK_START}, "
          f"at most {100 * MK_ROUND_FRAC:.1f}% of the budget per round, a site stays within +/-{100 * MK_DRIFT:.0f}% of its start:")
    for kind, (cnt, held, cost) in _kinds.items():
        print(f"    {kind:<34} {cnt:3d} sites | hold {held / 1e6:9.2f}M in exact units | one unit = {cost / 1e6:.3f}M parameters")
else:
    print("\n(C2) the market is off" + ("" if not V16_MARKET else " (no site with something to trade)"))


def _mk_probe_reset():
    for pp, pm in _MK["probes"].values():
        pp.grad = None
        pm.grad = None
    _MK["head_g"] = [0.0, 0.0]


def _mk_reset():
    """Forget every observed price (the budgets or the weights changed under them)."""
    for k_ in _MK["acc"]:
        _MK["acc"][k_] = []
    _mk_probe_reset()


def _mk_harvest():
    """Called right before every optimizer step: the probes' gradients of this update become one price observation.
    For every site:  gp = d loss if it had one unit MORE,  gm = d loss if it had one unit LESS  (first order)."""
    if _WAL.get("on"):
        _wal_harvest()
    if not _MK.get("on"):
        return
    for s_ in MK_SITES:
        if s_["key"] == "head":
            gp, gm = (float(v) for v in (_MK.get("head_g") or [0.0, 0.0]))
            seen = (gp != 0.0 or gm != 0.0)
        else:
            pp, pm = _MK["probes"][id(s_["mod"])]
            seen = pp.grad is not None or pm.grad is not None
            gp = float(pp.grad) if pp.grad is not None else 0.0
            gm = float(pm.grad) if pm.grad is not None else 0.0
        if seen and math.isfinite(gp) and math.isfinite(gm):
            a_ = _MK["acc"][s_["key"]]
            a_.append((gp, gm))
            if len(a_) > 4 * int(MK_EVERY):
                del a_[0]
    _mk_probe_reset()


# ================================================================ THE FUSE: a number that is not finite never reaches the weights
# Every micro-batch: is the loss finite, is every gradient finite? If not, the update is NOT applied (the weights stay
# exactly as they were), the place is named — which loss term, which tensor, which module — and the run goes on.
V16_FUSE = {"skip": False, "why": "", "skipped": 0, "held": False, "poison": [], "strikes": {}, "frozen": [], "names": None, "log": []}
V16_FUSE_MAX = int(globals().get("V16_FUSE_MAX", 6))        # updates the fuse may hold back before the governor takes over


def _fuse_names():
    if V16_FUSE["names"] is None:
        nm = {id(p_): n_ for n_, p_ in model.named_parameters()}
        for n_, p_ in getattr(globals().get("_LMH"), "named_parameters", lambda: [])():
            nm.setdefault(id(p_), "lm_head." + n_)
        V16_FUSE["names"] = nm
    return V16_FUSE["names"]


def _fuse_params():
    opt = globals().get("optimizer")
    return [] if opt is None else [p_ for g_ in opt.param_groups for p_ in g_["params"]]


def _fuse_say(msg):
    V16_FUSE["log"].append(msg)
    if len(V16_FUSE["log"]) <= 12:
        print(msg)


def _fuse_bad(ts):
    """Indices of the tensors in ts that hold a NaN / inf (one sync for all of them)."""
    if not ts:
        return []
    with torch.no_grad():
        fl = torch.stack([t.detach().float().sum() for t in ts]).isfinite().tolist()
    return [i for i, ok in enumerate(fl) if not ok]


def _fuse_trace(ids):
    """Where does it start? One more forward (no gradient) with a check behind every module."""
    first, hs = [], []

    def hook(name):
        def f(mod, inp, out):
            o = out[0] if isinstance(out, (tuple, list)) and len(out) else out
            if len(first) < 3 and torch.is_tensor(o) and o.is_floating_point() and not bool(torch.isfinite(o).all()):
                i0 = inp[0] if isinstance(inp, (tuple, list)) and len(inp) else None
                ok_in = bool(torch.isfinite(i0).all()) if (torch.is_tensor(i0) and i0.is_floating_point()) else None
                first.append(f"{name} [{type(mod).__name__}] (its input: {'finite' if ok_in else 'NOT finite' if ok_in is not None else '?'})")
        return f
    try:
        for n_, m_ in model.named_modules():
            if n_:
                hs.append(m_.register_forward_hook(hook(n_)))
        with torch.no_grad():
            model(input_ids=ids)
    except Exception as _e:
        first.append(f"(trace stopped: {type(_e).__name__}: {str(_e)[:80]})")
    finally:
        for h_ in hs:
            h_.remove()
    return first


def v16_fuse_loss(loss, parts, hidden=None, ids=None):
    """Called by the training step with the loss, its terms and the hidden states."""
    if bool(torch.isfinite(loss.detach()).all()):
        return
    V16_FUSE["skip"] = True
    bad = [k_ for k_, v_ in parts.items() if torch.is_tensor(v_) and not bool(torch.isfinite(v_.detach().float()).all())]
    hid = None if hidden is None else bool(torch.isfinite(hidden.detach()).all())
    V16_FUSE["why"] = f"the loss is not finite (terms: {', '.join(bad) or '?'}; hidden states: {'finite' if hid else 'NOT finite' if hid is not None else '?'})"
    if V16_FUSE.get("traced"):
        return
    V16_FUSE["traced"] = True                               # the full picture once
    ps, nm = _fuse_params(), _fuse_names()
    pb = [nm.get(id(ps[i]), "?") for i in _fuse_bad(ps)]
    V16_FUSE["poison"] = pb
    _fuse_say(f"[fuse] {V16_FUSE['why']}")
    _fuse_say(f"[fuse]   trained tensors holding NaN/inf: {len(pb)}" + (f" -> {', '.join(pb[:6])}" + (" ..." if len(pb) > 6 else "") if pb else " (the weights are clean)"))
    if hid is False and ids is not None:
        for line in _fuse_trace(ids) or ["(the repeat forward was finite)"]:
            _fuse_say(f"[fuse]   first module with a non-finite output: {line}")


V16_FUSE_SHARE = float(globals().get("V16_FUSE_SHARE", 0.02))   # more than this share of tensors hit -> the whole update is skipped


def _fuse_layer(n_):
    import re as _re
    m_ = _re.search(r"layers\.(\d+)\.", n_)
    return int(m_.group(1)) if m_ else 10 ** 6             # the head and other non-layer tensors sort as the deepest


def v16_fuse_grads():
    """Called after every backward. A NaN that starts in one place runs down through the hidden states into every
    tensor below it: those tensors are victims, not causes, so nothing is ever frozen for it. A few tensors hit ->
    they sit this update out; many -> the whole update is skipped. The DEEPEST tensor hit is named: that is where it starts."""
    if V16_FUSE["skip"]:
        return
    ps = [p_ for p_ in _fuse_params() if p_.grad is not None]
    bad = _fuse_bad([p_.grad for p_ in ps])
    if not bad:
        return
    nm = _fuse_names()
    names = sorted((nm.get(id(ps[i]), "?") for i in bad), key=_fuse_layer, reverse=True)
    for n_ in names:
        V16_FUSE["strikes"][n_] = V16_FUSE["strikes"].get(n_, 0) + 1
    if len(bad) > V16_FUSE_SHARE * len(ps):
        V16_FUSE["skip"] = True
        V16_FUSE["why"] = f"gradient not finite in {len(bad)} of {len(ps)} tensors (deepest: {names[0]})"
    else:
        for i in bad:
            ps[i].grad = None
    _fuse_say(f"[fuse] gradient not finite in {len(bad)} of {len(ps)} tensors -> "
              + ("the whole update is skipped" if V16_FUSE["skip"] else "they sit this update out")
              + f" | deepest first: {', '.join(names[:4])}" + (" ..." if len(names) > 4 else ""))


def _fuse_heal():
    """After the governor put the best state back: the tensors that went non-finite stay as they are there (frozen)."""
    nm = {n_: p_ for p_ in _fuse_params() for n_ in [_fuse_names().get(id(p_), "?")]}
    done = []
    for n_ in V16_FUSE["poison"]:
        p_ = nm.get(n_)
        if p_ is not None and p_.requires_grad:
            p_.requires_grad_(False)
            p_.grad = None
            V16_FUSE["frozen"].append(n_)
            done.append(n_)
    V16_FUSE["poison"], V16_FUSE["traced"] = [], False
    if done:
        print(f"[fuse] {len(done)} tensors that went non-finite are frozen at their restored values: {', '.join(done[:6])}" + (" ..." if len(done) > 6 else ""))



def v16_wrap_optimizer(opt):
    """The market listens at the optimizer: each step first turns the probe gradients into a price observation."""
    if getattr(opt, "_v16_wrapped", False):
        return opt
    _orig = opt.step

    def _step(self, *a, **k):
        try:
            _mk_harvest()
        except Exception as _e:
            if _WAL.get("on"):
                _WAL["on"] = False
                WAL_INFO["closed"] = f"harvest: {type(_e).__name__}: {str(_e)[:100]}"
                print(f"[wallet] closed ({WAL_INFO['closed']}) — every token buys the quotas again")
            if _MK.get("on"):
                _MK["on"], _MK["closed"] = False, f"harvest: {type(_e).__name__}: {str(_e)[:100]}"
                print(f"[market] closed ({_MK['closed']}) — the budgets stay where they are")
        if V16_FUSE["skip"]:                               # THE FUSE: this update is not applied
            V16_FUSE["skip"], V16_FUSE["held"] = False, True
            V16_FUSE["skipped"] += 1
            for p_ in _fuse_params():
                p_.grad = None
            _fuse_say(f"[fuse] update not applied ({V16_FUSE['skipped']}/{V16_FUSE_MAX}): {V16_FUSE['why']} — the weights are untouched")
            return None
        return _orig(*a, **k)

    import types as _types
    opt.step = _types.MethodType(_step, opt)   # a bound method: the LR scheduler re-wraps step.__func__
    opt._v16_wrapped = True
    return opt


def _mk_prices():
    """Per site: what one more parameter there is worth (lower bound) and what one parameter less costs (upper bound)."""
    out = {}
    for s_ in MK_SITES:
        a_ = _MK["acc"][s_["key"]]
        n = len(a_)
        if n < int(MK_MIN_OBS):
            continue
        gp, gm = [x[0] for x in a_], [x[1] for x in a_]
        mp, mm_ = sum(gp) / n, sum(gm) / n
        sp_ = (sum((x - mp) ** 2 for x in gp) / max(1, n - 1)) ** 0.5 / n ** 0.5
        sm_ = (sum((x - mm_) ** 2 for x in gm) / max(1, n - 1)) ** 0.5 / n ** 0.5
        c = float(s_["cost"])
        out[s_["key"]] = {"buy": (-mp - MK_Z * sp_) / c, "sell": (max(mm_, 0.0) + MK_Z * sm_) / c,
                          "buy_mean": -mp / c, "sell_mean": mm_ / c, "n": n}
    return out


def _mk_round(st):
    """One trading round. Returns the log entry."""
    P = _mk_prices()
    cap = int(_MK["cap"])
    pool = cap - int(measured_active())
    before = {s_["key"]: int(s_["get"]()) for s_ in MK_SITES}
    buyers = sorted((s_ for s_ in MK_SITES if s_["key"] in P and P[s_["key"]]["buy"] > 0 and before[s_["key"]] + s_["unit"] <= s_["hi"]),
                    key=lambda s_: -P[s_["key"]]["buy"])
    sellers = sorted((s_ for s_ in MK_SITES if s_["key"] in P and before[s_["key"]] - s_["unit"] >= s_["lo"]),
                     key=lambda s_: P[s_["key"]]["sell"])
    used, buys, sells, moved, gain, loss = set(), [], [], 0, 0.0, 0.0
    vol = float(MK_ROUND_FRAC) * cap
    for B in buyers:
        if B["key"] in used or moved + B["cost"] > vol:
            continue
        need, pick = B["cost"] - pool, []
        if need > 0:
            for S_ in sellers:
                if need <= 0:
                    break
                if S_["key"] in used or S_ is B or any(S_ is x for x in pick):
                    continue
                if P[S_["key"]]["sell"] * float(MK_RHO) >= P[B["key"]]["buy"]:
                    break                                  # the sellers are sorted: nobody further down is cheap enough
                pick.append(S_)
                need -= S_["cost"]
            if need > 0:
                continue
        for S_ in pick:
            S_["set"](before[S_["key"]] - S_["unit"])
            used.add(S_["key"]); sells.append(S_["key"])
            pool += S_["cost"]; loss += P[S_["key"]]["sell"] * S_["cost"]
        B["set"](before[B["key"]] + B["unit"])
        used.add(B["key"]); buys.append(B["key"])
        pool -= B["cost"]; moved += B["cost"]; gain += P[B["key"]]["buy"] * B["cost"]
    census = int(measured_active())
    ok = bool(buys) and census <= cap and gain > float(MK_RHO) * loss
    if buys and not ok:                                    # the round as a whole does not pay (or broke the ceiling): undone
        for s_ in MK_SITES:
            if int(s_["get"]()) != before[s_["key"]]:
                s_["set"](before[s_["key"]])
        buys, sells, census = [], [], int(measured_active())
    entry = {"step": int(st), "buys": buys, "sells": sells, "gain": float(gain) if ok else 0.0, "loss": float(loss) if ok else 0.0,
             "census": census, "priced": len(P)}
    _MK["rounds"] += 1
    _MK["log"].append(entry)
    if buys:
        _MK["trades"] += len(buys)
        _mk_reset()                                        # the prices belonged to the old allocation
        kind_of = {s_["key"]: s_["kind"] for s_ in MK_SITES}
        cnt = lambda ks: ", ".join(f"{v} x {k}" for k, v in sorted({kk: sum(1 for x in ks if kind_of[x] == kk) for kk in {kind_of[x] for x in ks}}.items()))
        print(f"[market] update {st}: bought {cnt(buys)}" + (f" | sold {cnt(sells)}" if sells else " | from the free budget")
              + f" | predicted {-gain:+.5f} nats for the buys, {loss:+.5f} for the sells | census {census / 1e9:.4f}B")
    return entry


class V16Market(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if not _MK.get("on"):
            return
        st = _abs_step(state)
        if st < int(MK_START) or st > float(MK_STOP_FRAC) * STEPS or st % int(MK_EVERY) != 0:
            return
        try:
            _mk_round(st)
        except Exception as _e:
            _MK["on"], _MK["closed"] = False, f"round: {type(_e).__name__}: {str(_e)[:100]}"
            print(f"[market] closed ({_MK['closed']}) — the budgets stay where they are")


# ================================================================ (C3) THE WALLET: every token spends its own 3.0B
WAL_INFO = {"on": False, "sites": 0, "restored": False, "log": [], "closed": None}
_wal_restored = bool(_WAL.get("order")) and bool(_WAL.get("on"))
def _wal_setup():
    global _wal_restored
    _t0 = time.time()
    if not _wal_restored:
        _wal_build(span_ffn=WAL_SPAN_FFN, more_blocks=WAL_MORE_BLOCKS, more_heads=WAL_MORE_HEADS)
        # where does each site's quota cut its scores? (median over tokens = the opening price; spread = its scale)
        _WAL.update({"on": False, "calib": {}, "beta": float(WAL_BETA), "rho": float(WAL_RHO), "tau": float(WAL_TAU),
                     "bref_frac": float(WAL_BREF), "violations": 0})
        set_world("serving")
        _calb = [teach_blocks[0:2]] + [b_[0:2] for b_ in (globals().get("math_blocks"), globals().get("trace_blocks")) if b_ is not None]
        try:
            with torch.no_grad():
                for _cb in _calb:
                    for _i in range(_cb.shape[0]):
                        model(input_ids=_cb[_i:_i + 1].to(DEV))
            _cal = _WAL["calib"]
        finally:
            _WAL["calib"] = None
        _by_key = {st["key"]: st for st in _WAL["sites"].values()}
        _WAL["order"] = []
        for _k, _v in _cal.items():                           # insertion order = the order in which a token meets the sites
            st = _by_key[_k]
            c = torch.cat(_v).float()
            st["th"] = float(c.median())
            st["sig"] = max(float(c.std()), 1e-3 * abs(st["th"]) + 1e-3)
            _WAL["order"].append(st)
        _keep = {id(st) for st in _WAL["order"]}
        _WAL["sites"] = {k_: st for k_, st in _WAL["sites"].items() if id(st) in _keep}
        _WAL["b0"] = float(max(0, int(_MK["cap"]) - int(measured_active())))
        _wal_open(0.0)
        _WAL["on"] = bool(_WAL["order"])
    WAL_INFO.update({"on": bool(_WAL["on"]), "sites": len(_WAL.get("order", [])), "restored": _wal_restored})
    if _WAL["on"]:
        _MK["on"] = False                                     # the quota market stands by (it takes over if the wallet is switched off)
        _wk = {}
        for st in _WAL["order"]:
            e = _wk.setdefault(st["kind"], [0, 0.0, 0.0])
            e[0] += 1; e[1] += st["cost"] * st["n0"](); e[2] += st["cost"] * (min(st["hi"], st["n0"]() + st["span"]) - max(st["lo"], st["n0"]() - st["span"]))
        print(f"\n(C3) THE WALLET — {len(_WAL['order'])} sites sell per token" + (" (restored with the state)" if _wal_restored else "")
              + f". Ceiling for EVERY token: {(_MK['cap']) / 1e9:.4f}B = census at the quotas {measured_active() / 1e9:.4f}B + "
              f"{_WAL['b0'] / 1e6:.2f}M free. The window opens from update {WAL_START} to {WAL_OPEN} (now {100 * _WAL['omega']:.0f}%); "
              f"a token may run a debt of {WAL_RHO:g} x what later sites can refund | {time.time() - _t0:.0f}s")
        for kind, (cnt, held, rng) in _wk.items():
            print(f"    {kind:<34} {cnt:3d} sites | quota {held / 1e6:9.2f}M | a token may move {rng / 1e6:9.2f}M of it")
    else:
        print("\n(C3) the wallet has no site to work on -> quota mode")


if V16_WALLET:
    try:
        _wal_setup()
    except Exception as _e:                                    # the wallet is optional: any failure here leaves the quota model untouched
        if is_oom(_e):
            free_cuda("wallet setup OOM")
        _WAL["on"], _WAL["calib"] = False, None
        WAL_INFO.update({"on": False, "closed": f"setup: {type(_e).__name__}: {str(_e)[:160]}"})
        print(f"\n(C3) THE WALLET — !! could not be set up ({WAL_INFO['closed']}) -> switched off; the quota market trades instead. "
              "Please send this line.")
        set_world("serving")
else:
    _WAL["on"] = False
    print("\n(C3) the wallet is off (V16_WALLET = False) -> the quota market above trades")
if not _WAL.get("on"):
    _MK["on"] = bool(V16_MARKET) and bool(MK_SITES)


def _wal_harvest():
    """Right before every optimizer step: the price probes' gradients of this update."""
    for st in _WAL.get("order", []):
        g = st["probe"].grad
        if g is not None:
            gv = float(g)
            if math.isfinite(gv):
                st["acc"]["g"] += gv
                st["acc"]["steps"] += 1
        st["probe"].grad = None


def _wal_acc_reset():
    for st in _WAL.get("order", []):
        st["acc"] = {"g": 0.0, "steps": 0, "mass": 0.0, "dem": 0.0, "n": 0.0, "clip": 0.0, "fwd": 0}
        st["probe"].grad = None


def _wal_prices(step):
    """Tatonnement. For every site the probe says what one more parameter per token is worth AT ITS PRICE:
        m_s = - d loss / d probe / (cost x mass of the items at the price)
    A site whose marginal parameter is worth more than the typical site's lowers its price, the others raise theirs
    (so the same parameter is worth the same everywhere a token can spend it); then all prices move together until
    what the tokens want to spend equals what there is to spend. Prices move in units of each site's own score spread."""
    obs = []
    for st in _WAL["order"]:
        a = st["acc"]
        if a["steps"] >= 1 and a["fwd"] >= 1 and a["mass"] > 0:
            obs.append((st, -(a["g"] / a["steps"]) / (st["cost"] * a["mass"] / a["fwd"])))
    if len(obs) < 2:
        return None
    vals = sorted(m for _, m in obs)
    med = vals[len(vals) // 2]
    dev = sorted(abs(m - med) for m in vals)
    mad = max(dev[len(dev) // 2], 1e-30)
    for st, m in obs:
        st["th"] -= float(WAL_LR) * st["sig"] * max(-1.0, min(1.0, (m - med) / (3.0 * mad)))
    demand = sum(st["acc"]["dem"] / max(1, st["acc"]["fwd"]) for st in _WAL["order"])       # parameters per token above the quotas
    # v17: the level of all prices follows what the tokens LEAVE UNSPENT. v16 followed what they WANTED: the wants
    # came from the few tokens whose wallets were already empty, so prices rose, everybody else sold, and ~157M per
    # token was left unspent (mean 2.84B of a 2.999B ceiling). Unspent budget is wasted capacity: lower the prices.
    _un_n = int(_WAL.get("unspent_n", 0))
    unspent = float(_WAL.get("unspent_sum", 0.0)) / _un_n if _un_n else None
    _WAL["unspent_sum"], _WAL["unspent_n"] = 0.0, 0
    if unspent is not None and V17_WAL_UNSPENT:
        _tgt = float(V17_WAL_TARGET) * float(_MK["cap"])
        shift = -float(WAL_GLR) * max(-1.0, min(1.0, (unspent - _tgt) / max(1.0, 0.005 * float(_MK["cap"]))))
    else:
        shift = float(WAL_GLR) * max(-1.0, min(1.0, (demand - _WAL["b0"]) / max(1.0, 0.05 * _WAL.get("vol", 1.0))))
    for st in _WAL["order"]:
        st["th"] += shift * st["sig"]
    clip = sum(st["acc"]["clip"] / max(1, st["acc"]["fwd"]) for st in _WAL["order"]) / max(1, len(_WAL["order"]))
    kinds = {}
    for st in _WAL["order"]:
        e = kinds.setdefault(st["kind"], [0.0, 0.0, 0])
        e[0] += st["acc"]["n"] / max(1, st["acc"]["fwd"]); e[1] += st["n0"](); e[2] += 1
    entry = {"step": int(step), "omega": float(_WAL["omega"]), "demand": float(demand), "free": float(_WAL["b0"]), "shift": float(shift),
             "unspent": (None if unspent is None else float(unspent)),
             "clipped": float(clip), "worth_median": float(med),
             "mean_items": {k: [v[0] / v[2], v[1] / v[2]] for k, v in kinds.items()}}
    WAL_INFO["log"].append(entry)
    _wal_acc_reset()
    return entry


class V16Wallet(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if not _WAL.get("on"):
            return
        st = _abs_step(state)
        try:
            om = max(0.0, min(1.0, (st - int(WAL_START)) / max(1, int(WAL_OPEN) - int(WAL_START)))) * float(WAL_MAX)
            if om > _WAL["omega"] + 1e-9:                     # the window only ever opens during a run
                _wal_open(om)
            if _WAL["omega"] > 0 and st % int(WAL_EVERY) == 0:
                e = _wal_prices(st)
                if e is not None and (len(WAL_INFO["log"]) % int(WAL_PRINT) == 1 or st % V15_EVAL_EVERY == 0):
                    print(f"[wallet] update {st}: window {100 * e['omega']:.0f}% | "
                          + (f"tokens left {e['unspent'] / 1e6:.1f}M unspent " if e.get('unspent') is not None else
                             f"tokens want {e['demand'] / 1e6:+.2f}M above the quotas ({e['free'] / 1e6:.2f}M is free) ")
                          + f"-> all prices {'up' if e['shift'] > 0 else 'down'} {abs(e['shift']):.2f} | wallet-limited "
                          f"{100 * e['clipped']:.1f}% | mean items/token vs quota: "
                          + ", ".join(f"{k.split(' ')[0]} {v[0]:.1f}/{v[1]:.0f}" for k, v in list(e["mean_items"].items())[:6]))
            if _WAL.get("violations"):
                print(f"[wallet] !! a token ended below zero {_WAL['violations']}x -> the credit line is closed (rho = 0); the cap holds without it")
                _WAL["violations"] = 0
        except Exception as _e:
            _WAL["on"] = False
            WAL_INFO["closed"] = f"{type(_e).__name__}: {str(_e)[:100]}"
            print(f"[wallet] closed ({WAL_INFO['closed']}) — every token buys the quotas again")


def _v16_budget_get():
    return {s_["key"]: int(s_["get"]()) for s_ in MK_SITES}


def _v16_budget_set(saved):
    if not saved:
        return
    for s_ in MK_SITES:
        if s_["key"] in saved and int(s_["get"]()) != int(saved[s_["key"]]):
            s_["set"](int(saved[s_["key"]]))


# ================================================================ (D) pre-flight: every new gradient path runs once, on a short prefix, before the long run
def _v15_morm_get():
    """The state the training loop moves OUTSIDE the optimizer: the write policies' controllers and — v16 — every
    budget the market trades."""
    out = []
    for _, m in RESONA_MORM:
        c = getattr(m, "controller", None)
        out.append({"m_bar": getattr(m, "m_bar", None), "ctrl": (dict(c.__dict__) if c is not None else None),
                    "thr": (m.rate_threshold.detach().clone() if hasattr(m, "rate_threshold") else None),
                    "strict_m": getattr(m, "strict_m", None)})
    return {"morm": out, "budget": _v16_budget_get(), "wallet": (_wal_state() if _WAL.get("order") else None)}


def _v15_morm_set(saved):
    lst = saved["morm"] if isinstance(saved, dict) else saved
    for (_, m), d in zip(RESONA_MORM, lst):
        if d["m_bar"] is not None:
            m.m_bar = d["m_bar"]
        if d["ctrl"] is not None:
            m.controller.__dict__.update(d["ctrl"])
        if d["thr"] is not None:
            with torch.no_grad():
                m.rate_threshold.copy_(d["thr"])
        if "strict_m" in d:
            m.strict_m = d["strict_m"]
    if isinstance(saved, dict):
        _v16_budget_set(saved.get("budget"))
        if _MK.get("on"):
            _mk_reset()
        ws = saved.get("wallet")
        if ws and _WAL.get("order"):                          # prices and window of that state (the switch itself is not touched)
            _on = bool(_WAL["on"])
            for st in _WAL["order"]:
                sv = ws["sites"].get(st["key"])
                if sv is not None:
                    st["th"], st["sig"] = float(sv["th"]), float(sv["sig"])
            _wal_open(float(ws.get("omega", 0.0)))
            _WAL["on"] = _on
            _wal_acc_reset()



# ================================================================ (V17) the teacher in the step, LOOP, per-source KL, the long curve
V17_SRC = {"text": 0, "problems": 1, "traces": 2, "fresh": 3, "loop": 4}
V17_SRC_NAME = {v: k for k, v in V17_SRC.items()}
V17_SRCLOG = {"sum": {}, "n": {}}
V17_LOOP = {"queue": [], "buf": [], "mb": 0, "rounds": 0, "made": 0, "right": 0, "sec": 0.0, "off": None, "hist": []}


@torch.no_grad()
def v17_teacher_topk(h, chunk=256):
    """Top-K log-probabilities of the DENSE head (the teacher's) from final hidden states h [B, L, d]."""
    W = _LMH.weight
    B_, L_, _ = h.shape
    K = int(TEACHER_TOPK)
    ids = torch.empty(B_, L_, K, dtype=torch.long, device=h.device)
    lps = torch.empty(B_, L_, K, dtype=torch.float32, device=h.device)
    for c0 in range(0, L_, int(chunk)):
        lg = (h[:, c0:c0 + chunk].to(W.dtype) @ W.t()).float()
        tv, ti = lg.log_softmax(-1).topk(K, dim=-1)
        ids[:, c0:c0 + chunk], lps[:, c0:c0 + chunk] = ti, tv
        del lg, tv, ti
    return ids, lps


@torch.no_grad()
def v17_teacher(mdl, input_ids, need_states):
    """One dense-teacher pass on the very batch being trained: its stream (for the anchor) and its top-K targets.
    v16 read the targets from a cache built before training, so the data could not grow; the teacher was already
    running in every step for the anchor — now the same pass also labels the batch."""
    if not V17_INSTEP:
        return (v15_teacher_states(mdl, input_ids) if need_states else None), None, None
    set_world("dense_teacher")
    try:
        out = mdl(input_ids=input_ids, output_hidden_states=bool(need_states))
        hs = getattr(out, "hidden_states", None) if need_states else None
        st = None if hs is None else [hs[i].detach() for i in _v15_idx(len(hs))]
        h = out.logits
        hfin = None
        if h.shape[-1] == _LMH.weight.shape[0]:            # the head was not swapped out: these are logits already
            lsm = h.float().log_softmax(-1)
            tv, ti = lsm.topk(int(TEACHER_TOPK), dim=-1)
            tt = (ti, tv)
        else:
            tt = v17_teacher_topk(h)
            hfin = h.detach()
        del out, hs
        return st, tt, hfin
    finally:
        set_world("serving")


def v17_src_note(src, kl_pos):
    """Per-source KL of the training batches (kl_pos [B, L-1])."""
    if src is None or kl_pos is None:
        return
    with torch.no_grad():
        per = kl_pos.detach().float().mean(-1)
        for b in range(per.shape[0]):
            k = V17_SRC_NAME.get(int(src[b]), "?")
            V17_SRCLOG["sum"][k] = V17_SRCLOG["sum"].get(k, 0.0) + float(per[b])
            V17_SRCLOG["n"][k] = V17_SRCLOG["n"].get(k, 0) + 1


def v17_src_report(reset=True):
    rows = {k: V17_SRCLOG["sum"][k] / max(1, V17_SRCLOG["n"][k]) for k in V17_SRCLOG["sum"]}
    if reset:
        V17_SRCLOG["sum"], V17_SRCLOG["n"] = {}, {}
    return rows


# ---------------------------------------------------------------- LOOP: the student answers, the teacher grades every token
_V17_PROBE_QS = set()
_r0 = _random.Random(1234)                                   # the math probe's own problems are never practised
for _i in range(200):
    _k = _i % 4
    if _k == 0:
        _a, _b = _r0.randint(11, 99), _r0.randint(11, 99); _V17_PROBE_QS.add(f"{_a} + {_b}")
    elif _k == 1:
        _a, _b = _r0.randint(50, 99), _r0.randint(11, 49); _V17_PROBE_QS.add(f"{_a} - {_b}")
    elif _k == 2:
        _a, _b = _r0.randint(2, 9), _r0.randint(11, 30); _V17_PROBE_QS.add(f"{_a} * {_b}")
    else:
        _b = _r0.randint(2, 9); _g = _r0.randint(3, 20); _V17_PROBE_QS.add(f"{_b * _g} / {_b}")
_V17_INSTR = "Compute the value. Reply with exactly one line of the form 'Answer: <number>'.\n\n"
_V17_ANS = _re15.compile(r"Answer:\s*\$?\s*(-?[\d,]*\.?\d+)")


def _v17_problem(rng):
    """One-step problems in the probe's four kinds (other numbers), and some two-step ones."""
    while True:
        k = rng.random()
        if k < 0.20:
            a, b = rng.randint(11, 99), rng.randint(11, 99); q, g = f"{a} + {b}", a + b
        elif k < 0.40:
            a, b = rng.randint(50, 99), rng.randint(11, 49); q, g = f"{a} - {b}", a - b
        elif k < 0.60:
            a, b = rng.randint(2, 9), rng.randint(11, 30); q, g = f"{a} * {b}", a * b
        elif k < 0.80:
            b = rng.randint(2, 9); g = rng.randint(3, 20); q = f"{b * g} / {b}"
        elif k < 0.90:
            a, b, c = rng.randint(11, 99), rng.randint(11, 99), rng.randint(11, 99); q, g = f"{a} + {b} - {c}", a + b - c
        else:
            a, b, c = rng.randint(2, 9), rng.randint(11, 30), rng.randint(11, 99); q, g = f"{a} * {b} + {c}", a * b + c
        if q not in _V17_PROBE_QS:
            return q, g


def _v17_prompt_ids(content):
    msg = [{"role": "user", "content": content}]
    try:
        s_ = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except Exception:
        s_ = content + "\n"
    return list(tok(s_, add_special_tokens=False)["input_ids"])


def _v17_stop_ids():
    out = set()
    for t_ in ("<|im_end|>", "<|endoftext|>"):
        try:
            i_ = tok.convert_tokens_to_ids(t_)
            if isinstance(i_, int) and i_ >= 0 and i_ != getattr(tok, "unk_token_id", None):
                out.add(i_)
        except Exception:
            pass
    if tok.eos_token_id is not None:
        out.add(int(tok.eos_token_id))
    return out


@torch.no_grad()
def _v17_decode(prompts, max_new):
    """Greedy, batched, cached decoding of equal-length prompts in the SERVED world. Returns lists of new ids."""
    ids = torch.tensor(prompts, dtype=torch.long, device=DEV)
    stop = _v17_stop_ids()
    o = model(input_ids=ids, use_cache=True)
    past = getattr(o, "past_key_values", None)             # no cache -> a full forward per token (slow, same function)
    outs = [[] for _ in prompts]
    done = [False] * len(prompts)
    nxt = o.logits[:, -1].float().argmax(-1)
    for _ in range(int(max_new)):
        for b, t_ in enumerate(nxt.tolist()):
            if done[b]:
                continue
            if t_ in stop:
                done[b] = True
                continue
            outs[b].append(t_)
            if _V17_ANS.search(tok.decode(outs[b], skip_special_tokens=True) + " ") and tok.decode(outs[b][-1:]).strip() == "":
                done[b] = True
        if all(done):
            break
        if past is not None:
            o = model(input_ids=nxt[:, None], past_key_values=past, use_cache=True)
            past = o.past_key_values
        else:
            ids = torch.cat([ids, nxt[:, None]], 1)
            o = model(input_ids=ids, use_cache=False)
        nxt = o.logits[:, -1].float().argmax(-1)
    del o, past
    return outs


def _v17_loop_round():
    """The student answers V17_LOOP_N problems; every exchange (prompt + its own answer + end of turn) goes into the
    stream the next LOOP rows are cut from. The teacher then grades every token the student wrote, in the step."""
    t0 = time.time()
    rng = _random.Random(7919 * (V17_LOOP["rounds"] + 1) + int(V15_STEP0))
    probs = [_v17_problem(rng) for _ in range(int(V17_LOOP_N))]
    groups = {}
    for q, g in probs:
        p_ = _v17_prompt_ids(_V17_INSTR + q)
        groups.setdefault(len(p_), []).append((p_, q, g))
    was = model.training
    model.eval()
    set_world("serving")
    end = [i_ for i_ in _v17_stop_ids()][:1]
    nl = list(tok("\n", add_special_tokens=False)["input_ids"])
    try:
        for _, items in sorted(groups.items()):
            for i0 in range(0, len(items), int(V17_LOOP_BATCH)):
                part = items[i0:i0 + int(V17_LOOP_BATCH)]
                outs = _v17_decode([p_ for p_, _, _ in part], V17_LOOP_NEW)
                for (p_, q, g), o_ in zip(part, outs):
                    m_ = _V17_ANS.search(tok.decode(o_, skip_special_tokens=True))
                    ok = m_ is not None and abs(float(m_.group(1).replace(",", "")) - g) < 1e-6
                    V17_LOOP["made"] += 1
                    V17_LOOP["right"] += int(ok)
                    V17_LOOP["buf"] += p_ + o_ + end + nl
    finally:
        model.train(was)
        set_world("serving")
    while len(V17_LOOP["buf"]) >= int(SEQ_LEN):
        V17_LOOP["queue"].append(V17_LOOP["buf"][:int(SEQ_LEN)])
        V17_LOOP["buf"] = V17_LOOP["buf"][int(SEQ_LEN):]
    V17_LOOP["rounds"] += 1
    V17_LOOP["sec"] += time.time() - t0


def v17_loop_swap(inputs):
    """Every V17_LOOP_EVERY-th micro-batch: row 0 becomes a stream of the student's own answers."""
    if not (V17_LOOP_ON and V17_INSTEP) or V17_LOOP.get("off"):
        return inputs
    V17_LOOP["mb"] += 1
    if V17_LOOP["mb"] % int(V17_LOOP_EVERY):
        return inputs
    try:
        tries = 0
        while not V17_LOOP["queue"] and tries < 6:
            _v17_loop_round()
            tries += 1
        if not V17_LOOP["queue"]:
            return inputs
        row = torch.tensor(V17_LOOP["queue"].pop(0), dtype=inputs["input_ids"].dtype, device=inputs["input_ids"].device)
        ids = inputs["input_ids"].clone()
        L_ = min(ids.shape[1], row.shape[0])
        ids[0, :L_] = row[:L_]
        out = {**inputs, "input_ids": ids}
        if "src" in inputs:
            src = inputs["src"].clone()
            src[0] = V17_SRC["loop"]
            out["src"] = src
        return out
    except Exception as _e:
        V17_LOOP["off"] = f"{type(_e).__name__}: {str(_e)[:120]}"
        print(f"[loop] off ({V17_LOOP['off']}) — the run goes on without it")
        model.train()
        set_world("serving")
        return inputs


class V17Watch(TrainerCallback):
    """Every V17_WATCH_EVERY updates: KL per source of the training batches, and how often the student's own answers were right."""
    def on_step_end(self, args, state, control, **kw):
        st = _abs_step(state)
        if st % int(V17_WATCH_EVERY):
            return
        rows = v17_src_report()
        lp = V17_LOOP
        acc = (f" | LOOP: the student was right on {lp['right']}/{lp['made']} of its own answers "
               f"({100 * lp['right'] / max(1, lp['made']):.0f}%), {lp['sec'] / 60:.1f} min spent writing" if lp["made"] else "")
        dl = V17_DRIFT_LOG
        if dl["pos"]:
            acc += (f" | DRIFT: the student forked at {100 * dl['forks'] / dl['pos']:.1f}% of the positions it read, "
                    f"{dl['made']} of its own tokens were written into {dl['seqs']} sequences")
            lp.setdefault("drift_hist", []).append((st, dl["forks"] / dl["pos"], dl["made"]))
            dl["seqs"], dl["pos"], dl["forks"], dl["made"] = 0, 0, 0, 0
        lp["hist"].append((st, lp["right"], lp["made"]))
        lp["right"], lp["made"] = 0, 0
        if rows:
            print(f"[v17 step {st}] KL by source: " + ", ".join(f"{k} {v:.3f}" for k, v in sorted(rows.items())) + acc)



# ---------------------------------------------------------------- DRIFT: the student's own mistakes, in every kind of text
V17_DRIFT_LOG = {"mb": 0, "seqs": 0, "pos": 0, "forks": 0, "made": 0}


@torch.no_grad()
def v17_drift(inputs):
    """The student reads the batch once (served, no gradient). Where its first choice differs from the text — its own
    fork — the next token is replaced, at random, by the student's choice. The teacher then labels the batch as it
    now reads: every position after a replaced token asks "how does the teacher go on from THIS mistake?". Teacher
    forcing never shows the student its own errors; long answers fail exactly there. DRIFT does it for text, code,
    science and math alike, for one extra forward and without generating."""
    if not (V17_INSTEP and float(V17_DRIFT) > 0) or V17_LOOP.get("drift_off"):
        return inputs
    V17_DRIFT_LOG["mb"] += 1
    if V17_DRIFT_LOG["mb"] % int(V17_DRIFT_EVERY):
        return inputs
    try:
        ids = inputs["input_ids"]
        was = model.training
        model.eval()
        set_world("serving")
        try:
            pred = model(input_ids=ids).logits[:, :-1].argmax(-1)          # the student's choice for positions 1..L-1
        finally:
            model.train(was)
        nxt = ids[:, 1:]
        fork = pred != nxt
        fork[:, :8] = False                                                   # keep the first tokens as written
        g = torch.Generator(device="cpu").manual_seed(1000003 * V17_DRIFT_LOG["mb"] + int(V15_STEP0))
        take = fork & (torch.rand(fork.shape, generator=g).to(fork.device) < float(V17_DRIFT))
        cap = max(1, int(float(V17_DRIFT_MAX) * ids.shape[1]))
        for b in range(take.shape[0]):                                      # at most `cap` replacements per sequence
            pos = take[b].nonzero().flatten()
            if pos.numel() > cap:
                keep = pos[torch.randperm(pos.numel(), generator=g)[:cap].to(pos.device)]
                take[b] = False
                take[b, keep] = True
        new = ids.clone()
        new[:, 1:] = torch.where(take, pred, nxt)
        V17_DRIFT_LOG["seqs"] += int(ids.shape[0])
        V17_DRIFT_LOG["pos"] += int(nxt.numel())
        V17_DRIFT_LOG["forks"] += int(fork.sum())
        V17_DRIFT_LOG["made"] += int(take.sum())
        del pred, fork, take
        return {**inputs, "input_ids": new}
    except Exception as _e:
        V17_LOOP["drift_off"] = f"{type(_e).__name__}: {str(_e)[:120]}"
        print(f"[drift] off ({V17_LOOP['drift_off']}) — the run goes on without it")
        model.train()
        set_world("serving")
        return inputs


# ---------------------------------------------------------------- the long-context curve
V17_LONG = {"ids": globals().get("V17_LONG_IDS"), "src": globals().get("V17_LONG_SRC"), "curves": {}}


@torch.no_grad()
def v17_long_curve(tag, bins=(0, 1024, 4096, 8192, 16384, 32768)):
    """KL(teacher || student) and first-token agreement by POSITION in one long document, served exactly as counted.
    Tries the full length, then halves it when the GPU says no."""
    ids_all = V17_LONG["ids"]
    if ids_all is None or len(ids_all) < 2048:
        print(f"[long] {tag}: no long document available — skipped")
        return None
    was = model.training
    model.eval()
    lm = model.lm_head
    res = None
    for Lm in (int(V17_LONG_MAX), int(V17_LONG_MAX) // 2, int(V17_LONG_MAX) // 4):
        L_ = min(len(ids_all), Lm)
        if L_ < 2048 and Lm != int(V17_LONG_MAX):
            break
        ids = torch.tensor([ids_all[:L_]], dtype=torch.long, device=DEV)
        try:
            model.lm_head = torch.nn.Identity()
            set_world("dense_teacher")
            ht = model(input_ids=ids).logits[0]
            set_world("serving")
            hs = model(input_ids=ids).logits[0]
            model.lm_head = lm
            kl = torch.empty(L_, dtype=torch.float32, device=DEV)
            ag = torch.empty(L_, dtype=torch.float32, device=DEV)
            Pb, Ub = _fh_factors() if V16.get("head_strict") else (None, None)
            k_ = min(int(funnel.shortlist), int(funnel.V))
            for c0 in range(0, L_, 256):
                zt_ = (ht[c0:c0 + 256].to(_LMH.weight.dtype) @ _LMH.weight.t()).float()
                if V16.get("head_strict"):
                    zs_, _, _, _, _ = _fh_mix(hs[c0:c0 + 256].to(_LMH.weight.dtype), _LMH.weight, Pb, Ub, _LMH.ns_fb.float(), k_)
                else:
                    zs_ = (hs[c0:c0 + 256].to(_LMH.weight.dtype) @ _LMH.weight.t()).float()
                pt_ = zt_.log_softmax(-1)
                ps_ = zs_.log_softmax(-1)
                kl[c0:c0 + 256] = (pt_.exp() * (pt_ - ps_)).sum(-1)
                ag[c0:c0 + 256] = (pt_.argmax(-1) == ps_.argmax(-1)).float()
                del zt_, zs_, pt_, ps_
            rows = []
            for a, b in zip(bins[:-1], bins[1:]):
                if a >= L_:
                    break
                bb = min(b, L_)
                rows.append((a, bb, float(kl[a:bb].mean()), float(ag[a:bb].mean())))
            res = {"length": L_, "rows": rows, "src": V17_LONG["src"]}
            del ht, hs, kl, ag
            break
        except torch.cuda.OutOfMemoryError:
            model.lm_head = lm
            free_cuda(f"long curve at {L_} tokens")
            continue
        finally:
            model.lm_head = lm
            set_world("serving")
    model.train(was)
    if res is None:
        print(f"[long] {tag}: did not fit in GPU memory even at {int(V17_LONG_MAX) // 4} tokens — skipped")
        return None
    V17_LONG["curves"][tag] = res
    print(f"[long] {tag} — one document of {res['length']} tokens ({res['src']}), KL / same first token by position:")
    for a, b, k_, g_ in res["rows"]:
        print(f"    tokens {a:>6d}-{b:<6d}  KL {k_:.4f}  same first token {100 * g_:.1f}%")
    return res


def v16_head_loss(hidden, lm, labels, tid, chunk, tlp=None, ht=None):
    """(nll, slog, px, fkl) of the head that is SERVED: the strict head when the law is on (px = its proxy's distance to
    the full head), the dense head otherwise (px = None). Used by the pre-flight and by the training step."""
    if V16.get("head_strict") and V16.get("loss_strict", True):
        dk = int(MK_DK_HEAD) if (_MK.get("on") and "head" in _MK["acc"]) else 0
        k = min(int(funnel.shortlist), int(funnel.V))
        return _StrictHeadLoss.apply(hidden, lm.weight, lm.ns_fP, lm.ns_fU, lm.ns_fb, labels, tid,
                                     max(16, int(chunk) // 2), k, min(dk, k, int(funnel.V) - k),
                                     tlp, (float(LAMBDA_LIFT) / float(LAMBDA_PX)) if (tlp is not None and float(LAMBDA_PX) > 0) else 0.0,
                                     ht if V17_FULLKL else None)
    lg = None
    if "_ChunkedHeadLoss" in globals():
        nll, slog = _ChunkedHeadLoss.apply(hidden, lm.weight, labels, tid, int(chunk))
    else:                                                  # (the pre-flight runs before the training cell defines it)
        lg = (hidden.to(lm.weight.dtype) @ lm.weight.t()).float()
        lse = lg.logsumexp(-1)
        nll = lse - lg.gather(-1, labels[..., None]).squeeze(-1)
        slog = lg.gather(-1, tid) - lse[..., None]
    return nll, slog, None, None


def _v15_preflight():
    """One training-mode forward + backward through the whole model on a short prefix, with the loss pieces the real
    step uses: the served head inside the loss, the fork loss, the lens regression, and the market's probes.
    Returns the per-family count of tensors that received a finite gradient. No optimizer step: nothing is changed."""
    was, morm0 = model.training, _v15_morm_get()
    lm = getattr(model, "lm_head", None)
    n_ = min(int(SEQ_LEN), 128)
    sh = V15_EXTRA_SHARDS[0] if V15_EXTRA_SHARDS else None
    if sh is not None:
        ids = sh["input_ids"][0:1, :n_].long().to(DEV)
        if "teacher_ids" in sh:
            tid = sh["teacher_ids"][0:1, :n_ - 1].long().to(DEV)
            tlp = sh["teacher_logprobs"][0:1, :n_ - 1].float().to(DEV)
        else:                                                  # v17: no cache — the teacher labels the block here
            _lmh_keep = model.lm_head
            model.lm_head = torch.nn.Identity()
            try:
                _, (_ti, _tl), _pf_ht = v17_teacher(model, ids, False)
            finally:
                model.lm_head = _lmh_keep
            tid, tlp = _ti[:, :n_ - 1], _tl[:, :n_ - 1]
    else:
        ids = teach_blocks[0:1, :n_].to(DEV)
        _V = int(lm.weight.shape[0])
        tid = torch.stack([ids[:, 1:], (ids[:, 1:] + 1) % _V], -1)
        tlp = torch.tensor([math.log(0.7), math.log(0.2)], device=DEV).expand(1, n_ - 1, 2)
    _ac = getattr(torch, "autocast", None)
    ctx = _ac(device_type="cuda", dtype=torch.bfloat16) if (_ac is not None and DEV.type == "cuda") else contextlib.nullcontext()
    try:
        model.train()
        set_world("serving")
        model.zero_grad(set_to_none=True)
        _mk_probe_reset()
        if _WAL.get("on"):                                     # the wallet's paths run with the window fully open
            _wal_open(1.0)
            _wal_acc_reset()
        for _, m in PRISM_MODULES:
            m._store_aux = True
        if lm is not None:
            model.lm_head = torch.nn.Identity()
        with ctx:
            hid = model(input_ids=ids).logits[:, :-1]        # hidden states: the head is swapped out, as in the real step
            _pfh = locals().get("_pf_ht")
            nll, slog, px, fkl = v16_head_loss(hid, lm, ids[:, 1:], tid, LOSS_CHUNK, tlp=tlp,
                                               ht=(None if _pfh is None else _pfh[:, :-1]))
            loss = (fkl.mean() if fkl is not None and _pfh is not None else partition_kl(tlp, slog).mean()) + 0.0 * nll.mean()
            if LAMBDA_FORK > 0:
                loss = loss + float(LAMBDA_FORK) * v16_fork(tlp, slog, V16_FORK_ALPHA).mean()
            if px is not None:
                loss = loss + float(LAMBDA_PX) * px
            for _, m in PRISM_MODULES:
                aux, m.last_aux = getattr(m, "last_aux", None), None
                if aux is not None and aux.get("ns_fit") is not None:
                    loss = loss + 0.01 * aux["ns_fit"].float()
        loss.backward()
        ok = lambda ps: sum(1 for p in ps if p.grad is not None and bool(p.grad.isfinite().all()) and float(p.grad.abs().sum()) > 0)
        fin = lambda g: g is not None and math.isfinite(float(g))
        n_pr = sum(1 for pp, pm in _MK["probes"].values() if fin(pp.grad) and fin(pm.grad)) if _MK.get("on") else 0
        n_hd = int(_MK.get("on") and "head" in _MK["acc"] and all(math.isfinite(float(v)) for v in _MK["head_g"])
                   and any(float(v) != 0.0 for v in _MK["head_g"]))
        n_wal = sum(1 for st in _WAL.get("order", []) if fin(st["probe"].grad)) if _WAL.get("on") else 0
        _rc = _wal_receipt() if _WAL.get("on") else None
        if _rc is not None and _rc[3] < -0.5:
            raise RuntimeError(f"wallet: a token ended {-_rc[3]:.0f} parameters below zero")
        return {"loss": float(loss), "FFN lens shadows": (ok(V15_LENS_PARAMS), len(V15_LENS_PARAMS)),
                "consequence head": (ok(V15_HEAD_W), len(V15_HEAD_W)), "ghost gains": (ok(V15_GHOST_PARAMS), len(V15_GHOST_PARAMS)),
                "attention shadows": (ok(V15_ATTN_PARAMS), len(V15_ATTN_PARAMS)), "lens predictor": (ok(V15_PRED_PARAMS), len(V15_PRED_PARAMS)),
                "strict head": (ok(V16_HEAD_PARAMS), len(V16_HEAD_PARAMS)), "own neurons": (ok(V16_OWN_PARAMS), len(V16_OWN_PARAMS)),
                "market probes": (n_pr + n_hd, (len(_MK["probes"]) + int("head" in _MK["acc"])) if _MK.get("on") else 0),
                "wallet sites": (n_wal, len(_WAL.get("order", [])) if _WAL.get("on") else 0),
                "wallet receipt": _rc}
    finally:
        if lm is not None:
            model.lm_head = lm
        for _, m in PRISM_MODULES:
            m._store_aux = False
            m.last_aux = None
        model.zero_grad(set_to_none=True)
        _mk_probe_reset()
        model.train(was)
        _v15_morm_set(morm0)
        set_world("serving")


def _pf_head(m, k, cap, want):
    """Put the consequence head on (when wanted and installed) and pay for it with exact neurons, inside `cap`."""
    if want and hasattr(m, "ns_w") and V15_HEAD:
        m.ns_hd = True
        k = int(k) - int(math.ceil(max(0, _ns_cost(m, k) - cap) / (3 * m.d_model)))
    m.ns_k = int(k)
    m.ns_st = bool(m.ns_hd)
    m.ns_active = _ns_cost(m)
    assert m.ns_active <= cap


def _pf_ghosts_off():
    """The ghosts' share of the budget goes to exact neurons instead (second sight and the head stay)."""
    for li, (_, m) in enumerate(PRISM_MODULES):
        if not _has[li] or not _PF_PLAN[li][3]:
            continue
        r1, n_s, k, k2, act, hd0 = _PF_PLAN[li]
        m.ns_k2, m.ns_hd = 0, hd0
        k0 = min(k + (m.d_model * k2 + m.D) // (3 * m.d_model), n_s)
        V15_K0[li] = (k0, _ns_cost(m, k0), hd0)
        assert V15_K0[li][1] <= act
        _pf_head(m, k0, act, V15_K1[li][2])
    if isinstance(globals().get("SSIGHT"), dict) and SSIGHT.get("kept"):
        SSIGHT["choice"] = str(SSIGHT.get("choice")) + " — ghosts switched off by the pre-flight, their budget read as exact neurons"


def _pf_reading_base(head):
    """Second sight and ghosts off: the FFNs read as the restored state did (head=True keeps the consequence head)."""
    for li, (_, m) in enumerate(PRISM_MODULES):
        if not _has[li]:
            continue
        k, act = globals().get("_SS_BASE", {}).get(li, (_PF_PLAN[li][2], _PF_PLAN[li][4]))
        m.ns_k2, m.ns_r1, m.ns_ns, m.ns_hd = 0, 0, int(m.D), _PF_PLAN[li][5]
        m.ns_k, m.ns_active = int(k), int(act)
        V15_K0[li] = (int(k), int(act), _PF_PLAN[li][5])
        if head:
            _pf_head(m, k, act, V15_K1[li][2])
        else:
            m.ns_hd = m.ns_st = False
            m.ns_active = _ns_cost(m)
            V15_K0[li] = (int(k), int(m.ns_active), False)
    if isinstance(globals().get("SSIGHT"), dict):
        SSIGHT["kept"] = False


def _pf_new_reading_off():
    _pf_reading_base(False)
    if isinstance(globals().get("SSIGHT"), dict):
        SSIGHT["choice"] = "switched off by the pre-flight"


def _pf_market_off():
    _MK["on"], _MK["closed"] = False, "switched off by the pre-flight"


def _pf_wallet_off():
    """Every token buys the quotas again; the quota market (verified separately) takes the budget over."""
    _WAL["on"] = False
    WAL_INFO.update({"on": False, "closed": "switched off by the pre-flight"})
    _MK["on"] = bool(V16_MARKET) and bool(MK_SITES)


def _pf_own_off():
    """The own copies are dropped from the forward pass and from the optimizer: those neurons are the teacher's again."""
    global V16_OWN_PARAMS
    for _, m in _owners:
        m.ns_own = False
    for p in V16_OWN_PARAMS:
        p.requires_grad_(False)
    V16_OWN_PARAMS = []
    V16_OWN_INFO["on"] = False


def _pf_head_train_off():
    """The strict head stays in every evaluation, but it is neither trained nor used inside the loss (the hidden state
    is then trained through the dense head, as in v15)."""
    global V16_HEAD_PARAMS
    for p in V16_HEAD_PARAMS:
        p.requires_grad_(False)
    V16_HEAD_PARAMS = []
    V16["loss_strict"] = False


V16["loss_strict"] = True
V15_PF = {"ok": None, "fallbacks": [], "grads": None}
if V15_PREFLIGHT:
    _t0 = time.time()
    for _name, _undo in (("", None), ("the wallet", _pf_wallet_off), ("the market", _pf_market_off), ("own neurons", _pf_own_off),
                         ("training of the strict head", _pf_head_train_off), ("ghost neurons", _pf_ghosts_off),
                         ("second sight + consequence head", _pf_new_reading_off)):
        if _undo is not None:
            _undo()
            V15_PF["fallbacks"].append(_name)
            print(f"    -> switched off: {_name}; trying again")
        try:
            V15_PF["grads"] = _v15_preflight()
            V15_PF["ok"] = True
            break
        except Exception as _e:
            V15_PF["ok"] = False
            _msg = f"{type(_e).__name__}: {str(_e)[:400]}"
            print(f"\n!! pre-flight failed: {_msg}")
            if is_oom(_e):
                free_cuda("pre-flight OOM")
            if "triton" in _msg.lower() or "hopper" in _msg.lower():
                raise RuntimeError("역전파 커널(fla-core / Triton) 오류입니다. H100·H200에서는 triton >= 3.7.1 이 필요합니다 "
                                   "(v13 실행 때 고친 것과 같은 문제). 환경을 고친 뒤 다시 실행하세요.\n" + _msg) from None
            _last_pf_err = _msg
    if not V15_PF["ok"]:
        raise RuntimeError("학습 단계의 순전파/역전파가 새 장치를 모두 꺼도 실패합니다. 아직 학습은 시작하지 않았습니다. "
                           "이 셀의 출력을 그대로 보내 주세요.\n" + _last_pf_err)
    _g = V15_PF["grads"]
    print(f"\n(D) PRE-FLIGHT — one training-mode forward + backward on {min(int(SEQ_LEN), 128)} tokens ran clean ({time.time() - _t0:.0f}s"
          + (f"; switched off first: {', '.join(V15_PF['fallbacks'])}" if V15_PF["fallbacks"] else "") + "). Tensors that received a gradient: "
          + " | ".join(f"{k} {v[0]}/{v[1]}" for k, v in _g.items() if k not in ("loss", "wallet receipt") and v[1]))
    if _g.get("wallet receipt") is not None:
        _r = _g["wallet receipt"]
        print(f"    wallet, window fully open: a token spent between {_r[0] / 1e6:+.3f}M and {_r[2] / 1e6:+.3f}M relative to the quotas "
              f"(mean {_r[1] / 1e6:+.3f}M; {_WAL['b0'] / 1e6:.3f}M is free) — lowest balance at the end {_r[3] / 1e6:.3f}M (never below zero)")
    for k, v in _g.items():
        if k not in ("loss", "wallet receipt") and v[1] and v[0] == 0:
            print(f"    !! no gradient reached: {k} — it will not train in this run")
    if _MK.get("on") and _g["market probes"][0] < _g["market probes"][1]:
        print(f"    !! only {_g['market probes'][0]} of {_g['market probes'][1]} market probes answered: sites without an answer do not trade")
    if V15_PF["fallbacks"]:
        V15_K1 = [(int(m.ns_k), int(m.ns_active), bool(getattr(m, "ns_hd", False))) if _has[li] else None
                  for li, (_, m) in enumerate(PRISM_MODULES)]
        V15_EXTRA_GROUPS = _v16_groups()
    _active = measured_active()
    assert _active <= max(B_START, globals().get("B_RESTORED", 0)), "census grew"
else:
    print("\n(D) pre-flight is off (V15_PREFLIGHT = False)")


def _v15_head_off():
    """Back to the reading rule and the neuron counts the model had when it reached this cell."""
    for li, (_, m) in enumerate(PRISM_MODULES):
        if V15_K0[li] is not None:
            m.ns_k, m.ns_active, m.ns_hd = V15_K0[li]


def _v15_head_on():
    for li, (_, m) in enumerate(PRISM_MODULES):
        if V15_K1[li] is not None:
            m.ns_k, m.ns_active, m.ns_hd = V15_K1[li]


# ================================================================ (E) the model to beat = the restored model UNDER THE LAW (own copies = originals, nothing trained)
_v15_head_off()
_ns_apply(V12_MASK)
set_world("serving")
E_REF_TEXT = v15_eval(heldout, NS_VERDICT_BLOCKS)
D_REF_TEXT = dict(v15_eval.last)
E_REF_MATH = v15_eval(heldout_math, V15_MATH_HOLD) if heldout_math is not None else None
D_REF_MATH = dict(v15_eval.last) if E_REF_MATH else {}
KL_REF = float(E_REF_TEXT[0])
if V15_PF["fallbacks"] and "KL_RESTORED" in globals() and KL_REF > KL_RESTORED * 1.002 and any(_ns_prog(m)[0] for _, m in PRISM_MODULES if hasattr(m, "ns_B")):
    # what is left of second sight after the pre-flight's cuts is not better than the restored reading: take that back too
    print(f"    second sight without its ghosts reads worse than the restored model (KL {KL_REF:.4f} vs {KL_RESTORED:.4f}) -> undone")
    _pf_reading_base(True)
    SSIGHT["choice"] = "undone (without ghosts it did not beat the restored reading)"
    V15_K1 = [(int(m.ns_k), int(m.ns_active), bool(getattr(m, "ns_hd", False))) if _has[li] else None
              for li, (_, m) in enumerate(PRISM_MODULES)]
    _v15_head_off()
    E_REF_TEXT = v15_eval(heldout, NS_VERDICT_BLOCKS)
    E_REF_MATH = v15_eval(heldout_math, V15_MATH_HOLD) if heldout_math is not None else None
    KL_REF = float(E_REF_TEXT[0])
PPL_REF = float(ppl(heldout, "topk"))                         # under the law (PPL_V12 above was read with the dense head)
set_world("serving")
SCORE_REF = v15_score(E_REF_TEXT, E_REF_MATH)
print(f"\n(E) MODEL TO BEAT (the restored model under the law, lens on {sum(V12_MASK)}/{_L} layers) — text: KL {KL_REF:.4f}, "
      f"same first token as the teacher {100 * E_REF_TEXT[1]:.1f}%, PPL {PPL_REF:.2f} (dense {PPL.get('teacher_dense', float('nan')):.2f}; "
      f"with the loopholes it read {float(PPL_V12):.2f})"
      + (f" | reasoning: KL {E_REF_MATH[0]:.4f}, same first token {100 * E_REF_MATH[1]:.1f}%, DECISIVE positions "
         f"{100 * D_REF_MATH.get('decisive', 0.0):.1f}% (runs of {V16_RUN} without a fork: {100 * D_REF_MATH.get('run', 0.0):.1f}%)" if E_REF_MATH else "")
      + f" | score {SCORE_REF:.4f} | census {measured_active() / 1e9:.3f}B")
_ns_apply(_has)
_v15_head_on()
set_world("serving")
if V15_K0 != V15_K1 or V12_MASK != _has:
    E_HEAD_TEXT = v15_eval(heldout, NS_VERDICT_BLOCKS)
    print(f"    as it enters the training (lens on {sum(_has)}/{_L} layers, consequence head and its cost): text KL {E_HEAD_TEXT[0]:.4f} | census {measured_active() / 1e9:.3f}B")
else:
    E_HEAD_TEXT = E_REF_TEXT


# ================================================================ (F) rollback copy, resume
def _v15_get():
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def _v15_load(state):
    """Write a saved set of tensors back by name — into the parameter, or (when this session trains a smaller set than
    the one that saved it) into the buffer of the same name."""
    named, n_ok = dict(model.named_parameters()), 0
    with torch.no_grad():
        for n, t in state.items():
            p = named.get(n)
            if p is None:
                path, _, attr = n.rpartition(".")
                try:
                    p = getattr(model.get_submodule(path), attr, None)
                except Exception:
                    p = None
            if p is None or tuple(p.shape) != tuple(t.shape):
                continue
            p.copy_(t.to(p.device, p.dtype))
            n_ok += 1
    return n_ok


V15_SNAP_MORM = _v15_morm_get()
V15_SNAP = _v15_get()                                         # the restored model, for an exact rollback
V15_BASE_SIG = f"{os.path.basename(STATE_PATH)}|{STATE.get('config', {}).get('__PRISM_VER__')}|{len(STATE.get('lineage', []))}"
V15_RESUME = os.path.join(V15_DIR, "resume.pt")
V15_STEP0, V15_RESUMED = 0, None
_cands = [p for p in ([V15_RESUME, os.path.join(LOCAL_DIR, "resume.pt")]
                      + ([os.path.join(os.environ["PRISM_STATE_ALT"], "v17_state", "resume.pt")] if os.environ.get("PRISM_STATE_ALT") else []))
          if os.path.exists(p)]
if V15_RESUME_ON and _cands:
    _ck = torch.load(_cands[0], map_location="cpu", weights_only=False)
    if _ck.get("base") != V15_BASE_SIG:
        print(f"\n(F) a resume file exists but belongs to another starting state ({_ck.get('base')!r} vs {V15_BASE_SIG!r}) -> ignored")
    else:
        _oi = _ck.get("own_idx") or {}
        _named_m = dict(PRISM_MODULES)
        for _nm_, _ix in _oi.items():                         # the own neurons of the cut session (same neurons, their trained copies follow)
            _m_ = _named_m.get(_nm_)
            if _m_ is not None and getattr(_m_, "ns_oi", None) is not None and tuple(_m_.ns_oi.shape) == tuple(_ix.shape):
                _m_.ns_oi.copy_(_ix.to(_m_.ns_oi.device))
        _n = _v15_load(_ck["trainable"])
        if _ck.get("morm") is not None:
            _v15_morm_set(_ck["morm"])
        V15_STEP0, V15_RESUMED = int(_ck["step"]), _ck
        print(f"\n(F) RESUMED from {_cands[0]}: {_n}/{len(_ck['trainable'])} tensors of the best state so far (step "
              f"{_ck['best_step']}, score {_ck['best_score']:.4f}); the schedule continues at step {V15_STEP0} of {V15_STEPS}")
    del _ck
if V15_RESUMED is None:
    print("\n(F) fresh start (no resume file for this starting state)")
V15_START = _v15_get() if V15_RESUMED is not None else V15_SNAP
V15_START_MORM = _v15_morm_get()

STEPS = int(V15_STEPS)
GRAD_ACCUM = int(V15_GRAD_ACCUM)
if V15_FINALIZE_ONLY:
    assert V15_RESUMED is not None, "V15_FINALIZE_ONLY needs a resume file (v17_state/resume.pt) for this starting state"
    STEPS = int(V15_STEP0)
    print("    V15_FINALIZE_ONLY: no training in this session — the resumed best state goes straight to the verdict")
WARMUP_STEPS = int(V15_WARMUP)
TRAIN_PATH = "hard"


def v15_lr_lambda(step):
    """Warm-up (again after a resume: the optimizer's moments start empty), full rate, then a cosine to a FLOOR —
    v13's rate was already at 9% of its peak when the healing curve flattened; here the run never coasts to zero.
    The governor's cuts multiply everything."""
    step = int(step)
    warm = min(1.0, (step - V15["warm0"] + 1) / max(1, V15_WARMUP))
    el = 0.0 if V15["t0"] is None else (time.time() - V15["t0"]) / 60.0
    prog = max(step / max(1, STEPS), el / max(1e-9, float(TRAIN_BUDGET_MIN)))     # the GPU's speed is not known in advance:
    if prog <= V15_FLAT_FRAC:                                                      # the run is as far as its steps OR its clock say
        shape = 1.0
    else:
        p2 = min(1.0, (prog - V15_FLAT_FRAC) / max(1e-9, 1.0 - V15_FLAT_FRAC))
        shape = V15_LR_FLOOR + (1.0 - V15_LR_FLOOR) * 0.5 * (1.0 + math.cos(math.pi * p2))
    return V15["lr_scale"] * warm * shape


# ================================================================ (G) the stream anchor (v13), called inside the step
def _v15_idx(n_states):
    return list(range(int(ANCHOR_STRIDE), n_states, int(ANCHOR_STRIDE)))


@torch.no_grad()
def v15_teacher_states(mdl, input_ids):
    """The dense teacher's residual stream on this micro-batch (every ANCHOR_STRIDE-th layer), no gradient."""
    set_world("dense_teacher")
    try:
        hs = getattr(mdl(input_ids=input_ids, output_hidden_states=True), "hidden_states", None)
        return None if hs is None else [hs[i].detach() for i in _v15_idx(len(hs))]
    finally:
        set_world("serving")


_v15_warned = [False]


def v15_anchor(hs_student, hs_teacher):
    """Mean over the anchored layers of ||h_student - h_teacher||^2 / ||h_teacher||^2."""
    if hs_student is None or not hs_teacher:
        if not _v15_warned[0]:
            _v15_warned[0] = True
            print("[anchor] the model returns no hidden states -> the stream anchor is OFF for this run")
        return None
    tot = 0.0
    for j, i in enumerate(_v15_idx(len(hs_student))):
        t = hs_teacher[j].float()
        tot = tot + (hs_student[i].float() - t).pow(2).sum() / t.pow(2).sum().clamp_min(1e-12)
    return tot / max(1, len(hs_teacher))


# ================================================================ (H) watch: eval, best state, governor, resume file
V16_DEC = {}                                                   # decisive agreement / runs of the last quick eval (reasoning blocks)


def _v15_quick():
    et = v15_eval(heldout, V15_EVAL_BLOCKS)
    V16_DEC.clear()
    V16_DEC.update({"text": float(v15_eval.last.get("decisive", 0.0))})
    em = v15_eval(heldout_math, V15_EVAL_MATH) if heldout_math is not None else None
    if em:
        V16_DEC.update({"reasoning": float(v15_eval.last.get("decisive", 0.0)), "run": float(v15_eval.last.get("run", 0.0))})
    if v15_eval.last.get("spent_max") is not None:
        _cq = float(measured_active())
        V16_DEC.update({"touched_mean": _cq + v15_eval.last["spent_mean"], "touched_max": _cq + v15_eval.last["spent_max"]})
    return et, em, v15_score(et, em)


_dec_s = lambda: ((f" | decisive: reasoning {100 * V16_DEC['reasoning']:.1f}%, runs of {V16_RUN} {100 * V16_DEC['run']:.1f}%" if "reasoning" in V16_DEC
                   else f" | decisive (text) {100 * V16_DEC.get('text', 0.0):.1f}%")
                  + (f" | touched per token: mean {V16_DEC['touched_mean'] / 1e9:.4f}B, max {V16_DEC['touched_max'] / 1e9:.4f}B" if "touched_max" in V16_DEC else ""))
_et, _em, _s0 = _v15_quick()
V15 = {"curve": [(V15_STEP0, _s0, _et[0], (_em[0] if _em else None), _et[1], (_em[1] if _em else None), V16_DEC.get("reasoning"), V16_DEC.get("run"))],
       "best": _s0, "best_step": V15_STEP0, "best_state": None, "best_morm": None, "t0": None, "stopped": None,
       "lr_scale": 1.0, "strikes": 0, "cuts": 0, "persisted": V15_STEP0, "start_score": _s0, "warm0": V15_STEP0}
if V15_RESUMED is not None:
    V15["curve"] = [tuple(c) for c in V15_RESUMED.get("curve", []) if c[0] < V15_STEP0] + V15["curve"]
    V15["lr_scale"], V15["cuts"] = float(V15_RESUMED.get("lr_scale", 1.0)), int(V15_RESUMED.get("cuts", 0))
print(f"\n(H) START of this session (small eval set) — text KL {_et[0]:.4f} / first-token agreement {100 * _et[1]:.1f}%"
      + (f" | reasoning KL {_em[0]:.4f} / {100 * _em[1]:.1f}%" if _em else "") + _dec_s() + f" | score {_s0:.4f}")


def _v15_persist(st, why=""):
    """The best state so far -> v17_state/resume.pt (what a later session continues from if this one is cut)."""
    state = V15["best_state"]
    if state is None or V15["best_step"] <= V15["persisted"]:
        return
    need = sum(int(t.numel()) * 4 for t in state.values()) * 1.15
    path = V15_RESUME
    try:
        if _shutil.disk_usage(os.path.dirname(path)).free < need:
            path = os.path.join(LOCAL_DIR, "resume.pt")
            print(f"[resume] not enough room in {V15_DIR} ({need / 2**30:.1f} GiB needed) -> {path} (this disk does NOT survive the session)")
        _t = time.time()
        torch.save({"step": int(st), "best_step": int(V15["best_step"]), "best_score": float(V15["best"]),
                    "trainable": state, "morm": V15["best_morm"], "curve": V15["curve"], "base": V15_BASE_SIG,
                    "own_idx": {n: m.ns_oi.detach().cpu() for n, m in PRISM_MODULES if getattr(m, "ns_oi", None) is not None},
                    "market_log": list(_MK.get("log", [])),
                    "lr_scale": float(V15["lr_scale"]), "cuts": int(V15["cuts"])}, path + ".tmp")
        os.replace(path + ".tmp", path)
        V15["persisted"] = int(V15["best_step"])
        print(f"[resume] best state (step {V15['best_step']}) written at step {st}{why}: {os.path.getsize(path) / 2**30:.2f} GiB, "
              f"{time.time() - _t:.0f}s")
        if (V15_PUSH_RESUME or path != V15_RESUME) and os.environ.get("PRISM_PUSH_REPO"):     # no persistent room here -> the Hub keeps it
            from huggingface_hub import HfApi
            HfApi().upload_file(path_or_fileobj=path, path_in_repo="v17_state/resume.pt", repo_id=os.environ["PRISM_PUSH_REPO"])
            print(f"[resume] uploaded to {os.environ['PRISM_PUSH_REPO']}/v17_state/resume.pt")
    except Exception as _e:
        print(f"[resume] not written (non-fatal): {type(_e).__name__}: {str(_e)[:120]}")


class V15Watch(TrainerCallback):
    def __init__(self):
        self._last = V15_STEP0

    def on_step_end(self, args, state, control, **kw):
        if V15["t0"] is None:
            V15["t0"] = time.time()
        st = _abs_step(state)
        el = (time.time() - V15["t0"]) / 60
        over = el > TRAIN_BUDGET_MIN
        tot = _LP["sum"].get("total") if _LP.get("n") else None
        broken = tot is not None and not math.isfinite(float(tot))          # the training loss itself is NaN / inf
        if not (broken or ((st % V15_EVAL_EVERY == 0 or over or st >= STEPS) and st > self._last)):
            return
        self._last = st
        if broken and V16_FUSE["held"] and not V16_FUSE["poison"] and V16_FUSE["skipped"] < V16_FUSE_MAX:
            V16_FUSE["held"] = False                           # the fuse held the update back: nothing to roll back
            _LP["n"], _LP["sum"] = 0, {}
            return
        V16_FUSE["held"] = False
        if broken:
            print(f"[v16] step {st}: the training loss is not finite")
            _LP["n"], _LP["sum"] = 0, {}                       # the poisoned running sums must not trigger this twice
            et, em, sc = (float("nan"), 0.0), None, float("nan")
        else:
            was = model.training
            model.eval()
            try:
                et, em, sc = _v15_quick()
            finally:
                set_world("serving")
                model.train(was)
        V15["curve"].append((st, sc, et[0], (em[0] if em else None), et[1], (em[1] if em else None),
                             (None if broken else V16_DEC.get("reasoning")), (None if broken else V16_DEC.get("run"))))
        tag = ""
        if not math.isfinite(sc):                              # NaN / inf: not a number to wait on
            V15["strikes"] = int(V15_GOV_PATIENCE)
            tag = "  (not finite -> governor now)"
        elif sc < V15["best"]:
            V15["best"], V15["best_step"], V15["strikes"] = sc, st, 0
            V15["best_state"], V15["best_morm"] = _v15_get(), _v15_morm_get()
            tag = "  <- best"
        elif sc > V15_GOV_TOL * V15["best"]:
            V15["strikes"] += 1
            tag = f"  (strike {V15['strikes']}/{V15_GOV_PATIENCE})"
        else:
            V15["strikes"] = 0
        print(f"[v16] step {st}: score {sc:.4f} | text KL {et[0]:.4f} agree {100 * et[1]:.1f}%"
              + (f" | reasoning KL {em[0]:.4f} agree {100 * em[1]:.1f}%" if em else "") + ("" if broken else _dec_s())
              + f" | start {V15['start_score']:.4f}, to beat {SCORE_REF:.4f} | "
              + (f"ceiling {_MK['cap'] / 1e9:.4f}B" if _WAL.get("on") else f"census {measured_active() / 1e9:.4f}B") + f" | {el:.0f} min{tag}")
        if V15["strikes"] >= V15_GOV_PATIENCE:                 # GOVERNOR: the run went the wrong way twice in a row
            V15["strikes"] = 0
            V15["cuts"] += 1
            _v15_load(V15["best_state"] if V15["best_state"] is not None else V15_START)
            _v15_morm_set(V15["best_morm"] if V15["best_morm"] is not None else V15_START_MORM)
            V15["lr_scale"] *= 0.5
            V15["warm0"] = st                                  # ... and a fresh warm-up, because
            try:                                               # ... the optimizer's moments belong to the run that went wrong
                model.zero_grad(set_to_none=True)
                globals()["optimizer"].state.clear()
            except Exception as _e:
                print(f"[governor] optimizer state not cleared ({type(_e).__name__})")
            _fuse_heal()
            V16_FUSE["skipped"] = 0
            print(f"[governor] back to the best state (step {V15['best_step']}), optimizer moments cleared, every learning rate "
                  f"x0.5 (now x{V15['lr_scale']:g}) — cut {V15['cuts']}/{V15_GOV_MAX_CUTS}")
            if V15["cuts"] >= V15_GOV_MAX_CUTS and not control.should_training_stop:
                control.should_training_stop = True
                V15["stopped"] = st
                print(f"[governor] {V15_GOV_MAX_CUTS} cuts used -> the training stops here; the best state is kept")
        if st % V15_PERSIST_EVERY == 0 or over or st >= STEPS or control.should_training_stop:
            _v15_persist(st)
        if over and not control.should_training_stop:
            control.should_training_stop = True
            V15["stopped"] = st
            print(f"[v16] wall-clock budget of {TRAIN_BUDGET_MIN} min reached at step {st} -> stopping the training")


V15_CALLBACKS = [V16Wallet(), V16Market(), V15Watch(), V17Watch()]        # prices / budgets move first: an eval at the same update sees them
if globals().get("V17_LONG_ON", True) and not globals().get("V15_FINALIZE_ONLY"):
    try:
        v17_long_curve("before training")
    except Exception as _e:
        print(f"[long] before training: skipped ({type(_e).__name__}: {str(_e)[:120]})")
        set_world("serving")
free_cuda("v17 ready to train")


## Training

In [ ]:
# Training — Sinkhorn-OT self-distillation (Spectrum Consolidation) + v9 Partition-KL / EMRD / LTD
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments, TrainerCallback

class PrismDistillSet(Dataset):
    """Per-SEQUENCE items: every stored sequence is a teaching sample —
    items are (shard i, row j) pairs over all stored batches."""
    def __init__(self, shards):
        self.data = shards
        self.index = [(i, j) for i, d in enumerate(shards)
                      for j in range(d["input_ids"].shape[0])]
    def __len__(self): return len(self.index)
    def __getitem__(self, k):
        i, j = self.index[k]
        d = self.data[i]
        x = d["input_ids"][j].long()
        if "teacher_ids" in d:
            ti, tl = d["teacher_ids"][j].long(), d["teacher_logprobs"][j].float()
        else:                                   # v17: labelled by the dense teacher inside the step
            ti = torch.zeros(x.shape[0], TEACHER_TOPK, dtype=torch.long)
            tl = torch.full((x.shape[0], TEACHER_TOPK), -30.0)
        return {"input_ids": x, "teacher_ids": ti, "teacher_logprobs": tl,
                "src": torch.tensor(V17_SRC.get(d.get("src", "text"), 0))}

def collate(feats):
    return {"input_ids": torch.stack([f["input_ids"] for f in feats]),
            "teacher_ids": torch.stack([f["teacher_ids"] for f in feats]),
            "teacher_logprobs": torch.stack([f["teacher_logprobs"] for f in feats]),
            "src": torch.stack([f["src"] for f in feats])}

class _ChunkedHeadLoss(torch.autograd.Function):
    """The head GEMM itself, chunked (the Liger / Cut-Cross-Entropy design,
    cf. arXiv:2410.10989 / arXiv:2411.09009).

    DistillTrainer swaps lm_head for an Identity during the training
    forward, so the model returns HIDDEN states [B, L, d] (20 MiB), and
    THIS Function computes
        logits_chunk = hidden[:, c0:c1] @ W_head.t()   (bf16, [B*ch, V])
    chunk-by-chunk, cast to fp32 for the loss arithmetic; backward rebuilds
    the exact per-token gradient per chunk and contracts it back through
    W_head into a [B, L, d] hidden gradient — the exact bf16 GEMM the stock
    lm_head backward would have run. The full-vocab tensor NEVER exists, in
    forward OR backward (chunking the arithmetic of an already-materialized
    tensor cannot shrink what backward keeps). Peak transient: two
    [B, ch, V] fp32 chunks (~0.5 GiB at ch=256, B=1), genuinely governed by
    LOSS_CHUNK.

    Math is IDENTICAL per token: every logit is an independent dot product
    over d, so row-blocking the GEMM changes only cuBLAS tile selection
    (bf16 last-bit drift); the per-token gradient formula is unchanged:
      d/dlogits = g_nll*(softmax - onehot(label))
                + sum_k g_slog[k]*(onehot(tid_k) - softmax)"""
    @staticmethod
    def forward(ctx, hidden, W, labels, tid, chunk):
        B, L, d = hidden.shape
        V = W.shape[0]
        K = tid.shape[-1]
        nll = torch.empty(B, L, dtype=torch.float32, device=hidden.device)
        slog = torch.empty(B, L, K, dtype=torch.float32, device=hidden.device)
        with torch.no_grad():
            for c0 in range(0, L, chunk):
                c1 = min(c0 + chunk, L)
                hc = hidden[:, c0:c1].reshape(-1, d).to(W.dtype)   # [B*ch, d]
                lg = (hc @ W.t()).float()                           # [B*ch, V]
                lse = lg.logsumexp(-1)                              # [B*ch]
                lab = labels[:, c0:c1].reshape(-1)
                nll[:, c0:c1] = (lse - lg.gather(-1, lab[:, None])
                                 .squeeze(-1)).reshape(B, c1 - c0)
                tidc = tid[:, c0:c1].reshape(-1, K)
                slog[:, c0:c1] = (lg.gather(-1, tidc) - lse[:, None]) \
                    .reshape(B, c1 - c0, K)
                del hc, lg, lse, lab, tidc
        ctx.save_for_backward(hidden, W, labels, tid)
        ctx.chunk = int(chunk)
        return nll, slog

    @staticmethod
    def backward(ctx, g_nll, g_slog):
        hidden, W, labels, tid = ctx.saved_tensors
        B, L, d = hidden.shape
        K = tid.shape[-1]
        gh = torch.empty_like(hidden)          # [B, L, d] — 20 MiB, not [B, L, V]
        with torch.no_grad():
            for c0 in range(0, L, ctx.chunk):
                c1 = min(c0 + ctx.chunk, L)
                hc = hidden[:, c0:c1].reshape(-1, d).to(W.dtype)
                lg = (hc @ W.t()).float()                           # [B*ch, V]
                p = lg.softmax(-1)                                  # [B*ch, V]
                g = g_nll[:, c0:c1].reshape(-1, 1).to(p.dtype) * p
                lab = labels[:, c0:c1].reshape(-1)
                g.scatter_add_(-1, lab[:, None],
                               (-g_nll[:, c0:c1].reshape(-1)).to(p.dtype)[:, None])
                if g_slog is not None:
                    tidc = tid[:, c0:c1].reshape(-1, K)
                    gs = g_slog[:, c0:c1].reshape(-1, K).to(p.dtype)
                    g.scatter_add_(-1, tidc, gs)
                    g -= gs.sum(-1, keepdim=True) * p
                gh[:, c0:c1] = (g.to(W.dtype) @ W).reshape(B, c1 - c0, d) \
                    .to(hidden.dtype)
                del hc, lg, p, g
        return gh, None, None, None, None


_IDENTITY = torch.nn.Identity()
_PHASE = {"hard": False}      # v9: True once AnnealCallback switched every family to the serving path
_LP = {"n": 0, "sum": {}}     # v9: running sums of every loss term (detached, on the GPU; printed by LossPartsCallback)


def _legacy_head_loss(logits, labels, tid):
    """Fallback for wrappers without a swappable lm_head attribute (and
    the return_outputs/eval shape): chunked arithmetic on the materialized
    logits, as plain autograd — mathematically identical to
    _ChunkedHeadLoss; only the memory profile differs (full-vocab tensors
    exist). The stock Qwen3.8 wrapper always takes the headless path."""
    lg = logits.float()
    lse = lg.logsumexp(-1)
    nll = lse - lg.gather(-1, labels[..., None]).squeeze(-1)
    slog = lg.gather(-1, tid) - lse[..., None]
    return nll, slog


class DistillTrainer(Trainer):
    def _move_model_to_device(self, model, device):
        """No blanket model.to(device): the substrate is already resident on
        `device`; the pre-training CPU-parking of the text-inert subtrees
        (vision tower / MTP) must stand — they are never executed by the
        text-only distillation."""
        return

    # ---- v8.0.3: per-micro-batch OOM shield --------------------------------
    # One CUDA OOM inside ONE micro-batch used to abort the optimizer step (and,
    # after a single recovery, the run). Now that micro-batch is retried on a
    # causal PREFIX (1/2, then 1/4 of the tokens — every layer of this model is
    # causal, so a prefix is a valid, shorter training sample) and only skipped
    # if even that does not fit. Every OOM drops the half-built gradient
    # accumulation window (partial backward grads would be biased toward the
    # last layers).
    _oom_skips = 0           # consecutive micro-batches lost to OOM
    _oom_shrinks = 0         # micro-batches that only fit as a shorter prefix

    @staticmethod
    def _prefix(inputs, frac):
        L = inputs["input_ids"].shape[1]
        n = max(64, int(L * frac))
        out = {}
        for k, v in inputs.items():
            if torch.is_tensor(v) and v.dim() >= 2 and v.shape[1] == L:
                out[k] = v[:, :n]
            else:
                out[k] = v
        return out

    def training_step(self, model, inputs, *args, **kwargs):
        for frac in (1.0, 0.5, 0.25):
            cur = inputs if frac == 1.0 else self._prefix(inputs, frac)
            failed = False
            try:
                loss = super().training_step(model, cur, *args, **kwargs)
            except RuntimeError as e:
                if not is_oom(e):
                    raise
                failed = True
            if not failed:
                v16_fuse_grads()
                DistillTrainer._oom_skips = 0
                if frac < 1.0:
                    DistillTrainer._oom_shrinks += 1
                    print(f"[oom-shield] micro-batch trained on a {frac:g}-length prefix "
                          f"({DistillTrainer._oom_shrinks} so far)")
                return loss
            # ---- outside the except block: the dead graph is releasable ----
            model.zero_grad(set_to_none=True)
            for _m in (m for _, m in PRISM_MODULES):
                _m.last_aux = None
            free_cuda(f"OOM in training_step (length x{frac:g})")
        DistillTrainer._oom_skips += 1
        print(f"[oom-shield] micro-batch skipped: OOM even at 1/4 length "
              f"({DistillTrainer._oom_skips} in a row)")
        if DistillTrainer._oom_skips >= 4:
            raise torch.cuda.OutOfMemoryError(
                "training_step: 4 consecutive micro-batches OOM even at 1/4 length "
                "- see the [mem ...] lines above")
        return torch.zeros((), device=self.args.device)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # Headless forward: lm_head is swapped for an Identity so out.logits
        # IS the hidden state [B, L, d] (20 MiB); the vocab GEMM happens
        # per-chunk inside _ChunkedHeadLoss (exception-safe swap, restored
        # in `finally`).
        #
        # Telemetry contract (the +24.2 GiB OOM class, fixed): the aux
        # side-channel is switched ON only for THIS forward; every value is
        # consumed here and the attributes cleared before returning — the
        # checkpoint recomputes that run during backward store NOTHING, so
        # each recomputed layer graph dies with its own backward.
        if not return_outputs:
            inputs = v17_loop_swap(inputs)          # v17 LOOP: some rows are the student's own answers (written before the head is swapped out)
            inputs = v17_drift(inputs)              # v17 DRIFT: the student's own forks written into the batch
        _prism_mods = [m for _, m in PRISM_MODULES]
        _hard = bool(_PHASE["hard"])
        for _m in _prism_mods:
            _m._store_aux = True
            # v9 side channels: detached oracle energies / dense twin, computed under no_grad inside the
            # module (so the checkpoint recompute during backward stays consistent whatever these flags are)
            _m.side_en = LAMBDA_EMRD > 0
            _m.side_ltd = (LAMBDA_LTD > 0) and _hard
        _lm = getattr(model, "lm_head", None)
        _headless = (not return_outputs) and (_lm is not None) \
            and (getattr(_lm, "weight", None) is not None)
        if _headless:
            model.lm_head = _IDENTITY
        try:
            # the dense teacher runs FIRST (so the student forward is the last one the MoRM controllers see)
            _anc_t, _tt, _ht = v17_teacher(model, inputs["input_ids"], LAMBDA_ANCHOR > 0)     # v17: the teacher labels THIS batch
            if _tt is not None:
                inputs = {**inputs, "teacher_ids": _tt[0], "teacher_logprobs": _tt[1]}
            out = model(input_ids=inputs["input_ids"], output_hidden_states=_anc_t is not None)
            labels = inputs["input_ids"][:, 1:]
            tid = inputs["teacher_ids"][:, :-1]      # align with hidden[:, :-1]
            tlp = inputs["teacher_logprobs"][:, :-1]
            if _headless:
                hidden = out.logits[:, :-1]          # [B, L-1, d] VIEW (20 MiB)
                nll_tok, slog, _px, _fkl = v16_head_loss(hidden, _lm, labels, tid, LOSS_CHUNK, tlp=tlp,
                                                        ht=(None if _ht is None else _ht[:, :-1]))     # the head that is SERVED (+ LIFT, + FULL KL)
            else:
                nll_tok, slog = _legacy_head_loss(out.logits[:, :-1],
                                                  labels, tid)
                _px, _fkl = None, None
            ce = nll_tok.mean()
            if KL_MODE == "partition":
                # v9 PARTITION-KL: KL of the coarse-grained distributions on {each teacher top-K id} U
                # {everything else}. Chain rule: KL_part = P_S * KL_support + KL_Bernoulli(P_S || Q_S), so
                # (unlike the v8 KL renormalised inside the support) it also sees the probability MASS the
                # student keeps on the teacher's support; still a lower bound of the full-vocab KL.
                _klp = partition_kl(tlp, slog)
                if _fkl is not None and _ht is not None:     # v17 FULL KL: the whole 248k distribution of the teacher
                    _LP["sum"]["kl_top64"] = _LP["sum"].get("kl_top64", 0.0) + _klp.mean().detach().float()
                    _klp = _fkl
                kl = _klp.mean()
                v17_src_note(inputs.get("src"), _klp)
            else:                                   # v8: both sides renormalised inside the support
                kl = kl_support(tlp, slog).mean()
            bal = z = zs = emrd = ltd = nll_tok.new_zeros((), dtype=torch.float32)
            n = n_en = n_ltd = 0
            _lens_fit, _n_fit = 0.0, 0
            for m in _prism_mods:
                aux = getattr(m, "last_aux", None)
                m.last_aux = None          # consume-and-clear
                if aux is not None and aux.get("ns_fit") is not None:     # v15: the lens predictor against the exact activations
                    _lens_fit = _lens_fit + aux["ns_fit"].float()
                    _n_fit += 1
                if aux is not None and "p" in aux:
                    p, lg = aux["p"], aux["logits"]
                    Pplan = sinkhorn_balanced(lg.detach().float(), n_iter=3)
                    bal = bal + (Pplan * (Pplan.clamp_min(1e-9).log()
                                          - p.float().clamp_min(1e-9).log())).sum(-1).mean()
                    z = z + torch.logsumexp(lg.float(), -1).pow(2).mean()
                    zs = zs + zero_sum_penalty(m.alpha) + zero_sum_penalty(m.beta)
                    n += 1
                    if "en" in aux:
                        # v9 EMRD: -log of the router mass on experts that own the oracle output energy.
                        # With k=1 the served gate renormalises to 1, so the output loss sends the router's
                        # CHOICE no gradient at all — this term is its supervision.
                        emrd = emrd + emrd_loss(p, aux["en"], K_CHOSEN)
                        n_en += 1
                    if "y_dense" in aux and "y" in aux:
                        # v9 LTD (hard phase): ||y - y_dense||^2 / ||y_dense||^2 against this layer's own
                        # dense twin on the same input — a dense local target for all 64 FFNs.
                        ltd = ltd + ltd_loss(aux["y"], aux["y_dense"])
                        n_ltd += 1
            # v9.1: LAMBDA_CE = 0 by default -> PURE TEACHER DISTILLATION. The corpus only supplies the inputs;
            # every target is the dense Qwen3.8 itself (its next-token distribution here, its own FFN outputs in
            # LTD). The hard-label CE (the corpus' next token) is still computed and logged, but not trained on:
            # fitting a small corpus' text would pull a strong model towards that text.
            _aux_reg = (LAMBDA_BAL * bal + LAMBDA_Z * z + LAMBDA_ZS * zs) / max(n, 1)
            loss = LAMBDA_CE * ce + LAMBDA_KL * kl + _aux_reg
            if n_en:
                loss = loss + LAMBDA_EMRD * emrd / n_en
            if n_ltd:
                loss = loss + LAMBDA_LTD * ltd / n_ltd
            if _n_fit:                              # v15 self-refitting lens (reaches B / Ag / Au only)
                loss = loss + LAMBDA_LENS * _lens_fit / _n_fit
                _LP["sum"]["lens"] = _LP["sum"].get("lens", 0.0) + (_lens_fit / _n_fit).detach().float()
            if _anc_t is not None:                  # stream anchor (v13)
                _anc = v15_anchor(getattr(out, "hidden_states", None), _anc_t)
                if _anc is not None:
                    loss = loss + LAMBDA_ANCHOR * _anc
                    _LP["sum"]["anchor"] = _LP["sum"].get("anchor", 0.0) + _anc.detach().float()
            if LAMBDA_FORK > 0:                     # v16 fork loss: where the student leaves the teacher's acceptable tokens
                _fk = v16_fork(tlp, slog, V16_FORK_ALPHA).mean()
                loss = loss + float(LAMBDA_FORK) * _fk
                _LP["sum"]["fork"] = _LP["sum"].get("fork", 0.0) + _fk.detach().float()
            if _px is not None:                     # v16: the head's proxy against the student's own full head (reaches P, U, b only)
                loss = loss + float(LAMBDA_PX) * _px
                _LP["sum"]["head_px"] = _LP["sum"].get("head_px", 0.0) + _px.detach().float()
            for _k, _v in (("ce", ce), ("kl", kl), ("bal", bal / max(n, 1)), ("z", z / max(n, 1)),
                           ("emrd", emrd / max(n_en, 1)), ("ltd", ltd / max(n_ltd, 1)),
                           ("reg(w)", _aux_reg)):
                _LP["sum"][_k] = _LP["sum"].get(_k, 0.0) + _v.detach().float()
            _LP["n"] += 1
            # v8 (MoRM 2.0): critic regression on the EXACT influence labels
            # (log-space MSE against the calibration WIK stash; the distill
            # STE gradient flows separately through the mask sites).
            if MORM2_LAMBDA_CRITIC > 0:
                crit, nc = 0.0, 0
                for _nm, _m in RESONA_MORM:
                    fit = getattr(_m, "_crit_fit", None)
                    if not isinstance(_m, MoRM2Policy) or fit is None:
                        continue
                    phi_c, lab_c = fit
                    idx = torch.randint(0, phi_c.shape[0],
                                        (min(256, phi_c.shape[0]),),
                                        device=phi_c.device)
                    Lh = _m.scores(phi_c[idx]).clamp_min(1e-12).log()
                    crit = crit + (Lh - lab_c[idx].clamp_min(1e-12).log()
                                   ).pow(2).mean()
                    nc += 1
                if nc:
                    loss = loss + MORM2_LAMBDA_CRITIC * crit / nc
                    _LP["sum"]["crit"] = _LP["sum"].get("crit", 0.0) + (crit / nc).detach().float()
            v16_fuse_loss(loss, {"kl": kl, "ce": ce, "fork": locals().get("_fk"), "head_px": _px, "reg": _aux_reg,
                                 "crit": locals().get("crit"), "lens": (_lens_fit if _n_fit else None)},
                          hidden=(hidden if _headless else None), ids=inputs["input_ids"])
            _LP["sum"]["total"] = _LP["sum"].get("total", 0.0) + loss.detach().float()
            return (loss, out) if return_outputs else loss
        finally:
            for _m in _prism_mods:
                _m._store_aux = False
                _m.side_en = False
                _m.side_ltd = False
            if _headless:
                model.lm_head = _lm


class MoRM2Callback(TrainerCallback):
    """v8 elastic-budget governance for MoRM 2.0 policies: anneals m_bar
    from exact (all writes — pure distillation, the critic learns from the
    label regression) down to the solver's target by mid-anneal, and feeds
    each policy's rate controller the observed mean write count of the last
    training forward (Theorem D in the loop)."""
    def on_step_end(self, args, state, control, **kw):
        if not RESONA_MORM or not isinstance(RESONA_MORM[0][1], MoRM2Policy):
            return
        frac = min(1.0, _abs_step(state)
                   / max(1, 0.5 * ANNEAL_FRAC * STEPS))      # v9: absolute step (resume-safe)
        for _nm, m in RESONA_MORM:
            if not isinstance(m, MoRM2Policy):
                continue
            tgt = max(1, int(round(m.H + (min(M_WRITE, m.H) - m.H) * frac)))
            m.set_budget(tgt, m_max=(max(tgt, int(m.m_max)) if m.m_max is not None else None))
            if m.last_m is not None:
                m.controller_update(float(m.last_m.float().mean()))


class AnnealCallback(TrainerCallback):
    """Last ANNEAL_FRAC of training: hard serving path for ALL families
    (FFN top-k + resonant top-k linears + MoRM write-sparsity). The switch
    reads the ABSOLUTE optimizer step from the LR scheduler (resume-safe:
    a per-call TrainerState counter restarts at 0 on every resumed
    trainer.train())."""
    def __init__(self): self.switched = False
    def on_step_end(self, args, state, control, **kw):
        sched = getattr(self, "sched", None)
        abs_step = int(sched.last_epoch) if sched is not None else state.global_step
        if not self.switched and abs_step >= ANNEAL_FRAC * STEPS:
            set_world("serving")
            _PHASE["hard"] = True        # v9: LTD (layer-local dense twin) switches on with the hard path
            self.switched = True
            print(f"\n[anneal] absolute step {abs_step}: switched ALL families "
                  "to the hard serving path")

_CKPT_EXEC = [0, 0]   # (checkpointed, plain) decoder layer-calls — a flag
#                      can be present while nothing executes; an execution
#                      counter cannot lie. Printed by the [vram] pulse.


class VRAMPulse(TrainerCallback):
    """Observe, don't govern: with the telemetry leak fixed the heap
    simply returns to the baseline between steps — this pulse proves it
    every 25 steps (alloc/peak/free + the checkpoint execution counter),
    and the leak canary (>+5 GiB over baseline at a step boundary, well
    above the optimizer's ~1.2 GiB band) turns any future pin into a
    one-line diagnosis."""

    def __init__(self, base_gib: float):
        self.base = float(base_gib)
        self.tick = 0

    def on_step_end(self, args, state, control, **kw):
        self.tick += 1
        if not torch.cuda.is_available() or self.tick % 25 != 0:
            return
        alloc = torch.cuda.memory_allocated() / 2**30
        peak = torch.cuda.max_memory_allocated() / 2**30
        free = torch.cuda.mem_get_info()[0] / 2**30
        print(f"[vram step {_abs_step(state)}] alloc {alloc:.1f} GiB | "
              f"peak {peak:.1f} GiB | free {free:.1f} GiB | ckpt-exec "
              f"{_CKPT_EXEC[0]}")
        if alloc - self.base > 5.0 + 8.0 * _n_trainable / 2**30:      # v16: the optimizer state of what really trains is not a leak
            print(f"[vram step {_abs_step(state)}] WARNING: "
                  f"{alloc - self.base:+.1f} GiB over baseline at a step "
                  "boundary — something is pinning a step graph")


def _abs_step(state):
    """ABSOLUTE optimizer step. TrainerState.global_step restarts at 0 on every trainer.train() — i.e. after each
    OOM recovery — while the LR scheduler (restored from the light checkpoint) carries the true step. Everything
    schedule-like (MoRM anneal, light checkpoints, logs) must read this, not state.global_step."""
    sched = globals().get("scheduler")
    return int(sched.last_epoch) if sched is not None else int(state.global_step)


class LossPartsCallback(TrainerCallback):
    """v9: running mean of every loss term, printed every `every` optimizer steps — one host sync per print.
    ce/kl/bal/z/emrd/ltd/crit are RAW (unweighted) values, reg(w) is the weighted bal+z+zs sum, and
    total is what the Trainer shows as 'Training Loss'. ce is logged even when LAMBDA_CE = 0."""
    every = 5      # same cadence as the Trainer's 'Training Loss' column

    def on_step_end(self, args, state, control, **kw):
        st = _abs_step(state)
        if _LP["n"] and st % self.every == 0:
            parts = {k: float(v) / _LP["n"] for k, v in _LP["sum"].items()}
            print(f"[loss parts step {st}] "
                  + " | ".join(f"{k} {v:.4f}" for k, v in parts.items())
                  + f" | {'hard' if _PHASE['hard'] else 'soft'} path")
            _LP["n"], _LP["sum"] = 0, {}


train_ds = PrismDistillSet(list(torch.load(TEACHER_SHARD, weights_only=False)) + list(globals().get("V15_EXTRA_SHARDS", [])))
print(f"training set: {len(train_ds)} sequences ({len(globals().get('V15_EXTRA_SHARDS', []))} shards of problems / traces mixed in)")

# frozen-substrate leak check: if the dense substrate ever became
# trainable, AdamW fp32 states alone (27B x 12 B/param) would demand ~300
# GiB and OOM at optimizer construction. Fail with a diagnosis instead.
_n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _n_trainable < 3_000_000_000, (
    f"{_n_trainable/1e6:.0f}M trainable params — the frozen substrate leaked "
    f"into the optimizer (AdamW fp32 states ≈ {_n_trainable*12/2**30:.0f} GiB). "
    "Check the surgery cells' requires_grad wiring.")
print(f"trainable parameters: {_n_trainable/1e6:.1f}M "
      f"(AdamW fp32 states ≈ {_n_trainable*12/2**30:.2f} GiB)")

# ============================================================================
# ============================================================================
# Pre-training baseline reclaim: free every pre-training resident, then
# name what is left.
#   (a) probe/recall/surgery leftovers bound in the notebook namespace,
#   (b) per-module telemetry caches pinning the LAST no-grad forward,
#   (c) calibration dicts that stayed CUDA-resident,
#   (d) stale teacher-cache tensors restored onto CUDA by torch.load —
#       forced back to host RAM,
#   (e) text-inert subtrees (vision tower / merger / MTP) CPU-parked for
#       the whole train() — restored right after training.
# ============================================================================
# ============================================================================
def _reclaim():
    for _name in ("anchor", "exact2", "serv", "out", "h_last", "ref",
                  "got", "sl", "b", "x",
                  # surgery-cell locals that pin the last calibration
                  # forward's tensors (e.g. the y_e [T, E, d] stacks)
                  "hs", "asub", "e_routed", "energy", "logits0", "topk_idx",
                  "y_e", "y_dense_sub", "Wr", "tau", "fus", "c",
                  "cal", "imp_l", "S_l"):
        _g = globals()
        if _name in _g:
            del _g[_name]
    # defensively resolve the stash dicts (missing names skip)
    for _dname in ("sketch", "stats", "mean", "std", "duty", "sqmean", "row_energy",
                   "dn_row_energy", "h_sub", "site_stash", "assign_store"):
        _d = globals().get(_dname)
        if not isinstance(_d, dict):
            continue
        for _k in list(_d):
            _t = _d[_k]
            if torch.is_tensor(_t) and _t.is_cuda:
                _d[_k] = _t.cpu()
    for _shard in train_ds.data:
        for _k in list(_shard):
            _t = _shard[_k]
            if torch.is_tensor(_t) and _t.is_cuda:
                _shard[_k] = _t.cpu()
    for _nm, _m in PRISM_MODULES + RESONA_LINEARS + RESONA_SECTIONED \
            + RESONA_MORM:
        for _attr in ("last_out", "last_mask", "last_aux", "last_cert"):
            if getattr(_m, _attr, None) is not None:
                setattr(_m, _attr, None)
_reclaim(); del _reclaim
gc.collect()

def cuda_census(tag: str = "", top: int = 15) -> float:
    """Name every live CUDA tensor group: aggregates bytes by (shape,
    dtype) over the GC heap (shared-storage views are counted once per
    wrapper, so the total is an UPPER bound — good enough to name the
    holder of any multi-GiB block). If a phase OOMs, call cuda_census()
    there and the culprit is NAMED, not guessed."""
    groups = {}
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj) and obj.is_cuda:
                key = (tuple(obj.shape), str(obj.dtype))
                groups[key] = groups.get(key, 0) \
                    + obj.numel() * obj.element_size()
        except Exception:
            pass
    total = float(sum(groups.values()))
    print(f"[census{(' ' + tag) if tag else ''}] {total / 2**30:.2f} GiB live "
          f"in {len(groups)} groups (upper bound; views double-counted)")
    for (_shape, _dt), _nb in sorted(groups.items(),
                                     key=lambda kv: -kv[1])[:top]:
        print(f"    {str(_shape):>30s}  {_dt:>15s}  {_nb / 2**30:8.3f} GiB")
    return total

_PARK_PAT = re.compile(r"(visual|vision|merger|multi_modal|mm_|mtp)", re.I)
def _park_inert(model) -> list:
    """Top-2-level subtrees the TEXT-ONLY distillation never executes.
    Parking moves storage only — weights are untouched, so state_dict(),
    the export, and the Theorem-7 anchors are all unaffected. Restored by
    the epilogue at the end of this cell."""
    parked = []
    for _name, _mod in list(model.named_children()):
        if _PARK_PAT.search(_name):
            parked.append((_name, _mod))
            continue
        for _c_name, _c_mod in list(_mod.named_children()):
            if _PARK_PAT.search(_c_name):
                parked.append((f"{_name}.{_c_name}", _c_mod))
    return parked
_PARKED = _park_inert(model)
del _park_inert
_before = torch.cuda.mem_get_info()[0] if torch.cuda.is_available() else 0
for _name, _mod in _PARKED:
    _mod.to("cpu")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
_after = torch.cuda.mem_get_info()[0] if torch.cuda.is_available() else 0
print(f"{__PRISM_VER__} parked {len(_PARKED)} text-inert subtree(s) "
      f"{[_n for _n, _ in _PARKED]} -> freed "
      f"{(_after - _before) / 2**30:.2f} GiB (restored after training)")
del _before, _after


# leak-canary baseline: the heap the training phase starts from
# (post-parking); the optimizer adds its ~1.2 GiB at the first step and
# stays — the [vram] canary tolerates that band.
_base_gib = (torch.cuda.memory_allocated() / 2**30
             if torch.cuda.is_available() else 0.0)

groups = [
    {"params": [p for _, m in PRISM_MODULES for p in (m.U, m.P)], "lr": LR_FIELDS},
    {"params": [p for _, m in PRISM_MODULES for p in (m.alpha, m.beta,
                                                      m.alpha_shared, m.beta_shared)],
     "lr": LR_FIELDS},
    {"params": [m.router.weight for _, m in PRISM_MODULES], "lr": LR_ROUTER},
    {"params": [m.gains for _, m in PRISM_MODULES], "lr": LR_ROUTER * 0.1},
    {"params": [m.log_tau for _, m in PRISM_MODULES], "lr": LR_TAU},
    # RESONA trainables: resonant-linear routers/gains, section q/z
    # routers/gains, MoRM write-policy heads, v-head gains
    {"params": [m.router.weight for _, m in RESONA_LINEARS], "lr": LR_ROUTER},
    {"params": [m.gains for _, m in RESONA_LINEARS], "lr": LR_ROUTER * 0.1},
    {"params": [getattr(m, f"{t}_router").weight
                for _, m in RESONA_SECTIONED for t in ("q", "z")
                if hasattr(m, f"{t}_B")], "lr": LR_ROUTER},
    {"params": [getattr(m, f"{t}_gains")
                for _, m in RESONA_SECTIONED for t in ("q", "z")
                if hasattr(m, f"{t}_B")], "lr": LR_ROUTER * 0.1},
    {"params": [m.v_gains for _, m in RESONA_SECTIONED], "lr": LR_ROUTER * 0.1},
    {"params": [p for _, m in RESONA_MORM
                for p in (([m.critic_phi.weight, m.critic_obs.weight]
                           + ([m.critic_phi.bias] if m.critic_phi.bias is not None else []))
                          if isinstance(m, MoRM2Policy)
                          else [m.router.weight])], "lr": LR_ROUTER},
]
groups += list(globals().get("V15_EXTRA_GROUPS", []))     # lens shadows, consequence head, attention shadows, predictor, own neurons, head
args = TrainingArguments(
    output_dir=os.path.join(DRIVE_DIR, "ckpt"),
    max_steps=STEPS,
    warmup_steps=WARMUP_STEPS,          # warmup_ratio is DEPRECATED -> warmup_steps
    lr_scheduler_type="cosine",
    bf16=(torch.cuda.is_available()),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=5, save_strategy="no", report_to=[],
    remove_unused_columns=False, optim="adamw_torch",
    weight_decay=0.0, max_grad_norm=1.0, seed=int(V15_STEP0),     # a resumed session draws the data in a new order
)
model.config.use_cache = False
# also kill KV-cache building on the TEXT config (VL wrappers hide the
# decoder's flag one level down — a silent per-step activation tax).
for _cfg in (model.config, getattr(model.config, "text_config", None)):
    if _cfg is not None:
        _cfg.use_cache = False

from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

# 8-bit PAGED optimizer: blockwise 8-bit states shrink the fp32 twin
# states (1.18 GiB -> ~0.3 GiB) and PAGED states spill to host RAM under
# allocator pressure instead of OOMing. Fallback chain (no bitsandbytes:
# plain AdamW, foreach OFF — no transient per-group tensor lists). The
# training signal, per-group LRs and the cosine schedule are UNCHANGED.
_OPT_GROUPS = [{"params": g["params"], "lr": g["lr"], **({"eps": g["eps"]} if "eps" in g else {})} for g in groups]
try:
    import bitsandbytes as _bnb
    _bnb_err = ""
except Exception as _e:
    _bnb, _bnb_err = None, str(_e)
if _bnb is not None and hasattr(_bnb.optim, "PagedAdamW8bit"):
    optimizer = _bnb.optim.PagedAdamW8bit(_OPT_GROUPS, weight_decay=0.0)
    _opt_name = "PagedAdamW8bit (bitsandbytes)"
elif _bnb is not None and hasattr(_bnb.optim, "AdamW8bit"):
    optimizer = _bnb.optim.AdamW8bit(_OPT_GROUPS, weight_decay=0.0)
    _opt_name = "AdamW8bit (bitsandbytes)"
else:
    optimizer = AdamW(_OPT_GROUPS, weight_decay=0.0, foreach=False)
    _opt_name = "AdamW (torch, foreach=False)" + \
        (f" [bnb unavailable: {_bnb_err}]" if _bnb is None else "")
print(f"[{__PRISM_VER__}] optimizer: {_opt_name}")
v16_wrap_optimizer(optimizer)           # the market reads its probes right before every optimizer step
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, v15_lr_lambda)     # warm-up, flat, cosine to a floor
if V15_STEP0 > 0:                       # a resumed session: the schedule continues where the cut one stopped
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for _ in range(int(V15_STEP0)):
            scheduler.step()

# ============================================================================
# the v5.x memory-governance machinery (65 GiB governor target,
# allocator guard cap, dry-run probe, 3-level self-heal ladder,
# census-as-retry loop) is DELETED — the RESONA-core telemetry fix removed
# the actual leak, and those knobs never touched it.
# ============================================================================

VER = __PRISM_VER__
LIGHT_CKPT_EVERY = 10**9      # off. An OOM leaves the parameters in memory intact (the handler drops the
#                              half-built gradients), and resume.pt covers a lost session; a file with
#                              the optimizer state of ~1B parameters would be ~12 GB per write.
# version-tagged: a state saved under a different memory plan must never
# resume into this one
LIGHT_CKPT = os.path.join(LOCAL_DIR, "v17_train_state.pt")     # in-run OOM recovery only: the fast local disk
# v8.0.4: this file exists ONLY for the in-run OOM recovery below. One left on Drive by an
# EARLIER run would be "recovered" as if it were this run's state (old weights + old step).
if os.path.exists(LIGHT_CKPT):
    os.replace(LIGHT_CKPT, LIGHT_CKPT + ".prev")
    print("[light-ckpt] checkpoint from an earlier run moved to .prev — this run starts clean")

def _save_light(step, mdl, opt, sched):
    tmp = LIGHT_CKPT + ".tmp"
    torch.save({"step": int(step),
                "trainables": {n: p.detach().cpu()
                               for n, p in mdl.named_parameters()
                               if p.requires_grad},
                "opt": opt.state_dict(), "sched": sched.state_dict()}, tmp)
    os.replace(tmp, LIGHT_CKPT)               # atomic — never a torn file
    print(f"[light-ckpt] step {int(step)} saved "
          f"({os.path.getsize(LIGHT_CKPT) / 2**20:.0f} MiB)")

def _load_light(mdl, opt, sched):
    ck = torch.load(LIGHT_CKPT, map_location="cpu", weights_only=False)
    with torch.no_grad():
        for n, p in mdl.named_parameters():
            if p.requires_grad and n in ck["trainables"]:
                p.copy_(ck["trainables"][n].to(p.device, p.dtype))
    opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"])
    return int(ck["step"])

class _LightCkptCallback(TrainerCallback):
    """L2 — resumable state every LIGHT_CKPT_EVERY global steps."""
    def __init__(self):
        self._last = -1
    def on_step_end(self, args, state, control, **kw):
        st = _abs_step(state)       # v9: ABSOLUTE step — a checkpoint written after a recovery must say so
        if st % LIGHT_CKPT_EVERY == 0 and st > self._last and st < STEPS:
            self._last = st
            try:
                _save_light(st, model, optimizer, scheduler)
            except Exception as e:            # NEVER kill training over a save
                print(f"[light-ckpt] save failed (non-fatal): {e}")

def _build_trainer(bs, accum):
    args.per_device_train_batch_size = bs
    args.gradient_accumulation_steps = accum
    _anneal = AnnealCallback()
    _trainer = DistillTrainer(model=model, args=args, train_dataset=train_ds,
                              data_collator=collate,
                              callbacks=[_anneal, MoRM2Callback(),
                                         VRAMPulse(_base_gib),
                                         _LightCkptCallback(),
                                         LossPartsCallback()] + list(globals().get("V15_CALLBACKS", [])),
                              optimizers=(optimizer, scheduler))
    _anneal.sched = scheduler               # absolute-step anneal (resume-safe)
    return _trainer

# NO allocator cap: expandable_segments is already active (the setup cell
# set PYTORCH_CUDA_ALLOC_CONF before torch ever touched the GPU) and the
# lean step footprint leaves >20 GiB of true headroom.

# ---------------- engage + VERIFY HF layer checkpointing ---------------------
# The flag alone is not trusted: it is engaged here AND counted; if it
# reaches no module, every RESONA module still self-checkpoints and the
# at-OOM census names the rest.
try:
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()   # frozen embeddings: keep the
        # checkpoint graph differentiable from token 0 (also lets layer-1
        # self-checkpointing engage immediately)
except Exception as _gc_err:
    print(f"[{VER}] WARNING: gradient_checkpointing_enable failed "
          f"({_gc_err}) — module self-checkpointing (L1) is the bound")
_n_gc = sum(1 for _m in model.modules()
            if getattr(_m, "_gradient_checkpointing_func", None) is not None)
print(f"[{VER}] HF gradient checkpointing engaged on {_n_gc} module(s)")
if _n_gc == 0:
    print(f"[{VER}] NOTE: HF checkpointing reached no module — every RESONA "
          "module still self-checkpoints (L1); the stock GDN/full-attn "
          "cores are the residual consumers named by the at-OOM census")

# TRUST EXECUTION, NOT FLAGS: a present HF checkpointing flag proves
# nothing about the call path (the wrap decision lives in the container's
# forward loop, which the VL/PRISM surgery does not have to execute the way
# the flag walk assumed). Every decoder layer forward is therefore wrapped
# UNCONDITIONALLY with torch.utils.checkpoint(use_reentrant=False) — the
# per-layer saved set is the [B, L, d] input BY CONSTRUCTION — and the wrap
# counts its own EXECUTIONS (_CKPT_EXEC, printed by the [vram] pulse; zero
# executions under grad would be a loud stop, never a silent bare run).
def _text_tower(m):
    """Locate the text decoder: the largest .layers ModuleList not living
    under a parked inert subtree (visual/vision/merger/mtp). VL wrapper:
    model.model.language_model; plain LM: model.model. Returns (path, mod)
    or (None, None)."""
    _best = (0, None, None)
    for _p, _mod in m.named_modules():
        _ls = getattr(_mod, "layers", None)
        if (isinstance(_ls, torch.nn.ModuleList) and len(_ls) > _best[0]
                and len(_ls) >= 8 and not _PARK_PAT.search(_p)):
            _best = (len(_ls), _p, _mod)
    return _best[1], _best[2]

_tt_path, _tt = _text_tower(model)
for _ly, _f in globals().get("_FORCED_CKPT", []):     # this cell run a second time: never wrap a wrapped layer
    _ly.forward = _f
_FORCED_CKPT = []          # (layer, original forward) — restored post-training
if _tt is not None:
    _tt_layers = list(_tt.layers)
    _tt_flagged = sum(
        1 for _ly in _tt_layers
        if getattr(_ly, "_gradient_checkpointing_func", None) is not None
        and getattr(_ly, "gradient_checkpointing", False))
    print(f"[{VER}] text-decoder checkpoint audit ({_tt_path}): "
          f"{_tt_flagged}/{len(_tt_layers)} layers carry the HF flag "
          "(informational only — the wrap below runs regardless)")
    from torch.utils.checkpoint import checkpoint as _ck_fn
    for _ly in _tt_layers:
        _orig_fwd = _ly.forward
        def _ckpt_fwd(*a, __ly=_ly, __f=_orig_fwd, **kw):
            if __ly.training and torch.is_grad_enabled():
                _CKPT_EXEC[0] += 1
                return _ck_fn(__f, *a, use_reentrant=False, **kw)
            _CKPT_EXEC[1] += 1
            return __f(*a, **kw)
        _ly.forward = _ckpt_fwd
        _FORCED_CKPT.append((_ly, _orig_fwd))
    print(f"[{VER}] #30 FORCE-wrapped {len(_FORCED_CKPT)} decoder layer "
          "forwards UNCONDITIONALLY (torch.utils.checkpoint, "
          "use_reentrant=False) — per-layer saved set is the [B, L, d] "
          "input BY CONSTRUCTION; values unchanged (the suite's "
          "self-checkpoint contract, RNG preserved); _CKPT_EXEC counts "
          "every execution — "
          "printed by the [vram] pulse every 25 steps")
else:
    print(f"[{VER}] #30 NOTE: no text decoder .layers found to wrap "
          "(unexpected topology — L1 module self-checkpointing stands)")

# -------- telemetry diet for the training phase ------------------------------
# SectionedProjector.last_out pins [T, q+k+v+z] tensors per GDN layer
# across the whole fwd+bwd and the trainer never reads it (compute_loss
# reads PRISM_MODULES[].last_aux only); restored after training.
for _, _m in RESONA_SECTIONED:
    _m.telemetry = False
print(f"[{VER}] #28 telemetry diet: last_out disabled on "
      f"{len(RESONA_SECTIONED)} SectionedProjector(s) for training "
      "(restored after)")

t0 = time.time()
free_cuda("before training")   # phase boundary
cuda_census(f"before training ({VER})")   # names every live CUDA block
if torch.cuda.is_available():
    print(f"[{VER} budget] baseline {_base_gib:.1f} GiB | free "
          f"{torch.cuda.mem_get_info()[0] / 2**30:.1f} GiB — expected "
          "steady-state peak ≈ baseline + ~3 GiB (optimizer 1.2 + one-step "
          "transients). No cap, no governor, no dry-run: step 1 is the "
          "measurement, the [vram] pulse verifies it, the OOM handler "
          "below survives the rest.")

# self-check: chunked-head path vs full-GEMM oracle (CPU mocks; the
# notebook refuses to train before it passes).
_dsc, _Vsc, _Ksc = 32, 97, 6
_hin = torch.randn(2, 17, _dsc, requires_grad=True)
_Wsc = torch.randn(_Vsc, _dsc)                      # the frozen head
_lab = torch.randint(0, _Vsc, (2, 16))
_tid = torch.randint(0, _Vsc, (2, 16, _Ksc))
_hv = _hin[:, :-1]
_nsc, _ssc = _ChunkedHeadLoss.apply(_hv, _Wsc, _lab, _tid, 7)
_lgsc = (_hv.reshape(-1, _dsc) @ _Wsc.t()).reshape(2, 16, _Vsc)
_lse = _lgsc.logsumexp(-1)
_nref = _lse - _lgsc.gather(-1, _lab[..., None]).squeeze(-1)
_sref = _lgsc.gather(-1, _tid) - _lse[..., None]
assert float((_nsc - _nref).detach().abs().max()) < 1e-5, "chunked-head nll drift"
assert float((_ssc - _sref).detach().abs().max()) < 1e-5, "chunked-head slog drift"
_gn = torch.randn_like(_nsc); _gs = torch.randn_like(_ssc) * 0.1
(_gn * _nsc + (_gs * _ssc).sum(-1)).sum().backward()
_gh = _hin.grad.clone(); _hin.grad = None
(_gn * _nref + (_gs * _sref).sum(-1)).sum().backward()
_gd = float((_gh - _hin.grad).norm()
            / (_hin.grad.norm() + 1e-12))
assert _gd < 1e-5, "chunked-head hidden-grad drift"
print(f"[{VER} self-check] chunked-head loss == full-GEMM oracle: "
      f"nll max|d| {float((_nsc - _nref).detach().abs().max()):.2e}, "
      f"slog max|d| {float((_ssc - _sref).detach().abs().max()):.2e}, "
      f"hidden-grad rel {_gd:.2e} (fp32 mocks, chunk=7)")
del _dsc, _Vsc, _Ksc, _hin, _Wsc, _lab, _tid, _hv, _nsc, _ssc, _lgsc, _lse
del _nref, _sref, _gn, _gs, _gh, _gd

import traceback

# ============================================================================
# ============================================================================
# Training loop — plain run + ONE clean OOM recovery.
#   * no dry-run probe (it doubled the OOM surface and leaked its own graph)
#   * the handler keeps STRINGS only (`traceback.format_exc()`), never the
#     exception object: cleaning up INSIDE the except block pins the dead
#     graph via the very frames being handled — and `raise ... from e`
#     chains the pinned graph into the final error. Both avoided here.
#   * after one recovery (heap scrub + light-ckpt resume) a second OOM
#     raises with the census naming every live block.
# ============================================================================
# ============================================================================
# ---- v9.1 TRAINING PATH ------------------------------------------------------------------------------
# 'soft_then_hard' (v8): differentiable all-block path first, hard serving path for the last part.
# 'hard': train the function that is SERVED from step 0 (KL to the teacher + LTD + EMRD).
# 'auto': measured, not assumed — the soft path weights blocks by softmax(router)*B*gain, which equals the
#   dense model only for an untrained router. With calibrated routers/gains it can start FAR from the
#   teacher (a loss above ln(vocab) ~ 12.4 is that symptom). If the soft student is not better than the
#   served student on held-out text, the soft phase is skipped.
_ps, _pt = PPL.get("student_soft@init"), PPL.get("student_topk@init")
_path = TRAIN_PATH
if _path == "auto":
    _path = "hard" if (_ps is None or _pt is None or not (_ps < _pt)) else "soft_then_hard"
print(f"[{VER}] training path: {_path} (TRAIN_PATH={TRAIN_PATH!r}; held-out PPL at init: "
      f"dense {PPL.get('teacher_dense', float('nan')):.2f} | served top-k {_pt if _pt is not None else float('nan'):.2f} | "
      f"soft {_ps if _ps is not None else float('nan'):.2f}) | LAMBDA_CE={LAMBDA_CE} LAMBDA_KL={LAMBDA_KL}")
if _path == "hard":
    ANNEAL_FRAC = 0.0                 # MoRM budget at its target from step 1; no soft phase
    set_world("serving")
    _PHASE["hard"] = True
else:
    set_world("soft_train")
    _PHASE["hard"] = False
_MAX_OOM_RECOVERIES = 3     # v8.0.3: was ONE; each recovery also halves LOSS_CHUNK
_done, _oom_retries = int(V15_STEP0), 0
_V16_MANUAL_STOP = False
while _done < STEPS:
    trainer = _build_trainer(BATCH_SIZE, GRAD_ACCUM)
    if _done > 0:
        args.max_steps = STEPS - _done        # continue the remaining steps
        print(f"[{VER} resume] {STEPS - _done} steps left")
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    try:
        trainer.train()
        _done = STEPS
    except KeyboardInterrupt:               # v16: the stop button ends the TRAINING, not the run
        _V16_MANUAL_STOP = True
    except RuntimeError as _e:
        if not is_oom(_e):
            raise
        _err = traceback.format_exc()   # STRING only — the exception dies here
        _oom_retries += 1
        if _oom_retries > _MAX_OOM_RECOVERIES:
            cuda_census("at-OOM (final)")
            raise RuntimeError(
                f"{VER}: training OOMed again after {_MAX_OOM_RECOVERIES} clean recoveries. "
                "The census above names every live block — paste it with "
                f"the last [vram] line.\n{_err}") from None
    else:
        break
    if globals().get("_V16_MANUAL_STOP"):   # outside the except block: the interrupted step's graph is released
        _done = STEPS
        V15["stopped"] = int(scheduler.last_epoch)
        model.zero_grad(set_to_none=True)
        optimizer.zero_grad(set_to_none=True)
        _mk_probe_reset()
        for _m in (m for _, m in PRISM_MODULES):
            _m.last_aux = None
            _m._store_aux = False
        _lmh_now = getattr(model, "lm_head", None)
        if _lmh_now is not _LMH:          # the stop may land inside a step, where the head is swapped out
            model.lm_head = _LMH
        print(f"\n[{VER}] training STOPPED BY HAND at update {V15['stopped']}. Nothing is lost: run the cells below — the verdict "
              "takes the best state so far (or the last one), then evaluation, save, push and probe run as usual.")
        break
    # ---- cleanup AFTER the except block: the OOM exception and its
    # graph-pinning traceback have been released by now ----
    del trainer
    optimizer.zero_grad(set_to_none=True)
    for _m in (m for _, m in PRISM_MODULES):
        _m.last_aux = None
    LOSS_CHUNK = max(64, LOSS_CHUNK // 2)   # smaller vocab-GEMM chunks (math-neutral)
    print(f"[{VER} OOM recovery] LOSS_CHUNK -> {LOSS_CHUNK}")
    free_cuda(f"OOM recovery {_oom_retries}/{_MAX_OOM_RECOVERIES}")
    _done = _load_light(model, optimizer, scheduler) \
        if os.path.exists(LIGHT_CKPT) else int(scheduler.last_epoch)
    print(f"[{VER} OOM recovery] heap back to "
          f"{torch.cuda.memory_allocated() / 2**30:.1f} GiB (baseline "
          f"{_base_gib:.1f}); resuming from step {_done}")

print(f"training wall time: {(time.time()-t0)/60:.1f} min")
free_cuda("after training")    # release optimizer/activation transients

# restore what the training phase borrowed: the forced decoder
# checkpoints and the SectionedProjector telemetry diet.
for _ly, _f in _FORCED_CKPT:            # identity math, restored
    _ly.forward = _f                     # so later phases see stock modules
_FORCED_CKPT.clear()
for _, _m in RESONA_SECTIONED:           # telemetry back on
    _m.telemetry = True

# bring the parked text-inert subtrees home: the export, the parity
# reload, and vLLM all expect the full VL model resident again (parking
# moved storage only — weight-neutral).
for _name, _mod in _PARKED:
    _mod.to(DEV)
_PARKED.clear()
free_cuda(f"{__PRISM_VER__} parked subtrees restored")
model.config.use_cache = True


## Verdict

In [ ]:
# v16 — VERDICT: does the trained model beat the restored model — both UNDER THE LAW — on held-out text AND reasoning?
model.eval()
set_world("serving")
V15_STEPS_DONE = int(scheduler.last_epoch)


def _v15_full():
    et = v15_eval(heldout, NS_VERDICT_BLOCKS)
    dt = dict(v15_eval.last)
    em = v15_eval(heldout_math, V15_MATH_HOLD) if heldout_math is not None else None
    _v15_full.last = (dt, dict(v15_eval.last) if em else {})
    return et, em, v15_score(et, em)


_last = _v15_full()
_cands = [("last step", None, None, _last)]
if V15["best_state"] is not None and V15["best_step"] != V15["curve"][-1][0]:
    _last_state, _last_morm = _v15_get(), _v15_morm_get()
    _v15_load(V15["best_state"])
    _v15_morm_set(V15["best_morm"])
    _cands.append((f"best checkpoint (step {V15['best_step']})", V15["best_state"], V15["best_morm"], _v15_full()))
    _v15_load(_last_state)
    _v15_morm_set(_last_morm)
    del _last_state
elif V15["best_state"] is None and V15_STEPS_DONE > V15_STEP0:
    # nothing in this session beat the state it started from (after a resume that is the best state of the earlier
    # sessions): that state is a candidate too
    _last_state, _last_morm = _v15_get(), _v15_morm_get()
    _v15_load(V15_START)
    _v15_morm_set(V15_START_MORM)
    _cands.append((f"start of this session (step {V15_STEP0})", V15_START, V15_START_MORM, _v15_full()))
    _v15_load(_last_state)
    _v15_morm_set(_last_morm)
    del _last_state
print("training curve (small eval set) — step: score [text KL / reasoning KL]")
print("    " + "   ".join(f"{c[0]}: {c[1]:.3f} [{c[2]:.3f}/{(c[3] if c[3] is not None else float('nan')):.3f}]" for c in V15["curve"]))
for _t, _, _, _e in _cands:
    print(f"    {_t:<28} score {_e[2]:.4f} | text KL {_e[0][0]:.4f}" + (f" | reasoning KL {_e[1][0]:.4f}" if _e[1] else ""))
_pick = min(_cands, key=lambda c: c[3][2] if math.isfinite(c[3][2]) else float("inf"))
V15_KEPT = bool(math.isfinite(_pick[3][2]) and _pick[3][2] < SCORE_REF)


def _v15_rollback():
    _v15_load(V15_SNAP)                 # every trained tensor as it was restored
    _v15_morm_set(V15_SNAP_MORM)        # ... the write-policy budgets / thresholds as they were
    _v15_head_off()                     # ... the v12 reading rule and the neuron counts of the restored state
    _ns_apply(V12_MASK)                 # ... and the lens exactly where the state had it


if V15_KEPT:
    if _pick[1] is not None:
        _v15_load(_pick[1])
        _v15_morm_set(_pick[2])
    _ns_apply(_has)
else:
    _v15_rollback()
_active = measured_active()
if V15_KEPT and RUN_MODE == "full" and TARGET_ACTIVE and _active > TARGET_ACTIVE:
    print(f"!! census {_active / 1e9:.3f}B is over the {TARGET_ACTIVE / 1e9:.1f}B mandate -> rolled back")
    _v15_rollback()
    V15_KEPT = False
    _active = measured_active()
E_V15_TEXT, E_V15_MATH, SCORE_V15 = _v15_full()
D_V15_TEXT, D_V15_MATH = _v15_full.last
KL_V15 = float(E_V15_TEXT[0])
PPL_V15 = ppl(heldout, "topk")
PPL["student_topk@v15"] = PPL_V15
_hs = bool(V16.get("head_strict"))
V16["head_strict"] = False                                    # the number the NEXT rebuild will read before its law cell runs
PPL_V16_DENSE_HEAD = ppl(heldout, "topk") if _hs else float(PPL_V15)
V16["head_strict"] = _hs
set_world("serving")
_m = lambda e: (f"KL {e[0]:.4f}, same first token {100 * e[1]:.1f}%" if e else "n/a")
if V15_KEPT:
    print(f"\nV16 KEPT ({_pick[0]}, schedule at step {V15_STEPS_DONE}, {V15['cuts']} governor cuts):")
    print(f"    text       {_m(E_REF_TEXT)}  ->  {_m(E_V15_TEXT)} | held-out PPL {PPL_REF:.2f} -> {PPL_V15:.2f} "
          f"(dense teacher {PPL.get('teacher_dense', float('nan')):.2f})")
    print(f"    reasoning  {_m(E_REF_MATH)}  ->  {_m(E_V15_MATH)}")
    if E_V15_MATH and D_V15_MATH.get("n_decisive"):
        print(f"    DECISIVE positions of the reasoning blocks (the teacher is >= {V16_DEC_P:g} sure): agreement "
              f"{100 * D_REF_MATH.get('decisive', 0.0):.1f}% -> {100 * D_V15_MATH.get('decisive', 0.0):.1f}% | runs of {V16_RUN} without a fork "
              f"{100 * D_REF_MATH.get('run', 0.0):.1f}% -> {100 * D_V15_MATH.get('run', 0.0):.1f}%   (gate 1 needs roughly 94% agreement)")
    print(f"    score {SCORE_REF:.4f} -> {SCORE_V15:.4f} | active {_active / 1e9:.3f}B (every token; the law)")
    if KL_V15 > KL_REF:
        print("    NOTE: the text KL alone got WORSE; the model was kept because the reasoning KL fell by more "
              f"(score = text + {V15_MATH_W:g} x reasoning). Read the two lines above separately.")
else:
    print(f"\nV16 ROLLED BACK: best score {_pick[3][2]:.4f} did not beat the restored model ({SCORE_REF:.4f}) — the model is the "
          f"restored one again | text KL {KL_V15:.4f} | held-out PPL {PPL_V15:.2f} | schedule at step {V15_STEPS_DONE}")
# ---- which part of the training mattered? ONE family at a time is put back to its restored values (the rest stays
#      trained) and the held-out numbers are read again. The families were trained together, so this over-states each
#      one's share — it is a ranking, not a decomposition.
V15_ABLATION = {}
if V15_KEPT and globals().get("V15_ABLATE", True):
    _nm = {id(p): n for n, p in model.named_parameters()}
    print("\nwhat mattered — one family back to its restored values, everything else as trained:")
    print(f"    {'(nothing reverted: the kept model)':<44} text KL {E_V15_TEXT[0]:.4f}"
          + (f" | reasoning KL {E_V15_MATH[0]:.4f}" if E_V15_MATH else ""))
    _k2 = [int(getattr(m, "ns_k2", 0) or 0) for _, m in PRISM_MODULES]
    if any(_k2):                                              # the ghosts silenced (their budget simply unused)
        for _, m in PRISM_MODULES:
            m.ns_k2 = 0
        _et, _em, _ = _v15_full()
        for (_, m), _v in zip(PRISM_MODULES, _k2):
            m.ns_k2 = _v
        V15_ABLATION["ghost neurons silenced"] = {"text_kl": float(_et[0]), "reasoning_kl": (float(_em[0]) if _em else None)}
        print(f"    {'ghost neurons silenced':<44} text KL {_et[0]:.4f} ({_et[0] - E_V15_TEXT[0]:+.4f})"
              + (f" | reasoning KL {_em[0]:.4f} ({_em[0] - E_V15_MATH[0]:+.4f})" if _em else ""))
    if _WAL.get("on") and _WAL.get("omega", 0) > 0:           # every token buys the quotas again (prices and weights as trained)
        _om = float(_WAL["omega"])
        _wal_open(0.0)
        _et, _em, _ = _v15_full()
        _wal_open(_om)
        V15_ABLATION["wallet (-> every token buys the quotas)"] = {"text_kl": float(_et[0]), "reasoning_kl": (float(_em[0]) if _em else None)}
        print(f"    {'wallet (-> every token buys the quotas)':<44} text KL {_et[0]:.4f} ({_et[0] - E_V15_TEXT[0]:+.4f})"
              + (f" | reasoning KL {_em[0]:.4f} ({_em[0] - E_V15_MATH[0]:+.4f})" if _em else ""))
    if _MK.get("trades"):                                     # the market's allocation back to the planned one (weights as trained)
        _bud = _v16_budget_get()
        _v16_budget_set(V15_SNAP_MORM.get("budget") if isinstance(V15_SNAP_MORM, dict) else None)
        _et, _em, _ = _v15_full()
        _v16_budget_set(_bud)
        V15_ABLATION["market (-> the planned budgets)"] = {"text_kl": float(_et[0]), "reasoning_kl": (float(_em[0]) if _em else None)}
        print(f"    {'market (-> the planned budgets)':<44} text KL {_et[0]:.4f} ({_et[0] - E_V15_TEXT[0]:+.4f})"
              + (f" | reasoning KL {_em[0]:.4f} ({_em[0] - E_V15_MATH[0]:+.4f})" if _em else ""))
    for _tag, _ps in (("own neurons (-> the teacher's copies)", V16_OWN_PARAMS), ("strict head (-> untrained proxy)", V16_HEAD_PARAMS),
                      ("consequence head (-> read by size only)", V15_HEAD_W + V15_HEAD_P), ("ghost gains (-> as measured)", V15_GHOST_PARAMS),
                      ("attention shadows", V15_ATTN_PARAMS), ("lens predictor", V15_PRED_PARAMS), ("FFN lens shadows", V15_LENS_PARAMS)):
        if not _ps:
            continue
        _cur = {_nm[id(p)]: p.detach().clone() for p in _ps}
        _v15_load({n: V15_SNAP[n] for n in _cur})
        _et, _em, _ = _v15_full()
        _v15_load(_cur)
        del _cur
        V15_ABLATION[_tag] = {"text_kl": float(_et[0]), "reasoning_kl": (float(_em[0]) if _em else None)}
        print(f"    {_tag:<44} text KL {_et[0]:.4f} ({_et[0] - E_V15_TEXT[0]:+.4f})"
              + (f" | reasoning KL {_em[0]:.4f} ({_em[0] - E_V15_MATH[0]:+.4f})" if _em else ""))
    set_world("serving")
# how far did the reading rule move? (the head starts at 0 = "read the largest")
_w = [m.ns_w.detach().float() for _, m in PRISM_MODULES if getattr(m, "ns_hd", False)]
V15_HEAD_STATS = None
if _w:
    _wa = torch.cat([t.abs().flatten().cpu() for t in _w])
    V15_HEAD_STATS = {"mean_abs_w": float(_wa.mean()), "max_abs_w": float(_wa.max()),
                      "share_moved_10pct": float((_wa > 0.0953).float().mean())}
    print(f"consequence head: mean |log importance| {V15_HEAD_STATS['mean_abs_w']:.4f}, max {V15_HEAD_STATS['max_abs_w']:.2f}, "
          f"{100 * V15_HEAD_STATS['share_moved_10pct']:.1f}% of the neurons re-weighted by more than 10%")
    del _wa
# where the budget went: what every kind of site holds now, against the plan it started from
V16_MARKET_REPORT = {"rounds": int(_MK.get("rounds", 0)), "trades": int(_MK.get("trades", 0)), "closed": _MK.get("closed"), "kinds": {}}
if MK_SITES:
    _kr = {}
    for s_ in MK_SITES:
        e = _kr.setdefault(s_["kind"], [0, 0, 0, 0, 0])
        n1 = int(s_["get"]())
        e[0] += s_["n0"] * s_["cost"] // s_["unit"]; e[1] += n1 * s_["cost"] // s_["unit"]
        e[2] += int(n1 > s_["n0"]); e[3] += int(n1 < s_["n0"]); e[4] += 1
    print(f"\nthe market — {V16_MARKET_REPORT['rounds']} rounds, {V16_MARKET_REPORT['trades']} units bought"
          + (f" (closed early: {_MK['closed']})" if _MK.get("closed") else "") + ("" if V15_KEPT else " — rolled back with the model") + ":")
    for kind, (b0, b1, up, dn, cnt_) in _kr.items():
        V16_MARKET_REPORT["kinds"][kind] = {"before": int(b0), "after": int(b1), "sites_up": int(up), "sites_down": int(dn), "sites": int(cnt_)}
        print(f"    {kind:<34} {b0 / 1e6:9.2f}M -> {b1 / 1e6:9.2f}M ({(b1 - b0) / 1e6:+.2f}M) | {up} sites grew, {dn} shrank, of {cnt_}")
# the wallet: what the tokens actually bought
V16_WALLET_STATE = _wal_state() if _WAL.get("order") else None
V16_WALLET_REPORT = {"on": bool(_WAL.get("on")), "omega": float(_WAL.get("omega", 0.0)), "closed": WAL_INFO.get("closed"),
                     "updates": len(WAL_INFO.get("log", [])), "last": (WAL_INFO["log"][-1] if WAL_INFO.get("log") else None), "receipt": None}
if _WAL.get("on"):
    set_world("serving")
    with torch.no_grad():
        model(input_ids=(heldout_math if heldout_math is not None else heldout)[0:1].to(DEV))
    _rc = _wal_receipt()
    _cq = float(measured_active())
    if _rc is not None:
        V16_WALLET_REPORT["receipt"] = {"quota_census": _cq, "min": _cq + _rc[0], "mean": _cq + _rc[1], "max": _cq + _rc[2], "ceiling": float(_MK["cap"])}
        print(f"\nthe wallet — window {100 * _WAL['omega']:.0f}%, {V16_WALLET_REPORT['updates']} price updates"
              + (f" (closed: {WAL_INFO['closed']})" if WAL_INFO.get("closed") else "") + f". On one held-out block a token touched between "
              f"{(_cq + _rc[0]) / 1e9:.4f}B and {(_cq + _rc[2]) / 1e9:.4f}B (mean {(_cq + _rc[1]) / 1e9:.4f}B); ceiling {_MK['cap'] / 1e9:.4f}B")
        _per = {}
        for st in _WAL["order"]:
            got = _WAL["store"].get((st["key"], _WAL["fid"]))
            if got is not None:
                e = _per.setdefault(st["kind"], [0.0, 0.0, 0.0, 0.0, 0])
                nf = got[0].float()
                e[0] += float(nf.mean()); e[1] += float(nf.min()); e[2] += float(nf.max()); e[3] += st["n0"](); e[4] += 1
        for kind, (mn, lo_, hi_, q, c_) in _per.items():
            print(f"    {kind:<34} quota {q / c_:7.1f} per token | bought: mean {mn / c_:7.1f}, the thriftiest token {lo_ / c_:7.1f}, "
                  f"the hungriest {hi_ / c_:7.1f}")
        V16_WALLET_REPORT["per_kind"] = {k: {"quota": v[3] / v[4], "mean": v[0] / v[4], "min": v[1] / v[4], "max": v[2] / v[4]} for k, v in _per.items()}
XRAY_AFTER = {}
try:
    print("\nwhere the model stands locally now (the same x-ray as before the training):")
    XRAY_AFTER = v15_xray(quiet=True)
    for _k, _v in XRAY_AFTER.items():
        _b = globals().get("XRAY_BEFORE", {}).get(_k)
        print(f"        {_k:<34} relative error " + (f"{_b:.3f} -> " if _b is not None else "") + f"{_v:.3f}")
except Exception as _e:
    print(f"    (x-ray skipped: {type(_e).__name__}: {str(_e)[:120]})")
# ---- v17: LOOP, the sources, and the long-context curve after the training
_lh = V17_LOOP.get("hist") or []
if _lh:
    print(f"\nLOOP — how often the student's OWN answers were right while it trained ({int(V17_WATCH_EVERY)} updates each):")
    print("    " + "   ".join(f"{st_}: {r_}/{m_}" for st_, r_, m_ in _lh if m_))
    if V17_LOOP.get("off"):
        print(f"    (LOOP was switched off: {V17_LOOP['off']})")
if globals().get("V17_LONG_ON", True):
    try:
        v17_long_curve("after training")
        _b, _a = V17_LONG["curves"].get("before training"), V17_LONG["curves"].get("after training")
        if _b and _a:
            print("    change by position: " + " | ".join(f"{x[0]}-{x[1]}: {y[2] - x[2]:+.3f}" for x, y in zip(_b["rows"], _a["rows"])))
    except Exception as _e:
        print(f"[long] after training: skipped ({type(_e).__name__}: {str(_e)[:120]})")
        set_world("serving")
V15["best_state"] = None
free_cuda("v17 verdict")


## Evaluation: PPL, census, forward timing

In [ ]:
# Evaluation — PPL table, parity re-check, routing health, census-based cost (v9)
PPL_AFTER = {}
PPL_AFTER["teacher_dense"]   = ppl(heldout, "uniform")
PPL_AFTER["student_topk"]    = ppl(heldout, "topk")
PPL_AFTER["student_soft"]    = ppl(heldout, "soft")
PPL_BEFORE = torch.load(os.path.join(CACHE_DIR, "ppl_before.pt"), weights_only=False)

print("\n=========================== PERPLEXITY ===========================")
print(f"{'model path':<26}{'before':>10}{'after':>10}")
print("(before = the v10 student as first rebuilt, read with the dense head; after = this run's model UNDER THE LAW)")
print(f"{'dense teacher (uniform)':<26}{PPL_BEFORE['teacher_dense']:>10.3f}"
      f"{PPL_AFTER['teacher_dense']:>10.3f}")
print(f"{'PRISM top-k (serving)':<26}{PPL_BEFORE['student_topk@init']:>10.3f}"
      f"{PPL_AFTER['student_topk']:>10.3f}")
gap0 = PPL_BEFORE["student_topk@init"] - PPL_BEFORE["teacher_dense"]
gap1 = PPL_AFTER["student_topk"] - PPL_AFTER["teacher_dense"]
recovered = 100 * (1 - gap1 / gap0) if gap0 > 0 else 100.0
if gap0 <= 0:
    _rec_txt = "(the sparse student was not worse than the dense teacher at init — nothing to recover)"
elif recovered >= 0:
    _rec_txt = f"({recovered:.1f}% of the sparsity damage recovered)"
else:
    _rec_txt = (f"(WORSE than at init by {-recovered:.1f}% — training did not close the gap; "
                "see the [loss parts] lines above)")
print(f"\ninit routing gap {gap0:.3f} -> {gap1:.3f} PPL points " + _rec_txt)

# ------------------- Theorem 3 re-check on trained fields -------------------
worst = 0.0
x = heldout[0, :64].to(next(model.parameters()).device)
with torch.no_grad():
    set_mode("topk")
    y_mod = model(input_ids=x[None]).logits
    set_mode("soft")
h0 = x
for li, (name, m) in enumerate(PRISM_MODULES[:3] if RUN_MODE == "full" else PRISM_MODULES):
    with torch.no_grad():
        for e in (0, m.E // 2):
            W = m.materialize_expert(e)
            hs = torch.randn(8, m.d_model, device=x.device, dtype=m.W_gate.dtype)
            y_ref = m.expert_outputs(hs, True)[:, e, :] * m.gains[e]
            y_mat = (F.silu(hs @ W["gate_proj"].t()) *
                     (hs @ W["up_proj"].t())) @ W["down_proj"].t()
            worst = max(worst, float((y_mat - y_ref).norm() / y_ref.norm()))
# Tolerance = 2 ULPs of the STORAGE dtype: runtime fuses the fp32 correction
# AFTER the gate GEMM, materialization rounds it INTO the bf16 weight — both
# are valid roundings of the same math and cannot be bitwise-equal in bf16
# (in fp32 they agree to ~1e-7, matching the unit suite).  A drift beyond
# 2 ULPs would mean a REAL fusion bug (wrong slicing/gain), not rounding.
_T3_SUB = PRISM_MODULES[0][1].W_gate.dtype if PRISM_MODULES else torch.float32
_T3_TOL = max(2.0 * torch.finfo(_T3_SUB).eps, 1e-7)
print(f"\nTheorem 3 re-check after training: worst materialization rel-err {worst:.2e}"
      f"  ({'PASS' if worst < _T3_TOL else 'FAIL'} — tol {_T3_TOL:.1e} = 2 ULP of {_T3_SUB})")

# ------------------------------ routing health ------------------------------
loads, entropies = [], []
with torch.no_grad():
    set_mode("topk")
    for i in range(min(4, heldout.shape[0])):
        b = heldout[i:i+1].to(next(model.parameters()).device)
        _ = model(input_ids=b)
        for name, m in PRISM_MODULES:
            if hasattr(m, "last_aux") and "load" in m.last_aux:
                loads.append(m.last_aux["load"].float().cpu())
                p_ = m.last_aux["p"]
                entropies.append(float(-(p_ * p_.clamp_min(1e-9).log()).sum(-1).mean()))
    set_mode("soft")
if loads:
    L = torch.stack(loads).mean(0)
    cv = float(L.std() / L.mean())
    dead = int((L == 0).sum())
    maxent = math.log(E_CHOSEN)
    print(f"\nrouting health: load CV={cv:.3f} (0=perfect balance) | dead experts: {dead}"
          f" | mean gate entropy {sum(entropies)/len(entropies):.2f} / {maxent:.2f}")
    top = L.topk(min(8, len(L))).values
    bar = lambda v: "#" * max(1, int(40 * v / L.max()))
    print("busiest experts:", "  ".join(f"{float(v):.0f}" for v in top))
else:
    print("\nrouting health: n/a — every FFN is served by the neuron lens (no expert groups in use)")

# --------------------------- MoRM 2.0 telemetry (v8) ------------------------
if RESONA_MORM and isinstance(RESONA_MORM[0][1], MoRM2Policy):
    print("\n=================== MoRM 2.0 (oracle-scheduled writes) ============")
    for name, m in RESONA_MORM[:6]:
        fit = getattr(m, "_crit_fit", None)
        r2 = cap = orc = None
        if fit is not None:
            with torch.no_grad():
                # v9: the critic predicts LOG influence -> judge it there (pooled over heads, so each head's
                # level counts), and by what the policy is for: the share of the influence mass that its
                # elastic write set captures (oracle = the best top-m with the same mean m).
                lh = m.scores(fit[0]).clamp_min(1e-12).log()
                ly = fit[1].clamp_min(1e-12).log()
                r2 = float(1 - (lh - ly).pow(2).sum() / (ly - ly.mean()).pow(2).sum().clamp_min(1e-9))
                _thr = float(m.rate_threshold)
                if _thr > 0:
                    msk = (lh.exp() > _thr).to(fit[1].dtype)
                    tot_ = fit[1].sum().clamp_min(1e-12)
                    cap = float((msk * fit[1]).sum() / tot_)
                    _mb = max(1, min(int(round(float(msk.sum(-1).mean()))), fit[1].shape[-1]))
                    orc = float(fit[1].gather(-1, fit[1].topk(_mb, dim=-1).indices).sum() / tot_)
        mm = (float(m.last_m.float().mean())
              if m.last_m is not None else float("nan"))
        print(f"  {name.split('.')[-2] if '.' in name else name:<28} "
              + (f"exact writes per token: {int(m.strict_m)} (strict) | " if getattr(m, 'strict_m', None) is not None
                 else f"m_bar={m.m_bar:>3} | train mean m={mm:.2f} | ") +
              f"thr={float(m.rate_threshold):.3e} | "
              f"critic log-R^2={'n/a' if r2 is None else f'{r2:.3f}'}"
              + ("" if cap is None else f" | captured influence {cap:.3f} (oracle top-m {orc:.3f})"))
    print("  (critic log-R^2 / captured influence: measured on the calibration label stash, i.e. in-sample)")
    # scheduler simulation on the calibration labels: deferral + staleness
    _any = next(iter(MORM2_STASH.values()), None) if MORM2_STASH else None
    if _any is not None:
        lab = _any["labels"].to(DEV)
        T_, H_ = lab.shape
        sched_sim = schedule_writes(
            lab, torch.randn(T_, H_, 8, device=DEV),
            torch.randn(T_, H_, 8, device=DEV),
            torch.ones(T_, H_, device=DEV) * 0.7,
            torch.full((T_, H_), 0.97, device=DEV),
            m_bar=max(1, min(M_WRITE, H_)), staleness_k=MORM2_STALENESS_K,
            m_max=H_)
        st = sched_sim["stats"]
        print(f"  scheduler sim (label-driven): {st.summary()}")
        print(f"  => elastic writes hit the same mean budget as v7's fixed "
              f"top-m while deferral recovers late-critical writes "
              f"(contracts M6/M7/M8 in the MoRM 2.0 verification cell)")

# ------------------------------ cost accounting -----------------------------
# v9: ONE census of the live module graph (see census_active): embedding lookup, norms, head funnel,
# sketch applications, router reads and every dense site the surgery left behind are all inside
# TOTAL_ACTIVE (v8 added funnel / embedding / norms to a hand-picked registry sum).
set_world("serving")
CEN = measured_active(detail=True)
act_native = CEN["TOTAL_ACTIVE"]
set_world("compat_parity")
# COMPAT, same text-token convention as the census: one embedding ROW (not the table), the dense head
# (tied checkpoints: the embedding table IS the head), dense attention/DeltaNet, sparse FFN fields.
# The vision tower and the MTP draft head are idle for a plain text token (v8 charged them in full).
_head_dense = wf.get("lm_head", 0) or wf.get("embeddings", 0)
act_compat = (HID + _head_dense + wf.get("attention", 0) + wf.get("norms", 0)
              + N_LAY * 3 * HID * SHARED_SIZE + N_LAY * K_CHOSEN * 3 * HID * M_E
              + N_LAY * E_CHOSEN * HID)
set_world("soft_train")
print("\n==================== ACTIVE-PARAMETER WATERFALL ====================")
print(f"{'family':<26}{'plan (waterfall)':>20}{'census (built)':>20}")
for k, v in WF.items():
    if k != "TOTAL_ACTIVE":
        print(f"{k:<26}{v/1e9:>19.3f}B{CEN.get(k, 0)/1e9:>19.3f}B")
print(f"{'TOTAL (' + MAST_ACCOUNTING + ' sketch)':<26}{WF['TOTAL_ACTIVE']/1e9:>19.3f}B{act_native/1e9:>19.3f}B")
if census_active.last["unclassified"]:
    print("!! unclassified 2-D+ parameters were charged in full:", census_active.last["unclassified"][:6])
print(f"\nCOMPAT artifact (stock vLLM): ~{act_compat/1e9:.2f}B active "
      f"(FFN-sparse; attention/head dense by stock-arch necessity)")
print(f"NATIVE artifact (RESONA execution): ~{act_native/1e9:.3f}B active "
      f"| dense touched: {total_dense/1e9:.2f}B "
      f"| compression {total_dense/max(act_native,1):.1f}x")
if RUN_MODE == "full" and TARGET_ACTIVE:
    if act_native > TARGET_ACTIVE:
        _msg = (f"3B mandate violated at eval: census {act_native/1e9:.3f}B > {TARGET_ACTIVE/1e9:.2f}B "
                f"(families above)")
        if ENFORCE_BUDGET:
            raise AssertionError(_msg)
        print("!! " + _msg)
    else:
        print(f"===> 3B MANDATE (census, after training): {act_native/1e9:.3f}B <= {TARGET_ACTIVE/1e9:.2f}B <===")
print(f"\nhead: rank-{int(_LMH.ns_fU.shape[1])} proxy + bias, {int(funnel.shortlist)} exact rows per token "
      f"(top-64 recall when v10 built the funnel: {FUNNEL_RECALL:.4f}; the head as it is now is measured below)")

# head funnel on the TRAINED student: how much of the exact next-token distribution survives the shortlist
try:
    _fq_mass, _fq_top1, _fq_n = 0.0, 0.0, 0
    _lw = model.get_parameter("lm_head.weight")
    with torch.no_grad():
        set_mode("topk")
        for _i in range(min(4, heldout.shape[0])):
            _o = model(input_ids=heldout[_i:_i+1, :256].to(DEV), output_hidden_states=True)
            _h = _o.hidden_states[-1][0]
            _p = (_h.to(_lw.dtype) @ _lw.to(_h.device).t()).float().softmax(-1)               # [L, V] exact (the dense head on the student's state)
            _sl = _fh_proxy(_h.to(_lw.dtype), *_fh_factors(), _LMH.ns_fb).topk(min(int(funnel.shortlist), int(funnel.V)), dim=-1).indices
            _fq_mass += float(_p.gather(-1, _sl).sum())
            _fq_top1 += float((_sl == _p.argmax(-1, keepdim=True)).any(-1).float().sum())
            _fq_n += int(_h.shape[0])
            del _o, _h, _p, _sl
        set_mode("soft")
    FUNNEL_QUALITY = {"mass_kept": _fq_mass / max(_fq_n, 1), "top1_kept": _fq_top1 / max(_fq_n, 1)}
    print(f"head funnel on the trained student (held-out): next-token probability mass inside the shortlist "
          f"{FUNNEL_QUALITY['mass_kept']:.5f} | argmax inside {FUNNEL_QUALITY['top1_kept']:.5f}")
except Exception as _e:
    set_mode("soft")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"(funnel quality probe skipped: {type(_e).__name__}: {str(_e)[:120]})")

if torch.cuda.is_available():
    import time as _t
    xb = heldout[0, :256].to(DEV)[None]
    for mode in ("uniform", "topk"):
        set_mode(mode)
        with torch.no_grad():
            _ = model(input_ids=xb); torch.cuda.synchronize()
            t0 = _t.time()
            for _ in range(3): _ = model(input_ids=xb)
            torch.cuda.synchronize()
        dt = (_t.time() - t0) / 3
        print(f"training-path forward ({mode:>7}): {dt*1000:.0f} ms "
              f"({256/dt:,.0f} tok/s)")
    set_mode("soft")
    print("(serving-path speed is measured by vLLM below — its grouped MoE "
          "kernels are the real deployment target)")
free_cuda("evaluation complete")   # phase boundary before the export


## Save the v17 state (and push it to the Hub)

In [ ]:
# v16 — save the state (same file format as v10..v15: a later notebook rebuilds the model from it)
import shutil
_t0 = time.time()
_BIG = 20_000_000
_ATT_SH = ("sh_U", "sh_M", "q_sh_U", "q_sh_M", "v_sh_U", "v_sh_M", "z_sh_U", "z_sh_M")
_LENS_BUF = ("ns_B", "ns_Ag", "ns_Au", "ns_w", "ns_Pc", "ns_Qc", "ns_gc", "ns_og", "ns_ou", "ns_od")
_HEAD_T = ("ns_fP", "ns_fU", "ns_fb")                                   # the strict head: saved once, under extra["head"]


def _cpu(t):
    return t.detach().to("cpu").clone()


def _attrs(m, names):
    out = {}
    for a in names:
        try:
            v = getattr(m, a, None)
            if isinstance(v, (bool, int, float, str)) or v is None:
                out[a] = v
            elif torch.is_tensor(v) and v.numel() == 1:
                out[a] = float(v)
        except Exception:
            pass
    return out


state = {"trainable": {}, "buffers": {}, "skipped_big": [], "errors": [], "meta": {}, "extra": {}}
for _n, _p in model.named_parameters():
    if not (_p.requires_grad or _n in STATE.get("trainable", {})):      # v10's frozen FFN fields are kept too
        continue
    _path, _, _attr = _n.rpartition(".")
    if _attr in _HEAD_T:
        continue
    if _attr in _ATT_SH:
        # attention shadows trained in fp32 are SERVED in the substrate dtype: that rounded value is the model, and it
        # goes where every earlier state keeps these tensors (the buffers), so the rebuild cell needs no change
        state["buffers"][_n] = _cpu(_p.detach().to(model.get_submodule(_path).weight.dtype))
    elif _attr in _LENS_BUF:
        state["buffers"][_n] = _cpu(_p)                                 # the lens cell reads these from either place
    else:
        state["trainable"][_n] = _cpu(_p)
for _n, _b in model.named_buffers():
    if _b is None:
        continue
    if _n.rpartition(".")[2] in _HEAD_T:
        continue
    if _b.numel() > _BIG:
        state["skipped_big"].append((_n, tuple(_b.shape)))
        continue
    state["buffers"][_n] = _cpu(_b)
_meta = state["meta"]
_meta["classes"] = {n: type(m).__name__ for n, m in model.named_modules() if type(m).__module__ == "__main__"}
_meta["prism"] = {n: _attrs(m, ("E", "m_e", "k_topk", "rank", "d_model", "default_mode", "use_shadow",
                                       "ns_on", "ns_k", "ns_r", "ns_R", "ns_active", "ns_hd", "ns_r1", "ns_ns", "ns_k2", "ns_own"))
                  for n, m in PRISM_MODULES}
_meta["linears"] = {n: _attrs(m, ("topk", "B", "r", "out_f", "in_f", "router_st")) for n, m in RESONA_LINEARS}
_meta["sectioned"] = {n: _attrs(m, ("topk", "v_head_rows", "n_v_heads", "q_rows", "k_rows", "v_rows", "z_rows",
                                    "q_B", "q_r", "z_B", "z_r", "router_st")) for n, m in RESONA_SECTIONED}
_meta["morm"] = {}
for n, m in RESONA_MORM:
    d_ = _attrs(m, ("H", "m_bar", "m_max", "ste_tau", "use_obs", "rate_threshold", "strict_m"))
    d_["controller"] = _attrs(m.controller, ("m_bar", "m_max", "threshold", "lr"))
    _meta["morm"][n] = d_
_meta["gates"] = {n: _attrs(g, ("H", "b_only", "b_first", "logit_floor", "shadow_writes", "enabled"))
                  for n, g in RESONA_MORM_GATES}
with torch.no_grad():                                       # the funnel object carries the trained proxy too (substrate dtype)
    funnel.U_r.copy_(_LMH.ns_fU.detach().to(funnel.U_r.dtype))
    funnel.P_r.copy_(_LMH.ns_fP.detach().to(funnel.P_r.dtype))
state["extra"]["head"] = {"P": _cpu(_LMH.ns_fP).float(), "U": _cpu(_LMH.ns_fU).float(), "b": _cpu(_LMH.ns_fb).float(),
                          "strict": bool(V16.get("head_strict")), "shortlist": int(funnel.shortlist)}
state["extra"]["funnel"] = {"U_r": _cpu(funnel.U_r), "P_r": _cpu(funnel.P_r),
                            "rho": (_cpu(funnel.rho) if getattr(funnel, "rho", None) is not None else None),
                            "kappa": float(funnel.kappa), "shortlist": int(funnel.shortlist)}
state["extra"]["mast"] = {"R": _cpu(MAST.R), "S": int(MAST.S), "seed": int(getattr(MAST, "seed", 0)),
                          "d_model": int(MAST.d_model)}
_cfg = dict(STATE.get("config", {}))                    # start from what the earlier versions recorded
for _k in ("__PRISM_VER__", "MODEL_ID", "RUN_MODE", "PLAN", "WF", "E_CHOSEN", "K_CHOSEN", "M_E", "SHARED_SIZE",
           "COMPLETION_RANK", "MAST_S", "LIN_BLOCKS", "LIN_TOPK", "CARRIER_FRAC", "SHADOW_RANK_LIN",
           "SHADOW_RANK_FFN", "SHADOW_V", "M_WRITE", "FUNNEL_RANK", "FUNNEL_SHORTLIST", "FUNNEL_KAPPA",
           "FUNNEL_RECALL", "TARGET_ACTIVE", "MAST_ACCOUNTING", "SEQ_LEN", "NS_SPLIT", "NS_CONFIG", "NS_KEPT",
           "PPL", "V15_KEPT", "V15_STEPS_DONE", "LAMBDA_ANCHOR", "LR_LENS", "LR_HEAD", "LR_ATTN_REL", "LR_PRED_REL",
           "V15_HEAD_RANK", "V15_ST_TEMP", "V15_W_CLAMP", "V15_MATH_INFO", "V15_TRAINED", "V15_HEAD_STATS", "V15_MATH_W", "SSIGHT", "V15_PF",
           "V16", "V16_OWN_INFO", "V16_TRACE_INFO", "V16_MARKET_REPORT", "V16_WALLET_STATE", "V16_WALLET_REPORT",
           "WAL_START", "WAL_OPEN", "WAL_SPAN_FFN", "WAL_MORE_BLOCKS", "WAL_MORE_HEADS", "WAL_RHO", "WAL_BETA", "WAL_LR", "WAL_GLR", "LAMBDA_FORK", "V16_FORK_ALPHA", "LAMBDA_PX", "LR_OWN_REL",
           "LR_FH_REL", "LR_FH_BIAS", "V16_OWN_M", "V16_DEC_P", "V16_RUN", "MK_EVERY", "MK_RHO", "MK_Z", "MK_ROUND_FRAC", "MK_DRIFT"):
    if _k in globals():
        try:
            _cfg[_k] = json.loads(json.dumps(globals()[_k], default=str))
        except Exception as _e:
            _cfg[_k] = f"<unserializable: {type(_e).__name__}>"
state["config"] = _cfg
state["train"] = dict(STATE.get("train", {}))
state["lineage"] = list(STATE.get("lineage", [])) + [{"from": STATE_PATH, "step": "v16 MOLT: the law (strict head + strict writes), "
                                                       "the wallet, own neurons, fork loss, reasoning traces",
                                                       "kept": bool(V15_KEPT), "law": {"head": bool(V16.get("head_strict")),
                                                                                       "writes": bool(V16.get("write_strict"))}}]
_need = sum(int(t.numel()) * t.element_size() for d in (state["trainable"], state["buffers"]) for t in d.values()) * 1.1
SAVE_DIR = V15_DIR
try:
    if shutil.disk_usage(SAVE_DIR).free < _need:
        SAVE_DIR = os.path.join(LOCAL_DIR, "v17_state")
        os.makedirs(SAVE_DIR, exist_ok=True)
        print(f"[v16 save] !! not enough room in {V15_DIR} for {_need / 2**30:.1f} GiB -> saving to {SAVE_DIR}, which does NOT "
              "survive the session: the copy on the Hub (below) is the one that counts")
except Exception:
    pass
_path = os.path.join(SAVE_DIR, "v17_state.pt")
torch.save(state, _path + ".tmp")
os.replace(_path + ".tmp", _path)
_ck = torch.load(_path, map_location="cpu", weights_only=False)
assert len(_ck["trainable"]) == len(state["trainable"]) and len(_ck["buffers"]) == len(state["buffers"])
del _ck
_f2 = lambda e: (None if e is None else [float(e[0]), float(e[1])])
_res = {"teacher_dense": float(PPL.get("teacher_dense", float("nan"))),
        "student_topk": float(PPL_V15), "v10_restored": float(PPL.get("student_topk@init", float("nan"))),
        "dense_head_check": float(globals().get("PPL_V16_DENSE_HEAD", float("nan"))),      # what the next rebuild reads BEFORE its law cell
        "law": {"head_strict": bool(V16.get("head_strict")), "write_strict": bool(V16.get("write_strict"))},
        "audit": V16.get("audit"), "decisive_at_restore": globals().get("D_REF_MATH"), "decisive": globals().get("D_V15_MATH"),
        "at_restore": float(PPL_REF), "as_restored_before_second_sight": float(globals().get("PPL_RESTORED", PPL_REF)),
        "kl_as_restored_before_second_sight": float(globals().get("KL_RESTORED", KL_REF)),
        "second_sight": globals().get("SSIGHT", {}).get("choice"), "kept": bool(V15_KEPT), "steps": int(V15_STEPS_DONE),
        "kl_text_at_restore": float(KL_REF), "kl_text": float(KL_V15),
        "text_at_restore": _f2(E_REF_TEXT), "text": _f2(E_V15_TEXT),
        "reasoning_at_restore": _f2(E_REF_MATH), "reasoning": _f2(E_V15_MATH),
        "score_at_restore": float(SCORE_REF), "score": float(SCORE_V15), "active": int(measured_active()),
        "lens_layers": [bool(getattr(m, "ns_on", False)) for _, m in PRISM_MODULES]}
with open(os.path.join(SAVE_DIR, "ppl_after.json"), "w") as _f:
    json.dump(_res, _f, indent=1)
with open(os.path.join(SAVE_DIR, "v17_report.json"), "w") as _f:
    json.dump({"result": _res, "curve": V15["curve"], "stopped_at": V15["stopped"], "governor_cuts": V15["cuts"],
               "trained": V15_TRAINED, "curriculum": V15_MATH_INFO, "head": V15_HEAD_STATS, "ablation": V15_ABLATION,
               "second_sight": globals().get("SSIGHT"), "preflight": globals().get("V15_PF"),
               "law_audit": V16.get("audit"), "dense_swap": V16.get("swap"), "head_quality": V16.get("head_quality"),
               "own_neurons": globals().get("V16_OWN_INFO"), "traces": globals().get("V16_TRACE_INFO"),
               "market": globals().get("V16_MARKET_REPORT"), "market_log": list(_MK.get("log", [])),
               "wallet": globals().get("V16_WALLET_REPORT"), "wallet_log": list(WAL_INFO.get("log", [])),
               "wallet_state": globals().get("V16_WALLET_STATE"),
               "v17": {"loop_hist": V17_LOOP.get("hist"), "loop_off": V17_LOOP.get("off"), "loop_rounds": V17_LOOP.get("rounds"),
                       "long": V17_LONG.get("curves"), "fresh": globals().get("V17_FRESH_INFO"), "instep": bool(V17_INSTEP)},
               "budgets": {s_["key"]: [int(s_["n0"]), int(s_["get"]())] for s_ in MK_SITES},
               "xray_before": globals().get("XRAY_BEFORE"), "xray_after": globals().get("XRAY_AFTER"),
               "ffn_ghosts": [int(getattr(m, "ns_k2", 0) or 0) for _, m in PRISM_MODULES],
               "ffn_look": [list(_ns_prog(m)) if hasattr(m, "ns_B") else None for _, m in PRISM_MODULES],
               "ffn_k": [int(getattr(m, "ns_k", 0)) for _, m in PRISM_MODULES]}, _f, indent=1, default=str)
try:
    os.sync()
except Exception:
    pass
print(f"[v16 save] {_path} ({os.path.getsize(_path) / 2**30:.2f} GiB) | {len(state['trainable'])} trained tensors, "
      f"{len(state['buffers'])} buffers | held-out PPL {_res['student_topk']:.2f} "
      f"(at restore {_res['at_restore']:.2f}, dense {_res['teacher_dense']:.2f}) | {time.time() - _t0:.0f}s")
V15_SAVED = True
_repo = os.environ.get("PRISM_PUSH_REPO")
V15_PUSHED = False
if _repo:                                   # the Hub copy is what the next notebook downloads
    try:
        from huggingface_hub import HfApi
        HfApi().upload_folder(folder_path=SAVE_DIR, repo_id=_repo, path_in_repo="v17_state",
                              allow_patterns=["v17_state.pt", "ppl_after.json", "v17_report.json"])
        V15_PUSHED = True
        print(f"[v16 save] pushed to the Hub: {_repo}/v17_state")
        try:
            if "v17_state/resume.pt" in HfApi().list_repo_files(_repo):
                HfApi().delete_file("v17_state/resume.pt", _repo)
        except Exception:
            pass
    except Exception as _e:
        print(f"[v16 save] !! push to the Hub FAILED ({type(_e).__name__}: {str(_e)[:160]}) — copy {SAVE_DIR} by hand "
              f"before the machine is stopped")
if SAVE_DIR == V15_DIR or V15_PUSHED:       # the finished state is safe: the resume file of this run has done its job
    for _p in [V15_RESUME, os.path.join(LOCAL_DIR, "resume.pt")] + (
            [os.path.join(os.environ["PRISM_STATE_ALT"], "v17_state", "resume.pt")] if os.environ.get("PRISM_STATE_ALT") else []):
        try:
            if os.path.exists(_p):
                os.remove(_p)
        except Exception:
            pass
print("[v16 save] DONE")


## Math probe — gates 1 and 2

In [ ]:
if globals().get("RUN_MATH_PROBE", True):
    # PRISM math probe — how often is the sparse student RIGHT on math, next to the dense teacher?
    # Run AFTER the rebuild cell of the v11 notebook (model restored from v10_state.pt).
    # Do NOT run the realignment cell before/after this — it is not needed.
    #   tier A: 20 one-step arithmetic problems (answer in a few tokens)
    #   tier B: GSM8K word problems (short reasoning + "Answer: <number>")
    # Greedy decoding, one problem at a time, teacher and student on the SAME problems.
    # A wall-clock budget stops the probe early and reports on what was finished.
    import re, json, time, random, os
    import torch

    MATH_BUDGET_MIN = globals().get("MATH_BUDGET_MIN", 15)
    MATH_B_MIN_A = globals().get("MATH_B_MIN_A", 0.3)       # tier B (long, ~1 min per problem) only runs when the student gets this share of tier A     # hard stop for the whole probe (minutes)
    MATH_N_A = globals().get("MATH_N_A", 20)
    MATH_N_B = globals().get("MATH_N_B", 20)
    MATH_MAX_NEW_A = globals().get("MATH_MAX_NEW_A", 16)
    MATH_MAX_NEW_B = globals().get("MATH_MAX_NEW_B", 200)

    _DEV = next(model.parameters()).device
    model.eval()
    _T0 = time.time()

    _EOS = {getattr(tok, "eos_token_id", None)}
    for _t in ("<|im_end|>", "<|endoftext|>"):
        try:
            _i = tok.convert_tokens_to_ids(_t)
            if isinstance(_i, int) and _i >= 0 and _i != getattr(tok, "unk_token_id", None):
                _EOS.add(_i)
        except Exception:
            pass
    _EOS.discard(None)

    _DONE = re.compile(r"Answer:\s*\$?\s*-?[\d,]*\.?\d+[^\d,\.]")
    _ANS = re.compile(r"Answer:\s*\$?\s*(-?[\d,]*\.?\d+)")
    _NUM = re.compile(r"(-?\d[\d,]*\.?\d*)")


    def _to_num(s):
        try:
            return float(s.replace(",", "").rstrip("."))
        except Exception:
            return None


    def _parse(text):
        """(strict, lenient): strict = the number after the last 'Answer:', lenient = last number anywhere."""
        a = _ANS.findall(text)
        n = _NUM.findall(text)
        return (_to_num(a[-1]) if a else None), (_to_num(n[-1]) if n else None)


    def _same(x, y):
        return x is not None and y is not None and abs(x - y) <= 1e-6 * max(1.0, abs(y))


    def _prompt_ids(content):
        msg = [{"role": "user", "content": content}]
        try:
            s = tok.apply_chat_template(msg, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except Exception:
            s = content + "\n"
        return torch.tensor(tok(s, add_special_tokens=False)["input_ids"], dtype=torch.long)


    @torch.no_grad()
    def _greedy(pids, max_new, cached):
        ids = pids[None].to(_DEV)
        cur, past, out = ids, None, []
        for _ in range(max_new):
            if cached:
                o = model(input_ids=cur, past_key_values=past, use_cache=True)
                past = o.past_key_values
            else:
                o = model(input_ids=ids, use_cache=False)
            nxt = int(o.logits[0, -1].float().argmax())
            del o
            if nxt in _EOS:
                break
            out.append(nxt)
            cur = torch.tensor([[nxt]], device=_DEV)
            ids = torch.cat([ids, cur], dim=1)
            if _DONE.search(tok.decode(out, skip_special_tokens=True)):
                break
        return out


    @torch.no_grad()
    def _cache_ok(world, pids, k=6):
        """Is incremental decoding with the KV / recurrent cache the same function as a full forward?"""
        set_world(world)
        try:
            ids = pids[None].to(_DEV)
            o = model(input_ids=ids, use_cache=True)
            past = getattr(o, "past_key_values", None)
            if past is None:
                return False, "no cache returned"
            got = [int(o.logits[0, -1].float().argmax())]
            for _ in range(k - 1):
                o = model(input_ids=torch.tensor([[got[-1]]], device=_DEV), past_key_values=past, use_cache=True)
                past = o.past_key_values
                got.append(int(o.logits[0, -1].float().argmax()))
            full = torch.cat([ids, torch.tensor([got[:-1]], device=_DEV)], dim=1)
            ref = model(input_ids=full, use_cache=False).logits[0, -k:].float().argmax(-1).tolist()
            agree = sum(int(a == b) for a, b in zip(got, ref))
            return agree >= k - 1, f"{agree}/{k} tokens agree with the full forward"
        except Exception as e:
            return False, f"{type(e).__name__}: {str(e)[:120]}"


    # ----------------------------------------------------------------- problems
    _rng = random.Random(1234)
    TIER_A = []
    for _i in range(MATH_N_A):
        _k = _i % 4
        if _k == 0:
            a, b = _rng.randint(11, 99), _rng.randint(11, 99); q, g = f"{a} + {b}", a + b
        elif _k == 1:
            a, b = _rng.randint(50, 99), _rng.randint(11, 49); q, g = f"{a} - {b}", a - b
        elif _k == 2:
            a, b = _rng.randint(2, 9), _rng.randint(11, 30); q, g = f"{a} * {b}", a * b
        else:
            b = _rng.randint(2, 9); g = _rng.randint(3, 20); q = f"{b * g} / {b}"
        TIER_A.append(("Compute the value. Reply with exactly one line of the form 'Answer: <number>'.\n\n" + q,
                       float(g), q))

    TIER_B = []
    try:
        from datasets import load_dataset
        _ds = load_dataset("openai/gsm8k", "main", split="test")
        for _j in range(MATH_N_B):
            _r = _ds[_j * 37 % len(_ds)]
            _gold = _to_num(_r["answer"].split("####")[-1].strip())
            if _gold is None:
                continue
            TIER_B.append(("Solve the problem. Write at most 5 short lines of reasoning, then finish with a final "
                           "line of the form 'Answer: <number>'.\n\nProblem: " + _r["question"], _gold, _r["question"]))
    except Exception as _e:
        print(f"GSM8K not available ({type(_e).__name__}: {str(_e)[:100]}) -> tier B skipped")

    # ----------------------------------------------------------------- cache check
    _probe = _prompt_ids(TIER_A[0][0])
    CACHED = {}
    for _name, _world in (("teacher", "dense_teacher"), ("student", "serving")):
        _ok, _why = _cache_ok(_world, _probe)
        CACHED[_name] = _ok
        print(f"[cache check] {_name:7s}: {'incremental decoding OK' if _ok else 'FULL forward per token (slow)'} — {_why}")

    # ----------------------------------------------------------------- run
    RESULT = {"A": [], "B": []}
    SPEED = {"teacher": [0.0, 0], "student": [0.0, 0]}
    _stopped = False
    for _tier, _probs, _mx in (("A", TIER_A, MATH_MAX_NEW_A), ("B", TIER_B, MATH_MAX_NEW_B)):
        if _tier == "B" and RESULT["A"] and sum(r["student"]["strict"] for r in RESULT["A"]) < MATH_B_MIN_A * len(RESULT["A"]):
            print(f"tier B skipped: the student is under {100 * MATH_B_MIN_A:.0f}% on tier A (gate 2 cannot pass before gate 1 does; "
                  "MATH_B_MIN_A=0 runs it anyway)")
            continue
        for _pi, (_content, _gold, _q) in enumerate(_probs):
            if (time.time() - _T0) / 60 > MATH_BUDGET_MIN:
                _stopped = True
                break
            _pids = _prompt_ids(_content)
            _row = {"q": _q, "gold": _gold}
            for _name, _world in (("teacher", "dense_teacher"), ("student", "serving")):
                set_world(_world)
                _t = time.time()
                _out = _greedy(_pids, _mx, CACHED[_name])
                SPEED[_name][0] += time.time() - _t
                SPEED[_name][1] += len(_out)
                _txt = tok.decode(_out, skip_special_tokens=True)
                _s, _l = _parse(_txt)
                _row[_name] = {"text": _txt, "strict": _same(_s, _gold), "lenient": _same(_l, _gold),
                               "tokens": len(_out)}
            RESULT[_tier].append(_row)
            print(f"  [{_tier} {_pi + 1}/{len(_probs)}] teacher {'O' if _row['teacher']['strict'] else 'X'} | "
                  f"student {'O' if _row['student']['strict'] else 'X'} | {(time.time() - _T0) / 60:.1f} min")
        if _stopped:
            print(f"time budget ({MATH_BUDGET_MIN} min) reached — reporting on what was finished")
            break
    set_world("serving")

    # ----------------------------------------------------------------- report
    print("\n================ MATH PROBE ================")
    SUMMARY = {}
    for _tier, _label in (("A", "one-step arithmetic"), ("B", "GSM8K word problems")):
        _rows = RESULT[_tier]
        if not _rows:
            continue
        n = len(_rows)
        SUMMARY[_tier] = {"n": n}
        for _name in ("teacher", "student"):
            s = sum(r[_name]["strict"] for r in _rows)
            l = sum(r[_name]["lenient"] for r in _rows)
            SUMMARY[_tier][_name] = {"strict": s, "lenient": l}
            print(f"tier {_tier} ({_label:22s}) {_name:7s}: {s}/{n} correct with 'Answer:' line "
                  f"({100 * s / n:.0f}%) | {l}/{n} if the last number counts")
    for _name in ("teacher", "student"):
        _sec, _n = SPEED[_name]
        if _n:
            print(f"decode speed {_name:7s}: {_n / _sec:.1f} tok/s ({_n} tokens, "
                  f"{'cached' if CACHED[_name] else 'no cache'})")
    print("\n---- what the student actually writes ----")
    for _tier in ("A", "B"):
        for r in RESULT[_tier][:3]:
            print(f"[{_tier}] Q: {r['q'][:110]!r}\n     gold {r['gold']:g}\n     teacher: {r['teacher']['text'][:160]!r}"
                  f"\n     student: {r['student']['text'][:300]!r}\n")

    _tb = SUMMARY.get("B") or SUMMARY.get("A")
    if _tb:
        _acc = _tb["student"]["strict"] / _tb["n"]
        _tacc = _tb["teacher"]["strict"] / _tb["n"]
        if _tacc < 0.6:
            print("VERDICT: the TEACHER is below 60% here — the prompt/parser does not fit this model; "
                  "do not read the student number yet.")
        elif _acc >= 0.10:
            print(f"VERDICT: student is right {100 * _acc:.0f}% of the time — a correctness reward has signal; "
                  "start from on-policy teaching (student writes, teacher grades every token).")
        else:
            print(f"VERDICT: student is right {100 * _acc:.0f}% of the time — a correctness reward is ~always zero; "
                  "start from distillation on teacher-solved math, not from RL.")

    try:
        _dir = globals().get("SAVE_DIR") or os.path.join(DRIVE_DIR, "v17_state")
        os.makedirs(_dir, exist_ok=True)
        with open(os.path.join(_dir, "math_probe.json"), "w") as f:
            json.dump({"summary": SUMMARY, "speed": SPEED, "cached": CACHED, "rows": RESULT}, f,
                      ensure_ascii=False, indent=1)
        print(f"saved -> {os.path.join(_dir, 'math_probe.json')}")
    except Exception as _e:
        print(f"(result not saved: {type(_e).__name__}: {str(_e)[:100]})")
    print(f"MATH PROBE DONE in {(time.time() - _T0) / 60:.1f} min")

    if os.environ.get("PRISM_PUSH_REPO") and globals().get("V15_PUSHED"):
        try:
            from huggingface_hub import HfApi
            HfApi().upload_file(path_or_fileobj=os.path.join(_dir, "math_probe.json"), path_in_repo="v17_state/math_probe.json",
                                repo_id=os.environ["PRISM_PUSH_REPO"])
        except Exception as _e:
            print(f"(probe result not uploaded: {type(_e).__name__})")
else:
    print("math probe skipped (RUN_MATH_PROBE = False)")


## End — summary, gates, and stop the machine

In [ ]:
# The end: the summary, the gates, and (on a pay-per-hour Studio) the stop.
AUTO_STOP = globals().get("AUTO_STOP", True)
_pc = lambda e: (f"{100 * e[1]:.1f}%" if e else "n/a")
print("================ v17 요약 ================")
print(f"판정: {'KEPT (학습된 모델을 저장)' if V15_KEPT else 'ROLLED BACK (법 아래의 복원 상태 그대로 저장)'} | 스케줄 {V15_STEPS_DONE}/{V15_STEPS} 스텝"
      f" | 거버너 개입 {V15['cuts']}회")
_au = V16.get("audit") or {}
if _au.get("as_before") and _au.get("law"):
    print(f"1) 법 (엄격 계수) — 같은 held-out 블록의 텍스트 KL: 지금까지 재던 방식 {_au['as_before']['text_kl']:.4f} -> 법 아래 {_au['law']['text_kl']:.4f}"
          + (f" (쓰기만 엄격 {_au['strict_writes']['text_kl']:.4f}, 머리만 엄격 {_au['strict_head']['text_kl']:.4f})"
             if _au.get("strict_writes") and _au.get("strict_head") else ""))
_sw = V16.get("swap") or {}
if _sw:
    _top = sorted(_sw.items(), key=lambda kv: -(kv[1]["text_gain"] + 0.5 * (kv[1]["reasoning_gain"] or 0.0)))[:3]
    print("   손상이 큰 부품 (하나만 dense로 바꿨을 때 줄어드는 텍스트 KL): " + ", ".join(f"{k} -{v['text_gain']:.3f}" for k, v in _top))
_wr = globals().get("V16_WALLET_REPORT") or {}
if _wr.get("on") or _wr.get("closed"):
    _rc = _wr.get("receipt") or {}
    print(f"2) 지갑 — {'켜짐' if _wr.get('on') else '꺼짐'}, 창 {100 * _wr.get('omega', 0):.0f}%, 가격 갱신 {_wr.get('updates', 0)}회"
          + (f" (종료 사유: {_wr.get('closed')})" if _wr.get("closed") else "")
          + (f" | 토큰 하나가 읽은 양: 최소 {_rc['min'] / 1e9:.4f}B, 평균 {_rc['mean'] / 1e9:.4f}B, 최대 {_rc['max'] / 1e9:.4f}B (상한 {_rc['ceiling'] / 1e9:.4f}B)" if _rc else ""))
    for _k, _v in (_wr.get("per_kind") or {}).items():
        print(f"     {_k}: 정해진 몫 {_v['quota']:.1f}개 -> 토큰별로 {_v['min']:.1f} ~ {_v['max']:.1f}개 (평균 {_v['mean']:.1f})")
_mr = globals().get("V16_MARKET_REPORT") or {}
print(f"   몫 시장(지갑이 꺼졌을 때만 거래) — {_mr.get('rounds', 0)}회 열림, {_mr.get('trades', 0)}단위 거래" + (f" (조기 종료: {_mr.get('closed')})" if _mr.get("closed") else ""))
for _k, _v in (_mr.get("kinds") or {}).items():
    if _v["after"] != _v["before"]:
        print(f"     {_k}: {_v['before'] / 1e6:.1f}M -> {_v['after'] / 1e6:.1f}M ({(_v['after'] - _v['before']) / 1e6:+.2f}M)")
_oi = globals().get("V16_OWN_INFO") or {}
print(f"3) 자기 뉴런 — {'켜짐' if _oi.get('on') else '꺼짐'}: {_oi.get('layers', 0)}개 층 x {_oi.get('m', 0)}개"
      + (f", 학습된 파라미터 {sum(int(p.numel()) for p in V16_OWN_PARAMS) / 1e6:.0f}M" if globals().get("V16_OWN_PARAMS") else ""))
if globals().get("V15_PF", {}).get("fallbacks"):
    print(f"사전 점검에서 끈 장치: {', '.join(V15_PF['fallbacks'])}")
print(f"학습 (모두 법 아래) — 텍스트 KL: {KL_REF:.4f} -> {KL_V15:.4f} | held-out PPL: {PPL_REF:.2f} -> {PPL_V15:.2f} | dense {PPL.get('teacher_dense', float('nan')):.2f}")
if E_REF_MATH:
    print(f"추론 문제 KL: {E_REF_MATH[0]:.4f} -> {E_V15_MATH[0]:.4f} | 교사와 첫 토큰 일치: {_pc(E_REF_MATH)} -> {_pc(E_V15_MATH)}")
    print(f"결정적 토큰(교사 확신 {V16_DEC_P:g} 이상) 일치: {100 * D_REF_MATH.get('decisive', 0.0):.1f}% -> {100 * D_V15_MATH.get('decisive', 0.0):.1f}% | "
          f"{V16_RUN}개 연속 무갈림: {100 * D_REF_MATH.get('run', 0.0):.1f}% -> {100 * D_V15_MATH.get('run', 0.0):.1f}%")
print(f"활성 파라미터: 몫 기준 {measured_active() / 1e9:.4f}B, 토큰별 상한 {(_MK['cap'] if _WAL.get('on') else measured_active()) / 1e9:.4f}B "
      "(한도 3.000B — 평균이 아니라 모든 토큰 하나하나가 이 상한 이하)")
_S = globals().get("SUMMARY") or {}
_ga = (_S.get("A") or {})
_gb = (_S.get("B") or {})
_rate = lambda d: (d["student"]["strict"] / d["n"]) if d and d.get("n") else None
print("---- 게이트 ----")
for _name, _d, _bar in (("1. 한 단계 산수 > 50%", _ga, 0.5), ("2. GSM8K > 50%", _gb, 0.5)):
    _r = _rate(_d)
    if _r is None:
        print(f"  {_name}: 측정 안 됨")
        continue
    _tr = _d["teacher"]["strict"] / _d["n"]
    print(f"  {_name}: {100 * _r:.0f}% ({_d['student']['strict']}/{_d['n']}, 교사 {_d['teacher']['strict']}/{_d['n']}) -> "
          + ("판정 불가 (교사도 60% 미만: 프롬프트/채점 문제)" if _tr < 0.6 else ("통과" if _r > _bar else "미통과"))
          + ("" if _d["n"] >= 20 else f"  [문제 {_d['n']}개만 측정: 시간 예산]"))
print("  3. GPQA > 76 / 4. GPQA > 82: 이 노트북에서는 측정하지 않습니다 (게이트 1·2 통과 뒤에 측정)")
_lh = [h for h in (V17_LOOP.get("hist") or []) if h[2]]
if _lh:
    print(f"LOOP (학생이 직접 쓴 답) 정답률: 처음 {_lh[0][1]}/{_lh[0][2]} -> 마지막 {_lh[-1][1]}/{_lh[-1][2]}")
_lc = V17_LONG.get("curves") or {}
if _lc.get("after training"):
    print("장문 곡선 (위치별 KL, 학습 후): " + " | ".join(f"{a}-{b}: {k:.3f}" for a, b, k, _ in _lc["after training"]["rows"]))
print("결과 파일:", SAVE_DIR, "(v17_report.json 에 감사표·학습 곡선·LOOP·장문 곡선)")
_safe = bool(globals().get("V15_SAVED")) and (SAVE_DIR == V15_DIR or globals().get("V15_PUSHED"))
if not _safe:
    AUTO_STOP = False
    print("\n!! 결과가 이 머신의 임시 디스크에만 있습니다 (영구 저장소·허브 어디에도 없음). 자동 정지를 하지 않습니다 — "
          f"{SAVE_DIR} 를 직접 옮긴 뒤 머신을 끄세요.")
if globals().get("PRISM_ENV") == "lightning":
    if AUTO_STOP:
        print("\n150초 뒤 이 Studio를 자동으로 정지합니다 (크레딧 보호). 끄지 않으려면 지금 이 셀을 중단하세요.")
        time.sleep(150)                      # let the notebook autosave its outputs first
        try:
            from lightning_sdk import Studio
            Studio().stop()
        except Exception as _e:
            print(f"!! 자동 정지 실패 ({type(_e).__name__}: {str(_e)[:120]}) — 화면에서 직접 Stop 을 눌러 주세요. "
                  "켜 두면 크레딧이 계속 나갑니다.")
    else:
        print("\nAUTO_STOP=False: 직접 Stop 을 눌러 주세요. 켜 두면 크레딧이 계속 나갑니다.")
elif globals().get("PRISM_ENV") == "colab":
    if AUTO_STOP:
        print("\n결과가 저장됐습니다. 60초 뒤 런타임을 끊습니다 (컴퓨팅 단위 보호).")
        time.sleep(60)
        try:
            from google.colab import runtime
            runtime.unassign()
        except Exception as _e:
            print(f"(자동으로 끊지 못했습니다: {type(_e).__name__} — 직접 '런타임 연결 해제 및 삭제'를 눌러 주세요)")
    else:
        print("\n이제 런타임 연결을 끊어도 됩니다.")
else:
    print("\n작업이 끝났습니다. 머신을 직접 꺼 주세요.")
